In [ ]:
import os
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import random
singlecol = 3.5
plt.rcParams.update({'font.size': 8})
#plt.rcParams.update({'font.sans-serif': 'Helvetica'})
plt.rcParams.update({'lines.linewidth':1.5})
plt.rcParams.update({'axes.linewidth':1})
plt.rcParams.update({'lines.markersize':5})
plt.rcParams.update({'savefig.dpi':200})
plt.rcParams.update({'legend.fontsize': 8})

In [ ]:
import re

# Function to read and parse LAMMPS log
def parse_lammps_log_diff(file_path):
    data = {'Step': [], 'Temp': [], 'TotEng': [], 'Press': [], 'Volume': [], 'v_nNa': [], 'MSD(PB)' : [], 'MSD(Na)' : [], 'MSD(H2O)' : []}

    with open(file_path, 'r') as file:
        for line in file:
            # Find thermo output section
            if re.match(r"^\s*Step\s+Temp\s+TotEng", line):
                # Skip header lines
                next(file)
                # Parse the actual data lines
                for data_line in file:
                    if re.match(r"^\s*\d+", data_line) and len(data_line.split())==9:
                        # Extract values (Assuming specific columns order here)
                        values = data_line.split()
                        data['Step'].append(int(values[0]))
                        data['Temp'].append(float(values[1]))
                        data['TotEng'].append(float(values[2]))
                        data['Press'].append(float(values[3]))
                        data['Volume'].append(float(values[4]))
                        data['v_nNa'].append(float(values[5]))
                        data['MSD(PB)'].append(float(values[6]))
                        data['MSD(Na)'].append(float(values[7]))
                        data['MSD(H2O)'].append(float(values[8]))
                    else:
                        break
    return data

In [ ]:
import re

# Function to read and parse LAMMPS log
def parse_lammps_free(file_path):
    data = {'Step': [], 'Temp': [], 'TotEng': [], 'Press': [], 'Volume': [], 'v_nNa': [], 'atoms': [], 'v_nO': [], 'Fe_PE': [] }

    with open(file_path, 'r', encoding='utf-8', errors='ignore') as file:
        for line in file:
            # Find thermo output section
            if re.match(r"^\s*Step\s+Temp\s+TotEng", line):
                next(file)
                for data_line in file:
                    if re.match(r"^\s*\d+", data_line) and len(data_line.split())==5:
                        values = data_line.split()
                        data['Step'].append(int(values[0]))
                        data['Temp'].append(float(values[1]))
                        data['TotEng'].append(float(values[2]))
                        data['Press'].append(float(values[3]))
                        data['Volume'].append(float(values[4]))
                    elif re.match(r"^\s*\d+", data_line) and len(data_line.split())==6:
                        # Extract values (Assuming specific columns order here)
                        values = data_line.split()
                        data['Step'].append(int(values[0]))
                        data['Temp'].append(float(values[1]))
                        data['TotEng'].append(float(values[2]))
                        data['Press'].append(float(values[3]))
                        data['Volume'].append(float(values[4]))
                        data['v_nNa'].append(float(values[5]))
                    elif re.match(r"^\s*\d+", data_line) and len(data_line.split())==7:
                        # Extract values (Assuming specific columns order here)
                        values = data_line.split()
                        data['Step'].append(int(values[0]))
                        data['Temp'].append(float(values[1]))
                        data['TotEng'].append(float(values[2]))
                        data['Press'].append(float(values[3]))
                        data['Volume'].append(float(values[4]))
                        data['v_nO'].append(float(values[5]))
                        data['v_nNa'].append(float(values[6]))
                    elif re.match(r"^\s*\d+", data_line) and len(data_line.split())==8:
                        # Extract values (Assuming specific columns order here)
                        values = data_line.split()
                        data['Step'].append(int(values[0]))
                        data['Temp'].append(float(values[1]))
                        data['TotEng'].append(float(values[2]))
                        data['Press'].append(float(values[3]))
                        data['Volume'].append(float(values[4]))
                        data['v_nNa'].append(float(values[5]))
                        data['atoms'].append(float(values[6]))
                        data['U_total'].append(float(values[7]))
                    elif re.match(r"^\s*\d+", data_line) and len(data_line.split())==9:
                        # Extract values (Assuming specific columns order here)
                        values = data_line.split()
                        data['Step'].append(int(values[0]))
                        data['Temp'].append(float(values[1]))
                        data['TotEng'].append(float(values[2]))
                        data['Press'].append(float(values[3]))
                        data['Volume'].append(float(values[4]))
                        data['v_nNa'].append(float(values[5]))
                        data['atoms'].append(float(values[6]))
                        data['Mn_PE'].append(float(values[7]))
                        data['Fe_PE'].append(float(values[8]))
                    #else:
                    #    break
                        
    return data

In [ ]:
import re

def parse_lammps_free(file_path):
    # Initialize dictionary
    data = {'Step': [], 'Temp': [], 'TotEng': [], 'Press': [], 'Volume': [], 
            'v_nNa': [], 'atoms': [], 'v_nO': [], 'Fe_PE': [], 'U_total': [], 'Mn_PE': []}

    with open(file_path, 'r', encoding='utf-8', errors='ignore') as file:
        in_thermo = False
        for line in file:
            # 1. Detect start of thermo block
            if re.match(r"^\s*Step\s+Temp\s+TotEng", line):
                in_thermo = True
                continue
            
            # 2. Detect end of thermo block
            if in_thermo and (re.match(r"^\s*Loop time", line) or re.match(r"^\s*ERROR", line)):
                in_thermo = False
                continue

            if in_thermo:
                parts = line.split()
                num_cols = len(parts)
                
                # Check for empty or header lines
                if num_cols == 0:
                    continue

                try:
                    # Validate that the line starts with an integer (Step)
                    _ = int(parts[0])
                    
                    # --- FIX FOR INCOMPLETE LINES ---
                    # Only process lines that match your expected column counts (6, 7, 8, or 9)
                    if num_cols < 6:
                        continue 
                    
                    # Global columns
                    data['Step'].append(int(parts[0]))
                    data['Temp'].append(float(parts[1]))
                    data['TotEng'].append(float(parts[2]))
                    data['Press'].append(float(parts[3]))
                    data['Volume'].append(float(parts[4]))

                    # Conditional columns based on row length
                    if num_cols == 6:
                        data['v_nNa'].append(float(parts[5]))
                    elif num_cols == 7:
                        data['v_nO'].append(float(parts[5]))
                        data['v_nNa'].append(float(parts[6]))
                    elif num_cols == 8:
                        data['v_nNa'].append(float(parts[5]))
                        data['atoms'].append(float(parts[6]))
                        data['U_total'].append(float(parts[7]))
                    elif num_cols == 9:
                        data['v_nNa'].append(float(parts[5]))
                        data['atoms'].append(float(parts[6]))
                        data['Mn_PE'].append(float(parts[7]))
                        data['Fe_PE'].append(float(parts[8]))

                except (ValueError, IndexError):
                    # This catches cases where the line exists but contains non-numeric junk
                    continue
                    
    return data

In [ ]:
import re

# Function to read and parse LAMMPS log
def parse_lammps_log(file_path):
    data = {'Step': [], 'Temp': [], 'TotEng': [], 'Press': [], 'Volume': [], 'v_nNa': [], 'v_nO': []}

    with open(file_path, 'r') as file:
        for line in file:
            # Find thermo output section
            if re.match(r"^\s*Step\s+Temp\s+TotEng", line):
                # Skip header lines
                next(file)
                # Parse the actual data lines
                for data_line in file:
                    if re.match(r"^\s*\d+", data_line) and len(data_line.split())==5:
                        # Extract values (Assuming specific columns order here)
                        values = data_line.split()
                        data['Step'].append(int(values[0]))
                        data['Temp'].append(float(values[1]))
                        data['TotEng'].append(float(values[2]))
                        data['Press'].append(float(values[3]))
                        data['Volume'].append(float(values[4]))
                    if re.match(r"^\s*\d+", data_line) and len(data_line.split())==7:
                        # Extract values (Assuming specific columns order here)
                        values = data_line.split()
                        data['Step'].append(int(values[0]))
                        data['Temp'].append(float(values[1]))
                        data['TotEng'].append(float(values[2]))
                        data['Press'].append(float(values[3]))
                        data['Volume'].append(float(values[4]))
                        data['v_nO'].append(float(values[5]))
                        data['v_nNa'].append(float(values[6]))
                    if re.match(r"^\s*\d+", data_line) and len(data_line.split())==6:
                        # Extract values (Assuming specific columns order here)
                        values = data_line.split()
                        data['Step'].append(int(values[0]))
                        data['Temp'].append(float(values[1]))
                        data['TotEng'].append(float(values[2]))
                        data['Press'].append(float(values[3]))
                        data['Volume'].append(float(values[4]))
                        data['v_nNa'].append(float(values[5]))
                    else:
                        break
    return data

In [ ]:
def plot_log(log_file_path):
    log_data = parse_lammps_log(log_file_path)
    
    #E_Na = -.13023100E+01
    E_Na = -.14023100E+01
    
    my_list = log_data['TotEng'][20:40]
    E_i = sum(my_list) / len(my_list)

    my_list = log_data['TotEng'][-20:]
    E_f = sum(my_list) / len(my_list)

    my_list = log_data['v_nNa'][20:40]
    n_Na_i = sum(my_list) / len(my_list)

    my_list = log_data['v_nNa'][-20:]
    n_Na_f = sum(my_list) / len(my_list)
    
    n_Na = n_Na_i-n_Na_f
    
    V_ave = -(E_i-E_f-n_Na*E_Na)/n_Na
    #-(log_data['TotEng'][20]-log_data['TotEng'][-1]-(log_data['v_nNa'][20]-log_data['v_nNa'][-1])*E_Na)/(log_data['v_nNa'][20]-log_data['v_nNa'][-1])
    # -(Ei-Ef-n_Na*E_Na)/n_Na
    fig, ax1 = plt.subplots(figsize=(10, 6))
    
    # Plot RMS(E/atom) on the first y-axis
    color = 'tab:blue'
    ax1.set_xlabel('Timestep (0.4 fs/step)')
    ax1.set_ylabel('Total E (eV)', color=color)
    ax1.plot(log_data['Step'], log_data['TotEng'], color=color, marker='o', markersize=3)
    ax1.tick_params(axis='y', labelcolor=color)
    
    # Create the second y-axis and plot RMS(f)
    ax2 = ax1.twinx()
    color = 'tab:orange'
    ax2.set_ylabel('Volume (A^3)', color=color)
    ax2.plot(log_data['Step'][10:], log_data['Volume'][10:], color=color, marker='s', markersize=3)
    ax2.tick_params(axis='y', labelcolor=color)
    
    ax3 = ax1.twinx()
    color = 'tab:green'
    ax3.spines['right'].set_position(('outward', 80)) 
    ax3.set_ylabel('Number of Na+', color=color)
    ax3.plot(log_data['Step'], log_data['v_nNa'], color=color, marker='*')
    ax3.tick_params(axis='y', labelcolor=color)
    plt.grid()
    plt.title(log_file_path + f'    Average V = {round(V_ave,3)}')
    plt.savefig(log_file_path+'.jpg', bbox_inches='tight')

In [ ]:
def plot_log_no_V(log_file_path):
    log_data = parse_lammps_log(log_file_path)
    
    fig, ax1 = plt.subplots(figsize=(10, 6))
    
    # Plot RMS(E/atom) on the first y-axis
    color = 'tab:blue'
    ax1.set_xlabel('Timestep (0.4 fs/step)')
    ax1.set_ylabel('Total E (eV)', color=color)
    ax1.plot(log_data['Step'], log_data['TotEng'], color=color, marker='o', markersize=3)
    ax1.tick_params(axis='y', labelcolor=color)
    
    # Create the second y-axis and plot RMS(f)
    ax2 = ax1.twinx()
    color = 'tab:orange'
    ax2.set_ylabel('Volume (A^3)', color=color)
    ax2.plot(log_data['Step'], log_data['Volume'], color=color, marker='s', markersize=3)
    ax2.tick_params(axis='y', labelcolor=color)
    
    ax3 = ax1.twinx()
    color = 'tab:green'
    ax3.spines['right'].set_position(('outward', 80)) 
    ax3.set_ylabel('Number of H2O', color=color)
    #ax3.plot(log_data['Step'], log_data['v_nO'], color=color, marker='*')
    ax3.plot(log_data['Step'], log_data['v_nNa'], color=color, marker='*')
    ax3.tick_params(axis='y', labelcolor=color)
    plt.grid()
    plt.title(log_file_path)
    plt.tight_layout()
    plt.savefig(log_file_path+'.jpg', bbox_inches='tight')

In [ ]:
def plot_log_E_V(log_file_path):
    log_data = parse_lammps_free(log_file_path)
    
    fig, ax1 = plt.subplots(figsize=(10, 6))
    
    # Plot RMS(E/atom) on the first y-axis
    color = 'tab:blue'
    ax1.set_xlabel('Timestep (0.4 fs/step)')
    ax1.set_ylabel('Total E (eV)', color=color)
    ax1.plot(log_data['Step'], log_data['TotEng'], color=color, marker='o', markersize=3)
    ax1.tick_params(axis='y', labelcolor=color)
    
    # Create the second y-axis and plot RMS(f)
    ax2 = ax1.twinx()
    color = 'tab:orange'
    ax2.set_ylabel('Volume (A^3)', color=color)
    ax2.plot(log_data['Step'], log_data['Volume'], color=color, marker='s', markersize=3)
    ax2.tick_params(axis='y', labelcolor=color)
    
    plt.grid()
    plt.title(log_file_path)
    plt.tight_layout()
    plt.savefig(log_file_path+'.jpg', bbox_inches='tight')

In [ ]:
def plot_log_diff(log_file_path):
    log_data = parse_lammps_log_diff(log_file_path)
    log_data_2 = parse_lammps_free(log_file_path)
    #num_Na = int(log_data_2['v_nNa'][0])
    
    fig, ax1 = plt.subplots(figsize=(10, 6))
    
    # Plot RMS(E/atom) on the first y-axis
    color = 'tab:blue'
    ax1.set_xlabel('Timestep (1 fs/step)')
    ax1.set_ylabel('Total E (eV)', color=color)
    ax1.plot(log_data['Step'], log_data['TotEng'], color=color, marker='o', markersize=3)
    ax1.plot(log_data_2['Step'], log_data_2['TotEng'], color=color, marker='o', markersize=3)
    ax1.tick_params(axis='y', labelcolor=color)
    
    # Create the second y-axis and plot RMS(f)
    ax2 = ax1.twinx()
    color = 'tab:orange'
    ax2.set_ylabel('Volume (A^3)', color=color)
    ax2.plot(log_data['Step'][10:], log_data['Volume'][10:], color=color, marker='s', markersize=3)
    ax2.plot(log_data_2['Step'][10:], log_data_2['Volume'][10:], color=color, marker='s', markersize=3)
    ax2.tick_params(axis='y', labelcolor=color)
    
    # MSD(PB) (Third y-axis, second right)
    ax3 = ax1.twinx()
    color3 = 'tab:green'
    ax3.spines['right'].set_position(('outward', 80))
    ax3.set_ylabel('MSD A^2', color=color3)
    ax3.plot(log_data['Step'], log_data['MSD(PB)'], color=color3, marker='*', markersize=3, label='MSD(PB)')
    ax3.tick_params(axis='y')
    
    # MSD(Na) (Fourth y-axis, third right)
    #data["c_msd_Na[1]"] = [value * num_Na for value in data["c_msd_Na[1]"]]
    color4 = 'tab:red'
    ax3.plot(log_data['Step'], log_data['MSD(Na)'], color=color4, marker='*', markersize=3, label='MSD(Na)')

    color4 = 'tab:purple'
    ax3.plot(log_data['Step'], log_data['MSD(H2O)'], color=color4, marker='*', markersize=3, label='MSD(H2O)')
    ax3.legend()

    plt.title(log_file_path + f'    MSD')
    plt.grid(True, linestyle='--', alpha=0.7)
    plt.tight_layout()
    plt.savefig(log_file_path+'.jpg', dpi=300, bbox_inches='tight')

In [ ]:
def plot_log_free(log_file_path, plot_natom=True):
    Nt_ls = range(0,505,5)
    log_data = parse_lammps_free(log_file_path)
    
    fig, ax1 = plt.subplots(figsize=(singlecol*2,singlecol), dpi=300)
    
    # Plot RMS(E/atom) on the first y-axis
    color = 'tab:blue'
    ax1.set_xlabel('Timestep (0.4 fs/step)')
    ax1.set_ylabel('Total E (eV)', color=color)
    ax1.plot(log_data['Step'], log_data['TotEng'], color=color, marker='o', markersize=3)
    ax1.tick_params(axis='y', labelcolor=color)
    
    # Create the second y-axis and plot RMS(f)
    ax2 = ax1.twinx()
    color = 'tab:orange'
    ax2.set_ylabel('Volume (A^3)', color=color)
    ax2.plot(log_data['Step'][10:], log_data['Volume'][10:], color=color, marker='s', markersize=3)
    ax2.tick_params(axis='y', labelcolor=color)
    
    ax3 = ax1.twinx()
    color = 'tab:green'
    ax3.spines['right'].set_position(('outward', 80)) 
    ax3.set_ylabel('Number of Na+', color=color)
    if plot_natom:
        atoms = np.array(log_data['v_nNa'])
        ax3.plot(log_data['Step'], atoms, color=color, marker='*')
    atoms = np.array(log_data['v_nO'])
    ax3.plot(log_data['Step'], atoms, color='purple', marker='*')
    ax3.tick_params(axis='y', labelcolor=color)

    #nNa = log_data['v_nNa'][-1]
    plt.grid()
    plt.title(log_file_path)#+f'  final nNa = {nNa}')
    plt.savefig(log_file_path+'.jpg', bbox_inches='tight')
    plt.tight_layout()

In [ ]:
def plot_log_free_E(log_file_path, plot_natom=True):
    Nt_ls = range(0,505,5)
    log_data = parse_lammps_free(log_file_path)
    
    fig, ax1 = plt.subplots(figsize=(singlecol*2,singlecol), dpi=300)
    
    # Plot RMS(E/atom) on the first y-axis
    color = 'tab:blue'
    ax1.set_xlabel('Timestep (0.4 fs/step)')
    ax1.set_ylabel('Total E (eV)', color=color)
    ax1.plot(log_data['Step'], log_data['TotEng'], color=color, marker='o', markersize=3)
    ax1.tick_params(axis='y', labelcolor=color)
    
    # Create the second y-axis and plot RMS(f)
    ax2 = ax1.twinx()
    color = 'tab:orange'
    ax2.set_ylabel('Volume (A^3)', color=color)
    ax2.plot(log_data['Step'][10:], log_data['Volume'][10:], color=color, marker='s', markersize=3)
    ax2.tick_params(axis='y', labelcolor=color)
    
    ax3 = ax1.twinx()
    color = 'tab:green'
    ax3.spines['right'].set_position(('outward', 80)) 
    ax3.set_ylabel('Number of Na+', color=color)
    if plot_natom:
        atoms = np.array(log_data['v_nNa'])
        ax3.plot(log_data['Step'], atoms, color=color, marker='*')

    plt.grid()
    plt.savefig(log_file_path+'.jpg', bbox_inches='tight')
    plt.tight_layout()

In [ ]:
def convert_dump_file(input_file, output_file, element_map):
    """
    Convert atom types to element names in a LAMMPS dump file.
    
    Parameters:
    input_file (str): Path to the original LAMMPS dump file
    output_file (str): Path to save the modified file
    element_map (dict): Dictionary mapping atom types to element names
    """
    with open(input_file, 'r') as infile, open(output_file, 'w') as outfile:
        in_atoms_section = False
        
        for line in infile:
            line = line.strip()
            
            # Check if we're starting the atoms section
            if line.startswith("ITEM: ATOMS"):
                in_atoms_section = True
                outfile.write(line + '\n')
                continue
            
            # Process atom lines when we're in the atoms section
            if in_atoms_section and not line.startswith("ITEM:"):
                parts = line.split()
                
                if len(parts) >= 2:  # Ensure we have at least atom ID and type
                    # Get the atom type (2nd column in your file)
                    atom_type = int(parts[1])
                    
                    # Replace type number with element name if in our map
                    if atom_type in element_map:
                        parts[1] = element_map[atom_type]
                    
                    # Write the modified line
                    outfile.write(' '.join(parts) + '\n')
                else:
                    # Just write the line unchanged if it doesn't match expected format
                    outfile.write(line + '\n')
            else:
                # If we encounter a new "ITEM:" line, we're leaving the atoms section
                if line.startswith("ITEM:") and not line.startswith("ITEM: ATOMS"):
                    in_atoms_section = False
                
                # Write all non-atom lines unchanged
                outfile.write(line + '\n')

## GCMC

In [ ]:
%matplotlib inline
for i in [34,33,39,35,37,36,4,9,5]:#range(0,40,4):
    file = f'./gcmc_{i}.log'
    plot_log_free(file),

In [ ]:
%matplotlib widget
for i in range(2):# range(0,40,4):
    file = f'./300_vDOS/300_vDOS_{i+1}.log'
    plot_log_E_V(file)
    

In [ ]:


data

In [ ]:
%matplotlib inline
for i in range(8):# range(0,40,4):
    file = f'./vacancy/pot_save/gcmc_{i+1}_17.log'
    plot_log_free(file)

### Find coordinated water

In [ ]:
import numpy as np
from ase.io import read
from scipy.sparse.csgraph import connected_components

# ==========================================
# CONFIGURATION
# ==========================================
FILE_PATH = './400_diffusion_vac_lv20_r02_65_2/400K_10.data'

# LAMMPS atom types
MN_TYPE = 2
N_TYPE = 4
Na_TYPE = 5
O_TYPE = 6
H_TYPE = 7

MN_N_CUTOFF = 2.8      # Max Mn-N bond distance (Å)
VACANCY_RADIUS = 5.3   # Radius from vacancy center to select O (Å)
O_H_CUTOFF = 1.25      # Max O-H bond distance (Å)
MN_CLUSTER_CUTOFF = 8.0  # Mn-Mn distance to connect vacancy cluster (Å)


# ==========================================
# READ LAMMPS DATA WITH ASE
# ==========================================
def read_structure(filepath):
    """
    Read a LAMMPS data file using ASE.

    ASE handles:
      - simulation cell
      - periodic boundary conditions
      - atomic positions
    """
    atoms = read(
        filepath,
        format='lammps-data',
        style='atomic',
        sort_by_id=True
    )

    # Make sure the system is periodic.
    # For a normal fully periodic LAMMPS box this should already be True.
    atoms.pbc = [True, True, True]

    return atoms


# ==========================================
# GET ATOM INDICES BY LAMMPS TYPE
# ==========================================
def get_indices_by_type(atoms, atom_type):
    """
    Return ASE atom indices corresponding to a given LAMMPS atom type.

    ASE stores the atom type from a LAMMPS data file in the 'type'
    array when reading the file.
    """
    if 'type' not in atoms.arrays:
        raise RuntimeError(
            "ASE did not find the LAMMPS atom 'type' array. "
            "Check the LAMMPS data file and ASE version."
        )

    return np.where(atoms.arrays['type'] == atom_type)[0]


# ==========================================
# PBC-AWARE DISTANCE
# ==========================================
def pbc_distance(atoms, i, j):
    """
    Minimum-image distance between atoms i and j.

    ASE handles PBC and the simulation cell.
    """
    return atoms.get_distance(i, j, mic=True)


# ==========================================
# PBC-AWARE DISPLACEMENT
# ==========================================
def pbc_displacement(atoms, i, j):
    """
    Minimum-image displacement vector from atom i to atom j.

    Returns a Cartesian vector.
    """
    return atoms.get_distance(i, j, mic=True, vector=True)


# ==========================================
# MAIN ANALYSIS
# ==========================================
def main():

    # --------------------------------------
    # 1. Read structure
    # --------------------------------------
    print(f"Reading {FILE_PATH}...")

    atoms = read_structure(FILE_PATH)

    print(f"Number of atoms: {len(atoms)}")
    print(f"Cell:\n{atoms.cell}")
    print(f"PBC: {atoms.pbc}")

    # Optional: explicitly wrap all atoms into the unit cell.
    # ASE handles the cell and PBC here.
    atoms.wrap()

    # --------------------------------------
    # 2. Separate atoms by LAMMPS type
    # --------------------------------------
    mn_indices = get_indices_by_type(atoms, MN_TYPE)
    n_indices = get_indices_by_type(atoms, N_TYPE)
    o_indices = get_indices_by_type(atoms, O_TYPE)
    h_indices = get_indices_by_type(atoms, H_TYPE)
    na_indices = get_indices_by_type(atoms, Na_TYPE)

    print(f"Mn atoms: {len(mn_indices)}")
    print(f"N atoms : {len(n_indices)}")
    print(f"Na atoms: {len(na_indices)}")
    print(f"O atoms : {len(o_indices)}")
    print(f"H atoms : {len(h_indices)}")

    # --------------------------------------
    # 3. Find undercoordinated Mn atoms
    # --------------------------------------
    print("\nFinding Mn atoms with exactly 5 N neighbors...")

    undercoord_mn = []

    for mn in mn_indices:

        # ASE calculates all Mn-N distances using MIC/PBC.
        distances = atoms.get_distances(
            mn,
            n_indices,
            mic=True
        )

        # Number of N neighbors within cutoff
        n_neighbors = np.count_nonzero(
            distances <= MN_N_CUTOFF
        )

        if n_neighbors in (4, 5):
            undercoord_mn.append(mn)

    undercoord_mn = np.array(
        undercoord_mn,
        dtype=int
    )

    print(
        f"Found {len(undercoord_mn)} "
        f"Mn atoms facing a vacancy."
    )

    if len(undercoord_mn) == 0:
        print(
            "No vacancies found based on Mn coordination. "
            "Exiting."
        )
        return

    # --------------------------------------
    # 4. Cluster undercoordinated Mn atoms
    # --------------------------------------
    print(
        f"\nClustering undercoordinated Mn atoms "
        f"using {MN_CLUSTER_CUTOFF} Å cutoff..."
    )

    n_mn = len(undercoord_mn)

    # Distance matrix
    dist_matrix = np.zeros(
        (n_mn, n_mn)
    )

    for i in range(n_mn):

        for j in range(i + 1, n_mn):

            d = atoms.get_distance(
                undercoord_mn[i],
                undercoord_mn[j],
                mic=True
            )

            dist_matrix[i, j] = d
            dist_matrix[j, i] = d

    # Connect Mn atoms if their PBC distance is < cutoff
    adjacency = dist_matrix < MN_CLUSTER_CUTOFF

    # Every atom is connected to itself
    np.fill_diagonal(adjacency, True)

    n_vacancies, labels = connected_components(
        adjacency,
        directed=False
    )

    print(
        f"Clustered into {n_vacancies} "
        f"distinct Fe(CN)6 vacancies."
    )

    # --------------------------------------
    # 5. Calculate PBC-aware vacancy centers
    # --------------------------------------
    vacancy_centers = []

    print("\nVacancy centers:")

    for vacancy_id in range(n_vacancies):

        # ASE indices belonging to this cluster
        cluster = undercoord_mn[
            labels == vacancy_id
        ]

        # ----------------------------------
        # Use first Mn as reference
        # ----------------------------------
        reference = cluster[0]

        reference_position = atoms.positions[
            reference
        ]

        unwrapped_positions = []

        for mn in cluster:

            # Minimum-image vector from reference -> Mn
            displacement = atoms.get_distance(
                reference,
                mn,
                mic=True,
                vector=True
            )

            # Put this Mn into the same local image
            # as the reference Mn
            position = (
                reference_position
                + displacement
            )

            unwrapped_positions.append(
                position
            )

        unwrapped_positions = np.array(
            unwrapped_positions
        )

        # Average in the locally unwrapped representation
        center = unwrapped_positions.mean(
            axis=0
        )

        # Convert center into fractional coordinates
        scaled_center = atoms.cell.scaled_positions(
            np.array([center])
        )[0]

        # Wrap fractional coordinates into [0,1)
        scaled_center %= 1.0

        # Back to Cartesian coordinates
        center = atoms.cell.cartesian_positions(
            np.array([scaled_center])
        )[0]

        vacancy_centers.append(center)

        print(
            f"  Vacancy {vacancy_id + 1}: "
            f"[{center[0]:.3f}, "
            f"{center[1]:.3f}, "
            f"{center[2]:.3f}]"
        )

    vacancy_centers = np.array(
        vacancy_centers
    )

    # --------------------------------------
    # 6. Find O atoms near vacancy centers
    # --------------------------------------
    print(
        f"\nSearching for O atoms within "
        f"{VACANCY_RADIUS} Å of vacancies..."
    )

    selected_o_indices = []

    for o in o_indices:

        nearest_vacancy_distance = np.inf

        for center in vacancy_centers:

            # ------------------------------------------------
            # Calculate PBC-aware distance from O to vacancy
            #
            # Vacancy center isn't an atom, so we calculate
            # the minimum-image displacement manually using
            # ASE's cell representation.
            # ------------------------------------------------
            displacement = (
                atoms.positions[o] - center
            )

            # Convert Cartesian displacement to scaled coords
            scaled = atoms.cell.scaled_positions(
                np.array([displacement])
            )[0]

            # Minimum image in fractional coordinates
            scaled -= np.round(scaled)

            # Back to Cartesian
            displacement_mic = atoms.cell.cartesian_positions(
                np.array([scaled])
            )[0]

            distance = np.linalg.norm(
                displacement_mic
            )

            nearest_vacancy_distance = min(
                nearest_vacancy_distance,
                distance
            )

        if nearest_vacancy_distance <= VACANCY_RADIUS:
            selected_o_indices.append(o)

    selected_o_indices = np.array(
        selected_o_indices,
        dtype=int
    )

    print(
        f"Found {len(selected_o_indices)} "
        f"O atoms within "
        f"{VACANCY_RADIUS} Å of a vacancy."
    )

    # --------------------------------------
    # 7. Find H atoms bonded to selected O
    # --------------------------------------
    print(
        f"\nFinding H atoms within "
        f"{O_H_CUTOFF} Å of selected O..."
    )

    selected_h_indices = []

    for o in selected_o_indices:

        distances = atoms.get_distances(
            o,
            h_indices,
            mic=True
        )

        h_neighbors = h_indices[
            distances <= O_H_CUTOFF
        ]

        for h in h_neighbors:
            selected_h_indices.append(h)

    # Remove duplicates
    selected_h_indices = np.unique(
        selected_h_indices
    )

    # Sort by ASE atom index
    selected_h_indices = np.sort(
        selected_h_indices
    )

    # --------------------------------------
    # 8. Convert ASE indices -> LAMMPS IDs
    # --------------------------------------
    #
    # ASE's atom index is zero-based:
    #
    #     atoms[0], atoms[1], ...
    #
    # LAMMPS IDs are generally one-based:
    #
    #     1, 2, 3, ...
    #
    # However, with sort_by_id=True, the ASE ordering
    # follows the LAMMPS atom IDs, so we can recover them
    # from the original ordering.
    #
    # Prefer the explicit 'id' array when available.
    # --------------------------------------

    if 'id' in atoms.arrays:

        selected_h_ids = atoms.arrays['id'][
            selected_h_indices
        ].astype(int)

    else:

        # Fallback if ASE did not store the ID array
        selected_h_ids = selected_h_indices + 1

    selected_h_ids = np.sort(
        np.unique(selected_h_ids)
    )

    # --------------------------------------
    # 9. Final output
    # --------------------------------------
    print(
        "\n=================================================="
    )

    print(
        f"Total H atoms selected: "
        f"{len(selected_h_ids)}"
    )

    print("LAMMPS Atom IDs:")

    print(
        ",".join(
            map(str, selected_o_indices + 1) #selected_h_ids)
        )
    )

    print(
        "=================================================="
    )


# ==========================================
# RUN
# ==========================================
if __name__ == "__main__":
    main()

In [ ]:
def find_atoms_by_neighbor_count(
    atoms,
    x_type,
    y_type,
    cutoff=3.5,
    neighbor_counts=(0, 1, 3)
):
    """
    Find Y atoms based on how many X atoms are within cutoff.

    Returns LAMMPS atom IDs.
    """

    x_indices = np.where(
        atoms.arrays['type'] == x_type
    )[0]

    y_indices = np.where(
        atoms.arrays['type'] == y_type
    )[0]

    result = {
        n: []
        for n in neighbor_counts
    }

    for y in y_indices:

        distances = atoms.get_distances(
            y,
            x_indices,
            mic=True
        )

        n_x_neighbors = np.count_nonzero(
            distances <= cutoff
        )

        if n_x_neighbors in result:
            y_id = int(atoms.arrays['id'][y])
            result[n_x_neighbors].append(y_id)

    return result

In [ ]:
atoms = read_structure('400_diffusion_20_r02_65/400K_9.data')
result = find_atoms_by_neighbor_count(
    atoms,
    x_type=5, #shell
    y_type=6, # center
    cutoff=2.8,
    neighbor_counts=(0, 1, 2)
)

In [ ]:
len(result[0]),len(result[1]),len(result[2])

In [ ]:
result[1]

In [ ]:
atoms.arrays['type'][3]

In [ ]:
import numpy as np
from scipy.spatial import cKDTree
from scipy.sparse.csgraph import connected_components

# ==========================================
# CONFIGURATION
# ==========================================
FILE_PATH = './400_diffusion_vac_lv20_r02_65_2/400K_10.data'

# Match your atom types based on: C Mn Fe N Na O H
MN_TYPE = 2
N_TYPE = 4
Na_TYPE = 5
O_TYPE = 6
H_TYPE = 7

MN_N_CUTOFF = 2.8      # Max expected Mn-N bond distance (Å)
VACANCY_RADIUS = 5.3   # Radius from missing Fe to grab H2O (Å)
O_H_CUTOFF = 1.25      # Max O-H bond distance for water (Å)
# ==========================================

def read_lammps_data(filepath):
    """Parses a LAMMPS data file to extract the periodic box and atoms."""
    box = []
    atoms = []
    read_atoms = False
    
    with open(filepath, 'r') as f:
        lines = f.readlines()
        
    for line in lines:
        parts = line.strip().split()
        if not parts:
            continue
            
        if len(parts) >= 4 and parts[-1] == 'xhi' and parts[-2] == 'xlo':
            box.append([float(parts[0]), float(parts[1])])
        elif len(parts) >= 4 and parts[-1] == 'yhi' and parts[-2] == 'ylo':
            box.append([float(parts[0]), float(parts[1])])
        elif len(parts) >= 4 and parts[-1] == 'zhi' and parts[-2] == 'zlo':
            box.append([float(parts[0]), float(parts[1])])
        elif parts[0] == 'Atoms':
            read_atoms = True
            continue
            
        if read_atoms and len(parts) >= 5 and parts[0].isdigit():
            atom_id = int(parts[0])
            atom_type = int(parts[1])
            x, y, z = float(parts[2]), float(parts[3]), float(parts[4])
            atoms.append([atom_id, atom_type, x, y, z])
            
    return np.array(box), np.array(atoms)

def pbc_dist(p1, p2, box_lengths):
    """Calculates distance between two points respecting periodic boundary conditions."""
    delta = np.abs(p1 - p2)
    delta = np.where(delta > 0.5 * box_lengths, box_lengths - delta, delta)
    return np.sqrt((delta**2).sum())

def main():
    print(f"Reading {FILE_PATH}...")
    box, atoms = read_lammps_data(FILE_PATH)
    box_lengths = box[:, 1] - box[:, 0]
    
    # Wrap all coordinates cleanly into the periodic box
    coords = atoms[:, 2:5] % box_lengths
    types = atoms[:, 1]
    ids = atoms[:, 0].astype(int)
    
    # Separate coordinates by type
    mn_coords = coords[types == MN_TYPE]
    mn_ids = ids[types == MN_TYPE]
    n_coords = coords[types == N_TYPE]
    o_coords = coords[types == O_TYPE]
    o_ids = ids[types == O_TYPE]
    na_coords = coords[types == Na_TYPE]
    na_ids = ids[types == Na_TYPE]
    h_coords = coords[types == H_TYPE]
    h_ids = ids[types == H_TYPE]

    # 1. Find undercoordinated Mn atoms (Exactly 5 Nitrogen neighbors)
    print("Finding 5-coordinated Mn atoms...")
    tree_n = cKDTree(n_coords, boxsize=box_lengths)
    
    undercoord_mn_coords = []
    for i, m_c in enumerate(mn_coords):
        neighbors = tree_n.query_ball_point(m_c, MN_N_CUTOFF)
        if len(neighbors) == 5 :
            undercoord_mn_coords.append(m_c)
            
    undercoord_mn_coords = np.array(undercoord_mn_coords)
    print(f"Found {len(undercoord_mn_coords)} Mn atoms facing a vacancy.")
    if len(undercoord_mn_coords) == 0:
        print("No vacancies found based on Mn coordination. Exiting.")
        return

    # 2. Cluster the undercoordinated Mn atoms to find individual vacancies
    # Mn atoms surrounding the same vacancy will be within ~8 Å of each other
    N_mn = len(undercoord_mn_coords)
    dist_mat = np.zeros((N_mn, N_mn))
    for i in range(N_mn):
        for j in range(N_mn):
            dist_mat[i,j] = pbc_dist(undercoord_mn_coords[i], undercoord_mn_coords[j], box_lengths)
            
    adj_matrix = dist_mat < 9.0
    n_vacancies, labels = connected_components(adj_matrix, directed=False)
    print(f"Clustered into {n_vacancies} distinct Fe(CN)6 vacancies.")

    # 3. Find the geometric center of each vacancy (Periodic Boundary Aware)
    vacancy_centers = []
    for i in range(n_vacancies):
        cluster_coords = undercoord_mn_coords[labels == i]
        # Map to circle to average safely over periodic boundaries
        angles = cluster_coords / box_lengths * 2 * np.pi
        mean_cos = np.cos(angles).mean(axis=0)
        mean_sin = np.sin(angles).mean(axis=0)
        mean_angles = np.arctan2(mean_sin, mean_cos)
        mean_angles = np.where(mean_angles < 0, mean_angles + 2 * np.pi, mean_angles)
        center = mean_angles / (2 * np.pi) * box_lengths
        vacancy_centers.append(center)
        print(f" -> Vacancy {i+1} Coordinate: [{center[0]:.3f}, {center[1]:.3f}, {center[2]:.3f}]")

    # 4. Find all Oxygen atoms within 6 Å of any vacancy center
    tree_vac = cKDTree(vacancy_centers, boxsize=box_lengths)
    selected_o_ids = []
    selected_o_coords = []
    for i, o_c in enumerate(o_coords):
        dist, _ = tree_vac.query(o_c)
        if dist <= VACANCY_RADIUS:
            selected_o_ids.append(o_ids[i])
            selected_o_coords.append(o_c)
            
    print(f"\nFound {len(selected_o_ids)} Oxygen atoms within {VACANCY_RADIUS} Å of the missing Fe.")

    # 5. Find corresponding bonded Hydrogen atoms
    tree_h = cKDTree(h_coords, boxsize=box_lengths)
    selected_h_ids = []
    
    for o_c in selected_o_coords:
        indices = tree_h.query_ball_point(o_c, O_H_CUTOFF)
        for idx in indices:
            selected_h_ids.append(h_ids[idx])

    # 6. Final Outputs
    final_selection = sorted(list(set(selected_h_ids)))
    
    print("\n==================================================")
    print(f"Total H2O atoms selected: {len(final_selection)}")
    print("LAMMPS Atom IDs:")
    print(" ".join(map(str, final_selection)))
    print("==================================================")

if __name__ == "__main__":
    main()

In [ ]:
def ids_to_ovito_expression(ids):
    ids = sorted(set(ids))

    ranges = []
    start = prev = ids[0]

    for current in ids[1:]:
        if current == prev + 1:
            prev = current
        else:
            ranges.append((start, prev))
            start = prev = current

    ranges.append((start, prev))

    expressions = []

    for start, end in ranges:
        if start == end:
            expressions.append(f"ParticleIdentifier == {start}")
        else:
            expressions.append(
                f"(ParticleIdentifier >= {start} && "
                f"ParticleIdentifier <= {end})"
            )

    return " ||\n".join(expressions)

In [ ]:
ids = np.array(result[1])-1
expression = ids_to_ovito_expression(ids)

print(expression)

In [ ]:
import numpy as np
from scipy.spatial import cKDTree
from scipy.sparse.csgraph import connected_components

# ==========================================
# CONFIGURATION
# ==========================================
FILE_PATH = './400_diffusion_vac_lv20_r02_65_2/400K_2.dump'

# Match your atom types based on: C Mn Fe N Na O H
MN_TYPE = 2
N_TYPE = 4
Na_TYPE = 5
O_TYPE = 6
H_TYPE = 7

MN_N_CUTOFF = 2.8      # Max expected Mn-N bond distance (Å)
VACANCY_RADIUS = 5.3   # Radius from missing Fe to grab H2O (Å)
O_H_CUTOFF = 1.25      # Max O-H bond distance for water (Å)
# ==========================================


def parse_lammps_dump(dump_data):
    """
    Parse LAMMPS dump data from string format
    """
    lines = dump_data.strip().split('\n')

    # Parse header information
    timestep = None
    num_atoms = None
    box_bounds = []
    atoms_data = []

    xy = xz = yz = 0.0

    i = 0
    while i < len(lines):
        line = lines[i].strip()

        if line.startswith("ITEM: TIMESTEP"):
            if i + 1 < len(lines):
                timestep = int(lines[i + 1])
            i += 2
        elif line.startswith("ITEM: NUMBER OF ATOMS"):
            if i + 1 < len(lines):
                num_atoms = int(lines[i + 1])
            i += 2
        elif line.startswith("ITEM: BOX BOUNDS"):
            # Parse box bounds (3 lines)
            box_bounds = []
            for j in range(3):
                if i + 1 + j < len(lines):
                    bounds_line = lines[i + 1 + j].split()
                    if len(bounds_line) >= 2:
                        box_bounds.append([float(bounds_line[0]), float(bounds_line[1])])
                        # Store tilt factors if present
                        if len(bounds_line) >= 3:
                            if j == 0:  # xy
                                xy = float(bounds_line[2])
                            elif j == 1:  # xz
                                xz = float(bounds_line[2])
                            elif j == 2:  # yz
                                yz = float(bounds_line[2])
            i += 4
        elif line.startswith("ITEM: ATOMS"):
            # Parse atom data
            headers = line.split()[2:]  # Skip "ITEM:" and "ATOMS"
            atom_start_line = i + 1
            for j in range(num_atoms):
                if atom_start_line + j < len(lines):
                    atom_line = lines[atom_start_line + j].split()
                    atom_dict = {}
                    for k, header in enumerate(headers):
                        if k < len(atom_line):
                            if header in ['id', 'type']:
                                atom_dict[header] = int(atom_line[k])
                            else:
                                atom_dict[header] = float(atom_line[k])
                    atoms_data.append(atom_dict)
            break
        else:
            i += 1

    return timestep, num_atoms, box_bounds, atoms_data, xy, xz, yz


def read_lammps_dump_file(filepath):
    """
    Reads a LAMMPS dump file from disk, parses it with parse_lammps_dump,
    and converts the result into the (box, atoms) format the rest of the
    pipeline expects: box is an (3,2) array of [lo, hi], atoms is an
    (N, 5) array of [id, type, x, y, z] in unwrapped/real coordinates.
    """
    with open(filepath, 'r') as f:
        dump_data = f.read()

    timestep, num_atoms, box_bounds, atoms_data, xy, xz, yz = parse_lammps_dump(dump_data)

    if not atoms_data:
        raise ValueError(f"No atoms parsed from {filepath} — check the dump format.")

    box = np.array(box_bounds)  # shape (3, 2): [lo, hi] per dimension

    if xy != 0.0 or xz != 0.0 or yz != 0.0:
        print("Warning: triclinic tilt factors detected (xy={:.3f}, xz={:.3f}, yz={:.3f}). "
              "This script assumes an orthogonal box for the PBC distance logic below; "
              "results near cell edges may be off for a triclinic cell.".format(xy, xz, yz))

    headers = atoms_data[0].keys()

    # Figure out which coordinate keys are present, preferring unwrapped,
    # then wrapped Cartesian, then scaled coordinates.
    if all(k in headers for k in ('xu', 'yu', 'zu')):
        cx, cy, cz = 'xu', 'yu', 'zu'
        scaled = False
    elif all(k in headers for k in ('x', 'y', 'z')):
        cx, cy, cz = 'x', 'y', 'z'
        scaled = False
    elif all(k in headers for k in ('xs', 'ys', 'zs')):
        cx, cy, cz = 'xs', 'ys', 'zs'
        scaled = True
    else:
        raise ValueError(f"Could not find x/y/z (or xu/yu/zu, xs/ys/zs) columns in dump headers: {list(headers)}")

    box_lengths = box[:, 1] - box[:, 0]

    atoms = np.zeros((len(atoms_data), 5))
    for idx, a in enumerate(atoms_data):
        atoms[idx, 0] = a['id']
        atoms[idx, 1] = a['type']
        if scaled:
            atoms[idx, 2] = a[cx] * box_lengths[0] + box[0, 0]
            atoms[idx, 3] = a[cy] * box_lengths[1] + box[1, 0]
            atoms[idx, 4] = a[cz] * box_lengths[2] + box[2, 0]
        else:
            atoms[idx, 2] = a[cx]
            atoms[idx, 3] = a[cy]
            atoms[idx, 4] = a[cz]

    # Sort by atom id for consistency with the data-file reader
    atoms = atoms[atoms[:, 0].argsort()]

    return box, atoms


def pbc_dist(p1, p2, box_lengths):
    """Calculates distance between two points respecting periodic boundary conditions."""
    delta = np.abs(p1 - p2)
    delta = np.where(delta > 0.5 * box_lengths, box_lengths - delta, delta)
    return np.sqrt((delta**2).sum())


def main():
    print(f"Reading {FILE_PATH}...")
    box, atoms = read_lammps_dump_file(FILE_PATH)
    box_lengths = box[:, 1] - box[:, 0]

    # Wrap all coordinates into [0, box_lengths) — this is the frame
    # cKDTree(..., boxsize=box_lengths) requires; it will raise
    # "input data are greater than the size of the periodic box" if any
    # point falls outside [0, box_lengths), which happens if you instead
    # shift back by box_lo (dump box bounds are often not zero-based).
    coords = (atoms[:, 2:5] - box[:, 0]) % box_lengths
    types = atoms[:, 1]
    ids = atoms[:, 0].astype(int)

    # Separate coordinates by type
    mn_coords = coords[types == MN_TYPE]
    n_coords = coords[types == N_TYPE]
    o_coords = coords[types == O_TYPE]
    o_ids = ids[types == O_TYPE]
    h_coords = coords[types == H_TYPE]
    h_ids = ids[types == H_TYPE]

    # 1. Find undercoordinated Mn atoms (Exactly 5 Nitrogen neighbors)
    print("Finding 5-coordinated Mn atoms...")
    tree_n = cKDTree(n_coords, boxsize=box_lengths)

    undercoord_mn_coords = []
    for i, m_c in enumerate(mn_coords):
        neighbors = tree_n.query_ball_point(m_c, MN_N_CUTOFF)
        if len(neighbors) == 5:
            undercoord_mn_coords.append(m_c)

    undercoord_mn_coords = np.array(undercoord_mn_coords)
    print(f"Found {len(undercoord_mn_coords)} Mn atoms facing a vacancy.")
    if len(undercoord_mn_coords) == 0:
        print("No vacancies found based on Mn coordination. Exiting.")
        return

    # 2. Cluster the undercoordinated Mn atoms to find individual vacancies
    N_mn = len(undercoord_mn_coords)
    dist_mat = np.zeros((N_mn, N_mn))
    for i in range(N_mn):
        for j in range(N_mn):
            dist_mat[i, j] = pbc_dist(undercoord_mn_coords[i], undercoord_mn_coords[j], box_lengths)

    adj_matrix = dist_mat < 8.0
    n_vacancies, labels = connected_components(adj_matrix, directed=False)
    print(f"Clustered into {n_vacancies} distinct Fe(CN)6 vacancies.")

    # 3. Find the geometric center of each vacancy (Periodic Boundary Aware)
    vacancy_centers = []
    for i in range(n_vacancies):
        cluster_coords = undercoord_mn_coords[labels == i]
        # Map to circle to average safely over periodic boundaries
        angles = (cluster_coords - box[:, 0]) / box_lengths * 2 * np.pi
        mean_cos = np.cos(angles).mean(axis=0)
        mean_sin = np.sin(angles).mean(axis=0)
        mean_angles = np.arctan2(mean_sin, mean_cos)
        mean_angles = np.where(mean_angles < 0, mean_angles + 2 * np.pi, mean_angles)
        center = mean_angles / (2 * np.pi) * box_lengths + box[:, 0]
        vacancy_centers.append(center)
        print(f" -> Vacancy {i+1} Coordinate: [{center[0]:.3f}, {center[1]:.3f}, {center[2]:.3f}]")

    # 4. Find all Oxygen atoms within 6 Å of any vacancy center
    tree_vac = cKDTree(vacancy_centers, boxsize=box_lengths)
    selected_o_ids = []
    selected_o_coords = []
    for i, o_c in enumerate(o_coords):
        dist, _ = tree_vac.query(o_c)
        if dist <= VACANCY_RADIUS:
            selected_o_ids.append(o_ids[i])
            selected_o_coords.append(o_c)

    print(f"\nFound {len(selected_o_ids)} Oxygen atoms within {VACANCY_RADIUS} Å of the missing Fe.")

    # 5. Find corresponding bonded Hydrogen atoms
    tree_h = cKDTree(h_coords, boxsize=box_lengths)
    selected_h_ids = []

    for o_c in selected_o_coords:
        indices = tree_h.query_ball_point(o_c, O_H_CUTOFF)
        for idx in indices:
            selected_h_ids.append(h_ids[idx])

    # 6. Final Outputs
    final_selection = sorted(list(set(selected_h_ids))) #sorted(list(set(selected_o_ids + selected_h_ids)))

    print("\n==================================================")
    print(f"Total H2O atoms selected: {len(final_selection)}")
    print("LAMMPS Atom IDs:")
    print(",".join(map(str, final_selection)))
    print("==================================================")


if __name__ == "__main__":
    main()

In [ ]:
final_selection

## Hydration

In [ ]:
fig, ax1 = plt.subplots(
    figsize=(singlecol*2, singlecol),
    dpi=300
)

import matplotlib as mpl

cmap = plt.get_cmap("viridis")

mu_values = np.linspace(-2, 1.9, 40)

norm = mpl.colors.Normalize(
    vmin=mu_values.min(),
    vmax=mu_values.max()
)

for i, mu in enumerate(mu_values):

    color = cmap(norm(mu))

    file = f'./lv16_hydration/hydration_{i+1}.log'

    log_data = parse_lammps_free(file)

    atoms = np.array(log_data['v_nO'])

    ax1.plot(
        log_data['Step'],
        atoms / 32,
        color=color,
        linewidth=1
    )

# colorbar
sm = mpl.cm.ScalarMappable(
    cmap=cmap,
    norm=norm
)

sm.set_array([])

cbar = plt.colorbar(sm, ax=ax1)

cbar.set_label(
    r'Water chemical potential $\mu_{\mathrm{H_2O}}$ (eV)'
)

cbar.set_ticks(np.linspace(-2, 1.9, 10))

ax1.set_xlabel('Timestep (0.4 fs/step)')
ax1.set_ylabel(
    r"$y$ in Na$_x$Mn[Fe(CN)$_6$]$\cdot$yH$_2$O"
)
plt.tight_layout()
#plt.savefig('water_free_E.svg')

### Hydration vacancy

In [ ]:
fig, ax1 = plt.subplots(
    figsize=(singlecol*2, singlecol),
    dpi=300
)

import matplotlib as mpl

cmap = plt.get_cmap("viridis")

mu_values = np.linspace(-2, 1.9, 40)

norm = mpl.colors.Normalize(
    vmin=mu_values.min(),
    vmax=mu_values.max()
)

for i, mu in enumerate(mu_values):

    color = cmap(norm(mu))

    file = f'./Nt_H_vac_300/1/loop_bias_1.log_{i+1}'

    log_data = parse_lammps_free(file)

    atoms = np.array(log_data['v_nO'])

    ax1.plot(
        log_data['Step'],
        atoms / 32,
        color=color,
        linewidth=1
    )

# colorbar
sm = mpl.cm.ScalarMappable(
    cmap=cmap,
    norm=norm
)

sm.set_array([])

cbar = plt.colorbar(sm, ax=ax1)

cbar.set_label(
    r'Water chemical potential $\mu_{\mathrm{H_2O}}$ (eV)'
)

cbar.set_ticks(np.linspace(-2, 1.9, 10))

ax1.set_xlabel('Timestep (0.4 fs/step)')
ax1.set_ylabel(
    r"$y$ in Na$_x$Mn[Fe(CN)$_6$]$\cdot$yH$_2$O"
)
plt.tight_layout()
#plt.savefig('water_free_E.svg')

In [ ]:
plot_log_free(log_file_path, plot_natom=True):
    Nt_ls = range(0,505,5)
    log_data = parse_lammps_free(log_file_path)
    
    fig, ax1 = plt.subplots(figsize=(singlecol*2,singlecol), dpi=300)
    
    # Plot RMS(E/atom) on the first y-axis
    color = 'tab:blue'
    ax1.set_xlabel('Timestep (0.4 fs/step)')
    ax1.set_ylabel('Total E (eV)', color=color)
    ax1.plot(log_data['Step'], log_data['TotEng'], color=color, marker='o', markersize=3)
    ax1.tick_params(axis='y', labelcolor=color)
    
    # Create the second y-axis and plot RMS(f)
    ax2 = ax1.twinx()
    color = 'tab:orange'
    ax2.set_ylabel('Volume (A^3)', color=color)
    ax2.plot(log_data['Step'][10:], log_data['Volume'][10:], color=color, marker='s', markersize=3)
    ax2.tick_params(axis='y', labelcolor=color)
    
    ax3 = ax1.twinx()
    color = 'tab:green'
    ax3.spines['right'].set_position(('outward', 80)) 
    ax3.set_ylabel('Number of Na+', color=color)
    if plot_natom:
        atoms = np.array(log_data['v_nNa'])
        ax3.plot(log_data['Step'], atoms, color=color, marker='*')
    atoms = np.array(log_data['v_nO'])
    ax3.plot(log_data['Step'], atoms, color='purple', marker='*')
    ax3.tick_params(axis='y', labelcolor=color)

    nNa = log_data['v_nNa'][-1]
    plt.grid()
    plt.title(log_file_path+f'  final nNa = {nNa}')
    plt.savefig(log_file_path+'.jpg', bbox_inches='tight')
    plt.tight_layout()

In [ ]:
%matplotlib inline
for i in range(10):
    file = f'./lv16_hydration/hydration_{i+1}.log'
    plot_log_free(file,False)

In [ ]:
%matplotlib inline
for i in range(10):
    file = f'./lv16_hydration_2/hydration_{i+1}.log'
    plot_log_free(file)

In [ ]:
%matplotlib inline
for i in range(20,40,2):
    file = f'./lv16_hydration_3/hydration_{i+1}.log'
    plot_log_free(file, False)

In [ ]:
%matplotlib inline
for i in range(38,39):
    file = f'./lv16_hydration/save/hydration_{i+1}.log'
    plot_log_free(file)

In [ ]:
%matplotlib inline
for i in range(20,40,1):
    file = f'./lv16_hydration/hydration_{i+1}.log'
    plot_log_free(file)

In [ ]:
%matplotlib inline
for i in range(0,40,4):
    file = f'./lv16_dehydration_2/hydration_{i+1}.log'
    plot_log_free(file)

## Diffraction

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
plt.rcParams.update({'font.size': 12})
plt.rcParams.update({'savefig.dpi':300})
plt.rcParams.update({'legend.fontsize': 12})
def plot_prf(filename):
    with open(filename, 'r') as f:
        lines = f.readlines()

# 1. Find the header line index dynamically
    header_index = -1
    for i, line in enumerate(lines):
        if "2Theta" in line and "Yobs" in line and "Ycal" in line:
            header_index = i
            break
            
    if header_index == -1:
        print("Error: Could not find the header line in the .prf file.")
        return

    # 2. Extract metadata and define data boundaries
    # Number of points is usually at index 1 (second line)
    n_points = int(lines[1].split()[1])
    data_start = header_index + 1
    data_end = data_start + n_points
    
    # Parse scan data
    data = []
    for i in range(data_start, data_end):
        parts = lines[i].split()
        if len(parts) >= 5:
            data.append([float(p) for p in parts[:5]])
    
    df = pd.DataFrame(data, columns=['2Theta', 'Yobs', 'Ycal', 'Diff_File', 'Backg'])
    df['Diff'] = df['Yobs'] - df['Ycal'] # Manual diff calculation
    
    # Parse reflection positions (starting after scan data)
    reflections = []
    for i in range(data_end, len(lines)):
        parts = lines[i].split()
        if parts:
            try:
                reflections.append(float(parts[0]))
            except ValueError:
                continue
    
    # Plotting
    plt.figure(figsize=(singlecol*2, singlecol))
    
    # Main Plot
    plt.plot(df['2Theta'], df['Yobs'], 'ko', markersize=2, fillstyle='none', 
             markeredgewidth=0.5, label='Observed', alpha=0.7)
    plt.plot(df['2Theta'], df['Ycal'], 'r-', linewidth=1, label='Calculated')
    #plt.plot(df['2Theta'], df['Backg'], 'g--', linewidth=0.8, label='Background', alpha=0.5)
    
    # Offset for Difference and Bragg markers
    y_max = df['Yobs'].max()
    y_min = df['Yobs'].min()
    data_range = y_max - y_min
    diff_offset = -0.2 * data_range
    bragg_y = -20#200 * data_range
    
    # Difference Curve
    plt.plot(df['2Theta'], df['Diff'] + diff_offset, 'b-', linewidth=0.8, label='Difference')
    plt.axhline(y=diff_offset, color='black', linestyle='-', linewidth=0.5)
    
    # Bragg Ticks
    plt.vlines(reflections, bragg_y - 0.03 * data_range, bragg_y + 0.03 * data_range, 
               color='green', linewidth=0.5, alpha=0.6, label='Bragg Positions')
    
    plt.xlabel(r'$2\theta$ (degrees)')
    plt.ylabel('Intensity (arb. unit)')
    plt.yticks([])
    #plt.title('Rietveld Refinement Plot')
    plt.legend(frameon=False)#bbox_to_anchor=(1.05, 1), loc='upper left')
    plt.xlim(10,70)
    #plt.ylim(-100,1200)
    plt.tight_layout()
    plt.show()
    plt.savefig('H2O_PBA_Na_FC_XRD.svg')

# Run the function
plot_prf('./diffraction/PBA_hydrated_XRD.prf')

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

# Formatting settings
plt.rcParams.update({'font.size': 12, 'savefig.dpi': 300, 'legend.fontsize': 12})
singlecol = 3.5  # Assuming a standard column width

def plot_prf(filename, mode='XRD', diff_offset=0, bragg_y=200):
    """
    Plots Rietveld refinement data in Q-space.
    mode: 'XRD' (1.54056 Å) or 'Neutron' (1.5949 Å)
    """
    # 1. Define Wavelength based on mode
    if mode.upper() == 'XRD':
        wavelength = 1.540560
        label_prefix = "XRD"
    elif mode.upper() == 'NEUTRON':
        wavelength = 1.5949
        label_prefix = "Neutron"
    else:
        print("Error: mode must be 'XRD' or 'Neutron'")
        return

    with open(filename, 'r') as f:
        lines = f.readlines()

    # 2. Find header dynamically
    header_index = -1
    for i, line in enumerate(lines):
        if "2Theta" in line and "Yobs" in line:
            header_index = i
            break
            
    if header_index == -1:
        print("Error: Could not find the header line.")
        return

    # 3. Parse Scan Data
    n_points = int(lines[1].split()[1])
    data_start = header_index + 1
    data_end = data_start + n_points
    
    data = []
    for i in range(data_start, data_end):
        parts = lines[i].split()
        if len(parts) >= 5:
            data.append([float(p) for p in parts[:5]])
    
    df = pd.DataFrame(data, columns=['2Theta', 'Yobs', 'Ycal', 'Diff_File', 'Backg'])
    
    # --- Q-CONVERSION LOGIC ---
    # Convert 2Theta to Theta in radians: (2Theta / 2) * (pi / 180)
    theta_rad = np.radians(df['2Theta'] / 2)
    df['q'] = (4 * np.pi * np.sin(theta_rad)) / wavelength
    # --------------------------

    df['Diff'] = df['Yobs'] - df['Ycal']
    
    # 4. Parse Bragg Positions and convert to Q
    reflections_2theta = []
    for i in range(data_end, len(lines)):
        parts = lines[i].split()
        if parts:
            try:
                reflections_2theta.append(float(parts[0]))
            except ValueError:
                continue
    
    # Convert Bragg ticks to Q
    ref_theta_rad = np.radians(np.array(reflections_2theta) / 2)
    reflections_q = (4 * np.pi * np.sin(ref_theta_rad)) / wavelength
    
    # 5. Plotting
    plt.figure(figsize=(singlecol*2, singlecol))
    plt.xlim(0.8,5)
    # Main Plot (using 'q' instead of '2Theta')
    plt.plot(df['q'], df['Yobs'], 'ko', markersize=2, fillstyle='none', 
             markeredgewidth=0.5, label=f'Observed ({label_prefix})', alpha=0.7)
    plt.plot(df['q'], df['Ycal'], 'r-', linewidth=1, label='Calculated')
    
    # Offsets
    y_max, y_min = df['Yobs'].max(), df['Yobs'].min()
    data_range = y_max - y_min
    diff_offset = diff_offset #y_min - (0.1 * data_range)
    bragg_y = bragg_y #diff_offset - (0.1 * data_range)
    
    # Difference Curve
    plt.plot(df['q'], df['Diff'] + diff_offset, 'b-', linewidth=0.8, label='Difference')
    plt.axhline(y=diff_offset, color='black', linestyle='-', linewidth=0.5)
    
    # Bragg Ticks in Q
    plt.vlines(reflections_q, bragg_y - 0.02 * data_range, bragg_y + 0.02 * data_range, 
               color='green', linewidth=0.5, alpha=0.6, label='Bragg Positions')
    
    plt.xlabel(r'$q$ ($\AA^{-1}$)')
    plt.ylabel('Intensity (arb. unit)')
        
    plt.yticks([])
    plt.legend(frameon=False, fontsize=10)
    
    # Set limits based on Q range (adjust 0.7 and 5.0 as needed for your sample)
    plt.xlim(df['q'].min(), df['q'].max())
    
    plt.tight_layout()
    #plt.savefig(f'D2O_PBA_Na_FC_{mode}.svg')
    plt.show()

In [ ]:
%matplotlib widget
plot_prf('./diffraction/PBA_deutered_mix_sites_NPD.prf', mode='Neutron', diff_offset=50, bragg_y=150)
plt.xlim(0.8,5)
plt.ylim(0,1500)
mode='Neutron'
plt.savefig(f'D2O_PBA_Na_FC_{mode}.svg')

In [ ]:
%matplotlib widget
plot_prf('./diffraction/PBA_deutered_water_face_center_NPD.prf', mode='Neutron', diff_offset=-500, bragg_y=150)
plt.xlim(0.8,5)
plt.ylim(-850,1500)
mode='Neutron'
plt.savefig(f'D2O_PBA_H2O_FC_{mode}.svg')

In [ ]:
%matplotlib widget
plot_prf('./diffraction/PBA_hydrated_from_deuterated.prf', mode='Neutron', diff_offset=120, bragg_y=250)
plt.xlim(0.8,5)
plt.ylim(0,1150)
mode='Neutron'
plt.savefig(f'H2O_PBA_Na_FC_{mode}.svg')

In [ ]:
%matplotlib widget
plot_prf('./diffraction/PBA_hydrated_D_H2O_FC_NPD.prf', mode='Neutron', diff_offset=120, bragg_y=250)
plt.xlim(0.8,5)
plt.ylim(0,1150)
mode='Neutron'
plt.savefig(f'H2O_PBA_H2O_FC_{mode}.svg')

In [ ]:
# Run the function
plot_prf('./diffraction/PBA_hydrated_XRD.prf', mode='XRD', diff_offset=-100, bragg_y=-50)
plt.xlim(0.8,5)
mode='XRD'
plt.savefig(f'H2O_PBA_Na_FC_{mode}.svg')

In [ ]:
# Run the function
plot_prf('./diffraction/PBA_deutered_mix_sites_XRD.prf', mode='XRD',diff_offset=-100, bragg_y=-50)
plt.xlim(0.8,5)
mode='XRD'
plt.savefig(f'D2O_PBA_Na_FC_{mode}.svg')

## Structure inspection

In [ ]:
import re

def parse_lammps_box(filename):
    """
    Simple LAMMPS data parser to extract only box bounds.
    Returns dict with keys: xlo, xhi, ylo, yhi, zlo, zhi, xy, xz, yz
    """
    box_params = {}
    
    with open(filename, 'r') as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
                
            # xlo xhi
            if 'xlo xhi' in line:
                parts = line.split()
                box_params['xlo'] = float(parts[0])
                box_params['xhi'] = float(parts[1])
                
            # ylo yhi  
            elif 'ylo yhi' in line:
                parts = line.split()
                box_params['ylo'] = float(parts[0])
                box_params['yhi'] = float(parts[1])
                
            # zlo zhi
            elif 'zlo zhi' in line:
                parts = line.split()
                box_params['zlo'] = float(parts[0])
                box_params['zhi'] = float(parts[1])
                
            # xy xz yz (tilt factors)
            elif 'xy xz yz' in line:
                parts = line.split()
                box_params['xy'] = float(parts[0])
                box_params['xz'] = float(parts[1])
                box_params['yz'] = float(parts[2])

            elif 'atoms' in line:
                parts = line.split()
                atoms = float(parts[0])
    
    # Set tilt factors to 0 if not found (orthogonal box)
    if 'xy' not in box_params:
        box_params['xy'] = 0.0
        box_params['xz'] = 0.0
        box_params['yz'] = 0.0
    
    return box_params, atoms

def compute_triclinic_angles(xlo, xhi, ylo, yhi, zlo, zhi, xy, xz, yz):
    """
    Compute the angles between the three axes of a triclinic simulation box
    from LAMMPS data file parameters.
    
    Returns: alpha, beta, gamma (angles in degrees)
    """
    import numpy as np
    import math
    
    # Calculate box lengths
    lx = xhi - xlo
    ly = yhi - ylo
    lz = zhi - zlo
    
    # Define the three lattice vectors
    # In LAMMPS triclinic format:
    # a = (lx, 0, 0)
    # b = (xy, ly, 0)
    # c = (xz, yz, lz)
    
    a = np.array([lx, 0, 0])
    b = np.array([xy, ly, 0])
    c = np.array([xz, yz, lz])
    
    # Calculate magnitudes
    mag_a = np.linalg.norm(a)
    mag_b = np.linalg.norm(b)
    mag_c = np.linalg.norm(c)
    
    # Calculate angles using dot product formula: cos(θ) = (u·v)/(|u||v|)
    # α is angle between b and c
    # β is angle between a and c  
    # γ is angle between a and b
    
    cos_alpha = np.dot(b, c) / (mag_b * mag_c)
    cos_beta = np.dot(a, c) / (mag_a * mag_c)
    cos_gamma = np.dot(a, b) / (mag_a * mag_b)
    
    # Convert to degrees
    alpha = math.degrees(math.acos(cos_alpha))
    beta = math.degrees(math.acos(cos_beta))
    gamma = math.degrees(math.acos(cos_gamma))
    
    return lx, ly, lz, alpha, beta, gamma

In [ ]:

for i in [9]:
# Test with your data
    filename = f'./400_diffusion_20_MTP/400_diffusion_{i}.data'
    print(filename)
    #500K_Na{i}_O_32.data'  # Replace with your file
    #filename = f'./production_run/free_E/test/full_Na_big_300K.data'
    try:
        box, atoms = parse_lammps_box(filename)
        print("Box parameters:")
        for key, value in box.items():
            print(f"{key}: {value}")
        
        # Compute angles
        lx, ly, lz, alpha, beta, gamma = compute_triclinic_angles(**box)
        print(f"\nTriclinic angles:")
        print(f"α = {alpha:.3f}°")
        print(f"β = {beta:.3f}°") 
        print(f"γ = {gamma:.3f}°")
        print(f"a = {lx:.3f} A")
        print(f"b = {ly:.3f} A")
        print(f"c = {lz:.3f} A")
        
    except FileNotFoundError:
        print("File not found - testing with sample data instead:")
        
    #print(f'atoms = {(atoms-17280)/1728*2}')
    print("")
    print("")

In [ ]:
a_ls = []
b_ls = []
c_ls = []
alpha_ls = []
beta_ls = []
gamma_ls = []
atom_ls = []
for i in range(1,70,1):
# Test with your data
    filename = f'./Nt_300/5/300loop_bias_{i+1}.data'
    print(filename)
    #500K_Na{i}_O_32.data'  # Replace with your file
    #filename = f'./production_run/free_E/test/full_Na_big_300K.data'
    try:
        box, atoms = parse_lammps_box(filename)
        print("Box parameters:")
        for key, value in box.items():
            print(f"{key}: {value}")
        
        # Compute angles
        lx, ly, lz, alpha, beta, gamma = compute_triclinic_angles(**box)
        a_ls.append(lx)
        b_ls.append(ly)
        c_ls.append(lz)
        alpha_ls.append(alpha)
        beta_ls.append(beta)
        gamma_ls.append(gamma)
        atom_ls.append((atoms-17280)/1728*2)
        print(f"\nTriclinic angles:")
        print(f"α = {alpha:.3f}°")
        print(f"β = {beta:.3f}°") 
        print(f"γ = {gamma:.3f}°")
        print(f"a = {lx:.3f} A")
        print(f"b = {ly:.3f} A")
        print(f"c = {lz:.3f} A")
        
    except FileNotFoundError:
        print("File not found - testing with sample data instead:")
        
    print(f'atoms = {(atoms-17280)/1728*2}')
    print("")
    print("")

In [ ]:
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(singlecol*2, singlecol*1), dpi=150)
ax1.plot(atom_ls, a_ls, label='a')
ax1.plot(atom_ls, b_ls, label='b')
ax1.plot(atom_ls, c_ls, label='c')
ax1.legend()
ax1.set_xlabel('x')
ax1.set_ylabel('length (A)')

ax2.plot(atom_ls, abs(np.array(alpha_ls) - 90), label='alpha')
ax2.plot(atom_ls, abs(np.array(beta_ls) - 90), label='beta')
ax2.plot(atom_ls, abs(np.array(gamma_ls) - 90), label='gamma')
ax2.legend()
ax2.set_xlabel('x')
ax2.set_ylabel('|angel - 90| (degree)')

## RDF

In [ ]:
for i in range(1,1):
    file = f'gcmc_{i}.log'
    plot_log_free(file)

In [ ]:
file = f'./RDF_lv16_E0_sf0.1_f0.1/Na_32_O_32_temp.log'
plot_log_free(file)

In [ ]:
file = f'./RDF_lv16_E0_sf0.1_f0.1/Na_32_O_32_temp.log'
plot_log_free(file)

In [ ]:
file = f'./RDF_lv16_E0_sf0.1_f0.1_2/Na_32_O_32_temp.log'
plot_log_free(file)

In [ ]:
data_1['Step'][:2000],data_1['Step'][-1]

In [ ]:
a = [1, 2, 3]
b = [4, 5, 6]

c = a + b
c

### Plot_sodiation

In [ ]:
%matplotlib inline
file_1 = f'./RDF_lv20/300_temp.log'
data_1 = parse_lammps_free(file_1)

fig, ax1 = plt.subplots(figsize=(singlecol*3, singlecol*2))

# Plot RMS(E/atom) on the first y-axis
color = 'tab:blue'
ax1.set_xlabel('Timestep (0.4 fs/step)')
ax1.set_ylabel('Total E (eV)', color=color)
ax1.plot(data_1['Step'], data_1['TotEng'], color=color, marker='o', markersize=3)
ax1.tick_params(axis='y', labelcolor=color)

# Create the second y-axis and plot RMS(f)
ax2 = ax1.twinx()
color = 'tab:orange'
ax2.set_ylabel('Volume (A^3)', color=color)
ax2.plot(data_1['Step'], data_1['Volume'], color=color, marker='s', markersize=3)
ax2.tick_params(axis='y', labelcolor=color)

ax3 = ax1.twinx()
color = 'tab:green'
ax3.spines['right'].set_position(('outward', 80)) 
ax3.set_ylabel('Number of Na+', color=color)
ax3.plot(data_1['Step'], np.array(data_1['v_nNa'])-17280, color=color, marker='*')
ax3.tick_params(axis='y', labelcolor=color)

#nNa = data_2['v_nNa'][-1]
plt.grid()
#plt.title(log_file_path+f'  final nNa = {nNa}')
#plt.savefig(log_file_path+'.jpg', bbox_inches='tight')
plt.tight_layout()

In [ ]:
%matplotlib widget
file_1 = f'./RDF_lv20_vac/300_temp.log'
data_1 = parse_lammps_free(file_1)

fig, ax1 = plt.subplots(figsize=(singlecol*3, singlecol*2))

# Plot RMS(E/atom) on the first y-axis
color = 'tab:blue'
ax1.set_xlabel('Timestep (0.4 fs/step)')
ax1.set_ylabel('Total E (eV)', color=color)
ax1.plot(data_1['Step'], data_1['TotEng'], color=color, marker='o', markersize=3)
ax1.tick_params(axis='y', labelcolor=color)

# Create the second y-axis and plot RMS(f)
ax2 = ax1.twinx()
color = 'tab:orange'
ax2.set_ylabel('Volume (A^3)', color=color)
ax2.plot(data_1['Step'], data_1['Volume'], color=color, marker='s', markersize=3)
ax2.tick_params(axis='y', labelcolor=color)

ax3 = ax1.twinx()
color = 'tab:green'
ax3.spines['right'].set_position(('outward', 80)) 
ax3.set_ylabel('Number of Na+', color=color)
ax3.plot(data_1['Step'], np.array(data_1['v_nNa'])-4936, color=color, marker='*')
ax3.tick_params(axis='y', labelcolor=color)

#nNa = data_2['v_nNa'][-1]
plt.grid()
#plt.title(log_file_path+f'  final nNa = {nNa}')
#plt.savefig(log_file_path+'.jpg', bbox_inches='tight')
plt.tight_layout()

In [ ]:
file_1 = f'./RDF_lv16_E0_sf0.1_f0.1/Na_32_O_32_temp.log'
data_1 = parse_lammps_free(file_1)

file_2 = f'./RDF_lv16_E0_sf0.1_f0.1_2/Na_32_O_32_temp.log'
data_2 = parse_lammps_free(file_2)

fig, ax1 = plt.subplots(figsize=(singlecol*3, singlecol*2))

# Plot RMS(E/atom) on the first y-axis
color = 'tab:blue'
ax1.set_xlabel('Timestep (0.4 fs/step)')
ax1.set_ylabel('Total E (eV)', color=color)
ax1.plot(data_1['Step'][:2000], data_1['TotEng'][:2000], color=color, marker='o', markersize=3)
ax1.plot(data_2['Step'], data_2['TotEng'], color=color, marker='o', markersize=3)
ax1.tick_params(axis='y', labelcolor=color)

# Create the second y-axis and plot RMS(f)
ax2 = ax1.twinx()
color = 'tab:orange'
ax2.set_ylabel('Volume (A^3)', color=color)
ax2.plot(data_1['Step'][10:2000], data_1['Volume'][10:2000], color=color, marker='s', markersize=3)
ax2.plot(data_2['Step'][10:], data_2['Volume'][10:], color=color, marker='s', markersize=3)
ax2.tick_params(axis='y', labelcolor=color)

ax3 = ax1.twinx()
color = 'tab:green'
ax3.spines['right'].set_position(('outward', 80)) 
ax3.set_ylabel('Number of Na+', color=color)
ax3.plot(data_1['Step'][:2000], np.array(data_1['v_nNa'][:2000]), color=color, marker='*')
ax3.plot(data_2['Step'], np.array(data_2['v_nNa']), color=color, marker='*')
ax3.tick_params(axis='y', labelcolor=color)

nNa = data_2['v_nNa'][-1]
plt.grid()
#plt.title(log_file_path+f'  final nNa = {nNa}')
#plt.savefig(log_file_path+'.jpg', bbox_inches='tight')
plt.tight_layout()

In [ ]:
data_2.keys()

In [ ]:
data_2

In [ ]:
singlecol = 3.5
plt.rcParams.update({'font.size': 8})
#plt.rcParams.update({'font.sans-serif': 'Helvetica'})
plt.rcParams.update({'lines.linewidth':1})
plt.rcParams.update({'axes.linewidth':0.5})
plt.rcParams.update({'lines.markersize':3})
plt.rcParams.update({'savefig.dpi':300})
plt.rcParams.update({'legend.fontsize': 8})

file_1 = f'./RDF_lv16_E0_sf0.1_f0.1/Na_32_O_32_temp.log'
data_1 = parse_lammps_free(file_1)

file_2 = f'./RDF_lv16_E0_sf0.1_f0.1_2/Na_32_O_32_temp.log'
data_2 = parse_lammps_free(file_2)

T = (np.array(data_1['Step'][:2000] + data_2['Step'])/1000)[10:]
E = np.array(data_1['TotEng'][:2000] + data_2['TotEng'])[10:]
V = np.array(data_1['Volume'][:2000] + data_2['Volume'])[10:]
atoms = np.array(data_1['v_nNa'][:2000] + data_2['v_nNa'])[10:]-17280

fig, axs = plt.subplots(2,1 ,figsize=(singlecol*2,singlecol*1.5), dpi=300, sharex=True)

# Plot RMS(E/atom) on the first y-axis
color = 'tab:blue'
#axs[0].set_xlabel('Simulation time (ps)')
axs[0].set_ylabel('Total E (keV)', color=color)
axs[0].plot(T, E/1000, color=color, marker='o', markersize=3)
axs[0].tick_params(axis='y', labelcolor=color)

# Create the second y-axis and plot RMS(f)
ax2 = axs[0].twinx()
color = 'tab:orange'
ax2.set_ylabel(r'Volume (nm$^3$)', color=color)
ax2.plot(T, V/1000, color=color, marker='s', markersize=3)
ax2.tick_params(axis='y', labelcolor=color)

ax3 = axs[0].twinx()
color = 'tab:green'
ax3.spines['right'].set_position(('outward', 40)) 
ax3.set_ylabel(r"$x$ in Na$_x$Mn[Fe(CN)$_6$]$\cdot y$H$_2$O", color=color)
ax3.plot(T, atoms/1728*2, color=color, marker='*')
ax3.tick_params(axis='y', labelcolor=color)

nNa_i = atoms[0]
nNa_f = atoms[-1]
axs[0].vlines(5,-133,-124, linestyle='--', color='r')
axs[0].vlines(750,-133,-124, linestyle='--', color='r')
axs[0].vlines(1500,-133,-124, linestyle='--', color='r')
axs[0].vlines(1950,-133,-124, linestyle='--', color='r')

axs[0].set_xlim(-5,2e3)
axs[0].set_ylim(-133,-124)


color = 'tab:red'
valid_data_1 = pd.read_csv("./lv16_Na_O.csv")
valid_data_2 = pd.read_csv("./lv16_Na_O_2.csv")
valid_data = pd.concat([valid_data_1, valid_data_2], ignore_index=True)

axs[1].plot(valid_data['Timestep']/1000, valid_data['Avg_Coordination'], '-o', color=color,
            markersize=3, linewidth=1, alpha=0.8)
axs[1].fill_between(valid_data['Timestep']/1000, valid_data['Avg_Coordination'] - valid_data['Std_Coordination'],
                    valid_data['Avg_Coordination'] + valid_data['Std_Coordination'],
                    alpha=0.2, color=color, label='Na-O coord.')

color = 'tab:blue'
valid_data_1 = pd.read_csv("./lv16_Na_N.csv")
valid_data_2 = pd.read_csv("./lv16_Na_N_2.csv")
valid_data = pd.concat([valid_data_1, valid_data_2], ignore_index=True)

axs[1].plot(valid_data['Timestep']/1000, valid_data['Avg_Coordination'], '-o', color=color,
            markersize=3, linewidth=1, alpha=0.8)
axs[1].fill_between(valid_data['Timestep']/1000, valid_data['Avg_Coordination'] - valid_data['Std_Coordination'],
                    valid_data['Avg_Coordination'] + valid_data['Std_Coordination'],
                    alpha=0.2, color=color, label='Na-N coord.')

axs[1].set_ylabel('Coordination number')#, color=color)
#axs[1].legend(frameon=False)
#axs[1].tick_params(axis='y', labelcolor=color)
ax4 = axs[1].twinx()

df_1 = pd.read_csv("./bond_lv16_desodiation.csv")
df_2 = pd.read_csv("./bond_lv16_desodiation_2.csv")
df = pd.concat([df_1, df_2], ignore_index=True)

ax4.plot(df['Timestep'][df['Pair']=='Na-O']/1000,df['Bond_Length'][df['Pair']=='Na-O'],
         label='d(Na-O)', color='orangered', marker='*', markersize=3, alpha=0.5)
ax4.plot(df['Timestep'][df['Pair']=='N-Na']/1000,df['Bond_Length'][df['Pair']=='N-Na'],
         label='d(Na-N)', color='darkcyan', marker='*', markersize=3, alpha=0.5)
ax4.plot(df['Timestep'][df['Pair']=='N-Mn']/1000,df['Bond_Length'][df['Pair']=='N-Mn'], 
         label='d(N-Mn)', color='tab:purple', marker='*', markersize=3, alpha=0.5)

handles1, labels1 = axs[1].get_legend_handles_labels()
handles2, labels2 = ax4.get_legend_handles_labels()

# Create a single combined legend
axs[1].legend(
    handles2 + handles1,
    labels2 + labels1,
    ncol=5,
    frameon=False,
    loc="best",
    columnspacing=1.2,
    handletextpad=0.6
)
ax4.set_ylabel('Bond distance (Å)')
axs[1].set_xlabel('Simulation time (ps)')
axs[1].set_ylim(0.5,5)
ax4.set_ylim(2,2.7)
#plt.grid()
#plt.title(log_file_path+f'  final nNa = {nNa}')
#plt.savefig('./production_run/free_E/test/desodiation'+'.svg', bbox_inches='tight')
plt.tight_layout()
plt.savefig('./desodiation'+'.svg', bbox_inches='tight')

In [ ]:
atoms

### Plot_hydration

In [ ]:
%matplotlib widget
singlecol = 3.5
plt.rcParams.update({'font.size': 8})
#plt.rcParams.update({'font.sans-serif': 'Helvetica'})
plt.rcParams.update({'lines.linewidth':1})
plt.rcParams.update({'axes.linewidth':0.5})
plt.rcParams.update({'lines.markersize':3})
plt.rcParams.update({'savefig.dpi':300})
plt.rcParams.update({'legend.fontsize': 8})

file_1 = f'./lv16_hydration_2/hydration_3.log'
data_1 = parse_lammps_free(file_1)

T = np.array(data_1['Step'])
E = np.array(data_1['TotEng'])
V = np.array(data_1['Volume'])
atoms = np.array(data_1['v_nO'])

fig, axs = plt.subplots(2,1 ,figsize=(singlecol*2,singlecol*1.5), dpi=200, sharex=True)

# Plot RMS(E/atom) on the first y-axis
color = 'tab:blue'
#axs[0].set_xlabel('Simulation time (ps)')
axs[0].set_ylabel('Total E (eV)', color=color)
axs[0].plot(T, E, color=color, marker='o', markersize=3)
axs[0].tick_params(axis='y', labelcolor=color)

# Create the second y-axis and plot RMS(f)
ax2 = axs[0].twinx()
color = 'tab:orange'
ax2.set_ylabel(r'Volume (Å$^3$)', color=color)
ax2.plot(T, V, color=color, marker='s', markersize=3)
ax2.tick_params(axis='y', labelcolor=color)

ax3 = axs[0].twinx()
color = 'tab:green'
ax3.spines['right'].set_position(('outward', 45)) 
ax3.set_ylabel(r"$y$ in Na$_x$Mn[Fe(CN)$_6$]$\cdot y$H$_2$O", color=color)
ax3.plot(T, atoms/32, color=color, marker='*')
ax3.tick_params(axis='y', labelcolor=color)

nNa_i = atoms[0]
nNa_f = atoms[-1]
axs[0].vlines(1000,-5600,-3800, linestyle='--', color='r')
axs[0].vlines(100*2000, -5600,-3800, linestyle='--', color='r')
axs[0].vlines(210*2000, -5600,-3800, linestyle='--', color='r')
axs[0].vlines(360*2000, -5600,-3800, linestyle='--', color='r')

axs[0].set_xlim(0,750000)
axs[0].set_ylim(-5600,-3800)


color = 'tab:red'
valid_data = pd.read_csv("./lv16_hydration_Na_O.csv")

axs[1].plot(valid_data['Timestep'], valid_data['Avg_Coordination'], '-o', color=color,
            markersize=3, linewidth=1, alpha=0.8)
axs[1].fill_between(valid_data['Timestep'], valid_data['Avg_Coordination'] - valid_data['Std_Coordination'],
                    valid_data['Avg_Coordination'] + valid_data['Std_Coordination'],
                    alpha=0.2, color=color, label='Na-O coord.')

color = 'tab:blue'
valid_data = pd.read_csv("./lv16_hydration_Na_N.csv")

axs[1].plot(valid_data['Timestep'], valid_data['Avg_Coordination'], '-o', color=color,
            markersize=3, linewidth=1, alpha=0.8)
axs[1].fill_between(valid_data['Timestep'], valid_data['Avg_Coordination'] - valid_data['Std_Coordination'],
                    valid_data['Avg_Coordination'] + valid_data['Std_Coordination'],
                    alpha=0.2, color=color, label='Na-N coord.')

axs[1].set_ylabel('Coordination number')#, color=color)
#axs[1].legend(frameon=False)
#axs[1].tick_params(axis='y', labelcolor=color)
ax4 = axs[1].twinx()

df = pd.read_csv("./bond_lv16_hydration.csv")

#ax4.plot(df['Timestep'][df['Pair']=='Na-O'],df['Bond_Length'][df['Pair']=='Na-O'],
#         label='d(Na-O)', color='orangered', marker='*', markersize=3, alpha=0.5)
ax4.plot(df['Timestep'][df['Pair']=='N-Na'],df['Bond_Length'][df['Pair']=='N-Na'],
         label='d(Na-N)', color='darkcyan', marker='*', markersize=3, alpha=0.5)
ax4.plot(df['Timestep'][df['Pair']=='N-Mn'],df['Bond_Length'][df['Pair']=='N-Mn'], 
         label='d(N-Mn)', color='tab:purple', marker='*', markersize=3, alpha=0.5)

handles1, labels1 = axs[1].get_legend_handles_labels()
handles2, labels2 = ax4.get_legend_handles_labels()

# Create a single combined legend
axs[1].legend(
    handles2 + handles1,
    labels2 + labels1,
    ncol=2,
    frameon=False,
    loc="center left",
    columnspacing=1.2,
    handletextpad=0.6
)
ax4.set_ylabel('Bond distance (Å)')
axs[1].set_xlabel('Simulation time (fs)')
#axs[1].set_ylim(-0.5,6.5)
#ax4.set_ylim(2,2.8)
#plt.grid()
#plt.title(log_file_path+f'  final nNa = {nNa}')
#plt.savefig('./production_run/free_E/test/desodiation'+'.svg', bbox_inches='tight')
plt.tight_layout()
plt.savefig('./hydration'+'.svg', bbox_inches='tight')

In [ ]:
i = 3600
atoms[i]/32

In [ ]:
T[3600]

In [ ]:
V_i, V_f

In [ ]:
V_i = V[50]
V_f = V[-1]

volume_change_percent = (V_f - V_i) / V_i * 100
volume_change_percent

In [ ]:
E_Na = -.13023100E+01
#E_Na = -.14023100E+01
V_ave_ls=[]
for i in range(20,2000,16):
    my_list = E[i-8:i]
    E_i = sum(my_list) / len(my_list)

    my_list = E[i:i+8]
    E_f = sum(my_list) / len(my_list)
    
    my_list = atoms[i-8:i]
    n_Na_i = sum(my_list) / len(my_list)
    
    my_list = atoms[i:i+8]
    n_Na_f = sum(my_list) / len(my_list)
    
    n_Na = n_Na_i-n_Na_f
    
    V_ave = -(E_i-E_f-n_Na*E_Na)/n_Na

    V_ave_ls.append(V_ave)

In [ ]:
plt.plot(V_ave_ls)

In [ ]:
ave(V_ave_ls)

In [ ]:
#E_Na = -.13023100E+01
E_Na = -.14023100E+01

my_list = log_data['TotEng'][30:40]
E_i = sum(my_list) / len(my_list)

my_list = log_data['TotEng'][520:540]
E_f = sum(my_list) / len(my_list)

my_list = np.array(log_data['atoms'][30:40])-28672
n_Na_i = sum(my_list) / len(my_list)

my_list = np.array(log_data['atoms'][520:540])-28672
n_Na_f = sum(my_list) / len(my_list)

n_Na = n_Na_i-n_Na_f

V_ave = -(E_i-E_f-n_Na*E_Na)/n_Na

### sf=1, f=0.01

In [ ]:
file = f'./RDF_lv12_E0_sf1_f0.01/Na_28_O_32_temp.log'
plot_log_no_V(file)
file = f'./RDF_lv12_E0_sf1_f0.01/Na_28_O_16_temp.log'
plot_log_no_V(file)
file = f'./RDF_lv12_E0_sf1_f0.01/Na_28_O_4_temp.log'
plot_log_no_V(file)

### sf=1,f=0.1

In [ ]:
file = f'./RDF_lv12_E0_sf1_f0.1/Na_28_O_32_temp.log'
plot_log_no_V(file)
file = f'./RDF_lv12_E0_sf1_f0.1/Na_28_O_16_temp.log'
plot_log_no_V(file)
file = f'./RDF_lv12_E0_sf1_f0.1/Na_28_O_4_temp.log'
plot_log_no_V(file)

### sf=0.1, f=0.1

In [ ]:
file = f'./RDF_lv12_E0_sf0.1_f0.1/Na_28_O_32_temp.log'
plot_log_no_V(file)
file = f'./RDF_lv12_E0_sf0.1_f0.1/Na_28_O_16_temp.log'
plot_log_no_V(file)
file = f'./RDF_lv12_E0_sf0.1_f0.1/Na_28_O_4_temp.log'
plot_log_no_V(file)

### E0

In [ ]:
%matplotlib inline
file = f'./RDF_lv12_E0/Na_32_O_32_temp.log'
plot_log_no_V(file)
file = f'./RDF_lv12_E0/Na_32_O_16_temp.log'
plot_log_no_V(file)
file = f'./RDF_lv12_E0/Na_28_O_4_temp.log'
plot_log_no_V(file)

In [ ]:
%matplotlib inline
file = f'./RDF_lv12_E0_2/Na_4_O_32_temp.log'
plot_log_no_V(file)
file = f'./RDF_lv12_E0_2/Na_4_O_16_temp.log'
plot_log_no_V(file)
file = f'./RDF_lv12_E0_2/Na_4_O_4_temp.log'
plot_log_no_V(file)

In [ ]:
%matplotlib inline
file = f'./RDF_lv12_E0_3/Na_4_O_32_temp.log'
plot_log_no_V(file)
file = f'./RDF_lv12_E0_3/Na_4_O_16_temp.log'
plot_log_no_V(file)
file = f'./RDF_lv12_E0_3/Na_4_O_4_temp.log'
plot_log_no_V(file)

In [ ]:
%matplotlib inline
file = f'./RDF_lv12_E0_4/Na_16_O_4_temp.log'
plot_log_no_V(file)
file = f'./RDF_lv12_E0_4/Na_16_O_16_temp.log'
plot_log_no_V(file)
file = f'./RDF_lv12_E0_4/Na_28_O_4_temp.log'
plot_log_no_V(file)
file = f'./RDF_lv12_E0_4/Na_28_O_16_temp.log'
plot_log_no_V(file)

In [ ]:
file = f'./RDF_lv20_E0/Na_32_O_32_temp.log'
plot_log_no_V(file)
file = f'./RDF_lv20_E0/Na_32_O_16_temp.log'
plot_log_no_V(file)
#file = f'./RDF_lv20_E0/Na_32_O_4_temp.log'
#plot_log_no_V(file)

In [ ]:
element_map = {
    1: "C",
    2: "Mn", 
    3: "Fe",
    4: "N",
    5: "Na",
    6: "O",
    7: "H"
}
for i in [4]:
    input_file = f'./RDF_lv12_E0/dump.Na_28_O_{i}_temp.trj'
    output_file = f'./RDF_lv12_E0/new_dump.Na_28_O_{i}_temp_lv12.trj'
# Convert the file
    convert_dump_file(input_file, output_file, element_map)
    print(f"Conversion complete. Modified file saved as {output_file}")

In [ ]:
element_map = {
    1: "C",
    2: "Mn", 
    3: "Fe",
    4: "N",
    5: "Na",
    6: "O",
    7: "H"
}
for i in [4,16,32]:
    input_file = f'./RDF_lv12_E0_sf0.1_f0.1/dump.Na_28_O_{i}_temp.trj'
    output_file = f'./RDF_lv12_E0_sf0.1_f0.1/new_dump.Na_28_O_{i}_temp_lv_12_E0.trj'
# Convert the file
    convert_dump_file(input_file, output_file, element_map)
    print(f"Conversion complete. Modified file saved as {output_file}")

In [ ]:
element_map = {
    1: "C",
    2: "Mn", 
    3: "Fe",
    4: "N",
    5: "Na",
    6: "O",
    7: "H"
}
for i in [32,16]:
    input_file = f'./RDF_lv20_E0/dump.Na_32_O_{i}_temp.trj'
    output_file = f'./RDF_lv20_E0/new_dump.Na_32_O_{i}_temp_lv_20_E0.trj'
# Convert the file
    convert_dump_file(input_file, output_file, element_map)
    print(f"Conversion complete. Modified file saved as {output_file}")

## Raise Temp

In [ ]:
file = f'./diffusion/Na_28_O_32_temp.log'
plot_log_diff(file)
file = f'./diffusion/Na_28_O_16_temp.log'
plot_log_diff(file)
file = f'./diffusion/Na_28_O_4_temp.log'
plot_log_diff(file)

## MC 

In [ ]:
%matplotlib inline
temp=300
for i in range(9):
    file = f'./{temp}_MC/{temp}_MC_{i+1}.log'
    plot_log_no_V(file)

In [ ]:
element_map = {
    1: "C",
    2: "Mn", 
    3: "Fe",
    4: "N",
    5: "Na",
    6: "O",
    7: "H"
}
temp=600
for i in [7,8]:#range(9):
    input_file = f'./{temp}_diffusion_3/dump.{temp}_diffuse_{i+1}.trj'
    output_file = f'./{temp}_diffusion_3/new_dump.{temp}_diffuse_{i+1}.trj'
# Convert the file
    convert_dump_file(input_file, output_file, element_map)
    print(f"Conversion complete. Modified file saved as {output_file}")

# Diffusion

In [ ]:
element_map = {
    1: "C",
    2: "Mn", 
    3: "Fe",
    4: "N",
    5: "Na",
    6: "O",
    7: "H"
}
temp=500
for i in range(9):
    input_file = f'./{temp}_diffusion_2/dump.{temp}_diffuse_{i+1}.trj'
    output_file = f'./{temp}_diffusion_2/new_dump.{temp}_diffuse_{i+1}.trj'
# Convert the file
    convert_dump_file(input_file, output_file, element_map)
    print(f"Conversion complete. Modified file saved as {output_file}")

### 800K

In [ ]:
temp=800
for i in range(9):
    file = f'./{temp}_diffusion/{temp}_diffusion_{i+1}.log'
    plot_log_diff(file)

### 800K_2

In [ ]:
temp=800
for i in range(9):
    file = f'./{temp}_diffusion_2/{temp}_diffusion_{i+1}.log'
    plot_log_diff(file)

### 700K

In [ ]:
temp=700
for i in range(9):
    file = f'./{temp}_diffusion/{temp}_diffusion_{i+1}.log'
    plot_log_diff(file)

### 700K_2

In [ ]:
temp=700
for i in range(9):
    file = f'./{temp}_diffusion_2/{temp}_diffusion_{i+1}.log'
    plot_log_diff(file)

### 600K

In [ ]:
for i in range(9):
    file = f'./600_diffusion/600_diffusion_{i+1}.log'
    plot_log_diff(file)

### 600K_2

In [ ]:
for i in range(9):
    file = f'./600_diffusion_2/600_diffusion_{i+1}.log'
    plot_log_diff(file)

In [ ]:
for i in range(9):
    file = f'./550_diffusion_3/550_diffusion_{i+1}.log'
    plot_log_diff(file)

### 500K

In [ ]:
for i in range(9):
    file = f'./500_diffusion/500_diffusion_{i+1}.log'
    plot_log_diff(file)

### 500K_2

## 550K

### 550K_3

In [ ]:
for i in range(9):
    file = f'./550_diffusion_3/550_diffusion_{i+1}.log'
    plot_log_diff(file)

### 500K_4

In [ ]:
for i in range(4,6):
    file = f'./550_diffusion_5/550_diffusion_{i+1}.log'
    plot_log_diff(file)

In [ ]:
for i in range(9):
    file = f'./550_diffusion_4/550_diffusion_{i+1}.log'
    plot_log_diff(file)

### 550K_5

In [ ]:
for i in range(4,6):
    file = f'./550_diffusion_5/550_diffusion_{i+1}.log'
    plot_log_diff(file)

## 500K

### 500K_3

In [ ]:
for i in range(9):
    file = f'./500_diffusion_3/500_diffusion_{i+1}.log'
    plot_log_diff(file)

### 500K_4

In [ ]:
for i in range(9):
    file = f'./500_diffusion_4/500_diffusion_{i+1}.log'
    plot_log_diff(file)

## 450K

### 450K_3

In [ ]:
for i in range(9):
    file = f'./450_diffusion_3/450_diffusion_{i+1}.log'
    plot_log_diff(file)

### 450K_4

In [ ]:
for i in range(9):
    file = f'./450_diffusion_4/450_diffusion_{i+1}.log'
    plot_log_diff(file)

In [ ]:
for i in range(3,6):
    file = f'./450_diffusion_5/450_diffusion_{i+1}.log'
    plot_log_diff(file)

## 400K

### 400K_3

In [ ]:
for i in range(9):
    file = f'./400_diffusion_3/400_diffusion_{i+1}.log'
    plot_log_diff(file)

### 400K_4

In [ ]:
for i in range(9):
    file = f'./400_diffusion_4/400_diffusion_{i+1}.log'
    plot_log_diff(file)

### 400K_6

In [ ]:
for i in [1,6,8,9]:
    file = f'./Diffusion_all/400_diffusion_6/400_diffusion_{i}.log'
    plot_log_diff(file)

In [ ]:
%matplotlib widget
for i in [9]:
    file = f'./400_diffusion_4/400_diffusion_{i}.log'
    plot_log_diff(file)

### 400_diffusion_test_MTP

In [ ]:
for i in [9]:
    file = f'./400_diffusion_test_MTP/400_diffusion_{i}.log'
    plot_log_diff(file)

In [ ]:
for i in [9]:
    file = f'./400_diffusion_20_MTP/400_diffusion_{i}.log'
    plot_log_diff(file)

In [ ]:
i=9
data = parse_lammps_log(f'./400_diffusion_20_MTP_2/400_diffusion_{i}.log')
data

### 400_diffusion_20_MTP_2

In [ ]:

for i in [6,9,10]:
    file = f'./400_diffusion_20_MTP_2/400_diffusion_{i}.log'
    plot_log_diff(file)

In [ ]:
for i in [9,10]:
    file = f'./400_diffusion_20_short/400_diffusion_{i}.log'
    plot_log_diff(file)

In [ ]:
data = parse_lammps_log(file)
data

### 400_diffusion_vac

In [ ]:
for i in range(8):
    file = f'./400_diffusion_vac/400_diffusion_{i+1}.log'
    plot_log_diff(file)

### 400_diffusion_vac_2

In [ ]:
for i in range(8):
    file = f'./400_diffusion_vac_2/400_diffusion_{i+1}.log'
    plot_log_diff(file)

### 400_diffusion_vac_lv20

In [ ]:
for i in [1,2,4,5,6,7,8]:
    file = f'./400_diffusion_vac_lv20/400_diffusion_{i}.log'
    plot_log_diff(file)

### 400_diffusion_vac_lv20_2

In [ ]:
for i in [1,2,4,5,6,7,8]:
    file = f'./400_diffusion_vac_lv20_2/400_diffusion_{i}.log'
    plot_log_diff(file)

### 400_diffusion_vac_lv20_3

In [ ]:
for i in [1,2,4,5,6]:
    file = f'./400_diffusion_vac_lv20_3/400_diffusion_{i}.log'
    plot_log_diff(file)

### 400_diffusion_vac_lv20_5

In [ ]:
for i in [1,2,4,5,6,10]:
    file = f'./400_diffusion_vac_lv20_5/400_diffusion_{i}.log'
    plot_log_diff(file)

### 400_diffusion_20_r02_70

In [ ]:
for i in [8,9]:
    file = f'./400_diffusion_20_r02_70/400_diffusion_{i}.log'
    plot_log_diff(file)

### 400_diffusion_20_r02_65

In [ ]:
#%matplotlib widget
for i in [6,8,9]:
    file = f'./400_diffusion_20_r02_65/400_diffusion_{i}.log'
    plot_log_diff(file)

In [ ]:
#%matplotlib widget
for i in [9]:
    file = f'./400_diffusion_20_r02_65/400_diffusion_{i}_short.log'
    plot_log_diff(file)

### 400_diffusion_vac_lv20_r02_65

In [ ]:
for i in [1,2,3,4,5,6,10]:
    file = f'./400_diffusion_vac_lv20_r02_65/400_diffusion_{i}.log'
    plot_log_diff(file)

In [ ]:
for i in [10,11,12]:
    file = f'./400_diffusion_vac_lv20_r02_65_2/400_diffusion_{i}.log'
    plot_log_diff(file)

### Amaurey

In [ ]:
%matplotlib widget
file = f'/W/ac245936/no_vacancy/run1/gpuv100/log.lammps'
plot_log_diff(file)

In [ ]:
%matplotlib widget
for i in [1,6,9]:
    file = f'./400_diffusion_20_MTP/400_diffusion_{i}.log'
    plot_log_diff(file)

In [ ]:
for i in [1,2]:
    file = f'./400_diffusion_9/400_diffusion_{i}.log'
    plot_log_diff(file)

In [ ]:
for i in [1]:
    file = f'./400_diffusion_10/400_diffusion_{i}.log'
    plot_log_diff(file)

In [ ]:
for i in [1,6,9]:
    file = f'./300_diffusion_6/300_diffusion_{i}.log'
    plot_log_diff(file)

## 350K

### 350K_3

### 350K_4

In [ ]:
for i in range(9):
    file = f'./350_diffusion_4/350_diffusion_{i+1}.log'
    plot_log_diff(file)

In [ ]:
for i in range(1):
    file = f'./350_diffusion_3/350_diffusion_{i+1}.log'
    plot_log_diff(file)
for i in range(2,9):
    file = f'./350_diffusion_3/350_diffusion_{i+1}.log'
    plot_log_diff(file)

## 300K

### 300K_3

### 300K_4

In [ ]:
for i in range(9):
    file = f'./300_diffusion_4/300_diffusion_{i+1}.log'
    plot_log_diff(file)

### 300K_5

In [ ]:
for i in range(3,5):
    file = f'./300_diffusion_5/300_diffusion_{i+1}.log'
    plot_log_diff(file)

In [ ]:
for i in range(1):
    file = f'./300_diffusion_3/300_diffusion_{i+1}.log'
    plot_log_diff(file)
for i in range(2,9):
    file = f'./300_diffusion_3/300_diffusion_{i+1}.log'
    plot_log_diff(file)

### 400K

In [ ]:
for i in range(9):
    file = f'./400_diffusion/400_diffusion_{i+1}.log'
    plot_log_diff(file)

### 400K_2

In [ ]:
for i in range(9):
    file = f'./400_diffusion_2/400_diffusion_{i+1}.log'
    plot_log_diff(file)

## MSD analysis

In [ ]:
#%matplotlib widget
def MSD_analysis_Na(temp, serie, Na_list, start, end, name):
    delta_t_ls = np.array([4, 5, 6, 7, 8,
                           9, 10, 15, 20, 30,
                           40, 50, 60, 70, 80,
                           90, 100, 200, 300, 400,
                           500,600])#,2000,5000]) #1, 2.5, 10, 25, 100, 250 ps
    
    fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(17,5))
    plt.subplots_adjust(wspace=1)
    
    temp = temp
    
    diffusivity_list = []
    num_Na_list = [216, 866, 1512,
                   205, 867, 1512,
                   205, 867, 1512]
    num_H2O_list = [216, 216, 216,
                    864, 864, 864,
                    1728,1728,1728]
    k=Na_list[0]-1
    for index in Na_list:
        if serie == 0:
            file = f'./{temp}_diffusion/{temp}_diffusion_{index}.log'
        else:
            file = f'./{temp}_diffusion_{serie}/{temp}_diffusion_{index}.log'
        log_data = parse_lammps_log_diff(file)
        MSD_Na = np.array(log_data['MSD(Na)'])
        MSD_H2O = np.array(log_data['MSD(H2O)'])
        timestep = np.array(log_data['Step'])
        t_intival_ls = []
        MSD_intival_ls = []
        for i in range(delta_t_ls.shape[0]):
            num_segment = int((MSD_Na.shape[0]-start)/delta_t_ls[i])
            t_intival = []
            MSD_intival = []
            for w in range(num_segment):
                t_intival.append(timestep[start + delta_t_ls[i]*w : start + delta_t_ls[i]*(w+1)] - timestep[start + delta_t_ls[i]*w])
                MSD_intival.append(MSD_Na[start + delta_t_ls[i]*w : start + delta_t_ls[i]*(w+1)] - MSD_Na[start + delta_t_ls[i]*w])
            t_intival_ls.append(t_intival)
            MSD_intival_ls.append(MSD_intival)
    
        delta_t_all = []
        delta_MSD_all = []
        for i in range(delta_t_ls.shape[0]):
            delta_t = []
            delta_MSD = []
            for j in range(len(MSD_intival_ls[i])):
                delta_MSD.append(MSD_intival_ls[i][j][-1])
                delta_t.append(t_intival_ls[i][j][-1])
    
            delta_t_all.append(delta_t)
            delta_MSD_all.append(delta_MSD)
    
        final_MSD = []
        final_t = []
        MSD_std = []
        diffusivity = []
        
        for i in range(delta_t_ls.shape[0]):
            final_t.append(sum(delta_t_all[i])/len(delta_t_all[i]))
            final_MSD.append(sum(delta_MSD_all[i])/len(delta_MSD_all[i]))
            MSD_std.append(np.std(delta_MSD_all[i]))
            
        final_MSD = np.array(final_MSD)
        final_t = np.array(final_t)
        MSD_std = np.array(MSD_std)
        diffusivity_list.append(final_MSD/final_t/6*1e-5)
        ax2.errorbar(final_t, final_MSD, yerr=MSD_std, capsize=5, fmt='-o', alpha=0.8)
        #ax2.plot(final_t, final_MSD, marker='*', markersize=3, label=f'{temp}K x = {round(num_Na_list[k]/864,3)} y = {round(num_H2O_list[k]/864,3)}')
        ax1.set_xlabel('Timestep (1 fs/step)')
        ax1.set_ylabel('MSD A^2')
        ax1.plot(log_data['Step'], log_data['MSD(Na)'],  marker='*', markersize=3, label=f'{temp}K x = {round(num_Na_list[k]/864,3)} y = {round(num_H2O_list[k]/864,3)}')
        ax1.plot(timestep[:start], MSD_Na[:start], color='black', alpha=0.5)
        k+=1
        
    ax1.grid(True, linestyle='--', alpha=0.7)
    ax2.grid(True, linestyle='--', alpha=0.7)
    #ax2.legend()
    ax1.legend()
    #ax2.set_xscale('log')
    #ax2.set_yscale('log')
    ax2.set_xlabel('delta t (fs)')
    ax2.set_ylabel('MSD (A^2)')
    plt.title(f'{temp} K')
    
    k=Na_list[0]-1
    for i in range(len(diffusivity_list)):
        #ax3.errorbar(final_t, diffusivity_list[i], yerr=MSD_std/final_t/6*1e-5 , capsize=5 , fmt='-o')
        ax3.plot(final_t, diffusivity_list[i], marker='*', markersize=3, label=f'{temp}K x = {round(num_Na_list[k]/864,3)} y = {round(num_H2O_list[k]/864,3)}')
        k+=1
    
    ax3.grid(True, linestyle='--', alpha=0.7)
    #ax3.legend()
    #ax3.set_xscale('log')
    #ax3.set_yscale('log')
    ax3.set_xlabel('delta t (fs)')
    ax3.set_ylabel('Diffusivity (m^2/s)')
    
    plt.tight_layout()
    if serie == 0:
        plt.savefig(f'./{temp}_diffusion/MSD_analysis_Na_{name}.png', bbox_inches='tight')
    else:
        plt.savefig(f'./{temp}_diffusion_{serie}/MSD_analysis_Na_{name}.png', bbox_inches='tight')
    return diffusivity_list

In [ ]:
temp, serie, Na_list, start, end, name = 400, 4, [9,8,7,6,5,1], 100, -1, 'O4' delta_t_ls = np.array([4, 5, 6, 7, 8, 9, 10, 15, 20, 30, 40, 50, 60, 70, 80, 90, 100])#,2000,5000]) #1, 2.5, 10, 25, 100, 250 ps fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(singlecol*2, singlecol*1), dpi=300) plt.subplots_adjust(wspace=1) temp = temp diffusivity_list = [] num_Na_list = [216, 866, 1512, 205, 867, 1512, 205, 867, 1512] num_H2O_list = [216, 216, 216, 864, 864, 864, 1728,1728,1728] color_ls=[] for i in range(6): index = Na_list[i] if serie != 0: file = f'./{temp}_diffusion_{serie}/{temp}_diffusion_{index}.log' else: file = f'./{temp}_diffusion/{temp}_diffusion_{index}.log' log_data = parse_lammps_log_diff(file) MSD_Na = np.array(log_data['MSD(Na)']) MSD_H2O = np.array(log_data['MSD(H2O)']) timestep = np.array(log_data['Step']) ax1.plot(log_data['Step'], log_data['MSD(Na)'], marker='*', markersize=3, label=f'x = {round(num_Na_list[(Na_list[i]-1])/864,3)} y = {round(num_H2O_list[(Na_list[i]-1])]/864,3)}', color='') ax1.plot(timestep[:start], MSD_Na[:start], color='black', alpha=0.5, marker='*', markersize=3) ax2.plot(log_data['Step'], log_data['MSD(H2O)'], marker='*', markersize=3, label=f'x = {round(num_Na_list[(Na_list[i]-1])]/864,3)} y = {round(num_H2O_list[(Na_list[i]-1])]/864,3)}', color='') ax2.plot(timestep[:start], MSD_H2O[:start], color='black', alpha=0.5,marker='*', markersize=3) ax1.set_xlabel('Timestep (1 fs/step)') ax1.set_ylabel('MSD A^2') ax2.set_xlabel('Timestep (1 fs/step)') ax2.set_ylabel('MSD A^2') ax2.legend(frameon=False) ax1.legend(frameon=False) ax2.set_xlabel('delta t (fs)') ax2.set_ylabel('MSD (A^2)') plt.tight_layout() #plt.savefig(f'./{temp}_diffusion_{serie}/MSD_analysis_Na_{name}.png', bbox_inches='tight')

In [ ]:
Na_list[i]-1

In [ ]:
temp, serie, Na_list, start, end, name = 400, 4, [9,8,7,6,5,1], 300, -1, 'O4'
delta_t_ls = np.array([4, 5, 6, 7, 8,
                       9, 10, 15, 20, 30,
                       40, 50, 60, 70, 80,
                       90, 100])#,2000,5000]) #1, 2.5, 10, 25, 100, 250 ps

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(singlecol*2, singlecol*1), dpi=300)
plt.subplots_adjust(wspace=1)

temp = temp

diffusivity_list = []
num_Na_list = [216, 866, 1512,
               205, 867, 1512,
               205, 867, 1512]
num_H2O_list = [216, 216, 216,
                864, 864, 864,
                1728,1728,1728]
marker_ls=['s','s','s','o','o','^',]
color_ls=['navy', 'royalblue', 'c', 'red', 'orangered', 'g']

for i in range(6):
    index = Na_list[i]
    if serie != 0:
        file = f'./{temp}_diffusion_{serie}/{temp}_diffusion_{index}.log'
    else:
        file = f'./{temp}_diffusion/{temp}_diffusion_{index}.log'
    log_data = parse_lammps_log_diff(file)
    MSD_Na = np.array(log_data['MSD(Na)'])
    MSD_H2O = np.array(log_data['MSD(H2O)'])
    timestep = np.array(log_data['Step'])
    x = round(num_Na_list[(Na_list[i]-1)]/864,3)
    y = round(num_H2O_list[(Na_list[i]-1)]/864,3)
    ax1.plot(log_data['Step'], log_data['MSD(Na)'], color=color_ls[i], marker=marker_ls[i], markersize=3, 
             label=f'x = {x} y = {y}')
    ax1.plot(timestep[:start], MSD_Na[:start], color='black', alpha=0.25, marker=marker_ls[i],  markersize=3)
    
    ax2.plot(log_data['Step'], log_data['MSD(H2O)'], color=color_ls[i], marker=marker_ls[i],  markersize=3, 
             label=f'x = {x} y = {y}')
    ax2.plot(timestep[:start], MSD_H2O[:start], color='black', alpha=0.2,marker=marker_ls[i], markersize=3)

    
ax1.set_xlabel('Timestep (1 fs/step)')
ax1.set_ylabel(f'Sodium MSD at {temp} K ($\\AA^2$)')
ax2.set_xlabel('Timestep (1 fs/step)')
ax2.set_ylabel(f'Water MSD at {temp} K ($\\AA^2$)')
ax2.legend(frameon=False)
ax1.legend(frameon=False)
plt.tight_layout()
fig.subplots_adjust(wspace=0.3)
plt.savefig(f'./{temp}_MSD.png')

In [ ]:
#%matplotlib widget

path, temp, Na_list, start, end, name = '400_diffusion_vac_lv20_r02_65_2', 400, [1,2,3,4,5], 500, 4000, 'vac'
delta_t_ls = np.array([4, 5, 6, 7, 8,
                       9, 10, 15, 20, 30,
                       40, 50, 60, 70, 80,
                       90, 100, 200, 300, 400,
                       500,600,700,800])#,2000,5000]) #1, 2.5, 10, 25, 100, 250 ps

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(singlecol*2, singlecol*1), dpi=300)
plt.subplots_adjust(wspace=1)

temp = temp
num_Na_list = [224,224,224,192,192,192,1,1,1,240]
num_H2O_list = [416,512,584,312,512,626,1,1,1,520]
marker_ls=['s','s','s','o','o','^',]
color_ls=['navy', 'royalblue', 'c', 'red', 'orangered', 'g']
diffusivity_list = []

for i in range(5):
    file = f'./{path}/{temp}_diffusion_{i+1}.log'
    log_data = parse_lammps_log_diff(file)
    MSD_Na = np.array(log_data['MSD(Na)'])
    MSD_H2O = np.array(log_data['MSD(H2O)'])
    timestep = np.array(log_data['Step'])
    ax1.plot(log_data['Step'][:end], log_data['MSD(H2O)'][:end], color=color_ls[i], marker=marker_ls[i], markersize=3, 
             label=f'y = {round(num_H2O_list[i]/256,3):.3f} □ = {1 - round(num_Na_list[i]/256,3):.3f}')
    ax1.plot(timestep[:start], MSD_H2O[:start], color='black', alpha=0.25, marker=marker_ls[i],  markersize=3)

i, index = 5, 10
file = f'./{path}/{temp}_diffusion_{index}.log'
log_data = parse_lammps_log_diff(file)
MSD_Na = np.array(log_data['MSD(Na)'])
MSD_H2O = np.array(log_data['MSD(H2O)'])
timestep = np.array(log_data['Step'])
ax2.plot(log_data['Step'][:end], log_data['MSD(H2O)'][:end], color=color_ls[1], marker=marker_ls[i],  markersize=3, 
           label=f'x = 1 y = {round(num_H2O_list[i]/256,3)} □ = 0.0625')
ax2.plot(timestep[:start], MSD_H2O[:start], color='black', alpha=0.2,marker=marker_ls[i], markersize=3)
ax22 = ax2.twinx()
ax22.plot(log_data['Step'][:end], log_data['MSD(Na)'][:end], color=color_ls[4], marker=marker_ls[i],  markersize=3, 
          label=f'NSD(Na): x = 1 y = {round(num_H2O_list[i]/256,3)} □ = 0.0625')
ax22.plot(timestep[:start], MSD_Na[:start], color='black', alpha=0.2,marker=marker_ls[i], markersize=3)

#ax1.set_ylim(0,5)
#ax2.set_ylim(0,2.5)
ax1.set_xlabel('Timestep (1 fs/step)')
ax1.set_ylabel(f'Water MSD at {temp} K ($\\AA^2$)')
ax2.set_xlabel('Timestep (1 fs/step)')
ax2.set_ylabel(f'Water MSD at {temp} K ($\\AA^2$)', color=color_ls[1])
ax22.set_ylabel(f'Sodium MSD at {temp} K ($\\AA^2$)', color=color_ls[4])
ax2.legend(frameon=False, loc = 'lower right')
ax1.legend(frameon=False, loc = 'lower right')
plt.tight_layout()
fig.subplots_adjust(wspace=0.3)

plt.tight_layout()
plt.savefig(f'./MSD_analysis_vac_long.png', bbox_inches='tight')

In [ ]:
#%matplotlib widget

path, temp, Na_list, start, end, name = '400_diffusion_vac_lv20_r02_65_2', 400, [10, 11, 12], 500, 20000, 'vac'
delta_t_ls = np.array([4, 5, 6, 7, 8,
                       9, 10, 15, 20, 30,
                       40, 50, 60, 70, 80,
                       90, 100, 200, 300, 400,
                       500,600,700,800])#,2000,5000]) #1, 2.5, 10, 25, 100, 250 ps

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(singlecol*2, singlecol*1), dpi=300)
plt.subplots_adjust(wspace=1)

temp = temp
num_Na_list = [224,224,224,192,192,192,1,1,1,240,0,59]
num_H2O_list = [416,512,584,312,512,626,1,1,1,520,520,520]
marker_ls=['s','s','s','o','o','^',]
color_ls=['royalblue', 'orangered', 'navy', 'red', 'orangered', 'g']
diffusivity_list = []

for i in [10,11,9]:
    file = f'./{path}/{temp}_diffusion_{i+1}.log'
    log_data = parse_lammps_log_diff(file)
    MSD_Na = np.array(log_data['MSD(Na)'])
    MSD_H2O = np.array(log_data['MSD(H2O)'])
    timestep = np.array(log_data['Step'])
    ax1.plot(log_data['Step'][:end], log_data['MSD(H2O)'][:end], color=color_ls[i-9], marker=marker_ls[i-9], markersize=3, 
             label=f'x = {round(num_Na_list[i]/256,3)}')
    ax1.plot(timestep[:start], MSD_H2O[:start], color='black', alpha=0.25, marker=marker_ls[i-9],  markersize=3)

for i in [11,9]:
    file = f'./{path}/{temp}_diffusion_{i+1}.log'
    log_data = parse_lammps_log_diff(file)
    MSD_Na = np.array(log_data['MSD(Na)'])
    MSD_H2O = np.array(log_data['MSD(H2O)'])
    timestep = np.array(log_data['Step'])
    ax2.plot(log_data['Step'][:end], log_data['MSD(Na)'][:end], color=color_ls[i-9], marker=marker_ls[i-9], markersize=3, 
             label=f'x = {round(num_Na_list[i]/256,3)}')
    ax2.plot(timestep[:start], MSD_Na[:start], color='black', alpha=0.25, marker=marker_ls[i-9],  markersize=3)


#ax1.set_ylim(0,5)
#ax2.set_ylim(0,2.5)
ax1.set_xlabel('Timestep (1 fs/step)')
ax1.set_ylabel(f'Water MSD at {temp} K ($\\AA^2$)')
ax2.set_xlabel('Timestep (1 fs/step)')
ax2.set_ylabel(f'Sodium MSD at {temp} K ($\\AA^2$)')
ax2.legend(frameon=False)#, loc = 'lower right')
ax1.legend(frameon=False, fontsize=8)#, loc = 'lower right')
plt.tight_layout()
fig.subplots_adjust(wspace=0.3)

plt.tight_layout()
plt.savefig(f'./MSD_analysis_vac_Na.png', bbox_inches='tight')

In [ ]:
#%matplotlib widget
def MSD_analysis_O(temp, serie, Na_list, start, end, name):
    delta_t_ls = np.array([4, 5, 6, 7, 8,
                           9, 10, 15, 20, 30,
                           40, 50, 60, 70, 80,
                           90, 100, 200, 300, 400,
                           500,600,700,800])#,2000,5000]) #1, 2.5, 10, 25, 100, 250 ps
    
    fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(17,5))
    plt.subplots_adjust(wspace=1)
    
    temp = temp
    
    diffusivity_list = []
    num_Na_list = [216, 866, 1512,
                   205, 867, 1512,
                   205, 867, 1512]
    num_H2O_list = [216, 216, 216,
                    864, 864, 864,
                    1728,1728,1728]
    k=Na_list[0]-1
    for index in Na_list:
        if serie == 0:
            file = f'./{temp}_diffusion/{temp}_diffusion_{index}.log'
        else:
            file = f'./{temp}_diffusion_{serie}/{temp}_diffusion_{index}.log'
        log_data = parse_lammps_log_diff(file)
        MSD_Na = np.array(log_data['MSD(H2O)'])
        timestep = np.array(log_data['Step'])
        t_intival_ls = []
        MSD_intival_ls = []
        for i in range(delta_t_ls.shape[0]):
            num_segment = int((MSD_Na.shape[0]-start)/delta_t_ls[i])
            t_intival = []
            MSD_intival = []
            for w in range(num_segment):
                t_intival.append(timestep[start + delta_t_ls[i]*w : start + delta_t_ls[i]*(w+1)] - timestep[start + delta_t_ls[i]*w])
                MSD_intival.append(MSD_Na[start + delta_t_ls[i]*w : start + delta_t_ls[i]*(w+1)] - MSD_Na[start + delta_t_ls[i]*w])
            t_intival_ls.append(t_intival)
            MSD_intival_ls.append(MSD_intival)
    
        delta_t_all = []
        delta_MSD_all = []
        for i in range(delta_t_ls.shape[0]):
            delta_t = []
            delta_MSD = []
            for j in range(len(MSD_intival_ls[i])):
                delta_MSD.append(MSD_intival_ls[i][j][-1])
                delta_t.append(t_intival_ls[i][j][-1])
    
            delta_t_all.append(delta_t)
            delta_MSD_all.append(delta_MSD)
    
        final_MSD = []
        final_t = []
        MSD_std = []
        diffusivity = []
        
        for i in range(delta_t_ls.shape[0]):
            final_t.append(sum(delta_t_all[i])/len(delta_t_all[i]))
            final_MSD.append(sum(delta_MSD_all[i])/len(delta_MSD_all[i]))
            MSD_std.append(np.std(delta_MSD_all[i]))
            
        final_MSD = np.array(final_MSD)
        final_t = np.array(final_t)
        MSD_std = np.array(MSD_std)
        diffusivity_list.append(final_MSD/final_t/6*1e-5)
        ax2.errorbar(final_t, final_MSD, yerr=MSD_std, capsize=5, fmt='-o', alpha=0.8)
        #ax2.plot(final_t, final_MSD, marker='*', markersize=3, label=f'{temp}K x = {round(num_Na_list[k]/864,3)} y = {round(num_H2O_list[k]/864,3)}')
        ax1.set_xlabel('Timestep (1 fs/step)')
        ax1.set_ylabel('MSD A^2')
        ax1.plot(log_data['Step'], log_data['MSD(H2O)'],  marker='*', markersize=3, label=f'{temp}K x = {round(num_Na_list[k]/864,3)} y = {round(num_H2O_list[k]/864,3)}')
        ax1.plot(timestep[:start], MSD_Na[:start], color='black', alpha=0.5)
        k+=1
        
    ax1.grid(True, linestyle='--', alpha=0.7)
    ax2.grid(True, linestyle='--', alpha=0.7)
    #ax2.legend()
    ax1.legend()
    #ax2.set_xscale('log')
    #ax2.set_yscale('log')
    ax2.set_xlabel('delta t (fs)')
    ax2.set_ylabel('MSD (A^2)')
    plt.title(f'{temp} K')
    
    k=Na_list[0]-1
    for i in range(len(diffusivity_list)):
        #ax3.errorbar(final_t, diffusivity_list[i], yerr=MSD_std/final_t/6*1e-5 , capsize=5 , fmt='-o')
        ax3.plot(final_t, diffusivity_list[i], marker='*', markersize=3, label=f'{temp}K x = {round(num_Na_list[k]/864,3)} y = {round(num_H2O_list[k]/864,3)}')
        k+=1
    
    ax3.grid(True, linestyle='--', alpha=0.7)
    #ax3.legend()
    #ax3.set_xscale('log')
    #ax3.set_yscale('log')
    ax3.set_xlabel('delta t (fs)')
    ax3.set_ylabel('Diffusivity (m^2/s)')
    
    plt.tight_layout()
    if serie == 0:
        plt.savefig(f'./{temp}_diffusion/MSD_analysis_O_{name}.png', bbox_inches='tight')
    else:
        plt.savefig(f'./{temp}_diffusion_{serie}/MSD_analysis_O_{name}.png', bbox_inches='tight')
    return diffusivity_list

In [ ]:
#%matplotlib widget
def MSD_analysis_vac_O(path, temp, Na_list, start, end, name):
    delta_t_ls = np.array([4, 5, 6, 7, 8,
                           9, 10, 15, 20, 30,
                           40, 50, 60, 70, 80,
                           90, 100, 200, 300, 400,
                           500,600,700,800])#,2000,5000]) #1, 2.5, 10, 25, 100, 250 ps
    
    fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(17,5))
    plt.subplots_adjust(wspace=1)
    
    temp = temp
    num_Na_list = [224,224,224,192,192,192,1,1,1,240,0,59]
    num_H2O_list = [416,512,584,312,512,626,1,1,1,520,520,520]
    diffusivity_list = []
    
    k=Na_list[0]-1
    for index in Na_list:
        file = f'./{path}/{temp}_diffusion_{index}.log'
        log_data = parse_lammps_log_diff(file)
        MSD_Na = np.array(log_data['MSD(H2O)'])
        timestep = np.array(log_data['Step'])
        t_intival_ls = []
        MSD_intival_ls = []
        for i in range(delta_t_ls.shape[0]):
            num_segment = int((MSD_Na.shape[0]-start)/delta_t_ls[i])
            t_intival = []
            MSD_intival = []
            for w in range(num_segment):
                t_intival.append(timestep[start + delta_t_ls[i]*w : start + delta_t_ls[i]*(w+1)] - timestep[start + delta_t_ls[i]*w])
                MSD_intival.append(MSD_Na[start + delta_t_ls[i]*w : start + delta_t_ls[i]*(w+1)] - MSD_Na[start + delta_t_ls[i]*w])
            t_intival_ls.append(t_intival)
            MSD_intival_ls.append(MSD_intival)
    
        delta_t_all = []
        delta_MSD_all = []
        for i in range(delta_t_ls.shape[0]):
            delta_t = []
            delta_MSD = []
            for j in range(len(MSD_intival_ls[i])):
                delta_MSD.append(MSD_intival_ls[i][j][-1])
                delta_t.append(t_intival_ls[i][j][-1])
    
            delta_t_all.append(delta_t)
            delta_MSD_all.append(delta_MSD)
    
        final_MSD = []
        final_t = []
        MSD_std = []
        diffusivity = []
        
        for i in range(delta_t_ls.shape[0]):
            final_t.append(sum(delta_t_all[i])/len(delta_t_all[i]))
            final_MSD.append(sum(delta_MSD_all[i])/len(delta_MSD_all[i]))
            MSD_std.append(np.std(delta_MSD_all[i]))
            
        final_MSD = np.array(final_MSD)
        final_t = np.array(final_t)
        MSD_std = np.array(MSD_std)
        diffusivity_list.append(final_MSD/final_t/6*1e-5)
        ax2.errorbar(final_t, final_MSD, yerr=MSD_std, capsize=5, fmt='-o', alpha=0.8)
        #ax2.plot(final_t, final_MSD, marker='*', markersize=3, label=f'{temp}K x = {round(num_Na_list[k]/864,3)} y = {round(num_H2O_list[k]/864,3)}')
        ax1.set_xlabel('Timestep (1 fs/step)')
        ax1.set_ylabel('MSD A^2')
        ax1.plot(log_data['Step'][:end], log_data['MSD(H2O)'][:end],  marker='*', markersize=3, label=f'{temp}K vac = {round(num_Na_list[k]/256,3)} y = {round(num_H2O_list[k]/256,3)}')
        ax1.plot(timestep[:start], MSD_Na[:start], color='black', alpha=0.5)
        k+=1
        
    ax1.grid(True, linestyle='--', alpha=0.7)
    ax2.grid(True, linestyle='--', alpha=0.7)
    #ax2.legend()
    ax1.legend()
    #ax2.set_xscale('log')
    #ax2.set_yscale('log')
    ax2.set_xlabel('delta t (fs)')
    ax2.set_ylabel('MSD (A^2)')
    plt.title(f'{temp} K')
    
    k=Na_list[0]-1
    for i in range(len(diffusivity_list)):
        #ax3.errorbar(final_t, diffusivity_list[i], yerr=MSD_std/final_t/6*1e-5 , capsize=5 , fmt='-o')
        ax3.plot(final_t, diffusivity_list[i], marker='*', markersize=3, label=f'{temp}K x = {round(num_Na_list[k]/864,3)} y = {round(num_H2O_list[k]/864,3)}')
        k+=1
    
    ax3.grid(True, linestyle='--', alpha=0.7)
    #ax3.legend()
    #ax3.set_xscale('log')
    #ax3.set_yscale('log')
    ax3.set_xlabel('delta t (fs)')
    ax3.set_ylabel('Diffusivity (m^2/s)')
    
    plt.tight_layout()
    plt.savefig(f'./{path}/MSD_analysis_O_{name}.png', bbox_inches='tight')
    return diffusivity_list

In [ ]:
#%matplotlib widget
def MSD_analysis_vac_Na(path, temp, Na_list, start, end, name):
    delta_t_ls = np.array([4, 5, 6, 7, 8,
                           9, 10, 15, 20, 30,
                           40, 50, 60, 70, 80,
                           90, 100, 200, 300, 400,
                           500,600])#,2000,5000]) #1, 2.5, 10, 25, 100, 250 ps
    
    fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(17,5))
    plt.subplots_adjust(wspace=1)
    
    temp = temp
    
    diffusivity_list = []
    num_Na_list = [216, 866, 1512,
                   205, 867, 1512,
                   205, 867, 1512,1]
    num_H2O_list = [216, 216, 216,
                    864, 864, 864,
                    1728,1728,1728,1]
    k=Na_list[0]-1
    for index in Na_list:
        file = f'./{path}/{temp}_diffusion_{index}.log'
        log_data = parse_lammps_log_diff(file)
        MSD_Na = np.array(log_data['MSD(Na)'])
        MSD_H2O = np.array(log_data['MSD(H2O)'])
        timestep = np.array(log_data['Step'])
        t_intival_ls = []
        MSD_intival_ls = []
        for i in range(delta_t_ls.shape[0]):
            num_segment = int((MSD_Na.shape[0]-start)/delta_t_ls[i])
            t_intival = []
            MSD_intival = []
            for w in range(num_segment):
                t_intival.append(timestep[start + delta_t_ls[i]*w : start + delta_t_ls[i]*(w+1)] - timestep[start + delta_t_ls[i]*w])
                MSD_intival.append(MSD_Na[start + delta_t_ls[i]*w : start + delta_t_ls[i]*(w+1)] - MSD_Na[start + delta_t_ls[i]*w])
            t_intival_ls.append(t_intival)
            MSD_intival_ls.append(MSD_intival)
    
        delta_t_all = []
        delta_MSD_all = []
        for i in range(delta_t_ls.shape[0]):
            delta_t = []
            delta_MSD = []
            for j in range(len(MSD_intival_ls[i])):
                delta_MSD.append(MSD_intival_ls[i][j][-1])
                delta_t.append(t_intival_ls[i][j][-1])
    
            delta_t_all.append(delta_t)
            delta_MSD_all.append(delta_MSD)
    
        final_MSD = []
        final_t = []
        MSD_std = []
        diffusivity = []
        
        for i in range(delta_t_ls.shape[0]):
            final_t.append(sum(delta_t_all[i])/len(delta_t_all[i]))
            final_MSD.append(sum(delta_MSD_all[i])/len(delta_MSD_all[i]))
            MSD_std.append(np.std(delta_MSD_all[i]))
            
        final_MSD = np.array(final_MSD)
        final_t = np.array(final_t)
        MSD_std = np.array(MSD_std)
        diffusivity_list.append(final_MSD/final_t/6*1e-5)
        ax2.errorbar(final_t, final_MSD, yerr=MSD_std, capsize=5, fmt='-o', alpha=0.8)
        #ax2.plot(final_t, final_MSD, marker='*', markersize=3, label=f'{temp}K x = {round(num_Na_list[k]/864,3)} y = {round(num_H2O_list[k]/864,3)}')
        ax1.set_xlabel('Timestep (1 fs/step)')
        ax1.set_ylabel('MSD A^2')
        ax1.plot(log_data['Step'], log_data['MSD(Na)'],  marker='*', markersize=3, label=f'{temp}K x = {round(num_Na_list[k]/864,3)} y = {round(num_H2O_list[k]/864,3)}')
        ax1.plot(timestep[:start], MSD_Na[:start], color='black', alpha=0.5)
        k+=1
        
    ax1.grid(True, linestyle='--', alpha=0.7)
    ax2.grid(True, linestyle='--', alpha=0.7)
    #ax2.legend()
    ax1.legend()
    #ax2.set_xscale('log')
    #ax2.set_yscale('log')
    ax2.set_xlabel('delta t (fs)')
    ax2.set_ylabel('MSD (A^2)')
    plt.title(f'{temp} K')
    
    k=Na_list[0]-1
    for i in range(len(diffusivity_list)):
        #ax3.errorbar(final_t, diffusivity_list[i], yerr=MSD_std/final_t/6*1e-5 , capsize=5 , fmt='-o')
        ax3.plot(final_t, diffusivity_list[i], marker='*', markersize=3, label=f'{temp}K x = {round(num_Na_list[k]/864,3)} y = {round(num_H2O_list[k]/864,3)}')
        k+=1
    
    ax3.grid(True, linestyle='--', alpha=0.7)
    #ax3.legend()
    #ax3.set_xscale('log')
    #ax3.set_yscale('log')
    ax3.set_xlabel('delta t (fs)')
    ax3.set_ylabel('Diffusivity (m^2/s)')
    
    plt.tight_layout()

    return diffusivity_list

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
import os

def MSD_analysis(temp, Na_list, timestep_fs=1.0, output_interval=500):
    """
    Analyze Mean Square Displacement (MSD) data and calculate diffusivity.
    
    Parameters:
    -----------
    temp : float
        Temperature in Kelvin
    Na_list : list
        List of Na configurations to analyze
    timestep_fs : float
        Timestep in femtoseconds (default: 1.0 fs)
    output_interval : int
        Output interval in timesteps (default: 500)
    
    Returns:
    --------
    dict : Analysis results including diffusivities and fit parameters
    """
    
    # Define time intervals for analysis (in ps, converted to timesteps)
    delta_t_ps = np.array([1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 15, 20, 30, 40, 50, 60, 70, 80, 90, 100])
    delta_t_steps = delta_t_ps * 1000 / timestep_fs  # Convert ps to timesteps
    
    # Create figure with better layout
    fig, axes = plt.subplots(2, 2, figsize=(15, 12))
    fig.suptitle(f'MSD Analysis at {temp} K', fontsize=16, fontweight='bold')
    
    # Configuration parameters
    composition_data = {
        'num_Na': [216, 856, 1509, 216, 864, 1512],
        'num_H2O': [216, 216, 216, 1512, 1512, 1512],
        'total_particles': 1728  # Assuming constant total
    }
    
    results = {
        'diffusivities': [],
        'compositions': [],
        'linear_fits': [],
        'r_squared': []
    }
    
    for idx, na_config in enumerate(Na_list):
        # Load and parse data
        file_path = f'./{temp}_diffusion_3/{temp}_diffusion_{na_config}.log'
        
        if not os.path.exists(file_path):
            print(f"Warning: File {file_path} not found. Skipping...")
            continue
            
        try:
            log_data = parse_lammps_log_diff(file_path)
        except Exception as e:
            print(f"Error parsing {file_path}: {e}")
            continue
        
        # Extract data
        timesteps = np.array(log_data['Step'])
        msd_na = np.array(log_data['MSD(Na)'])
        msd_h2o = np.array(log_data['MSD(H2O)']) if 'MSD(H2O)' in log_data else None
        
        # Convert timesteps to time in fs
        time_fs = timesteps * timestep_fs
        
        # Calculate composition fractions
        x_na = composition_data['num_Na'][idx] / composition_data['total_particles']
        y_h2o = composition_data['num_H2O'][idx] / composition_data['total_particles']
        label = f'x(Na)={x_na:.3f}, y(H₂O)={y_h2o:.3f}'
        
        # Plot raw MSD data
        axes[0, 0].plot(time_fs / 1000, msd_na, marker='o', markersize=2, 
                       label=label, alpha=0.7)
        
        # Calculate MSD differences for different time intervals
        delta_times = []
        delta_msds = []
        
        for dt_steps in delta_t_steps:
            dt_int = int(dt_steps)
            if dt_int >= len(timesteps):
                continue
                
            # Calculate time differences and MSD differences
            time_diffs = []
            msd_diffs = []
            
            for i in range(0, len(timesteps) - dt_int, dt_int):
                if i + dt_int < len(timesteps):
                    dt = timesteps[i + dt_int] - timesteps[i]
                    dmsd = msd_na[i + dt_int] - msd_na[i]
                    time_diffs.append(dt * timestep_fs)  # Convert to fs
                    msd_diffs.append(dmsd)
            
            if time_diffs:
                avg_dt = np.mean(time_diffs)
                avg_dmsd = np.mean(msd_diffs)
                delta_times.append(avg_dt)
                delta_msds.append(avg_dmsd)
        
        delta_times = np.array(delta_times)
        delta_msds = np.array(delta_msds)
        
        # Plot MSD vs delta_t
        axes[0, 1].plot(delta_times / 1000, delta_msds, marker='s', markersize=4, 
                       label=label, alpha=0.7)
        
        # Calculate diffusivity using Einstein relation: D = MSD/(6*t) for 3D
        # Convert to m²/s: Å²/fs * 1e-20 m²/Å² * 1e15 fs/s = 1e-5 m²/s
        diffusivities = delta_msds / (6 * delta_times) * 1e-5
        
        # Plot diffusivity vs time
        axes[1, 0].plot(delta_times / 1000, diffusivities, marker='^', markersize=4, 
                       label=label, alpha=0.7)
        
        # Linear fit for long-time behavior (use last 50% of data points)
        if len(delta_times) > 4:
            fit_start = len(delta_times) // 2
            fit_times = delta_times[fit_start:]
            fit_msds = delta_msds[fit_start:]
            
            if len(fit_times) > 2:
                slope, intercept, r_value, p_value, std_err = stats.linregress(fit_times, fit_msds)
                
                # Calculate average diffusivity from linear region
                avg_diffusivity = slope / 6 * 1e-5  # Convert to m²/s
                
                # Plot linear fit
                fit_line = slope * delta_times + intercept
                axes[0, 1].plot(delta_times / 1000, fit_line, '--', alpha=0.5, 
                               color=axes[0, 1].lines[-1].get_color())
                
                results['diffusivities'].append(avg_diffusivity)
                results['compositions'].append((x_na, y_h2o))
                results['linear_fits'].append((slope, intercept))
                results['r_squared'].append(r_value**2)
            else:
                results['diffusivities'].append(np.mean(diffusivities[-3:]) if len(diffusivities) >= 3 else np.mean(diffusivities))
                results['compositions'].append((x_na, y_h2o))
                results['linear_fits'].append((None, None))
                results['r_squared'].append(None)
    
    # Plot composition vs diffusivity
    if results['diffusivities']:
        x_compositions = [comp[0] for comp in results['compositions']]
        axes[1, 1].scatter(x_compositions, results['diffusivities'], 
                          s=60, alpha=0.7, edgecolors='black')
        axes[1, 1].set_xlabel('Na mole fraction')
        axes[1, 1].set_ylabel('Diffusivity (m²/s)')
        axes[1, 1].set_title('Diffusivity vs Composition')
        axes[1, 1].grid(True, alpha=0.3)
    
    # Format subplots
    axes[0, 0].set_xlabel('Time (ps)')
    axes[0, 0].set_ylabel('MSD (Ų)')
    axes[0, 0].set_title('Raw MSD Data')
    axes[0, 0].grid(True, alpha=0.3)
    axes[0, 0].legend(bbox_to_anchor=(1.05, 1), loc='upper left')
    
    axes[0, 1].set_xlabel('Δt (ps)')
    axes[0, 1].set_ylabel('ΔMSD (Ų)')
    axes[0, 1].set_title('MSD Differences vs Time Interval')
    axes[0, 1].grid(True, alpha=0.3)
    axes[0, 1].legend(bbox_to_anchor=(1.05, 1), loc='upper left')
    
    axes[1, 0].set_xlabel('Δt (ps)')
    axes[1, 0].set_ylabel('Diffusivity (m²/s)')
    axes[1, 0].set_title('Instantaneous Diffusivity')
    axes[1, 0].grid(True, alpha=0.3)
    axes[1, 0].legend(bbox_to_anchor=(1.05, 1), loc='upper left')
    
    # Apply scientific notation for diffusivity plots
    axes[1, 0].ticklabel_format(style='scientific', axis='y', scilimits=(0,0))
    axes[1, 1].ticklabel_format(style='scientific', axis='y', scilimits=(0,0))
    
    plt.tight_layout()
    
    # Save figure
    output_dir = f'./{temp}_diffusion'
    os.makedirs(output_dir, exist_ok=True)
    plt.savefig(f'{output_dir}/{temp}_MSD_analysis.png', dpi=300, bbox_inches='tight')
    
    # Print summary
    print(f"\n=== MSD Analysis Summary for {temp} K ===")
    for i, (comp, diff, r2) in enumerate(zip(results['compositions'], results['diffusivities'], results['r_squared'])):
        r2_str = f", R² = {r2:.4f}" if r2 is not None else ""
        print(f"Config {i+1}: x(Na) = {comp[0]:.3f}, D = {diff:.2e} m²/s{r2_str}")
    
    return results


def calculate_diffusivity_statistics(results_dict):
    """
    Calculate statistics for diffusivity measurements.
    
    Parameters:
    -----------
    results_dict : dict
        Results from MSD_analysis function
    
    Returns:
    --------
    dict : Statistical summary
    """
    diffusivities = np.array(results_dict['diffusivities'])
    
    stats = {
        'mean': np.mean(diffusivities),
        'std': np.std(diffusivities),
        'median': np.median(diffusivities),
        'min': np.min(diffusivities),
        'max': np.max(diffusivities),
        'cv': np.std(diffusivities) / np.mean(diffusivities) * 100  # Coefficient of variation
    }
    
    return stats


# Example usage:
# temp = 300
# Na_list = ['config1', 'config2', 'config3']
# results = MSD_analysis(temp, Na_list)
# statistics = calculate_diffusivity_statistics(results)

In [ ]:
results = MSD_analysis(400, [1,2,3])

### 800K: Na

In [ ]:
diff_800_Na_1=MSD_analysis_Na(800,0,[3],100,-1,'O4')

In [ ]:
diff_800_Na_2=MSD_analysis_Na(800,0,[4,5,6],500,-1,'O16')

In [ ]:
diff_800_Na_3=MSD_analysis_Na(800,0,[7,8,9],500,-1,'O32')

### 700K: Na

In [ ]:
diff_700_Na_1=MSD_analysis_Na(700,0,[3],500,-1,'O4')

In [ ]:
diff_700_Na_2=MSD_analysis_Na(700,0,[4,5,6],500,-1,'O16')

In [ ]:
diff_700_Na_3=MSD_analysis_Na(700,0,[7,8,9],500,-1,'O32')

### 600K: Na

In [ ]:
%matplotlib inline
diff_600_Na_1=MSD_analysis_Na(600,0,[3],500,-1,'O4')

In [ ]:
%matplotlib inline
diff_600_Na_2=MSD_analysis_Na(600,0,[4,5,6],500,-1,'O16')

In [ ]:
%matplotlib inline
diff_600_Na_3=MSD_analysis_Na(600,0,[7,8,9],200,'O32')

In [ ]:
%matplotlib inline
diff_600_Na_3=MSD_analysis_Na(600,2,[7,8,9],200,-1,'O32')

### 550K: Na

In [ ]:
%matplotlib inline
diff_550_Na_1=MSD_analysis_Na(550,4,[1],100,-1,'O4')

In [ ]:
diff_550_O_1=MSD_analysis_O(550,4,[1],100,-1,'O4')

In [ ]:
diff_550_Na_2

In [ ]:
diff_550_Na_2=MSD_analysis_Na(550,4,[5,6],100,-1,'O4')

In [ ]:
diff_550_Na_2_2=MSD_analysis_Na(550,5,[5,6],100,-1,'O4')

In [ ]:
plt.plot(diff_550_Na_2[-2])
plt.plot(diff_550_Na_2_2[-2])

In [ ]:
diff_550_Na_2.append(diff_550_Na_2_2[0])
diff_550_Na_2.append(diff_550_Na_2_2[1])
diff_550_Na_2

In [ ]:
diff_550_O_2=MSD_analysis_O(550,4,[5,6],100,-1,'O4')

In [ ]:
diff_550_Na_3=MSD_analysis_Na(550,4,[7,8,9],100,-1,'O4')

In [ ]:
diff_550_O_3=MSD_analysis_O(550,4,[7,8,9],200,-1,'O4')

### 500K: Na

In [ ]:
diff_500_Na_1=MSD_analysis_Na(500,4,[1],100,-1,'O4')

In [ ]:
diff_500_O_1=MSD_analysis_O(500,4,[1],100,-1,'O4')

In [ ]:
diff_500_Na_2=MSD_analysis_Na(500,4,[5,6],100,-1,'O4')

In [ ]:
diff_500_O_2=MSD_analysis_O(500,4,[5,6],500,-1,'O4')

In [ ]:
diff_500_Na_3=MSD_analysis_Na(500,4,[7,8,9],100,-1,'O4')

In [ ]:
diff_500_O_3=MSD_analysis_O(500,4,[7,8,9],400,-1,'O4')

In [ ]:
diff_500_Na_1=MSD_analysis_Na(500,0,[3],500,-1,'O4')

In [ ]:
diff_500_Na_2=MSD_analysis_Na(500,0,[4,5,6],1000,-1,'O16')

In [ ]:
diff_500_Na_3=MSD_analysis_Na(500,0,[7,8,9],1000,-1,'O32')

In [ ]:
diff_500_Na_3=MSD_analysis_Na(500,2,[7,8,9],500,-1,'O32')

### 450K: Na

In [ ]:
diff_450_Na_1=MSD_analysis_Na(450,4,[1],100,-1,'O4')

In [ ]:
diff_450_O_1=MSD_analysis_O(450,4,[1],500,-1,'O4')

In [ ]:
diff_450_Na_2=MSD_analysis_Na(450,4,[5,6],100,-1,'O4')

In [ ]:
diff_450_Na_2_2=MSD_analysis_Na(450,5,[5,6],100,-1,'O4')

In [ ]:
plt.plot(diff_450_Na_2[-2])
plt.plot(diff_450_Na_2_2[-2])

In [ ]:
diff_450_Na_2.append(diff_450_Na_2_2[0])
diff_450_Na_2.append(diff_450_Na_2_2[1])
diff_450_Na_2

In [ ]:
diff_450_O_2=MSD_analysis_O(450,4,[5,6],400,-1,'O4')

In [ ]:
diff_450_Na_3=MSD_analysis_Na(450,4,[7,8,9],100,-1,'O4')

In [ ]:
diff_450_O_3=MSD_analysis_O(450,4,[7,8,9],400,-1,'O4')

### 400K: Na

In [ ]:
diff_400_Na_1=MSD_analysis_Na(400,4,[1],100,-1,'O4')

In [ ]:
diff_400_O_1=MSD_analysis_O(400,4,[1],400,-1,'O4')

In [ ]:
diff_400_Na_2=MSD_analysis_Na(400,4,[5,6],100,-1,'O4')

In [ ]:
diff_400_O_2=MSD_analysis_O(400,4,[5,6],400,-1,'O4')

In [ ]:
diff_400_Na_3=MSD_analysis_Na(400,4,[7,8,9],100,-1,'O4')

In [ ]:
diff_400_O_3=MSD_analysis_O(400,4,[7,8,9],400,-1,'O4')

### 350K: Na

In [ ]:
diff_350_Na_1=MSD_analysis_Na(350,4,[1],100,-1,'O4')

In [ ]:
diff_350_O_1=MSD_analysis_O(350,4,[1],400,-1,'O4')

In [ ]:
diff_350_Na_2=MSD_analysis_Na(350,4,[5,6],100,-1,'O4')

In [ ]:
diff_350_O_2=MSD_analysis_O(350,4,[5,6],400,-1,'O4')

In [ ]:
diff_350_Na_3=MSD_analysis_Na(350,4,[7,8,9],100,-1,'O4')

In [ ]:
diff_350_O_3=MSD_analysis_O(350,4,[7,8,9],400,-1,'O4')

### 300K: Na

In [ ]:
diff_300_Na_1=MSD_analysis_Na(300,4,[1,3],100,-1,'O4')

In [ ]:
diff_300_O_1=MSD_analysis_O(300,4,[1,3],500,-1,'O4')

In [ ]:
diff_300_Na_2=MSD_analysis_Na(300,4,[5,6],100,-1,'O4')

In [ ]:
diff_300_Na_2_2=MSD_analysis_Na(300,5,[4,5],100,-1,'O4')

In [ ]:
plt.plot(diff_300_Na_2_2[0])
plt.plot(diff_300_Na_2[1])

In [ ]:
diff_300_Na_2.append(diff_300_Na_2_2[0])
diff_300_Na_2

In [ ]:
diff_300_O_2=MSD_analysis_O(300,4,[5,6],400,-1,'O4')

In [ ]:
diff_300_Na_3=MSD_analysis_Na(300,4,[7,8],100,-1,'O4')

In [ ]:
diff_300_O_3=MSD_analysis_O(300,4,[7,8],400,-1,'O4')

### 800K: O

In [ ]:
diff_800_O_1=MSD_analysis_O(800,[3],100,'O4')

In [ ]:
diff_800_O_2=MSD_analysis_O(800,[4,5,6],300,'O16')

In [ ]:
diff_800_O_3=MSD_analysis_O(800,[7,8,9],1000,'O32')

### 700K: O

In [ ]:
diff_700_O_1=MSD_analysis_O(700,[3],100,'O4')

In [ ]:
diff_700_O_2=MSD_analysis_O(700,[4,5,6],500,'O16')

In [ ]:
diff_700_O_3=MSD_analysis_O(700,[7,8,9],2000,'O32')

### 600K: O

In [ ]:
%matplotlib inline
diff_600_O_1=MSD_analysis_O(600,[3],1000,'O4')

In [ ]:
%matplotlib inline
diff_600_O_2=MSD_analysis_O(600, [4,5,6], 2000, 'O16')

In [ ]:
%matplotlib inline
diff_600_O_3=MSD_analysis_O(600,[7,8,9],2000,'O32')

### 500K: O

In [ ]:
diff_500_O_1=MSD_analysis_O(500,[2,3],500,'O4')

In [ ]:
diff_500_O_2=MSD_analysis_O(500,[4,5,6],1000,'O16')

In [ ]:
diff_500_O_3=MSD_analysis_O(500,[7,8,9],1000,'O32')

### 400K: O

In [ ]:
diff_400_O_1=MSD_analysis_O(400,[2,3],1000,'O4')

In [ ]:
diff_400_O_1=MSD_analysis_O(400,4,[1,2,3],1000,-1,'O4')

In [ ]:
diff_400_O_1=MSD_analysis_O(400,3,[1,2,3],1000,-1,'O4')

In [ ]:
diff_400_O_1=MSD_analysis_O(400,2,[4,5,6],1000,-1,'O4')

In [ ]:
diff_400_O_1=MSD_analysis_O(400,4,[5,6],1000,-1,'O4')

In [ ]:
diff_400_O_3=MSD_analysis_O(400,0,[7,8,9],2000,-1,'O4')

In [ ]:
diff_400_O_3=MSD_analysis_O(400, 2, [8,9], 2000, -1, 'O4')

### 400_diffusion_vac_20_r02_65

In [ ]:
diff_400_O_vac =  MSD_analysis_vac_O('400_diffusion_vac_lv20_r02_65_2', 400, [10,11,12], 500, 4000, 'vac')

In [ ]:
diff_400_O_vac[2][-4:].mean()

In [ ]:
plt.figure()
for i in range(6):
    plt.plot(diff_400_O_vac[i][-7:-1])

In [ ]:
diff_400_O_vac =  MSD_analysis_vac_Na('400_diffusion_vac_lv20_r02_65', 400, [10], 500, -1, 'vac')

In [ ]:
diff_400_O_vac

In [ ]:
diff_400_O_vac =  MSD_analysis_vac_Na('400_diffusion_20_r02_65', 400, [6,8,9], 500, -1, 'vac')

In [ ]:
diff_400_O_vac =  MSD_analysis_vac_O('400_diffusion_20_r02_65', 400, [6,8,9], 500, -1, 'vac')

In [ ]:
diff_400_O_vac

## Diffusivity & Activation energy

In [ ]:
Na_4_O_32_diff_ls=[diff_400_Na_3[i][p1:p2].mean(), diff_350_Na_3[i][p1:p2].mean(), diff_300_Na_3[i][p1:p2].mean()]
Na_4_O_32_diff_ls

In [ ]:
diff_550_Na_2

In [ ]:
singlecol = 3.5
plt.rcParams.update({'font.size': 8})
#plt.rcParams.update({'font.sans-serif': 'Helvetica'})
plt.rcParams.update({'lines.linewidth':1})
plt.rcParams.update({'axes.linewidth':1})
plt.rcParams.update({'lines.markersize':3})
plt.rcParams.update({'savefig.dpi':300})
plt.rcParams.update({'legend.fontsize': 6})

fig, (ax1) = plt.subplots(1, 1, figsize=(singlecol, singlecol),dpi=300)
fig.subplots_adjust(wspace=0.3)
num_Na_list = [216, 866, 1512,
               205, 867, 1512,
               205, 867, 1512]
num_O_list = [216, 216, 216,
                864, 864, 1728,
                1728, 1728, 1728,]

color_ls=['navy', 'royalblue', 'c', 'red', 'orangered', 'g']
p1, p2 = -8,-7
i=0
Na_4_O_4_diff_ls=[diff_550_Na_1[i][p1:p2].mean(), diff_500_Na_1[i][p1:p2].mean(), diff_450_Na_1[i][p1:p2].mean(),
                  diff_400_Na_1[i][p1:p2].mean(), diff_350_Na_1[i][p1:p2].mean(), diff_300_Na_1[i][p1:p2].mean()]
i=0
Na_16_O_16_diff_ls=[diff_550_Na_2[i][p1:p2].mean(), diff_500_Na_2[i][p1:p2].mean(), diff_450_Na_2[i][p1:p2].mean(),
                  diff_400_Na_2[i][p1:p2].mean(), diff_350_Na_2[i][p1:p2].mean(), diff_300_Na_2[i][p1:p2].mean()]
i=1
Na_28_O_16_diff_ls=[diff_550_Na_2[i][p1:p2].mean(), diff_500_Na_2[i][p1:p2].mean(), diff_450_Na_2[i][p1:p2].mean(),
                  diff_400_Na_2[i][p1:p2].mean(), diff_350_Na_2[i][p1:p2].mean(), diff_300_Na_2[i][p1:p2].mean()]

i=0
Na_4_O_32_diff_ls=[diff_550_Na_3[i][p1:p2].mean(), diff_500_Na_3[i][p1:p2].mean(), diff_450_Na_3[i][p1:p2].mean(),
                  diff_400_Na_3[i][p1:p2].mean(), diff_350_Na_3[i][p1:p2].mean(), diff_300_Na_3[i][p1:p2].mean()]
i=1
Na_16_O_32_diff_ls=[diff_550_Na_3[i][p1:p2].mean(), diff_500_Na_3[i][p1:p2].mean(), diff_450_Na_3[i][p1:p2].mean(),
                  diff_400_Na_3[i][p1:p2].mean(), diff_350_Na_3[i][p1:p2].mean(), diff_300_Na_3[i][p1:p2].mean()]
i=2
Na_28_O_32_diff_ls=[diff_550_Na_3[1][p1:p2].mean(), diff_500_Na_3[1][p1:p2].mean(), diff_450_Na_3[1][p1:p2].mean(),
                  diff_400_Na_3[i][p1:p2].mean(), diff_350_Na_3[i][p1:p2].mean()]#, diff_300_Na_3[i][p1:p2].mean()]

k=8
ax1.plot(1/np.array([550, 500, 450, 400, 350])*1000, Na_28_O_32_diff_ls, marker='s',label=f'x = {round(num_Na_list[k]/864,3)} y = {round(num_H2O_list[k]/864,3)}', color=color_ls[0])
k=7
ax1.plot(1/temp_1*1000, Na_16_O_32_diff_ls, marker='s', label=f'x = {round(num_Na_list[k]/864,3)} y = {round(num_H2O_list[k]/864,3)}', color=color_ls[1])
k=6
ax1.plot(1/temp_1*1000, Na_4_O_32_diff_ls, marker='s', label=f'x = {round(num_Na_list[k]/864,3)} y = {round(num_H2O_list[k]/864,3)}', color=color_ls[2])
k=5
ax1.plot(1/temp_1*1000, Na_28_O_16_diff_ls, '--', marker='o', label=f'x = {round(num_Na_list[k]/864,3)} y = {round(num_H2O_list[k]/864,3)}', color=color_ls[3])
k=4
ax1.plot(1/temp_1*1000, Na_16_O_16_diff_ls, '--',  marker='o',  label=f'x = {round(num_Na_list[k]/864,3)} y = {round(num_H2O_list[k]/864,3)}', color=color_ls[4])
k=0
ax1.plot(1/temp_1*1000, Na_4_O_4_diff_ls, ':', marker='^', label=f'x = {round(num_Na_list[k]/864,3)} y = {round(num_H2O_list[k]/864,3)}', color=color_ls[5])

ax1.legend(frameon=False)#fontsize=12)
#ax1.grid()
#ax1.set_title(f'delta t = {delta_t_ls[i]*0.5} ps')
#ax1.set_xscale('log')
ax1.set_yscale('log')
ax1.set_xlabel('1000/T (1/K)')
ax1.set_ylabel(r'Sodium diffusivity ($\mathrm{m^2\,s^{-1}}$)')

plt.tight_layout()
plt.savefig('Diffuvisity_lv16.svg')

In [ ]:
singlecol = 3.5
plt.rcParams.update({'font.size': 8})
#plt.rcParams.update({'font.sans-serif': 'Helvetica'})
plt.rcParams.update({'lines.linewidth':1})
plt.rcParams.update({'axes.linewidth':1})
plt.rcParams.update({'lines.markersize':3})
plt.rcParams.update({'savefig.dpi':300})
plt.rcParams.update({'legend.fontsize': 6})

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(singlecol*2, singlecol),dpi=300)
fig.subplots_adjust(wspace=0.3)
num_Na_list = [216, 866, 1512,
               205, 867, 1512,
               205, 867, 1512]
num_O_list = [216, 216, 216,
                864, 864, 1728,
                1728, 1728, 1728,]

p1, p2 = -8,-7
i=0
Na_4_O_4_diff_ls=[diff_550_Na_1[i][p1:p2].mean(), diff_500_Na_1[i][p1:p2].mean(), diff_450_Na_1[i][p1:p2].mean(),
                  diff_400_Na_1[i][p1:p2].mean(), diff_350_Na_1[i][p1:p2].mean(), diff_300_Na_1[i][p1:p2].mean()]
i=0
Na_16_O_16_diff_ls=[diff_550_Na_2[i][p1:p2].mean(), diff_500_Na_2[i][p1:p2].mean(), diff_450_Na_2[i][p1:p2].mean(),
                  diff_400_Na_2[i][p1:p2].mean(), diff_350_Na_2[i][p1:p2].mean(), diff_300_Na_2[i][p1:p2].mean()]
i=1
Na_28_O_16_diff_ls=[diff_550_Na_2[i][p1:p2].mean(), diff_500_Na_2[i][p1:p2].mean(), diff_450_Na_2[i][p1:p2].mean(),
                  diff_400_Na_2[i][p1:p2].mean(), diff_350_Na_2[i][p1:p2].mean(), diff_300_Na_2[i][p1:p2].mean()]

i=0
Na_4_O_32_diff_ls=[diff_550_Na_3[i][p1:p2].mean(), diff_500_Na_3[i][p1:p2].mean(), diff_450_Na_3[i][p1:p2].mean(),
                  diff_400_Na_3[i][p1:p2].mean(), diff_350_Na_3[i][p1:p2].mean(), diff_300_Na_3[i][p1:p2].mean()]
i=1
Na_16_O_32_diff_ls=[diff_550_Na_3[i][p1:p2].mean(), diff_500_Na_3[i][p1:p2].mean(), diff_450_Na_3[i][p1:p2].mean(),
                  diff_400_Na_3[i][p1:p2].mean(), diff_350_Na_3[i][p1:p2].mean(), diff_300_Na_3[i][p1:p2].mean()]
i=2
Na_28_O_32_diff_ls=[diff_550_Na_3[1][p1:p2].mean(), diff_500_Na_3[1][p1:p2].mean(), diff_450_Na_3[1][p1:p2].mean(),
                  diff_400_Na_3[i][p1:p2].mean(), diff_350_Na_3[i][p1:p2].mean()]#, diff_300_Na_3[i][p1:p2].mean()]

k=8
ax1.plot(1/np.array([550, 500, 450, 400, 350])*1000, Na_28_O_32_diff_ls, marker='s',label=f'x = {round(num_Na_list[k]/864,3)} y = {round(num_H2O_list[k]/864,3)}')
k=7
ax1.plot(1/temp_1*1000, Na_16_O_32_diff_ls, marker='s', label=f'x = {round(num_Na_list[k]/864,3)} y = {round(num_H2O_list[k]/864,3)}')
k=6
ax1.plot(1/temp_1*1000, Na_4_O_32_diff_ls, marker='s', label=f'x = {round(num_Na_list[k]/864,3)} y = {round(num_H2O_list[k]/864,3)}')
k=5
ax1.plot(1/temp_1*1000, Na_28_O_16_diff_ls, '--', marker='o', label=f'x = {round(num_Na_list[k]/864,3)} y = {round(num_H2O_list[k]/864,3)}')
k=4
ax1.plot(1/temp_1*1000, Na_16_O_16_diff_ls, '--',  marker='o',  label=f'x = {round(num_Na_list[k]/864,3)} y = {round(num_H2O_list[k]/864,3)}')
k=0
ax1.plot(1/temp_1*1000, Na_4_O_4_diff_ls, ':', marker='^', label=f'x = {round(num_Na_list[k]/864,3)} y = {round(num_H2O_list[k]/864,3)}')

ax1.legend(frameon=False)#fontsize=12)
#ax1.grid()
#ax1.set_title(f'delta t = {delta_t_ls[i]*0.5} ps')
#ax1.set_xscale('log')
ax1.set_yscale('log')
ax1.set_xlabel('1000/T (1/K)')
ax1.set_ylabel(r'Sodium diffusivity ($\mathrm{m^2\,s^{-1}}$)')


p1, p2 = -8,-7
i=0
Na_4_O_4_diff_ls=[diff_550_O_1[i][p1:p2].mean(), diff_500_O_1[i][p1:p2].mean(), diff_450_O_1[i][p1:p2].mean(),
                  diff_400_O_1[i][p1:p2].mean(), diff_350_O_1[i][p1:p2].mean(), diff_300_O_1[i][p1:p2].mean()]
i=0
Na_16_O_16_diff_ls=[diff_550_O_2[i][p1:p2].mean(), diff_500_O_2[i][p1:p2].mean(), diff_450_O_2[i][p1:p2].mean(),
                  diff_400_O_2[i][p1:p2].mean(), diff_350_O_2[i][p1:p2].mean(), diff_300_O_2[i][p1:p2].mean()]
i=1
Na_28_O_16_diff_ls=[diff_550_O_2[i][p1:p2].mean(), diff_500_O_2[i][p1:p2].mean(), diff_450_O_2[i][p1:p2].mean(),
                  diff_400_O_2[i][p1:p2].mean(), diff_350_O_2[i][p1:p2].mean(), diff_300_O_2[i][p1:p2].mean()]

i=0
Na_4_O_32_diff_ls=[diff_550_O_3[i][p1:p2].mean(), diff_500_O_3[i][p1:p2].mean(), diff_450_O_3[i][p1:p2].mean(),
                  diff_400_O_3[i][p1:p2].mean(), diff_350_O_3[i][p1:p2].mean(), diff_300_O_3[i][p1:p2].mean()]
i=1
Na_16_O_32_diff_ls=[diff_550_O_3[i][p1:p2].mean(), diff_500_O_3[i][p1:p2].mean(), diff_450_O_3[i][p1:p2].mean(),
                  diff_400_O_3[i][p1:p2].mean(), diff_350_O_3[i][p1:p2].mean(), diff_300_O_3[i][p1:p2].mean()]
i=2
Na_28_O_32_diff_ls=[diff_550_O_3[1][p1:p2].mean(), diff_500_O_3[1][p1:p2].mean(), diff_450_O_3[1][p1:p2].mean(),
                  diff_400_O_3[i][p1:p2].mean(), diff_350_O_3[i][p1:p2].mean()]#, diff_300_O_3[i][p1:p2].mean()]

k=8
ax2.plot(1/np.array([550, 500, 450, 400, 350])*1000, Na_28_O_32_diff_ls, marker='s', label=f'x = {round(num_Na_list[k]/864,3)} y = {round(num_H2O_list[k]/864,3)}')
k=7
ax2.plot(1/temp_1*1000, Na_16_O_32_diff_ls, marker='s',  label=f'x = {round(num_Na_list[k]/864,3)} y = {round(num_H2O_list[k]/864,3)}')
k=6
ax2.plot(1/temp_1*1000, Na_4_O_32_diff_ls, marker='s',  label=f'x = {round(num_Na_list[k]/864,3)} y = {round(num_H2O_list[k]/864,3)}')
k=5
ax2.plot(1/temp_1*1000, Na_28_O_16_diff_ls, '--', marker='o',  label=f'x = {round(num_Na_list[k]/864,3)} y = {round(num_H2O_list[k]/864,3)}')
k=4
ax2.plot(1/temp_1*1000, Na_16_O_16_diff_ls, '--',  marker='o', label=f'x = {round(num_Na_list[k]/864,3)} y = {round(num_H2O_list[k]/864,3)}')
k=0
ax2.plot(1/temp_1*1000, Na_4_O_4_diff_ls, ':', marker='^',label=f'x = {round(num_Na_list[k]/864,3)} y = {round(num_H2O_list[k]/864,3)}')

#ax2.legend(frameon=False)#fontsize=12)
ax2.set_yscale('log')
ax2.set_xlabel('1000/T (1/K)')
ax2.set_ylabel(r'Water diffusivity ($\mathrm{m^2\,s^{-1}}$)')
plt.tight_layout()
#plt.savefig('Diffuvisity_lv16.svg')

In [ ]:
temp_1 = np.array([550, 500, 450, 400, 350, 300])
temp_2 = np.array([450, 400, 350, 300])
temp_3 = np.array([550, 500, 450, 400])
temp_4 = np.array([400, 350, 300])

delta_t_ls = np.array([4, 5, 6, 7, 8,
                       9, 10, 15, 20, 30,
                       40, 50, 60, 70, 80,
                       90, 100, 200, 300, 400,
                       500, 600])

fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(18,5))

num_Na_list = [216, 866, 1512,
               205, 867, 1512,
               205, 867, 1512]
num_H2O_list = [216, 216, 216,
                864, 864, 1728,
                1728, 1728, 1728,]

p1, p2 = -8,-7
i=0
Na_4_O_4_diff_ls=[diff_550_Na_1[i][p1:p2].mean(), diff_500_Na_1[i][p1:p2].mean(), diff_450_Na_1[i][p1:p2].mean(),
                  diff_400_Na_1[i][p1:p2].mean(), diff_350_Na_1[i][p1:p2].mean(), diff_300_Na_1[i][p1:p2].mean()]
#i=1
#Na_16_O_4_diff_ls=[diff_550_Na_1[i][p1:p2].mean(), diff_500_Na_1[i][p1:p2].mean(), diff_450_Na_1[i][p1:p2].mean(),
#                  diff_400_Na_1[i][p1:p2].mean()]
#i=2
#Na_28_O_4_diff_ls=[diff_550_Na_1[i][p1:p2].mean(), diff_500_Na_1[i][p1:p2].mean(), diff_450_Na_1[i][p1:p2].mean(),
#                  diff_400_Na_1[i][p1:p2].mean(), diff_350_Na_1[1][p1:p2].mean(), diff_300_Na_1[1][p1:p2].mean()]

#i=0
#Na_4_O_16_diff_ls=[diff_550_Na_2[i][p1:p2].mean(), diff_500_Na_2[i][p1:p2].mean(), diff_450_Na_2[i][p1:p2].mean(),
#                  diff_400_Na_2[i][p1:p2].mean(), diff_350_Na_2[i][p1:p2].mean(), diff_300_Na_2[i][p1:p2].mean()]
i=0
Na_16_O_16_diff_ls=[diff_550_Na_2[i][p1:p2].mean(), diff_500_Na_2[i][p1:p2].mean(), diff_450_Na_2[i][p1:p2].mean(),
                  diff_400_Na_2[i][p1:p2].mean(), diff_350_Na_2[i][p1:p2].mean(), diff_300_Na_2[i][p1:p2].mean()]
i=1
Na_28_O_16_diff_ls=[diff_550_Na_2[i][p1:p2].mean(), diff_500_Na_2[i][p1:p2].mean(), diff_450_Na_2[i][p1:p2].mean(),
                  diff_400_Na_2[i][p1:p2].mean(), diff_350_Na_2[i][p1:p2].mean(), diff_300_Na_2[i][p1:p2].mean()]

i=0
Na_4_O_32_diff_ls=[diff_550_Na_3[i][p1:p2].mean(), diff_500_Na_3[i][p1:p2].mean(), diff_450_Na_3[i][p1:p2].mean(),
                  diff_400_Na_3[i][p1:p2].mean(), diff_350_Na_3[i][p1:p2].mean(), diff_300_Na_3[i][p1:p2].mean()]
i=1
Na_16_O_32_diff_ls=[diff_550_Na_3[i][p1:p2].mean(), diff_500_Na_3[i][p1:p2].mean(), diff_450_Na_3[i][p1:p2].mean(),
                  diff_400_Na_3[i][p1:p2].mean(), diff_350_Na_3[i][p1:p2].mean(), diff_300_Na_3[i][p1:p2].mean()]
i=2
Na_28_O_32_diff_ls=[diff_550_Na_3[1][p1:p2].mean(), diff_500_Na_3[1][p1:p2].mean(), diff_450_Na_3[1][p1:p2].mean(),
                  diff_400_Na_3[i][p1:p2].mean(), diff_350_Na_3[i][p1:p2].mean()]#, diff_300_Na_3[i][p1:p2].mean()]

k=8
ax1.plot(1/np.array([550, 500, 450, 400, 350])*1000, Na_28_O_32_diff_ls, marker='*', markersize=7, label=f'x = {round(num_Na_list[k]/864,3)} y = {round(num_H2O_list[k]/864,3)}')
k=7
ax1.plot(1/temp_1*1000, Na_16_O_32_diff_ls, marker='*', markersize=7, label=f'x = {round(num_Na_list[k]/864,3)} y = {round(num_H2O_list[k]/864,3)}')
k=6
ax1.plot(1/temp_1*1000, Na_4_O_32_diff_ls, marker='*', markersize=7, label=f'x = {round(num_Na_list[k]/864,3)} y = {round(num_H2O_list[k]/864,3)}')
k=5
ax1.plot(1/temp_1*1000, Na_28_O_16_diff_ls, marker='o', markersize=7, label=f'x = {round(num_Na_list[k]/864,3)} y = {round(num_H2O_list[k]/864,3)}')
k=4
ax1.plot(1/temp_1*1000, Na_16_O_16_diff_ls, marker='o', markersize=7, label=f'x = {round(num_Na_list[k]/864,3)} y = {round(num_H2O_list[k]/864,3)}')
#k=3
#ax1.plot(1/temp_1*1000, Na_4_O_16_diff_ls, marker='o', markersize=7, label=f'x = {round(num_Na_list[k]/864,3)} y = {round(num_H2O_list[k]/864,3)}')
#k=2
#ax1.plot(1/temp_1*1000, Na_28_O_4_diff_ls, marker='^', markersize=7, label=f'x = {round(num_Na_list[k]/864,3)} y = {round(num_H2O_list[k]/864,3)}')
#k=1
#ax1.plot(1/temp_3*1000, Na_16_O_4_diff_ls, marker='^', markersize=7, label=f'x = {round(num_Na_list[k]/864,3)} y = {round(num_H2O_list[k]/864,3)}')
k=0
ax1.plot(1/temp_1*1000, Na_4_O_4_diff_ls, marker='^', markersize=7, label=f'x = {round(num_Na_list[k]/864,3)} y = {round(num_H2O_list[k]/864,3)}')

ax1.legend(fontsize=5)
ax1.grid()
ax1.set_title(f'delta t = {delta_t_ls[i]*0.5} ps')
#ax1.set_xscale('log')
ax1.set_yscale('log')
ax1.set_xlabel('1000/T (1/K)')
ax1.set_ylabel('Diffusivity (m^2/s)')

p1, p2 = -7,-6
i=0
Na_4_O_4_diff_ls=[diff_550_Na_1[i][p1:p2].mean(), diff_500_Na_1[i][p1:p2].mean(), diff_450_Na_1[i][p1:p2].mean(),
                  diff_400_Na_1[i][p1:p2].mean(), diff_350_Na_1[i][p1:p2].mean(), diff_300_Na_1[i][p1:p2].mean()]
#i=1
#Na_16_O_4_diff_ls=[diff_550_Na_1[i][p1:p2].mean(), diff_500_Na_1[i][p1:p2].mean(), diff_450_Na_1[i][p1:p2].mean(),
#                  diff_400_Na_1[i][p1:p2].mean()]
#i=2
#Na_28_O_4_diff_ls=[diff_550_Na_1[i][p1:p2].mean(), diff_500_Na_1[i][p1:p2].mean(), diff_450_Na_1[i][p1:p2].mean(),
#                  diff_400_Na_1[i][p1:p2].mean(), diff_350_Na_1[1][p1:p2].mean(), diff_300_Na_1[1][p1:p2].mean()]

#i=0
#Na_4_O_16_diff_ls=[diff_550_Na_2[i][p1:p2].mean(), diff_500_Na_2[i][p1:p2].mean(), diff_450_Na_2[i][p1:p2].mean(),
#                  diff_400_Na_2[i][p1:p2].mean(), diff_350_Na_2[i][p1:p2].mean(), diff_300_Na_2[i][p1:p2].mean()]
i=0
Na_16_O_16_diff_ls=[diff_550_Na_2[i][p1:p2].mean(), diff_500_Na_2[i][p1:p2].mean(), diff_450_Na_2[i][p1:p2].mean(),
                  diff_400_Na_2[i][p1:p2].mean(), diff_350_Na_2[i][p1:p2].mean(), diff_300_Na_2[i][p1:p2].mean()]
i=1
Na_28_O_16_diff_ls=[diff_550_Na_2[i][p1:p2].mean(), diff_500_Na_2[i][p1:p2].mean(), diff_450_Na_2[i][p1:p2].mean(),
                  diff_400_Na_2[i][p1:p2].mean(), diff_350_Na_2[i][p1:p2].mean(), diff_300_Na_2[i][p1:p2].mean()]

i=0
Na_4_O_32_diff_ls=[diff_550_Na_3[i][p1:p2].mean(), diff_500_Na_3[i][p1:p2].mean(), diff_450_Na_3[i][p1:p2].mean(),
                  diff_400_Na_3[i][p1:p2].mean(), diff_350_Na_3[i][p1:p2].mean(), diff_300_Na_3[i][p1:p2].mean()]
i=1
Na_16_O_32_diff_ls=[diff_550_Na_3[i][p1:p2].mean(), diff_500_Na_3[i][p1:p2].mean(), diff_450_Na_3[i][p1:p2].mean(),
                  diff_400_Na_3[i][p1:p2].mean(), diff_350_Na_3[i][p1:p2].mean(), diff_300_Na_3[i][p1:p2].mean()]
i=2
Na_28_O_32_diff_ls=[diff_550_Na_3[1][p1:p2].mean(), diff_500_Na_3[1][p1:p2].mean(), diff_450_Na_3[1][p1:p2].mean(),
                  diff_400_Na_3[i][p1:p2].mean(), diff_350_Na_3[i][p1:p2].mean()]#, diff_300_Na_3[i][p1:p2].mean()]

k=8
ax2.plot(1/np.array([550, 500, 450, 400, 350])*1000, Na_28_O_32_diff_ls, marker='*', markersize=7, label=f'x = {round(num_Na_list[k]/864,3)} y = {round(num_H2O_list[k]/864,3)}')
k=7
ax2.plot(1/temp_1*1000, Na_16_O_32_diff_ls, marker='*', markersize=7, label=f'x = {round(num_Na_list[k]/864,3)} y = {round(num_H2O_list[k]/864,3)}')
k=6
ax2.plot(1/temp_1*1000, Na_4_O_32_diff_ls, marker='*', markersize=7, label=f'x = {round(num_Na_list[k]/864,3)} y = {round(num_H2O_list[k]/864,3)}')
k=5
ax2.plot(1/temp_1*1000, Na_28_O_16_diff_ls, marker='o', markersize=7, label=f'x = {round(num_Na_list[k]/864,3)} y = {round(num_H2O_list[k]/864,3)}')
k=4
ax2.plot(1/temp_1*1000, Na_16_O_16_diff_ls, marker='o', markersize=7, label=f'x = {round(num_Na_list[k]/864,3)} y = {round(num_H2O_list[k]/864,3)}')
#k=3
#ax2.plot(1/temp_1*1000, Na_4_O_16_diff_ls, marker='o', markersize=7, label=f'x = {round(num_Na_list[k]/864,3)} y = {round(num_H2O_list[k]/864,3)}')
#k=2
#ax2.plot(1/temp_1*1000, Na_28_O_4_diff_ls, marker='^', markersize=7, label=f'x = {round(num_Na_list[k]/864,3)} y = {round(num_H2O_list[k]/864,3)}')
#k=1
#ax2.plot(1/temp_3*1000, Na_16_O_4_diff_ls, marker='^', markersize=7, label=f'x = {round(num_Na_list[k]/864,3)} y = {round(num_H2O_list[k]/864,3)}')
k=0
ax2.plot(1/temp_1*1000, Na_4_O_4_diff_ls, marker='^', markersize=7, label=f'x = {round(num_Na_list[k]/864,3)} y = {round(num_H2O_list[k]/864,3)}')

ax2.legend(fontsize=5)
ax2.grid()
ax2.set_title(f'delta t = {delta_t_ls[i]*0.5} ps')
#ax2.set_xscale('log')
ax2.set_yscale('log')
ax2.set_xlabel('1000/T (1/K)')
ax2.set_ylabel('Diffusivity (m^2/s)')

p1, p2 = -8,-3
i=0
Na_4_O_4_diff_ls=[diff_550_Na_1[i][p1:p2].mean(), diff_500_Na_1[i][p1:p2].mean(), diff_450_Na_1[i][p1:p2].mean(),
                  diff_400_Na_1[i][p1:p2].mean(), diff_350_Na_1[i][p1:p2].mean(), diff_300_Na_1[i][p1:p2].mean()]
#i=1
#Na_16_O_4_diff_ls=[diff_550_Na_1[i][p1:p2].mean(), diff_500_Na_1[i][p1:p2].mean(), diff_450_Na_1[i][p1:p2].mean(),
#                  diff_400_Na_1[i][p1:p2].mean()]
#i=2
#Na_28_O_4_diff_ls=[diff_550_Na_1[i][p1:p2].mean(), diff_500_Na_1[i][p1:p2].mean(), diff_450_Na_1[i][p1:p2].mean(),
#                  diff_400_Na_1[i][p1:p2].mean(), diff_350_Na_1[1][p1:p2].mean(), diff_300_Na_1[1][p1:p2].mean()]

#i=0
#Na_4_O_16_diff_ls=[diff_550_Na_2[i][p1:p2].mean(), diff_500_Na_2[i][p1:p2].mean(), diff_450_Na_2[i][p1:p2].mean(),
#                  diff_400_Na_2[i][p1:p2].mean(), diff_350_Na_2[i][p1:p2].mean(), diff_300_Na_2[i][p1:p2].mean()]
i=0
Na_16_O_16_diff_ls=[diff_550_Na_2[i][p1:p2].mean(), diff_500_Na_2[i][p1:p2].mean(), diff_450_Na_2[i][p1:p2].mean(),
                  diff_400_Na_2[i][p1:p2].mean(), diff_350_Na_2[i][p1:p2].mean(), diff_300_Na_2[i][p1:p2].mean()]
i=1
Na_28_O_16_diff_ls=[diff_550_Na_2[i][p1:p2].mean(), diff_500_Na_2[i][p1:p2].mean(), diff_450_Na_2[i][p1:p2].mean(),
                  diff_400_Na_2[i][p1:p2].mean(), diff_350_Na_2[i][p1:p2].mean(), diff_300_Na_2[i][p1:p2].mean()]

i=0
Na_4_O_32_diff_ls=[diff_550_Na_3[i][p1:p2].mean(), diff_500_Na_3[i][p1:p2].mean(), diff_450_Na_3[i][p1:p2].mean(),
                  diff_400_Na_3[i][p1:p2].mean(), diff_350_Na_3[i][p1:p2].mean(), diff_300_Na_3[i][p1:p2].mean()]
i=1
Na_16_O_32_diff_ls=[diff_550_Na_3[i][p1:p2].mean(), diff_500_Na_3[i][p1:p2].mean(), diff_450_Na_3[i][p1:p2].mean(),
                  diff_400_Na_3[i][p1:p2].mean(), diff_350_Na_3[i][p1:p2].mean(), diff_300_Na_3[i][p1:p2].mean()]
i=2
Na_28_O_32_diff_ls=[diff_550_Na_3[1][p1:p2].mean(), diff_500_Na_3[1][p1:p2].mean(), diff_450_Na_3[1][p1:p2].mean(),
                  diff_400_Na_3[i][p1:p2].mean(), diff_350_Na_3[i][p1:p2].mean()]#, diff_300_Na_3[i][p1:p2].mean()]

k=8
ax3.plot(1/np.array([550, 500, 450, 400, 350])*1000, Na_28_O_32_diff_ls, marker='*', markersize=7, label=f'x = {round(num_Na_list[k]/864,3)} y = {round(num_H2O_list[k]/864,3)}')
k=7
ax3.plot(1/temp_1*1000, Na_16_O_32_diff_ls, marker='*', markersize=7, label=f'x = {round(num_Na_list[k]/864,3)} y = {round(num_H2O_list[k]/864,3)}')
k=6
ax3.plot(1/temp_1*1000, Na_4_O_32_diff_ls, marker='*', markersize=7, label=f'x = {round(num_Na_list[k]/864,3)} y = {round(num_H2O_list[k]/864,3)}')
k=5
ax3.plot(1/temp_1*1000, Na_28_O_16_diff_ls, marker='o', markersize=7, label=f'x = {round(num_Na_list[k]/864,3)} y = {round(num_H2O_list[k]/864,3)}')
k=4
ax3.plot(1/temp_1*1000, Na_16_O_16_diff_ls, marker='o', markersize=7, label=f'x = {round(num_Na_list[k]/864,3)} y = {round(num_H2O_list[k]/864,3)}')
#k=3
#ax3.plot(1/temp_1*1000, Na_4_O_16_diff_ls, marker='o', markersize=7, label=f'x = {round(num_Na_list[k]/864,3)} y = {round(num_H2O_list[k]/864,3)}')
#k=2
#ax3.plot(1/temp_1*1000, Na_28_O_4_diff_ls, marker='^', markersize=7, label=f'x = {round(num_Na_list[k]/864,3)} y = {round(num_H2O_list[k]/864,3)}')
#k=1
#ax3.plot(1/temp_3*1000, Na_16_O_4_diff_ls, marker='^', markersize=7, label=f'x = {round(num_Na_list[k]/864,3)} y = {round(num_H2O_list[k]/864,3)}')
k=0
ax3.plot(1/temp_1*1000, Na_4_O_4_diff_ls, marker='^', markersize=7, label=f'x = {round(num_Na_list[k]/864,3)} y = {round(num_H2O_list[k]/864,3)}')

ax3.legend(fontsize=5)
ax3.grid()
ax3.set_title(f'delta t = {delta_t_ls[i]*0.5} ps')
#ax3.set_xscale('log')
ax3.set_yscale('log')
ax3.set_xlabel('1000/T (1/K)')
ax3.set_ylabel('Diffusivity (m^2/s)')

In [ ]:
temp_1 = np.array([550, 500, 450, 400, 350, 300])
temp_2 = np.array([450, 400, 350, 300])
temp_3 = np.array([550, 500, 450, 400])
temp_4 = np.array([400, 350, 300])

delta_t_ls = np.array([4, 5, 6, 7, 8,
                       9, 10, 15, 20, 30,
                       40, 50, 60, 70, 80,
                       90, 100, 200, 300, 400,
                       500, 600])

fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(18,5))

num_Na_list = [216, 866, 1512,
               205, 867, 1512,
               205, 867, 1512]
num_H2O_list = [216, 216, 216,
                864, 864, 1728,
                1728, 1728, 1728,]

p1, p2 = -8,-7
i=0
Na_4_O_4_diff_ls=[diff_550_O_1[i][p1:p2].mean(), diff_500_O_1[i][p1:p2].mean(), diff_450_O_1[i][p1:p2].mean(),
                  diff_400_O_1[i][p1:p2].mean(), diff_350_O_1[i][p1:p2].mean(), diff_300_O_1[i][p1:p2].mean()]
#i=1
#Na_16_O_4_diff_ls=[diff_550_O_1[i][p1:p2].mean(), diff_500_O_1[i][p1:p2].mean(), diff_450_O_1[i][p1:p2].mean(),
#                  diff_400_O_1[i][p1:p2].mean()]
#i=2
#Na_28_O_4_diff_ls=[diff_550_O_1[i][p1:p2].mean(), diff_500_O_1[i][p1:p2].mean(), diff_450_O_1[i][p1:p2].mean(),
#                  diff_400_O_1[i][p1:p2].mean(), diff_350_O_1[1][p1:p2].mean(), diff_300_O_1[1][p1:p2].mean()]

#i=0
#Na_4_O_16_diff_ls=[diff_550_O_2[i][p1:p2].mean(), diff_500_O_2[i][p1:p2].mean(), diff_450_O_2[i][p1:p2].mean(),
#                  diff_400_O_2[i][p1:p2].mean(), diff_350_O_2[i][p1:p2].mean(), diff_300_O_2[i][p1:p2].mean()]
i=0
Na_16_O_16_diff_ls=[diff_550_O_2[i][p1:p2].mean(), diff_500_O_2[i][p1:p2].mean(), diff_450_O_2[i][p1:p2].mean(),
                  diff_400_O_2[i][p1:p2].mean(), diff_350_O_2[i][p1:p2].mean(), diff_300_O_2[i][p1:p2].mean()]
i=1
Na_28_O_16_diff_ls=[diff_550_O_2[i][p1:p2].mean(), diff_500_O_2[i][p1:p2].mean(), diff_450_O_2[i][p1:p2].mean(),
                  diff_400_O_2[i][p1:p2].mean(), diff_350_O_2[i][p1:p2].mean(), diff_300_O_2[i][p1:p2].mean()]

i=0
Na_4_O_32_diff_ls=[diff_550_O_3[i][p1:p2].mean(), diff_500_O_3[i][p1:p2].mean(), diff_450_O_3[i][p1:p2].mean(),
                  diff_400_O_3[i][p1:p2].mean(), diff_350_O_3[i][p1:p2].mean(), diff_300_O_3[i][p1:p2].mean()]
i=1
Na_16_O_32_diff_ls=[diff_550_O_3[i][p1:p2].mean(), diff_500_O_3[i][p1:p2].mean(), diff_450_O_3[i][p1:p2].mean(),
                  diff_400_O_3[i][p1:p2].mean(), diff_350_O_3[i][p1:p2].mean(), diff_300_O_3[i][p1:p2].mean()]
i=2
Na_28_O_32_diff_ls=[diff_550_O_3[1][p1:p2].mean(), diff_500_O_3[1][p1:p2].mean(), diff_450_O_3[1][p1:p2].mean(),
                  diff_400_O_3[i][p1:p2].mean(), diff_350_O_3[i][p1:p2].mean()]#, diff_300_O_3[i][p1:p2].mean()]

k=8
ax1.plot(1/np.array([550, 500, 450, 400, 350])*1000, Na_28_O_32_diff_ls, marker='*', markersize=7, label=f'x = {round(num_Na_list[k]/864,3)} y = {round(num_H2O_list[k]/864,3)}')
k=7
ax1.plot(1/temp_1*1000, Na_16_O_32_diff_ls, marker='*', markersize=7, label=f'x = {round(num_Na_list[k]/864,3)} y = {round(num_H2O_list[k]/864,3)}')
k=6
ax1.plot(1/temp_1*1000, Na_4_O_32_diff_ls, marker='*', markersize=7, label=f'x = {round(num_Na_list[k]/864,3)} y = {round(num_H2O_list[k]/864,3)}')
k=5
ax1.plot(1/temp_1*1000, Na_28_O_16_diff_ls, marker='o', markersize=7, label=f'x = {round(num_Na_list[k]/864,3)} y = {round(num_H2O_list[k]/864,3)}')
k=4
ax1.plot(1/temp_1*1000, Na_16_O_16_diff_ls, marker='o', markersize=7, label=f'x = {round(num_Na_list[k]/864,3)} y = {round(num_H2O_list[k]/864,3)}')
#k=3
#ax1.plot(1/temp_1*1000, Na_4_O_16_diff_ls, marker='o', markersize=7, label=f'x = {round(num_Na_list[k]/864,3)} y = {round(num_H2O_list[k]/864,3)}')
#k=2
#ax1.plot(1/temp_1*1000, Na_28_O_4_diff_ls, marker='^', markersize=7, label=f'x = {round(num_Na_list[k]/864,3)} y = {round(num_H2O_list[k]/864,3)}')
#k=1
#ax1.plot(1/temp_3*1000, Na_16_O_4_diff_ls, marker='^', markersize=7, label=f'x = {round(num_Na_list[k]/864,3)} y = {round(num_H2O_list[k]/864,3)}')
k=0
ax1.plot(1/temp_1*1000, Na_4_O_4_diff_ls, marker='^', markersize=7, label=f'x = {round(num_Na_list[k]/864,3)} y = {round(num_H2O_list[k]/864,3)}')

ax1.legend(fontsize=5)
ax1.grid()
ax1.set_title(f'delta t = {delta_t_ls[i]*0.5} ps')
#ax1.set_xscale('log')
ax1.set_yscale('log')
ax1.set_xlabel('1000/T (1/K)')
ax1.set_ylabel('Diffusivity (m^2/s)')

p1, p2 = -7,-6
i=0
Na_4_O_4_diff_ls=[diff_550_O_1[i][p1:p2].mean(), diff_500_O_1[i][p1:p2].mean(), diff_450_O_1[i][p1:p2].mean(),
                  diff_400_O_1[i][p1:p2].mean(), diff_350_O_1[i][p1:p2].mean(), diff_300_O_1[i][p1:p2].mean()]
#i=1
#Na_16_O_4_diff_ls=[diff_550_O_1[i][p1:p2].mean(), diff_500_O_1[i][p1:p2].mean(), diff_450_O_1[i][p1:p2].mean(),
#                  diff_400_O_1[i][p1:p2].mean()]
#i=2
#Na_28_O_4_diff_ls=[diff_550_O_1[i][p1:p2].mean(), diff_500_O_1[i][p1:p2].mean(), diff_450_O_1[i][p1:p2].mean(),
#                  diff_400_O_1[i][p1:p2].mean(), diff_350_O_1[1][p1:p2].mean(), diff_300_O_1[1][p1:p2].mean()]

#i=0
#Na_4_O_16_diff_ls=[diff_550_O_2[i][p1:p2].mean(), diff_500_O_2[i][p1:p2].mean(), diff_450_O_2[i][p1:p2].mean(),
#                  diff_400_O_2[i][p1:p2].mean(), diff_350_O_2[i][p1:p2].mean(), diff_300_O_2[i][p1:p2].mean()]
i=0
Na_16_O_16_diff_ls=[diff_550_O_2[i][p1:p2].mean(), diff_500_O_2[i][p1:p2].mean(), diff_450_O_2[i][p1:p2].mean(),
                  diff_400_O_2[i][p1:p2].mean(), diff_350_O_2[i][p1:p2].mean(), diff_300_O_2[i][p1:p2].mean()]
i=1
Na_28_O_16_diff_ls=[diff_550_O_2[i][p1:p2].mean(), diff_500_O_2[i][p1:p2].mean(), diff_450_O_2[i][p1:p2].mean(),
                  diff_400_O_2[i][p1:p2].mean(), diff_350_O_2[i][p1:p2].mean(), diff_300_O_2[i][p1:p2].mean()]

i=0
Na_4_O_32_diff_ls=[diff_550_O_3[i][p1:p2].mean(), diff_500_O_3[i][p1:p2].mean(), diff_450_O_3[i][p1:p2].mean(),
                  diff_400_O_3[i][p1:p2].mean(), diff_350_O_3[i][p1:p2].mean(), diff_300_O_3[i][p1:p2].mean()]
i=1
Na_16_O_32_diff_ls=[diff_550_O_3[i][p1:p2].mean(), diff_500_O_3[i][p1:p2].mean(), diff_450_O_3[i][p1:p2].mean(),
                  diff_400_O_3[i][p1:p2].mean(), diff_350_O_3[i][p1:p2].mean(), diff_300_O_3[i][p1:p2].mean()]
i=2
Na_28_O_32_diff_ls=[diff_550_O_3[1][p1:p2].mean(), diff_500_O_3[1][p1:p2].mean(), diff_450_O_3[1][p1:p2].mean(),
                  diff_400_O_3[i][p1:p2].mean(), diff_350_O_3[i][p1:p2].mean()]#, diff_300_O_3[i][p1:p2].mean()]

k=8
ax2.plot(1/np.array([550, 500, 450, 400, 350])*1000, Na_28_O_32_diff_ls, marker='*', markersize=7, label=f'x = {round(num_Na_list[k]/864,3)} y = {round(num_H2O_list[k]/864,3)}')
k=7
ax2.plot(1/temp_1*1000, Na_16_O_32_diff_ls, marker='*', markersize=7, label=f'x = {round(num_Na_list[k]/864,3)} y = {round(num_H2O_list[k]/864,3)}')
k=6
ax2.plot(1/temp_1*1000, Na_4_O_32_diff_ls, marker='*', markersize=7, label=f'x = {round(num_Na_list[k]/864,3)} y = {round(num_H2O_list[k]/864,3)}')
k=5
ax2.plot(1/temp_1*1000, Na_28_O_16_diff_ls, marker='o', markersize=7, label=f'x = {round(num_Na_list[k]/864,3)} y = {round(num_H2O_list[k]/864,3)}')
k=4
ax2.plot(1/temp_1*1000, Na_16_O_16_diff_ls, marker='o', markersize=7, label=f'x = {round(num_Na_list[k]/864,3)} y = {round(num_H2O_list[k]/864,3)}')
#k=3
#ax2.plot(1/temp_1*1000, Na_4_O_16_diff_ls, marker='o', markersize=7, label=f'x = {round(num_Na_list[k]/864,3)} y = {round(num_H2O_list[k]/864,3)}')
#k=2
#ax2.plot(1/temp_1*1000, Na_28_O_4_diff_ls, marker='^', markersize=7, label=f'x = {round(num_Na_list[k]/864,3)} y = {round(num_H2O_list[k]/864,3)}')
#k=1
#ax2.plot(1/temp_3*1000, Na_16_O_4_diff_ls, marker='^', markersize=7, label=f'x = {round(num_Na_list[k]/864,3)} y = {round(num_H2O_list[k]/864,3)}')
k=0
ax2.plot(1/temp_1*1000, Na_4_O_4_diff_ls, marker='^', markersize=7, label=f'x = {round(num_Na_list[k]/864,3)} y = {round(num_H2O_list[k]/864,3)}')

ax2.legend(fontsize=5)
ax2.grid()
ax2.set_title(f'delta t = {delta_t_ls[i]*0.5} ps')
#ax2.set_xscale('log')
ax2.set_yscale('log')
ax2.set_xlabel('1000/T (1/K)')
ax2.set_ylabel('Diffusivity (m^2/s)')

p1, p2 = -8,-3
i=0
Na_4_O_4_diff_ls=[diff_550_O_1[i][p1:p2].mean(), diff_500_O_1[i][p1:p2].mean(), diff_450_O_1[i][p1:p2].mean(),
                  diff_400_O_1[i][p1:p2].mean(), diff_350_O_1[i][p1:p2].mean(), diff_300_O_1[i][p1:p2].mean()]
#i=1
#Na_16_O_4_diff_ls=[diff_550_O_1[i][p1:p2].mean(), diff_500_O_1[i][p1:p2].mean(), diff_450_O_1[i][p1:p2].mean(),
#                  diff_400_O_1[i][p1:p2].mean()]
#i=2
#Na_28_O_4_diff_ls=[diff_550_O_1[i][p1:p2].mean(), diff_500_O_1[i][p1:p2].mean(), diff_450_O_1[i][p1:p2].mean(),
#                  diff_400_O_1[i][p1:p2].mean(), diff_350_O_1[1][p1:p2].mean(), diff_300_O_1[1][p1:p2].mean()]

#i=0
#Na_4_O_16_diff_ls=[diff_550_O_2[i][p1:p2].mean(), diff_500_O_2[i][p1:p2].mean(), diff_450_O_2[i][p1:p2].mean(),
#                  diff_400_O_2[i][p1:p2].mean(), diff_350_O_2[i][p1:p2].mean(), diff_300_O_2[i][p1:p2].mean()]
i=0
Na_16_O_16_diff_ls=[diff_550_O_2[i][p1:p2].mean(), diff_500_O_2[i][p1:p2].mean(), diff_450_O_2[i][p1:p2].mean(),
                  diff_400_O_2[i][p1:p2].mean(), diff_350_O_2[i][p1:p2].mean(), diff_300_O_2[i][p1:p2].mean()]
i=1
Na_28_O_16_diff_ls=[diff_550_O_2[i][p1:p2].mean(), diff_500_O_2[i][p1:p2].mean(), diff_450_O_2[i][p1:p2].mean(),
                  diff_400_O_2[i][p1:p2].mean(), diff_350_O_2[i][p1:p2].mean(), diff_300_O_2[i][p1:p2].mean()]

i=0
Na_4_O_32_diff_ls=[diff_550_O_3[i][p1:p2].mean(), diff_500_O_3[i][p1:p2].mean(), diff_450_O_3[i][p1:p2].mean(),
                  diff_400_O_3[i][p1:p2].mean(), diff_350_O_3[i][p1:p2].mean(), diff_300_O_3[i][p1:p2].mean()]
i=1
Na_16_O_32_diff_ls=[diff_550_O_3[i][p1:p2].mean(), diff_500_O_3[i][p1:p2].mean(), diff_450_O_3[i][p1:p2].mean(),
                  diff_400_O_3[i][p1:p2].mean(), diff_350_O_3[i][p1:p2].mean(), diff_300_O_3[i][p1:p2].mean()]
i=2
Na_28_O_32_diff_ls=[diff_550_O_3[1][p1:p2].mean(), diff_500_O_3[1][p1:p2].mean(), diff_450_O_3[1][p1:p2].mean(),
                  diff_400_O_3[i][p1:p2].mean(), diff_350_O_3[i][p1:p2].mean()]#, diff_300_O_3[i][p1:p2].mean()]

k=8
ax3.plot(1/np.array([550, 500, 450, 400, 350])*1000, Na_28_O_32_diff_ls, marker='*', markersize=7, label=f'x = {round(num_Na_list[k]/864,3)} y = {round(num_H2O_list[k]/864,3)}')
k=7
ax3.plot(1/temp_1*1000, Na_16_O_32_diff_ls, marker='*', markersize=7, label=f'x = {round(num_Na_list[k]/864,3)} y = {round(num_H2O_list[k]/864,3)}')
k=6
ax3.plot(1/temp_1*1000, Na_4_O_32_diff_ls, marker='*', markersize=7, label=f'x = {round(num_Na_list[k]/864,3)} y = {round(num_H2O_list[k]/864,3)}')
k=5
ax3.plot(1/temp_1*1000, Na_28_O_16_diff_ls, marker='o', markersize=7, label=f'x = {round(num_Na_list[k]/864,3)} y = {round(num_H2O_list[k]/864,3)}')
k=4
ax3.plot(1/temp_1*1000, Na_16_O_16_diff_ls, marker='o', markersize=7, label=f'x = {round(num_Na_list[k]/864,3)} y = {round(num_H2O_list[k]/864,3)}')
#k=3
#ax3.plot(1/temp_1*1000, Na_4_O_16_diff_ls, marker='o', markersize=7, label=f'x = {round(num_Na_list[k]/864,3)} y = {round(num_H2O_list[k]/864,3)}')
#k=2
#ax3.plot(1/temp_1*1000, Na_28_O_4_diff_ls, marker='^', markersize=7, label=f'x = {round(num_Na_list[k]/864,3)} y = {round(num_H2O_list[k]/864,3)}')
#k=1
#ax3.plot(1/temp_3*1000, Na_16_O_4_diff_ls, marker='^', markersize=7, label=f'x = {round(num_Na_list[k]/864,3)} y = {round(num_H2O_list[k]/864,3)}')
k=0
ax3.plot(1/temp_1*1000, Na_4_O_4_diff_ls, marker='^', markersize=7, label=f'x = {round(num_Na_list[k]/864,3)} y = {round(num_H2O_list[k]/864,3)}')

ax3.legend(fontsize=5)
ax3.grid()
ax3.set_title(f'delta t = {delta_t_ls[i]*0.5} ps')
#ax3.set_xscale('log')
ax3.set_yscale('log')
ax3.set_xlabel('1000/T (1/K)')
ax3.set_ylabel('Diffusivity (m^2/s)')

In [ ]:
temp_1 = np.array([550, 500, 450, 400, 350, 300])
temp_2 = np.array([450, 400, 350, 300])
temp_3 = np.array([550, 500, 450, 400])
temp_4 = np.array([400, 350, 300])

delta_t_ls = np.array([4, 5, 6, 7, 8,
                       9, 10, 15, 20, 30,
                       40, 50, 60, 70, 80,
                       90, 100, 200, 300, 400,
                       500, 600])

fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(18,5))

num_Na_list = [216, 866, 1512,
               205, 867, 1512,
               205, 867, 1512]
num_H2O_list = [216, 216, 216,
                864, 864, 1728,
                1728, 1728, 1728,]

p1, p2 = -8,-7
i=0
Na_4_O_4_diff_ls=[diff_550_O_1[i][p1:p2].mean(), diff_500_O_1[i][p1:p2].mean(), diff_450_O_1[i][p1:p2].mean(),
                  diff_400_O_1[i][p1:p2].mean(), diff_350_O_1[i][p1:p2].mean(), diff_300_O_1[i][p1:p2].mean()]
i=1
Na_16_O_4_diff_ls=[diff_550_O_1[i][p1:p2].mean(), diff_500_O_1[i][p1:p2].mean(), diff_450_O_1[i][p1:p2].mean(),
                  diff_400_O_1[i][p1:p2].mean()]
i=2
Na_28_O_4_diff_ls=[diff_550_O_1[i][p1:p2].mean(), diff_500_O_1[i][p1:p2].mean(), diff_450_O_1[i][p1:p2].mean(),
                  diff_400_O_1[i][p1:p2].mean(), diff_350_O_1[1][p1:p2].mean(), diff_300_O_1[1][p1:p2].mean()]

i=0
Na_4_O_16_diff_ls=[diff_550_O_2[i][p1:p2].mean(), diff_500_O_2[i][p1:p2].mean(), diff_450_O_2[i][p1:p2].mean(),
                  diff_400_O_2[i][p1:p2].mean(), diff_350_O_2[i][p1:p2].mean(), diff_300_O_2[i][p1:p2].mean()]
i=1
Na_16_O_16_diff_ls=[diff_550_O_2[i][p1:p2].mean(), diff_500_O_2[i][p1:p2].mean(), diff_450_O_2[i][p1:p2].mean(),
                  diff_400_O_2[i][p1:p2].mean(), diff_350_O_2[i][p1:p2].mean(), diff_300_O_2[i][p1:p2].mean()]
i=2
Na_28_O_16_diff_ls=[diff_450_O_2[i][p1:p2].mean(),
                  diff_400_O_2[i][p1:p2].mean(), diff_350_O_2[i][p1:p2].mean(), diff_300_O_2[i][p1:p2].mean()]

i=0
Na_4_O_32_diff_ls=[diff_400_O_3[i][p1:p2].mean(), diff_350_O_3[i][p1:p2].mean(), diff_300_O_3[i][p1:p2].mean()]

i=1
Na_16_O_32_diff_ls=[diff_550_O_3[i][p1:p2].mean(), diff_500_O_3[i][p1:p2].mean(), diff_450_O_3[0][p1:p2].mean(),
                  diff_400_O_3[i][p1:p2].mean(), diff_350_O_3[i][p1:p2].mean(), diff_300_O_3[i][p1:p2].mean()]
i=2
Na_28_O_32_diff_ls=[diff_550_O_3[1][p1:p2].mean(), diff_500_O_3[1][p1:p2].mean(), diff_450_O_3[1][p1:p2].mean(),
                  diff_400_O_3[i][p1:p2].mean(), diff_350_O_3[i][p1:p2].mean(), diff_300_O_3[i][p1:p2].mean()]

k=8
ax1.plot(1/temp_1*1000, Na_28_O_32_diff_ls, marker='*', markersize=7, label=f'x = {round(num_Na_list[k]/864,3)} y = {round(num_H2O_list[k]/864,3)}')
k=7
ax1.plot(1/temp_1*1000, Na_16_O_32_diff_ls, marker='*', markersize=7, label=f'x = {round(num_Na_list[k]/864,3)} y = {round(num_H2O_list[k]/864,3)}')
k=6
ax1.plot(1/temp_4*1000, Na_4_O_32_diff_ls, marker='*', markersize=7, label=f'x = {round(num_Na_list[k]/864,3)} y = {round(num_H2O_list[k]/864,3)}')
k=5
ax1.plot(1/temp_2*1000, Na_28_O_16_diff_ls, marker='o', markersize=7, label=f'x = {round(num_Na_list[k]/864,3)} y = {round(num_H2O_list[k]/864,3)}')
k=4
ax1.plot(1/temp_1*1000, Na_16_O_16_diff_ls, marker='o', markersize=7, label=f'x = {round(num_Na_list[k]/864,3)} y = {round(num_H2O_list[k]/864,3)}')
k=3
ax1.plot(1/temp_1*1000, Na_4_O_16_diff_ls, marker='o', markersize=7, label=f'x = {round(num_Na_list[k]/864,3)} y = {round(num_H2O_list[k]/864,3)}')
k=2
ax1.plot(1/temp_1*1000, Na_28_O_4_diff_ls, marker='^', markersize=7, label=f'x = {round(num_Na_list[k]/864,3)} y = {round(num_H2O_list[k]/864,3)}')
k=1
ax1.plot(1/temp_3*1000, Na_16_O_4_diff_ls, marker='^', markersize=7, label=f'x = {round(num_Na_list[k]/864,3)} y = {round(num_H2O_list[k]/864,3)}')
k=0
ax1.plot(1/temp_1*1000, Na_4_O_4_diff_ls, marker='^', markersize=7, label=f'x = {round(num_Na_list[k]/864,3)} y = {round(num_H2O_list[k]/864,3)}')

ax1.legend(fontsize=5)
ax1.grid()
ax1.set_title(f'delta t = {delta_t_ls[i]*0.5} ps')
#ax1.set_xscale('log')
ax1.set_yscale('log')
ax1.set_xlabel('1000/T (1/K)')
ax1.set_ylabel('Diffusivity (m^2/s)')

p1, p2 = -7,-6
i=0
Na_4_O_4_diff_ls=[diff_550_O_1[i][p1:p2].mean(), diff_500_O_1[i][p1:p2].mean(), diff_450_O_1[i][p1:p2].mean(),
                  diff_400_O_1[i][p1:p2].mean(), diff_350_O_1[i][p1:p2].mean(), diff_300_O_1[i][p1:p2].mean()]
i=1
Na_16_O_4_diff_ls=[diff_550_O_1[i][p1:p2].mean(), diff_500_O_1[i][p1:p2].mean(), diff_450_O_1[i][p1:p2].mean(),
                  diff_400_O_1[i][p1:p2].mean()]
i=2
Na_28_O_4_diff_ls=[diff_550_O_1[i][p1:p2].mean(), diff_500_O_1[i][p1:p2].mean(), diff_450_O_1[i][p1:p2].mean(),
                  diff_400_O_1[i][p1:p2].mean(), diff_350_O_1[1][p1:p2].mean(), diff_300_O_1[1][p1:p2].mean()]

i=0
Na_4_O_16_diff_ls=[diff_550_O_2[i][p1:p2].mean(), diff_500_O_2[i][p1:p2].mean(), diff_450_O_2[i][p1:p2].mean(),
                  diff_400_O_2[i][p1:p2].mean(), diff_350_O_2[i][p1:p2].mean(), diff_300_O_2[i][p1:p2].mean()]
i=1
Na_16_O_16_diff_ls=[diff_550_O_2[i][p1:p2].mean(), diff_500_O_2[i][p1:p2].mean(), diff_450_O_2[i][p1:p2].mean(),
                  diff_400_O_2[i][p1:p2].mean(), diff_350_O_2[i][p1:p2].mean(), diff_300_O_2[i][p1:p2].mean()]
i=2
Na_28_O_16_diff_ls=[diff_450_O_2[i][p1:p2].mean(),
                  diff_400_O_2[i][p1:p2].mean(), diff_350_O_2[i][p1:p2].mean(), diff_300_O_2[i][p1:p2].mean()]

i=0
Na_4_O_32_diff_ls=[diff_400_O_3[i][p1:p2].mean(), diff_350_O_3[i][p1:p2].mean(), diff_300_O_3[i][p1:p2].mean()]

i=1
Na_16_O_32_diff_ls=[diff_550_O_3[i][p1:p2].mean(), diff_500_O_3[i][p1:p2].mean(), diff_450_O_3[0][p1:p2].mean(),
                  diff_400_O_3[i][p1:p2].mean(), diff_350_O_3[i][p1:p2].mean(), diff_300_O_3[i][p1:p2].mean()]
i=2
Na_28_O_32_diff_ls=[diff_550_O_3[1][p1:p2].mean(), diff_500_O_3[1][p1:p2].mean(), diff_450_O_3[1][p1:p2].mean(),
                  diff_400_O_3[i][p1:p2].mean(), diff_350_O_3[i][p1:p2].mean(), diff_300_O_3[i][p1:p2].mean()]
k=8
ax2.plot(1/temp_1*1000, Na_28_O_32_diff_ls, marker='*', markersize=7, label=f'x = {round(num_Na_list[k]/864,3)} y = {round(num_H2O_list[k]/864,3)}')
k=7
ax2.plot(1/temp_1*1000, Na_16_O_32_diff_ls, marker='*', markersize=7, label=f'x = {round(num_Na_list[k]/864,3)} y = {round(num_H2O_list[k]/864,3)}')
k=6
ax2.plot(1/temp_4*1000, Na_4_O_32_diff_ls, marker='*', markersize=7, label=f'x = {round(num_Na_list[k]/864,3)} y = {round(num_H2O_list[k]/864,3)}')
k=5
ax2.plot(1/temp_2*1000, Na_28_O_16_diff_ls, marker='o', markersize=7, label=f'x = {round(num_Na_list[k]/864,3)} y = {round(num_H2O_list[k]/864,3)}')
k=4
ax2.plot(1/temp_1*1000, Na_16_O_16_diff_ls, marker='o', markersize=7, label=f'x = {round(num_Na_list[k]/864,3)} y = {round(num_H2O_list[k]/864,3)}')
k=3
ax2.plot(1/temp_1*1000, Na_4_O_16_diff_ls, marker='o', markersize=7, label=f'x = {round(num_Na_list[k]/864,3)} y = {round(num_H2O_list[k]/864,3)}')
k=2
ax2.plot(1/temp_1*1000, Na_28_O_4_diff_ls, marker='^', markersize=7, label=f'x = {round(num_Na_list[k]/864,3)} y = {round(num_H2O_list[k]/864,3)}')
k=1
ax2.plot(1/temp_3*1000, Na_16_O_4_diff_ls, marker='^', markersize=7, label=f'x = {round(num_Na_list[k]/864,3)} y = {round(num_H2O_list[k]/864,3)}')
k=0
ax1.plot(1/temp_1*1000, Na_4_O_4_diff_ls, marker='^', markersize=7, label=f'x = {round(num_Na_list[k]/864,3)} y = {round(num_H2O_list[k]/864,3)}')

ax2.legend(fontsize=5)
ax2.grid()
ax2.set_title(f'delta t = {delta_t_ls[i]*0.5} ps')
#ax1.set_xscale('log')
ax2.set_yscale('log')
ax2.set_xlabel('1000/T (1/K)')
ax2.set_ylabel('Diffusivity (m^2/s)')

p1, p2 = -6,-5
i=0
Na_4_O_4_diff_ls=[diff_550_O_1[i][p1:p2].mean(), diff_500_O_1[i][p1:p2].mean(), diff_450_O_1[i][p1:p2].mean(),
                  diff_400_O_1[i][p1:p2].mean(), diff_350_O_1[i][p1:p2].mean(), diff_300_O_1[i][p1:p2].mean()]
i=1
Na_16_O_4_diff_ls=[diff_550_O_1[i][p1:p2].mean(), diff_500_O_1[i][p1:p2].mean(), diff_450_O_1[i][p1:p2].mean(),
                  diff_400_O_1[i][p1:p2].mean()]
i=2
Na_28_O_4_diff_ls=[diff_550_O_1[i][p1:p2].mean(), diff_500_O_1[i][p1:p2].mean(), diff_450_O_1[i][p1:p2].mean(),
                  diff_400_O_1[i][p1:p2].mean(), diff_350_O_1[1][p1:p2].mean(), diff_300_O_1[1][p1:p2].mean()]

i=0
Na_4_O_16_diff_ls=[diff_550_O_2[i][p1:p2].mean(), diff_500_O_2[i][p1:p2].mean(), diff_450_O_2[i][p1:p2].mean(),
                  diff_400_O_2[i][p1:p2].mean(), diff_350_O_2[i][p1:p2].mean(), diff_300_O_2[i][p1:p2].mean()]
i=1
Na_16_O_16_diff_ls=[diff_550_O_2[i][p1:p2].mean(), diff_500_O_2[i][p1:p2].mean(), diff_450_O_2[i][p1:p2].mean(),
                  diff_400_O_2[i][p1:p2].mean(), diff_350_O_2[i][p1:p2].mean(), diff_300_O_2[i][p1:p2].mean()]
i=2
Na_28_O_16_diff_ls=[diff_450_O_2[i][p1:p2].mean(),
                  diff_400_O_2[i][p1:p2].mean(), diff_350_O_2[i][p1:p2].mean(), diff_300_O_2[i][p1:p2].mean()]

i=0
Na_4_O_32_diff_ls=[diff_400_O_3[i][p1:p2].mean(), diff_350_O_3[i][p1:p2].mean(), diff_300_O_3[i][p1:p2].mean()]

i=1
Na_16_O_32_diff_ls=[diff_550_O_3[i][p1:p2].mean(), diff_500_O_3[i][p1:p2].mean(), diff_450_O_3[0][p1:p2].mean(),
                  diff_400_O_3[i][p1:p2].mean(), diff_350_O_3[i][p1:p2].mean(), diff_300_O_3[i][p1:p2].mean()]
i=2
Na_28_O_32_diff_ls=[diff_550_O_3[1][p1:p2].mean(), diff_500_O_3[1][p1:p2].mean(), diff_450_O_3[1][p1:p2].mean(),
                  diff_400_O_3[i][p1:p2].mean(), diff_350_O_3[i][p1:p2].mean(), diff_300_O_3[i][p1:p2].mean()]

k=8
ax3.plot(1/temp_1*1000, Na_28_O_32_diff_ls, marker='*', markersize=7, label=f'x = {round(num_Na_list[k]/864,3)} y = {round(num_H2O_list[k]/864,3)}')
k=7
ax3.plot(1/temp_1*1000, Na_16_O_32_diff_ls, marker='*', markersize=7, label=f'x = {round(num_Na_list[k]/864,3)} y = {round(num_H2O_list[k]/864,3)}')
k=6
ax3.plot(1/temp_4*1000, Na_4_O_32_diff_ls, marker='*', markersize=7, label=f'x = {round(num_Na_list[k]/864,3)} y = {round(num_H2O_list[k]/864,3)}')
k=5
ax3.plot(1/temp_2*1000, Na_28_O_16_diff_ls, marker='o', markersize=7, label=f'x = {round(num_Na_list[k]/864,3)} y = {round(num_H2O_list[k]/864,3)}')
k=4
ax3.plot(1/temp_1*1000, Na_16_O_16_diff_ls, marker='o', markersize=7, label=f'x = {round(num_Na_list[k]/864,3)} y = {round(num_H2O_list[k]/864,3)}')
k=3
ax3.plot(1/temp_1*1000, Na_4_O_16_diff_ls, marker='o', markersize=7, label=f'x = {round(num_Na_list[k]/864,3)} y = {round(num_H2O_list[k]/864,3)}')
k=2
ax3.plot(1/temp_1*1000, Na_28_O_4_diff_ls, marker='^', markersize=7, label=f'x = {round(num_Na_list[k]/864,3)} y = {round(num_H2O_list[k]/864,3)}')
k=1
ax3.plot(1/temp_3*1000, Na_16_O_4_diff_ls, marker='^', markersize=7, label=f'x = {round(num_Na_list[k]/864,3)} y = {round(num_H2O_list[k]/864,3)}')
k=0
ax3.plot(1/temp_1*1000, Na_4_O_4_diff_ls, marker='^', markersize=7, label=f'x = {round(num_Na_list[k]/864,3)} y = {round(num_H2O_list[k]/864,3)}')

ax3.legend(fontsize=5)
ax3.grid()
ax3.set_title(f'delta t = {delta_t_ls[i]*0.5} ps')
#ax1.set_xscale('log')
ax3.set_yscale('log')
ax3.set_xlabel('1000/T (1/K)')
ax3.set_ylabel('Diffusivity (m^2/s)')

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(singlecol*4, singlecol*2))

num_Na_list = [216, 866, 1512,
               205, 867, 1512,
               205, 867, 1512]
num_O_list = [216, 216, 216,
                864, 864, 864,
                1728, 1728, 1728,]
p1, p2 = -6,-5
i=0
Na_4_O_4_diff_ls=[diff_550_Na_1[i][p1:p2].mean(), diff_500_Na_1[i][p1:p2].mean(), diff_450_Na_1[i][p1:p2].mean(),
                  diff_400_Na_1[i][p1:p2].mean(), diff_350_Na_1[i][p1:p2].mean(), diff_300_Na_1[i][p1:p2].mean()]
i=1
Na_16_O_4_diff_ls=[diff_550_Na_1[i][p1:p2].mean(), diff_500_Na_1[i][p1:p2].mean(), diff_450_Na_1[i][p1:p2].mean(),
                  diff_400_Na_1[i][p1:p2].mean()]
i=2
Na_28_O_4_diff_ls=[diff_550_Na_1[i][p1:p2].mean(), diff_500_Na_1[i][p1:p2].mean(), diff_450_Na_1[i][p1:p2].mean(),
                  diff_400_Na_1[i][p1:p2].mean(), diff_350_Na_1[1][p1:p2].mean(), diff_300_Na_1[1][p1:p2].mean()]

i=0
Na_4_O_16_diff_ls=[diff_550_Na_2[i][p1:p2].mean(), diff_500_Na_2[i][p1:p2].mean(), diff_450_Na_2[i][p1:p2].mean(),
                  diff_400_Na_2[i][p1:p2].mean(), diff_350_Na_2[i][p1:p2].mean(), diff_300_Na_2[i][p1:p2].mean()]
i=1
Na_16_O_16_diff_ls=[diff_550_Na_2[i][p1:p2].mean(), diff_500_Na_2[i][p1:p2].mean(), diff_450_Na_2[i][p1:p2].mean(),
                  diff_400_Na_2[i][p1:p2].mean(), diff_350_Na_2[i][p1:p2].mean(), diff_300_Na_2[i][p1:p2].mean()]
i=2
Na_28_O_16_diff_ls=[diff_450_Na_2[i][p1:p2].mean(),
                  diff_400_Na_2[i][p1:p2].mean(), diff_350_Na_2[i][p1:p2].mean(), diff_300_Na_2[i][p1:p2].mean()]

i=0
Na_4_O_32_diff_ls=[diff_400_Na_3[i][p1:p2].mean(), diff_350_Na_3[i][p1:p2].mean(), diff_300_Na_3[i][p1:p2].mean()]

i=1
Na_16_O_32_diff_ls=[diff_550_Na_3[i][p1:p2].mean(), diff_500_Na_3[i][p1:p2].mean(), diff_450_Na_3[0][p1:p2].mean(),
                  diff_400_Na_3[i][p1:p2].mean(), diff_350_Na_3[i][p1:p2].mean(), diff_300_Na_3[i][p1:p2].mean()]
i=2
Na_28_O_32_diff_ls=[diff_550_Na_3[1][p1:p2].mean(), diff_500_Na_3[1][p1:p2].mean(), diff_450_Na_3[1][p1:p2].mean(),
                  diff_400_Na_3[i][p1:p2].mean(), diff_350_Na_3[i][p1:p2].mean(), diff_300_Na_3[i][p1:p2].mean()]

k=8
ax1.plot(1/temp_1*1000, Na_28_O_32_diff_ls, marker='*', markersize=7, label=f'x = {round(num_Na_list[k]/864,3)} y = {round(num_H2O_list[k]/864,3)}')
k=7
ax1.plot(1/temp_1*1000, Na_16_O_32_diff_ls, marker='*', markersize=7, label=f'x = {round(num_Na_list[k]/864,3)} y = {round(num_H2O_list[k]/864,3)}')
k=6
ax1.plot(1/temp_4*1000, Na_4_O_32_diff_ls, marker='*', markersize=7, label=f'x = {round(num_Na_list[k]/864,3)} y = {round(num_H2O_list[k]/864,3)}')
k=5
ax1.plot(1/temp_2*1000, Na_28_O_16_diff_ls, marker='o', markersize=7, label=f'x = {round(num_Na_list[k]/864,3)} y = {round(num_H2O_list[k]/864,3)}')
k=4
ax1.plot(1/temp_1*1000, Na_16_O_16_diff_ls, marker='o', markersize=7, label=f'x = {round(num_Na_list[k]/864,3)} y = {round(num_H2O_list[k]/864,3)}')
k=3
ax1.plot(1/temp_1*1000, Na_4_O_16_diff_ls, marker='o', markersize=7, label=f'x = {round(num_Na_list[k]/864,3)} y = {round(num_H2O_list[k]/864,3)}')
k=2
ax1.plot(1/temp_1*1000, Na_28_O_4_diff_ls, marker='^', markersize=7, label=f'x = {round(num_Na_list[k]/864,3)} y = {round(num_H2O_list[k]/864,3)}')
k=1
ax1.plot(1/temp_3*1000, Na_16_O_4_diff_ls, marker='^', markersize=7, label=f'x = {round(num_Na_list[k]/864,3)} y = {round(num_H2O_list[k]/864,3)}')
k=0
ax1.plot(1/temp_1*1000, Na_4_O_4_diff_ls, marker='^', markersize=7, label=f'x = {round(num_Na_list[k]/864,3)} y = {round(num_H2O_list[k]/864,3)}')

ax1.legend(frameon=False)
ax1.grid()
ax1.set_title(f'delta t = {delta_t_ls[i]*0.5} ps')

p1, p2 = -6,-5
i=0
Na_4_O_4_diff_ls=[diff_550_O_1[i][p1:p2].mean(), diff_500_O_1[i][p1:p2].mean(), diff_450_O_1[i][p1:p2].mean(),
                  diff_400_O_1[i][p1:p2].mean(), diff_350_O_1[i][p1:p2].mean(), diff_300_O_1[i][p1:p2].mean()]
i=1
Na_16_O_4_diff_ls=[diff_550_O_1[i][p1:p2].mean(), diff_500_O_1[i][p1:p2].mean(), diff_450_O_1[i][p1:p2].mean(),
                  diff_400_O_1[i][p1:p2].mean()]
i=2
Na_28_O_4_diff_ls=[diff_550_O_1[i][p1:p2].mean(), diff_500_O_1[i][p1:p2].mean(), diff_450_O_1[i][p1:p2].mean(),
                  diff_400_O_1[i][p1:p2].mean(), diff_350_O_1[1][p1:p2].mean(), diff_300_O_1[1][p1:p2].mean()]

i=0
Na_4_O_16_diff_ls=[diff_550_O_2[i][p1:p2].mean(), diff_500_O_2[i][p1:p2].mean(), diff_450_O_2[i][p1:p2].mean(),
                  diff_400_O_2[i][p1:p2].mean(), diff_350_O_2[i][p1:p2].mean(), diff_300_O_2[i][p1:p2].mean()]
i=1
Na_16_O_16_diff_ls=[diff_550_O_2[i][p1:p2].mean(), diff_500_O_2[i][p1:p2].mean(), diff_450_O_2[i][p1:p2].mean(),
                  diff_400_O_2[i][p1:p2].mean(), diff_350_O_2[i][p1:p2].mean(), diff_300_O_2[i][p1:p2].mean()]
i=2
Na_28_O_16_diff_ls=[diff_450_O_2[i][p1:p2].mean(),
                  diff_400_O_2[i][p1:p2].mean(), diff_350_O_2[i][p1:p2].mean(), diff_300_O_2[i][p1:p2].mean()]

i=0
Na_4_O_32_diff_ls=[diff_400_O_3[i][p1:p2].mean(), diff_350_O_3[i][p1:p2].mean(), diff_300_O_3[i][p1:p2].mean()]

i=1
Na_16_O_32_diff_ls=[diff_550_O_3[i][p1:p2].mean(), diff_500_O_3[i][p1:p2].mean(), diff_450_O_3[0][p1:p2].mean(),
                  diff_400_O_3[i][p1:p2].mean(), diff_350_O_3[i][p1:p2].mean(), diff_300_O_3[i][p1:p2].mean()]
i=2
Na_28_O_32_diff_ls=[diff_550_O_3[1][p1:p2].mean(), diff_500_O_3[1][p1:p2].mean(), diff_450_O_3[1][p1:p2].mean(),
                  diff_400_O_3[i][p1:p2].mean(), diff_350_O_3[i][p1:p2].mean(), diff_300_O_3[i][p1:p2].mean()]


k=8
ax2.plot(1/temp_1*1000, Na_28_O_32_diff_ls, marker='*', markersize=7, label=f'x = {round(num_Na_list[k]/864,3)} y = {round(num_H2O_list[k]/864,3)}')
k=7
ax2.plot(1/temp_1*1000, Na_16_O_32_diff_ls, marker='*', markersize=7, label=f'x = {round(num_Na_list[k]/864,3)} y = {round(num_H2O_list[k]/864,3)}')
k=6
ax2.plot(1/temp_4*1000, Na_4_O_32_diff_ls, marker='*', markersize=7, label=f'x = {round(num_Na_list[k]/864,3)} y = {round(num_H2O_list[k]/864,3)}')
k=5
ax2.plot(1/temp_2*1000, Na_28_O_16_diff_ls, marker='o', markersize=7, label=f'x = {round(num_Na_list[k]/864,3)} y = {round(num_H2O_list[k]/864,3)}')
k=4
ax2.plot(1/temp_1*1000, Na_16_O_16_diff_ls, marker='o', markersize=7, label=f'x = {round(num_Na_list[k]/864,3)} y = {round(num_H2O_list[k]/864,3)}')
k=3
ax2.plot(1/temp_1*1000, Na_4_O_16_diff_ls, marker='o', markersize=7, label=f'x = {round(num_Na_list[k]/864,3)} y = {round(num_H2O_list[k]/864,3)}')
k=2
ax2.plot(1/temp_1*1000, Na_28_O_4_diff_ls, marker='^', markersize=7, label=f'x = {round(num_Na_list[k]/864,3)} y = {round(num_H2O_list[k]/864,3)}')
k=1
ax2.plot(1/temp_3*1000, Na_16_O_4_diff_ls, marker='^', markersize=7, label=f'x = {round(num_Na_list[k]/864,3)} y = {round(num_H2O_list[k]/864,3)}')
k=0
ax2.plot(1/temp_1*1000, Na_4_O_4_diff_ls, marker='^', markersize=7, label=f'x = {round(num_Na_list[k]/864,3)} y = {round(num_H2O_list[k]/864,3)}')

ax2.legend()
ax2.grid()
ax2.set_title(f'delta t = {delta_t_ls[i]*0.5} ps')
#ax1.set_xscale('log')
ax2.set_yscale('log')
ax2.set_xlabel('1000/T (1/K)')
ax2.set_ylabel('H2O Diffusivity (m^2/s)')


#ax1.set_xscale('log')
ax1.set_yscale('log')
ax1.set_xlabel('1000/T (1/K)')
ax1.set_ylabel('Na Diffusivity (m^2/s)')
plt.tight_layout()
#plt.savefig('Diffuvisity_Temp.png')

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit

def fit_diff(temp_ls, Na_diff_ls, k):
    num_Na_list = [216, 866, 1512,
               205, 867, 1512,
               205, 867, 1512]
    num_O_list = [216, 216, 216,
                864, 864, 864,
                1728, 1728, 1728,]
    # Define the exponential function
    def exp_func(x, a, b):
        return a + b * x
    
    # Prepare your data
    x_data = 1/temp_ls  # Reciprocal of temperature
    y_data = np.log(Na_diff_ls)  # Diffusivity values
    
    # Perform curve fitting
    popt, pcov = curve_fit(exp_func, x_data, y_data, p0=(1, 1))
    a_fit, b_fit = popt
    
    # Calculate standard errors
    perr = np.sqrt(np.diag(pcov))
    a_err, b_err = perr
    
    # Print fitted parameters with their uncertainties
    print(f"Fitted Parameters:")
    print(f"a = {a_fit} ± {a_err}")
    print(f"b = {b_fit:.5f} ± {b_err:.5f}")
    
    # Calculate R-squared to assess goodness of fit
    residuals = y_data - exp_func(x_data, a_fit, b_fit)
    ss_res = np.sum(residuals**2)
    ss_tot = np.sum((y_data - np.mean(y_data))**2)
    r_squared = 1 - (ss_res / ss_tot)
    print(f"R-squared: {r_squared:.5f}")
    
    # Plotting
    plt.figure(figsize=(6, 6))
    plt.scatter(1000*x_data, y_data, label='Experimental Data')
    x_fit = np.linspace(x_data.min(), x_data.max(), 100)
    y_fit = exp_func(x_fit, a_fit, b_fit)
    D_300K = exp_func(1/300, a_fit, b_fit)
    print(f'(D(300K)) = {np.exp(D_300K)} m^2/s')
    plt.plot(1000*x_fit, y_fit, 'r-', label='Fitted Curve')
    plt.scatter(1000/300, D_300K)
    plt.xlabel('1000/T')
    plt.ylabel('log(Diffusivity)')
    plt.title(f'Diffusivity Exponential Fit at x = {round(num_Na_list[k]/864,3)} y = {round(num_O_list[k]/864,3)}')
    plt.legend()
    #plt.savefig(f'./production_run/free_E/test/diffusion_fit_{SoC}'+'.png', bbox_inches='tight')
    #plt.xscale('log')
    #plt.yscale('log')
    plt.grid(True)
    plt.show()
    return a_fit, b_fit, b_err, np.exp(D_300K)

In [ ]:
p1, p2 = -8,-7
i=0
Na_4_O_4_diff_ls=[diff_550_Na_1[i][p1:p2].mean(), diff_500_Na_1[i][p1:p2].mean(), diff_450_Na_1[i][p1:p2].mean(),
                  diff_400_Na_1[i][p1:p2].mean(), diff_350_Na_1[i][p1:p2].mean(), diff_300_Na_1[i][p1:p2].mean()]
i=0
Na_16_O_16_diff_ls=[diff_550_Na_2[i][p1:p2].mean(), diff_500_Na_2[i][p1:p2].mean(), diff_450_Na_2[i][p1:p2].mean(),
                  diff_400_Na_2[i][p1:p2].mean(), diff_350_Na_2[i][p1:p2].mean(), diff_300_Na_2[i][p1:p2].mean()]
i=1
Na_28_O_16_diff_ls=[diff_550_Na_2[i][p1:p2].mean(), diff_500_Na_2[i][p1:p2].mean(), diff_450_Na_2[i][p1:p2].mean(),
                  diff_400_Na_2[i][p1:p2].mean(), diff_350_Na_2[i][p1:p2].mean(), diff_300_Na_2[i][p1:p2].mean()]

i=0
Na_4_O_32_diff_ls=[diff_550_Na_3[i][p1:p2].mean(), diff_500_Na_3[i][p1:p2].mean(), diff_450_Na_3[i][p1:p2].mean(),
                  diff_400_Na_3[i][p1:p2].mean(), diff_350_Na_3[i][p1:p2].mean(), diff_300_Na_3[i][p1:p2].mean()]
i=1
Na_16_O_32_diff_ls=[diff_550_Na_3[i][p1:p2].mean(), diff_500_Na_3[i][p1:p2].mean(), diff_450_Na_3[i][p1:p2].mean(),
                  diff_400_Na_3[i][p1:p2].mean(), diff_350_Na_3[i][p1:p2].mean(), diff_300_Na_3[i][p1:p2].mean()]
i=2
Na_28_O_32_diff_ls=[diff_550_Na_3[1][p1:p2].mean(), diff_500_Na_3[1][p1:p2].mean(), diff_450_Na_3[1][p1:p2].mean(),
                  diff_400_Na_3[i][p1:p2].mean(), diff_350_Na_3[i][p1:p2].mean()]#, diff_300_Na_3[i][p1:p2].mean()]


In [ ]:
a_fit_8, b_fit_8, b_err_8, D_300_8 = fit_diff(temp_1[:-1], Na_28_O_32_diff_ls, 8)
a_fit_7, b_fit_7, b_err_7, D_300_7 = fit_diff(temp_1, Na_16_O_32_diff_ls, 7)
a_fit_6, b_fit_6, b_err_6, D_300_6 = fit_diff(temp_1, Na_4_O_32_diff_ls, 6)

In [ ]:
a_fit_8, b_fit_8, b_err_8, D_300_8 = fit_diff(temp_1, Na_28_O_32_diff_ls, 8)
a_fit_7, b_fit_7, b_err_7, D_300_7 = fit_diff(temp_1, Na_16_O_32_diff_ls, 7)
a_fit_6, b_fit_6, b_err_6, D_300_6 = fit_diff(temp_4, Na_4_O_32_diff_ls, 6)

In [ ]:
del Na_4_O_16_diff_ls[0]

In [ ]:
[Na_28_O_16_diff_ls[1],Na_28_O_16_diff_ls[3],Na_28_O_16_diff_ls[4],Na_28_O_16_diff_ls[5]]

In [ ]:
a_fit_5, b_fit_5, b_err_5, D_300_5 = fit_diff(np.array([500,400,350,300]), [Na_28_O_16_diff_ls[1],Na_28_O_16_diff_ls[3],Na_28_O_16_diff_ls[4],Na_28_O_16_diff_ls[5]], 5)
a_fit_4, b_fit_4, b_err_4, D_300_4 = fit_diff(np.array([500,400,350,300]), [Na_16_O_16_diff_ls[1],Na_16_O_16_diff_ls[3],Na_16_O_16_diff_ls[4],Na_16_O_16_diff_ls[5]], 4)
#a_fit_3, b_fit_3, b_err_3, D_300_3 = fit_diff(np.array([550,500,450,350,300])[:-1], Na_4_O_16_diff_ls[:-1], 3)

In [ ]:
a_fit_5, b_fit_5, b_err_5, D_300_5 = fit_diff(temp_2, Na_28_O_16_diff_ls, 5)
a_fit_4, b_fit_4, b_err_4, D_300_4 = fit_diff(temp_1, Na_16_O_16_diff_ls, 4)
a_fit_3, b_fit_3, b_err_3, D_300_3 = fit_diff(np.array([550,500,450,350,300]), Na_4_O_16_diff_ls, 3)

In [ ]:
Na_28_O_4_diff_ls

In [ ]:
a_fit_0, b_fit_0, b_err_0, D_300_0 = fit_diff(temp_1, Na_4_O_4_diff_ls, 0)

In [ ]:
a_fit_2, b_fit_2, b_err_2, D_300_2 = fit_diff(temp_1[:-3], Na_28_O_4_diff_ls[:-3], 2)
a_fit_1, b_fit_1, b_err_1, D_300_1 = fit_diff(temp_3, Na_16_O_4_diff_ls, 1)
a_fit_0, b_fit_0, b_err_0, D_300_0 = fit_diff(temp_1[:-1], Na_4_O_4_diff_ls[:-1], 0)

### Activation energy

In [ ]:
[D_300_6, D_300_7, D_300_8]

In [ ]:
fig, axs = plt.subplots(
    1, 2,
    figsize=(2 * singlecol, singlecol),
    dpi=300
)
plt.rcParams.update({'lines.markersize':5})

# =========================
# Panel (a): Activation energy
# =========================
ax = axs[0]

x1 = [0.25, 1, 1.75]
color = 'tab:blue'
y1 = np.array([-b_fit_6, -b_fit_7, -b_fit_8]) * 8.617e-5
yerr1 = np.array([b_err_6, b_err_7, b_err_8]) * 8.617e-5
ax.errorbar(x1, y1, yerr=yerr1, fmt='s', capsize=5, color=color)
ax.plot(x1, y1, '--', color=color)
ax.scatter(x1, y1, marker='s', color=color, label='y = 2')

color = 'tab:red'
x2 = [1, 1.75]
y2 = np.array([-b_fit_4, -b_fit_5]) * 8.617e-5
yerr2 = np.array([b_err_4, b_err_5]) * 8.617e-5
ax.errorbar(x2, y2, yerr=yerr2, fmt='o', capsize=5, color=color)
ax.plot(x2, y2, '--', color=color)
ax.scatter(x2, y2,  marker='o', color=color, label='y = 1')

color = 'tab:green'
x3 = [0.25]
y3 = np.array([-b_fit_0]) * 8.617e-5
yerr3 = np.array([b_err_0]) * 8.617e-5
ax.scatter(x3, y3,  marker='^', color=color, label='y = 0.25')
ax.errorbar(x3, y3, yerr=yerr3, fmt='^', capsize=5, color=color)

color = 'tab:purple'
x4 = [0.195, 1.758]
y4 = [0.30995606, 0.59990472]
yerr4 = [0.00629274, 0.0289458]
ax.errorbar(x4, y4, yerr=yerr4, fmt='*', capsize=5, color=color)
ax.scatter(x4, y4,  marker='*', color=color, label='y = 0')

ax.set_xlabel('x')
ax.set_ylabel('Activation energy (eV)')
ax.legend(frameon=False)

# =========================
# Panel (b): Diffusivity
# =========================
ax = axs[1]

ax.plot([0.25, 1, 1.75],  [D_300_6, D_300_7, D_300_8], '--', color='tab:blue')
ax.scatter([0.25, 1, 1.75],  [D_300_6, D_300_7, D_300_8], marker='s', color='tab:blue', label='y = 2')

ax.plot([1, 1.75], [D_300_4, D_300_5], '--', color='tab:red')
ax.scatter([1, 1.75], [D_300_4, D_300_5], marker='o', color='tab:red', label='y = 1')

ax.plot([0.25], [D_300_0], '^', color='tab:green', label='y = 0.25')

ax.scatter([0.195, 1.758], [7.75e-14, 3.99e-18],marker='*', color='purple', label='y = 0')

ax.set_xlabel('x')
ax.set_ylabel(r'Sodium diffusivity at 300 K ($\mathrm{m^2\,s^{-1}}$)')
ax.set_yscale('log')
ax.legend(frameon=False)

# =========================
# Layout and save
# =========================
fig.subplots_adjust(wspace=0.3)
plt.tight_layout()
fig.savefig('Activation_energy_lv16.svg')


In [ ]:
plt.figure(figsize=(singlecol, singlecol), dpi=300)

x = [0.25,1,1.75]

color='tab:blue'
y_1 = np.array([-b_fit_6, -b_fit_7, -b_fit_8])*8.617e-5
yerr_1 = np.array([b_err_6, b_err_7, b_err_8])*8.617e-5
plt.errorbar([0.25, 1, 1.75], y_1, yerr=yerr_1, fmt='o', capsize=5, color=color)
plt.scatter([0.25, 1, 1.75], y_1, color=color, label='y = 2')
plt.plot([0.25, 1, 1.75], y_1, '--', color=color)

color='tab:green'
y_2 = np.array([ -b_fit_4, -b_fit_5])*8.617e-5
yerr_2 = np.array([ b_err_4, b_err_5])*8.617e-5
plt.errorbar([1,1.75], y_2, yerr=yerr_2, fmt='o', capsize=5, color=color)
plt.scatter([1,1.75], y_2, color=color, label='y = 1')
plt.plot([1,1.75], y_2, '--', color=color)

color='tab:orange'
y_3 = np.array([-b_fit_0])*8.617e-5
yerr_3 = np.array([ b_err_0])*8.617e-5
plt.errorbar([0.25], y_3, yerr=yerr_3, fmt='o', capsize=5, color=color)
plt.scatter([0.25],y_3, color=color, label='y = 0.25')

color='tab:red'
y_4 = np.array([0.30995606, 0.59990472])
yerr_4 = np.array([0.00629274, 0.0289458 ])
x = [0.195,1.758]
plt.errorbar(x, y_4, yerr=yerr_4, fmt='^', capsize=5, color=color)
plt.scatter(x,y_4, color=color, marker='^', label='y = 0')
plt.xlabel('x')
plt.ylabel('Activation energy (eV)')
#plt.grid()
plt.legend(frameon=False)
plt.tight_layout()

color='tab:blue'
y = [D_300_6, D_300_7, D_300_8]
axs.plot([0.25, 1, 1.75], y, '--', color=color, label='y = 2')

color='tab:green'
y = [D_300_4, D_300_5]
axs.plot([1, 1.75], y, '--', color=color, label='y = 1')

color='tab:orange'
y = [D_300_0]
axs.plot([0.25, 1, 1.75], y, '--', color=color, label='y = 0.25')

color='tab:orange'
y = [7.75e-14,3.99e-18]
axs.plot([0.195,1.758], y, '--', color=color, label='y = 0')

axs.set_xlabel('x')
axs.set_ylabel(r'Sodium diffusivity at 300K ($\mathrm{m^2\,s^{-1}}$)')


plt.savefig('Activation energy_lv16.svg')

In [ ]:
plt.figure(figsize=(singlecol*2, singlecol*2))


color='tab:blue'
plt.plot([0.25, 1, 1.75], 
         [diff_450_Na_3[0][p1:p2].mean(), diff_450_Na_3[1][p1:p2].mean(), diff_450_Na_3[2][p1:p2].mean()],
         '-o', color=color, label='y = 1.75')

color='tab:green'
plt.plot([0.25, 1, 1.75], 
         [diff_450_Na_2[0][p1:p2].mean(), diff_450_Na_2[1][p1:p2].mean(), diff_450_Na_2[2][p1:p2].mean()],
         '-o', color=color, label='y = 1')

color='tab:orange'
plt.plot([0.25, 1, 1.75], 
         [diff_450_Na_1[0][p1:p2].mean(), diff_450_Na_1[1][p1:p2].mean(), diff_450_Na_1[2][p1:p2].mean()],
         '-o', color=color, label='y = 0.25')

color='tab:red'
#plt.plot([0.2,1,1.8], [6.134320605892382e-10, 1.285289731933245e-13, 1.902743972652757e-14], '-o', color=color, label='y = 0')

plt.xlabel('x')
plt.yscale('log')
plt.ylabel('D(300K) (cm^2/s)')
plt.grid()
plt.legend(frameon=False)
plt.tight_layout()

#plt.savefig('Activation energy.png')

In [ ]:
plt.figure(figsize=(singlecol*2, singlecol*2))
diff_400_Na_3[i][p1:p2].mean()
x = [0.25,1,1.75]

color='tab:blue'
plt.plot([0.25, 1, 1.75], 
         [diff_400_Na_3[0][p1:p2].mean(), diff_400_Na_3[1][p1:p2].mean(), diff_400_Na_3[2][p1:p2].mean()],
         '-o', color=color, label='y = 1.75')

color='tab:green'
plt.plot([0.25, 1, 1.75], 
         [diff_400_Na_2[0][p1:p2].mean(), diff_400_Na_2[1][p1:p2].mean(), diff_400_Na_2[2][p1:p2].mean()],
         '-o', color=color, label='y = 1')

color='tab:orange'
plt.plot([0.25, 1, 1.75], 
         [diff_400_Na_1[0][p1:p2].mean(), diff_400_Na_1[1][p1:p2].mean(), diff_400_Na_1[2][p1:p2].mean()],
         '-o', color=color, label='y = 0.25')

color='tab:red'
#plt.plot([0.2,1,1.8], [6.134320605892382e-10, 1.285289731933245e-13, 1.902743972652757e-14], '-o', color=color, label='y = 0')

plt.xlabel('x')
plt.yscale('log')
plt.ylabel('D(300K) (cm^2/s)')
plt.grid()
plt.legend(frameon=False)
plt.tight_layout()

#plt.savefig('Activation energy.png')

In [ ]:
plt.figure(figsize=(singlecol*2, singlecol*2))
x = [0.25,1,1.75]

color='tab:blue'
plt.plot([0.25, 1, 1.75], 
         [diff_350_Na_3[0][p1:p2].mean(), diff_350_Na_3[1][p1:p2].mean(), diff_350_Na_3[2][p1:p2].mean()],
         '-o', color=color, label='y = 1.75')

color='tab:green'
plt.plot([0.25, 1, 1.75], 
         [diff_350_Na_2[0][p1:p2].mean(), diff_350_Na_2[1][p1:p2].mean(), diff_350_Na_2[2][p1:p2].mean()],
         '-o', color=color, label='y = 1')

color='tab:orange'
plt.plot([0.25, 1.75], 
         [diff_350_Na_1[0][p1:p2].mean(), diff_350_Na_1[1][p1:p2].mean()],
         '-o', color=color, label='y = 0.25')

color='tab:red'
#plt.plot([0.2,1,1.8], [6.134320605892382e-10, 1.285289731933245e-13, 1.902743972652757e-14], '-o', color=color, label='y = 0')

plt.xlabel('x')
plt.yscale('log')
plt.ylabel('D(300K) (cm^2/s)')
plt.grid()
plt.legend(frameon=False)
plt.tight_layout()

#plt.savefig('Activation energy.png')

In [ ]:
plt.figure(figsize=(singlecol*2, singlecol*2))
x = [0.25,1,1.75]

color='tab:blue'
plt.plot([0.25, 1, 1.75], 
         [diff_300_Na_3[0][p1:p2].mean(), diff_300_Na_3[1][p1:p2].mean(), diff_300_Na_3[2][p1:p2].mean()],
         '-o', color=color, label='y = 1.75')

color='tab:green'
plt.plot([0.25, 1, 1.75], 
         [diff_300_Na_2[0][p1:p2].mean(), diff_300_Na_2[1][p1:p2].mean(), diff_300_Na_2[2][p1:p2].mean()],
         '-o', color=color, label='y = 1')

color='tab:orange'
plt.plot([0.25, 1.75], 
         [diff_300_Na_1[0][p1:p2].mean(), diff_300_Na_1[1][p1:p2].mean()],
         '-o', color=color, label='y = 0.25')

color='tab:red'
#plt.plot([0.2,1,1.8], [6.134320605892382e-10, 1.285289731933245e-13, 1.902743972652757e-14], '-o', color=color, label='y = 0')

plt.xlabel('x')
plt.yscale('log')
plt.ylabel('D(300K) (cm^2/s)')
plt.grid()
plt.legend(frameon=False)
plt.tight_layout()

#plt.savefig('Activation energy.png')

In [ ]:
D_300_0, D_300_1, D_300_2

In [ ]:
plt.figure(figsize=(singlecol*2, singlecol*2))

x = [0.25,1,1.75]

color='tab:blue'
plt.plot([0.25, 1, 1.75], [D_300_6, D_300_7, D_300_8], '-o', color=color, label='y = 1.75')

color='tab:green'
plt.plot(x, [D_300_4, D_300_5, D_300_6], '-o',color=color, label='y = 1')

color='tab:orange'
plt.plot([0.25,1,1.75], [D_300_0, D_300_1, D_300_2], '-o',color=color, label='y = 0.25')

color='tab:red'
plt.plot([0.195,0.559,1.758], [5.854495981601016e-14, 7.721160349884454e-20, 8.440084701043811e-19], '-o', color=color, label='y = 0')

plt.xlabel('x')
plt.yscale('log')
plt.ylabel(r"Na$^+$ 300 K diffusivity cm$^2$/s)")
plt.grid()
plt.legend(frameon=False)
plt.tight_layout()

plt.savefig('300K_diffusivity.png')

In [ ]:
4.25E-08,	4.05E-10, 1.18E-08,	2.20E-08, 3.76E-08,	4.65E-08, 2.71E-08,	5.00E-09
6.134320605892382e-10, 1.285289731933245e-13, 1.902743972652757e-14

In [ ]:
element_map = {
    1: "C",
    2: "Mn", 
    3: "Fe",
    4: "N",
    5: "Na",
    6: "O",
    7: "H"
}
temp=700
for i in [1,2,3,4,5,6,7,8,9]:
    input_file = f'./{temp}_diffusion/dump.{temp}_diffuse_{i}.trj'
    output_file = f'./{temp}_diffusion/new_dump.{temp}_diffuse_{i}.trj'
# Convert the file
    convert_dump_file(input_file, output_file, element_map)
    print(f"Conversion complete. Modified file saved as {output_file}")

In [ ]:
element_map = {
    1: "C",
    2: "Mn", 
    3: "Fe",
    4: "N",
    5: "Na",
    6: "O",
    7: "H"
}
temp=800
for i in [1,2,3,4,5,6,7,8,9]:
    input_file = f'./{temp}_diffusion/dump.{temp}_diffuse_{i}.trj'
    output_file = f'./{temp}_diffusion/new_dump.{temp}_diffuse_{i}.trj'
# Convert the file
    convert_dump_file(input_file, output_file, element_map)
    print(f"Conversion complete. Modified file saved as {output_file}")

# Correlation Analysis

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.spatial.distance import cdist
import warnings
warnings.filterwarnings('ignore')
plt.rcParams.update({'font.size': 8})
plt.rcParams.update({'lines.linewidth':1.5})
plt.rcParams.update({'axes.linewidth':1})
plt.rcParams.update({'lines.markersize':3})
plt.rcParams.update({'savefig.dpi':300})
plt.rcParams.update({'legend.fontsize': 8})
class MDTrajectoryAnalyzer:
    def __init__(self, trajectory_file):
        """
        Initialize the analyzer with trajectory file
        
        Parameters:
        trajectory_file (str): Path to LAMMPS trajectory file
        """
        self.trajectory_file = trajectory_file
        self.trajectory_data = []
        self.timesteps = []
        
    def load_trajectory(self):
        """
        Load LAMMPS trajectory file
        Expected format: timestep, then atoms with id type x y z
        """
        print("Loading trajectory file...")
        
        with open(self.trajectory_file, 'r') as f:
            lines = f.readlines()
        
        i = 0
        while i < len(lines):
            line = lines[i].strip()
            
            # Look for timestep information
            if 'ITEM: TIMESTEP' in line:
                i += 1
                timestep = int(lines[i].strip())
                self.timesteps.append(timestep)
                
                # Skip to atoms section
                while i < len(lines) and 'ITEM: ATOMS' not in lines[i]:
                    i += 1
                
                if i >= len(lines):
                    break
                    
                i += 1  # Skip the ATOMS header line
                
                # Read atom data for this timestep
                frame_data = []
                while i < len(lines) and not lines[i].strip().startswith('ITEM:'):
                    if lines[i].strip():  # Skip empty lines
                        parts = lines[i].strip().split()
                        if len(parts) >= 5:  # id type x y z
                            atom_id = int(parts[0])
                            atom_type = int(parts[1])
                            x, y, z = float(parts[2]), float(parts[3]), float(parts[4])
                            frame_data.append([atom_id, atom_type, x, y, z])
                    i += 1
                
                if frame_data:
                    self.trajectory_data.append(np.array(frame_data))
            else:
                i += 1
        
        print(f"Loaded {len(self.trajectory_data)} frames with {len(self.timesteps)} timesteps")
        
    def extract_atom_trajectory(self, target_atom_id):
        """
        Extract position trajectory for a specific atom
        
        Parameters:
        target_atom_id (int): ID of the atom to track
        
        Returns:
        numpy.ndarray: Array of positions [timestep, x, y, z]
        """
        positions = []
        valid_timesteps = []
        
        for i, frame in enumerate(self.trajectory_data):
            # Find the target atom in this frame
            atom_mask = frame[:, 0] == target_atom_id
            if np.any(atom_mask):
                atom_data = frame[atom_mask][0]
                positions.append([self.timesteps[i], atom_data[2], atom_data[3], atom_data[4]])
                valid_timesteps.append(i)
        
        if not positions:
            raise ValueError(f"Atom {target_atom_id} not found in trajectory")
            
        return np.array(positions), valid_timesteps
    
    def detect_diffusion_jumps(self, positions, jump_threshold=1.1):
        """
        Detect diffusion jumps and compute jump distances
        
        Parameters:
        positions (numpy.ndarray): Position array [timestep, x, y, z]
        jump_threshold (float): Minimum displacement to consider as jump (Angstrom)
        
        Returns:
        tuple: (jump_timesteps, jump_distances, all_displacements)
        """
        if len(positions) < 2:
            return [], [], []
        
        # Calculate displacements between consecutive timesteps
        coords = positions[:, 1:4]  # x, y, z coordinates
        displacements = np.sqrt(np.sum(np.diff(coords, axis=0)**2, axis=1))
        
        # Identify jumps
        jump_mask = displacements > jump_threshold
        jump_indices = np.where(jump_mask)[0] + 1  # +1 because diff shifts indices
        jump_timesteps = positions[jump_indices, 0] if len(jump_indices) > 0 else []
        jump_distances = displacements[jump_mask] if len(jump_indices) > 0 else []
        
        return jump_timesteps, jump_distances, displacements
    
    def compute_msd(self, positions):
        """
        Compute Mean Square Displacement
        
        Parameters:
        positions (numpy.ndarray): Position array [timestep, x, y, z]
        
        Returns:
        tuple: (time_points, msd_values)
        """
        coords = positions[:, 1:4]  # x, y, z coordinates
        time_points = positions[:, 0]  # timesteps
        
        # Reference position (first timestep)
        ref_pos = coords[0]
        
        # Calculate MSD
        msd = np.sum(np.sqrt((coords - ref_pos)**2), axis=1)
        
        return time_points, msd
    
    def compute_total_jump_distance(self, positions):
        """
        Compute cumulative jump distance over time
        
        Parameters:
        positions (numpy.ndarray): Position array [timestep, x, y, z]
        
        Returns:
        tuple: (time_points, cumulative_distances)
        """
        if len(positions) < 2:
            return positions[:, 0], np.zeros(len(positions))
        
        coords = positions[:, 1:4]
        time_points = positions[:, 0]
        
        # Calculate step-wise distances
        step_distances = np.sqrt(np.sum(np.diff(coords, axis=0)**2, axis=1))
        
        # Cumulative sum (prepend 0 for first timestep)
        cumulative_distances = np.concatenate([[0], np.cumsum(step_distances)])
        
        return time_points, cumulative_distances
    
    def find_neighboring_atoms(self, target_atom_id, valid_timestep_indices, radius_type5=4.5, radius_type6=2.8):
        """
        Find neighboring type 5 and type 6 atoms within specified radius
        
        Parameters:
        target_atom_id (int): ID of the target atom
        valid_timestep_indices (list): Indices of frames where target atom exists
        radius (float): Search radius in Angstrom
        
        Returns:
        tuple: (type5_neighbors, type6_neighbors, type6_timesteps)
        """
        type5_neighbors = []
        type6_neighbors = []
        type5_timesteps = []
        type6_timesteps = []
        
        for i in valid_timestep_indices:
            frame = self.trajectory_data[i]
            timestep = self.timesteps[i]
            
            # Find target atom position
            target_mask = frame[:, 0] == target_atom_id
            if not np.any(target_mask):
                continue
                
            target_pos = frame[target_mask][0, 2:5]  # x, y, z
            
            # Calculate distances to all other atoms
            other_atoms = frame[frame[:, 0] != target_atom_id]
            if len(other_atoms) == 0:
                type5_neighbors.append(0)
                type6_neighbors.append(0)
                continue
            
            distances = cdist([target_pos], other_atoms[:, 2:5])[0]
            
            # Count neighbors within radius
            neighbors_mask_type5 = distances <= radius_type5
            neighbor_atoms_type5 = other_atoms[neighbors_mask_type5]

            neighbors_mask_type6 = distances <= radius_type6
            neighbor_atoms_type6 = other_atoms[neighbors_mask_type6]
            
            type5_count = np.sum(neighbor_atoms_type5[:, 1] == 5)
            type6_count = np.sum(neighbor_atoms_type6[:, 1] == 2)
            
            type5_neighbors.append(type5_count)
            type6_neighbors.append(type6_count)
            
            # Mark timesteps with type 6 neighbors
            if type5_count > 0:
                type5_timesteps.append(timestep)

            if type6_count > 0:
                type6_timesteps.append(timestep)
        
        return type5_neighbors, type5_timesteps, type6_neighbors, type6_timesteps
    
    def analyze_atom(self, target_atom_id, jump_threshold=1.5, neighbor_radius_1=4.5, neighbor_radius_2=2.8):
        """
        Complete analysis for a specific atom
        
        Parameters:
        target_atom_id (int): ID of the atom to analyze
        jump_threshold (float): Jump detection threshold in Angstrom
        neighbor_radius (float): Neighbor search radius in Angstrom
        
        Returns:
        dict: Complete analysis results
        """
        print(f"Analyzing atom {target_atom_id}...")
        
        # Extract atom trajectory
        positions, valid_indices = self.extract_atom_trajectory(target_atom_id)
        
        # Detect jumps
        jump_timesteps, jump_distances, all_displacements = self.detect_diffusion_jumps(
            positions, jump_threshold)

        if len(jump_distances) == 0 or np.max(jump_distances) > 15:
            print("Jump distance > 50. Skipping...")
            return None, None, None, None, None
        
        # Compute MSD
        time_points, msd = self.compute_msd(positions)
        
        # Compute total jump distance
        _, total_jump_dist = self.compute_total_jump_distance(positions)
        
        # Find neighbors
        type5_neighbors, type5_timesteps, type6_neighbors, type6_timesteps = self.find_neighboring_atoms(target_atom_id, valid_indices)
        
        results = {
            'positions': positions,
            'time_points': time_points,
            'msd': msd,
            'total_jump_distance': total_jump_dist,
            'jump_timesteps': jump_timesteps,
            'jump_distances': jump_distances,
            'all_displacements': all_displacements,
            'type5_neighbors': type5_neighbors,
            'type6_neighbors': type6_neighbors,
            'type5_timesteps': type5_timesteps,
            'type6_timesteps': type6_timesteps
        }
        
            
        ave_n5 = round(sum(results['type5_neighbors'])/len(results['type5_neighbors']),3)
        ave_n6 = round(sum(results['type6_neighbors'])/len(results['type6_neighbors']),3)
        final_msd = results['msd'][-1]
        final_distance = np.sum(results['jump_distances'])
        
        return results, ave_n5, ave_n6, final_msd, final_distance
    
    def plot_analysis(self, results, target_atom_id, temp, i):
        
        if results == None:
            print("Jump distance > 50. Skipping...")
            return None
             
        fig, ((ax1, ax2)) = plt.subplots(2, 1, figsize=(singlecol*2, singlecol*2), sharex=True)
        fig.suptitle(f'{temp}K_{i} Atom {target_atom_id}', fontsize=16)
        
        # Plot 1: MSD vs Time
        ax1.set_xlabel('Timestep')
        ax1.set_ylabel('MSD (Ų)')
        #ax1.set_title('Mean Square Displacement')
        ax1.grid(True, alpha=0.3)
        
        # Plot 2: Total Jump Distance vs Time
        correlation_factor = round(results['msd'][-1] / np.sum(results['jump_distances']), 3)
        line1, = ax1.plot(results['jump_timesteps'], np.cumsum(results['jump_distances']), 'r-', linewidth=2, alpha=0.7,
                 label=f'Total Jump Distance\nf = |Displacement| / Distance = {correlation_factor}')
        ax1.set_xlabel('Timestep')
        ax1.set_ylabel('Jump Distance (Å)')

        ax12=ax1.twinx()
        line2, = ax12.plot(results['time_points'], results['msd'], 'b-', linewidth=2, label='Displacement', alpha=0.7)
        
        ax12.set_ylabel('Displacement (Å)')
        
        handles1, labels1 = ax1.get_legend_handles_labels()
        handles2, labels2 = ax12.get_legend_handles_labels()
        ax1.legend(handles1 + handles2, labels1 + labels2, loc='best', frameon=False)
        
        # Plot 4: Neighbor count and Type 6 presence
        ave_n5 = round(sum(results['type5_neighbors'])/len(results['type5_neighbors']),3)
        line1 = ax2.scatter(results['time_points'], results['type5_neighbors'], color='purple', alpha=0.3, 
                    label=f'Na neighbors\nave={ave_n5}')
        
        ave_n6 = round(sum(results['type6_neighbors'])/len(results['type6_neighbors']),3)
        line2 = ax2.scatter(results['time_points'], results['type6_neighbors'], color='orange', alpha=0.3, 
                    label=f'H2O neighbors\nave={ave_n6}')

        ax22=ax2.twinx()


                # Plot 3: Displacement per step
        if len(results['all_displacements']) > 0:
            time_diffs = np.diff(results['time_points'])
            line3, = ax22.plot(results['time_points'][1:], results['all_displacements'], 'g-', alpha=0.4, label='Step Displacement')
            
            # Mark jump events
            if len(results['jump_timesteps']) > 0:
                jump_indices = []
                for jt in results['jump_timesteps']:
                    idx = np.where(results['time_points'] == jt)[0]
                    if len(idx) > 0:
                        jump_indices.append(idx[0] - 1)  # -1 because displacement is between steps
                
                if jump_indices:
                    line4 = ax22.scatter(results['time_points'][np.array(jump_indices)+1], 
                              results['all_displacements'][jump_indices], 
                              color='red', s=20, alpha=0.4, label='Jumps', zorder=5)
            
            line5 = ax22.axhline(y=1.5, color='red', linestyle='--', alpha=0.5, label='Jump Threshold')

        ax22.set_ylabel('Displacement (Å)')
        
        handles1, labels1 = ax2.get_legend_handles_labels()
        handles2, labels2 = ax22.get_legend_handles_labels()
        ax2.legend(handles1 + handles2, labels1 + labels2, loc='best', frameon=False)
        # Highlight timesteps with Type 6 neighbors
        #if results['type6_timesteps']:
        #    for ts in results['type6_timesteps']:
        #        ax4.axvline(x=ts, color='orange', alpha=0.3, linestyle='-', linewidth=1)
        
        ax2.set_xlabel('Timestep')
        ax2.set_ylabel('Number of Neighbors')
        
        fig, ax3 = plt.subplots(1, 1, figsize=(singlecol*2, singlecol*1), sharex=True)
        fig.suptitle(f'{temp}K_{i} Atom {target_atom_id}', fontsize=16)
        counts, edges, bars = ax3.hist(results['jump_distances'], edgecolor='black')
        ax3.set_xlabel('Jump Distance (Å)')
        ax3.set_ylabel('Count')
        #ax2.set_title('Neighboring Atoms (4 Å radius)')
        
        plt.tight_layout()

        return

        # Print summary statistics
        print("\n=== Analysis Summary ===")
        print(f"Total timesteps analyzed: {len(results['time_points'])}")
        print(f"Number of jumps detected: {len(results['jump_timesteps'])}")
        if len(results['jump_distances']) > 0:
            print(f"Average jump distance: {np.mean(results['jump_distances']):.2f} Å")
            print(f"Maximum jump distance: {np.max(results['jump_distances']):.2f} Å")
        print(f"Final displacement: {results['msd'][-1]:.2f} Ų")
        print(f"Total distance traveled: {np.sum(results['jump_distances'])} Å")
        print(f"Timesteps with H2O neighbors: {len(results['type6_timesteps'])}")
        print(f"Timesteps with Na+ neighbors: {len(results['type5_timesteps'])}")

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.spatial.distance import cdist
import warnings
warnings.filterwarnings('ignore')
plt.rcParams.update({'font.size': 8})
plt.rcParams.update({'lines.linewidth':1.5})
plt.rcParams.update({'axes.linewidth':1})
plt.rcParams.update({'lines.markersize':3})
plt.rcParams.update({'savefig.dpi':300})
plt.rcParams.update({'legend.fontsize': 8})
class MDTrajectoryAnalyzer_H:
    def __init__(self, trajectory_file):
        """
        Initialize the analyzer with trajectory file
        
        Parameters:
        trajectory_file (str): Path to LAMMPS trajectory file
        """
        self.trajectory_file = trajectory_file
        self.trajectory_data = []
        self.timesteps = []
        
    def load_trajectory(self):
        """
        Load LAMMPS trajectory file
        Expected format: timestep, then atoms with id type x y z
        """
        print("Loading trajectory file...")
        
        with open(self.trajectory_file, 'r') as f:
            lines = f.readlines()
        
        i = 0
        while i < len(lines):
            line = lines[i].strip()
            
            # Look for timestep information
            if 'ITEM: TIMESTEP' in line:
                i += 1
                timestep = int(lines[i].strip())
                self.timesteps.append(timestep)
                
                # Skip to atoms section
                while i < len(lines) and 'ITEM: ATOMS' not in lines[i]:
                    i += 1
                
                if i >= len(lines):
                    break
                    
                i += 1  # Skip the ATOMS header line
                
                # Read atom data for this timestep
                frame_data = []
                while i < len(lines) and not lines[i].strip().startswith('ITEM:'):
                    if lines[i].strip():  # Skip empty lines
                        parts = lines[i].strip().split()
                        if len(parts) >= 5:  # id type x y z
                            atom_id = int(parts[0])
                            atom_type = int(parts[1])
                            x, y, z = float(parts[2]), float(parts[3]), float(parts[4])
                            frame_data.append([atom_id, atom_type, x, y, z])
                    i += 1
                
                if frame_data:
                    self.trajectory_data.append(np.array(frame_data))
            else:
                i += 1
        
        print(f"Loaded {len(self.trajectory_data)} frames with {len(self.timesteps)} timesteps")
        
    def extract_atom_trajectory(self, target_atom_id):
        """
        Extract position trajectory for a specific atom
        
        Parameters:
        target_atom_id (int): ID of the atom to track
        
        Returns:
        numpy.ndarray: Array of positions [timestep, x, y, z]
        """
        positions = []
        valid_timesteps = []
        
        for i, frame in enumerate(self.trajectory_data):
            # Find the target atom in this frame
            atom_mask = frame[:, 0] == target_atom_id
            if np.any(atom_mask):
                atom_data = frame[atom_mask][0]
                positions.append([self.timesteps[i], atom_data[2], atom_data[3], atom_data[4]])
                valid_timesteps.append(i)
        
        if not positions:
            raise ValueError(f"Atom {target_atom_id} not found in trajectory")
            
        return np.array(positions), valid_timesteps
    
    def detect_diffusion_jumps(self, positions, jump_threshold=1.1):
        """
        Detect diffusion jumps and compute jump distances
        
        Parameters:
        positions (numpy.ndarray): Position array [timestep, x, y, z]
        jump_threshold (float): Minimum displacement to consider as jump (Angstrom)
        
        Returns:
        tuple: (jump_timesteps, jump_distances, all_displacements)
        """
        if len(positions) < 2:
            return [], [], []
        
        # Calculate displacements between consecutive timesteps
        coords = positions[:, 1:4]  # x, y, z coordinates
        displacements = np.sqrt(np.sum(np.diff(coords, axis=0)**2, axis=1))
        
        # Identify jumps
        jump_mask = displacements > jump_threshold
        jump_indices = np.where(jump_mask)[0] + 1  # +1 because diff shifts indices
        jump_timesteps = positions[jump_indices, 0] if len(jump_indices) > 0 else []
        jump_distances = displacements[jump_mask] if len(jump_indices) > 0 else []
        
        return jump_timesteps, jump_distances, displacements
    
    def compute_msd(self, positions):
        """
        Compute Mean Square Displacement
        
        Parameters:
        positions (numpy.ndarray): Position array [timestep, x, y, z]
        
        Returns:
        tuple: (time_points, msd_values)
        """
        coords = positions[:, 1:4]  # x, y, z coordinates
        time_points = positions[:, 0]  # timesteps
        
        # Reference position (first timestep)
        ref_pos = coords[0]
        
        # Calculate MSD
        msd = np.sum(np.sqrt((coords - ref_pos)**2), axis=1)
        
        return time_points, msd
    
    def compute_total_jump_distance(self, positions):
        """
        Compute cumulative jump distance over time
        
        Parameters:
        positions (numpy.ndarray): Position array [timestep, x, y, z]
        
        Returns:
        tuple: (time_points, cumulative_distances)
        """
        if len(positions) < 2:
            return positions[:, 0], np.zeros(len(positions))
        
        coords = positions[:, 1:4]
        time_points = positions[:, 0]
        
        # Calculate step-wise distances
        step_distances = np.sqrt(np.sum(np.diff(coords, axis=0)**2, axis=1))
        
        # Cumulative sum (prepend 0 for first timestep)
        cumulative_distances = np.concatenate([[0], np.cumsum(step_distances)])
        
        return time_points, cumulative_distances
    
    def find_neighboring_atoms(self, target_atom_id, valid_timestep_indices, radius_type5=4.5, radius_type6=2.8):
        """
        Find neighboring type 5 and type 6 atoms within specified radius
        
        Parameters:
        target_atom_id (int): ID of the target atom
        valid_timestep_indices (list): Indices of frames where target atom exists
        radius (float): Search radius in Angstrom
        
        Returns:
        tuple: (type5_neighbors, type6_neighbors, type6_timesteps)
        """
        type5_neighbors = []
        type6_neighbors = []
        type5_timesteps = []
        type6_timesteps = []
        
        for i in valid_timestep_indices:
            frame = self.trajectory_data[i]
            timestep = self.timesteps[i]
            
            # Find target atom position
            target_mask = frame[:, 0] == target_atom_id
            if not np.any(target_mask):
                continue
                
            target_pos = frame[target_mask][0, 2:5]  # x, y, z
            
            # Calculate distances to all other atoms
            other_atoms = frame[frame[:, 0] != target_atom_id]
            if len(other_atoms) == 0:
                type5_neighbors.append(0)
                type6_neighbors.append(0)
                continue
            
            distances = cdist([target_pos], other_atoms[:, 2:5])[0]
            
            # Count neighbors within radius
            neighbors_mask_type5 = distances <= radius_type5
            neighbor_atoms_type5 = other_atoms[neighbors_mask_type5]

            neighbors_mask_type6 = distances <= radius_type6
            neighbor_atoms_type6 = other_atoms[neighbors_mask_type6]
            
            type5_count = np.sum(neighbor_atoms_type5[:, 1] == 5)
            type6_count = np.sum(neighbor_atoms_type6[:, 1] == 6)
            
            type5_neighbors.append(type5_count)
            type6_neighbors.append(type6_count)
            
            # Mark timesteps with type 6 neighbors
            if type5_count > 0:
                type5_timesteps.append(timestep)

            if type6_count > 0:
                type6_timesteps.append(timestep)
        
        return type5_neighbors, type5_timesteps, type6_neighbors, type6_timesteps
    
    def analyze_atom(self, target_atom_id, jump_threshold=1.5, neighbor_radius_1=4.5, neighbor_radius_2=2.8):
        """
        Complete analysis for a specific atom
        
        Parameters:
        target_atom_id (int): ID of the atom to analyze
        jump_threshold (float): Jump detection threshold in Angstrom
        neighbor_radius (float): Neighbor search radius in Angstrom
        
        Returns:
        dict: Complete analysis results
        """
        print(f"Analyzing atom {target_atom_id}...")
        
        # Extract atom trajectory
        positions, valid_indices = self.extract_atom_trajectory(target_atom_id)
        
        # Detect jumps
        jump_timesteps, jump_distances, all_displacements = self.detect_diffusion_jumps(
            positions, jump_threshold)

        if len(jump_distances) == 0 or np.max(jump_distances) > 50:
            print("Jump distance > 50. Skipping...")
            return None, None, None, None, None
        
        # Compute MSD
        time_points, msd = self.compute_msd(positions)
        
        # Compute total jump distance
        _, total_jump_dist = self.compute_total_jump_distance(positions)
        
        # Find neighbors
        type5_neighbors, type5_timesteps, type6_neighbors, type6_timesteps = self.find_neighboring_atoms(target_atom_id, valid_indices)
        
        results = {
            'positions': positions,
            'time_points': time_points,
            'msd': msd,
            'total_jump_distance': total_jump_dist,
            'jump_timesteps': jump_timesteps,
            'jump_distances': jump_distances,
            'all_displacements': all_displacements,
            'type5_neighbors': type5_neighbors,
            'type6_neighbors': type6_neighbors,
            'type5_timesteps': type5_timesteps,
            'type6_timesteps': type6_timesteps
        }
        
            
        ave_n5 = round(sum(results['type5_neighbors'])/len(results['type5_neighbors']),3)
        ave_n6 = round(sum(results['type6_neighbors'])/len(results['type6_neighbors']),3)
        final_msd = results['msd'][-1]
        final_distance = np.sum(results['jump_distances'])
        
        return results, ave_n5, ave_n6, final_msd, final_distance
    
    def plot_analysis(self, results, target_atom_id, temp, i):
        
        if results == None:
            print("Jump distance > 50. Skipping...")
            return None
             
        fig, ((ax1, ax2)) = plt.subplots(2, 1, figsize=(singlecol*2, singlecol*2), sharex=True)
        fig.suptitle(f'{temp}K_{i} Atom {target_atom_id}', fontsize=16)
        
        # Plot 1: MSD vs Time
        ax1.set_xlabel('Timestep')
        ax1.set_ylabel('MSD (Ų)')
        #ax1.set_title('Mean Square Displacement')
        ax1.grid(True, alpha=0.3)
        
        # Plot 2: Total Jump Distance vs Time
        correlation_factor = round(results['msd'][-1] / np.sum(results['jump_distances']), 3)
        line1, = ax1.plot(results['jump_timesteps'], np.cumsum(results['jump_distances']), 'r-', linewidth=2, alpha=0.7,
                 label=f'Total Jump Distance\nf = |Displacement| / Distance = {correlation_factor}')
        ax1.set_xlabel('Timestep')
        ax1.set_ylabel('Jump Distance (Å)')

        ax12=ax1.twinx()
        line2, = ax12.plot(results['time_points'], results['msd'], 'b-', linewidth=2, label='Displacement', alpha=0.7)
        
        ax12.set_ylabel('Displacement (Å)')
        
        handles1, labels1 = ax1.get_legend_handles_labels()
        handles2, labels2 = ax12.get_legend_handles_labels()
        ax1.legend(handles1 + handles2, labels1 + labels2, loc='best', frameon=False)
        
        # Plot 4: Neighbor count and Type 6 presence
        ave_n5 = round(sum(results['type5_neighbors'])/len(results['type5_neighbors']),3)
        line1 = ax2.scatter(results['time_points'], results['type5_neighbors'], color='purple', alpha=0.3, 
                    label=f'Na neighbors\nave={ave_n5}')
        
        ave_n6 = round(sum(results['type6_neighbors'])/len(results['type6_neighbors']),3)
        line2 = ax2.scatter(results['time_points'], results['type6_neighbors'], color='orange', alpha=0.3, 
                    label=f'H2O neighbors\nave={ave_n6}')

        ax22=ax2.twinx()


                # Plot 3: Displacement per step
        if len(results['all_displacements']) > 0:
            time_diffs = np.diff(results['time_points'])
            line3, = ax22.plot(results['time_points'][1:], results['all_displacements'], 'g-', alpha=0.4, label='Step Displacement')
            
            # Mark jump events
            if len(results['jump_timesteps']) > 0:
                jump_indices = []
                for jt in results['jump_timesteps']:
                    idx = np.where(results['time_points'] == jt)[0]
                    if len(idx) > 0:
                        jump_indices.append(idx[0] - 1)  # -1 because displacement is between steps
                
                if jump_indices:
                    line4 = ax22.scatter(results['time_points'][np.array(jump_indices)+1], 
                              results['all_displacements'][jump_indices], 
                              color='red', s=20, alpha=0.4, label='Jumps', zorder=5)
            
            line5 = ax22.axhline(y=1.5, color='red', linestyle='--', alpha=0.5, label='Jump Threshold')

        ax22.set_ylabel('Displacement (Å)')
        
        handles1, labels1 = ax2.get_legend_handles_labels()
        handles2, labels2 = ax22.get_legend_handles_labels()
        ax2.legend(handles1 + handles2, labels1 + labels2, loc='best', frameon=False)
        # Highlight timesteps with Type 6 neighbors
        #if results['type6_timesteps']:
        #    for ts in results['type6_timesteps']:
        #        ax4.axvline(x=ts, color='orange', alpha=0.3, linestyle='-', linewidth=1)
        
        ax2.set_xlabel('Timestep')
        ax2.set_ylabel('Number of Neighbors')
        
        fig, ax3 = plt.subplots(1, 1, figsize=(singlecol*2, singlecol*1), sharex=True)
        fig.suptitle(f'{temp}K_{i} Atom {target_atom_id}', fontsize=16)
        counts, edges, bars = ax3.hist(results['jump_distances'], edgecolor='black')
        ax3.set_xlabel('Jump Distance (Å)')
        ax3.set_ylabel('Count')
        #ax2.set_title('Neighboring Atoms (4 Å radius)')
        
        plt.tight_layout()

        return

        # Print summary statistics
        print("\n=== Analysis Summary ===")
        print(f"Total timesteps analyzed: {len(results['time_points'])}")
        print(f"Number of jumps detected: {len(results['jump_timesteps'])}")
        if len(results['jump_distances']) > 0:
            print(f"Average jump distance: {np.mean(results['jump_distances']):.2f} Å")
            print(f"Maximum jump distance: {np.max(results['jump_distances']):.2f} Å")
        print(f"Final displacement: {results['msd'][-1]:.2f} Ų")
        print(f"Total distance traveled: {np.sum(results['jump_distances'])} Å")
        print(f"Timesteps with H2O neighbors: {len(results['type6_timesteps'])}")
        print(f"Timesteps with Na+ neighbors: {len(results['type5_timesteps'])}")

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.spatial.distance import cdist
import warnings
warnings.filterwarnings('ignore')
plt.rcParams.update({'font.size': 8})
plt.rcParams.update({'lines.linewidth':1.5})
plt.rcParams.update({'axes.linewidth':1})
plt.rcParams.update({'lines.markersize':3})
plt.rcParams.update({'savefig.dpi':300})
plt.rcParams.update({'legend.fontsize': 8})

class MDTrajectoryAnalyzer_dehydrate:
    def __init__(self, trajectory_file):
        """
        Initialize the analyzer with trajectory file
        
        Parameters:
        trajectory_file (str): Path to LAMMPS trajectory file
        """
        self.trajectory_file = trajectory_file
        self.trajectory_data = []
        self.timesteps = []
        
    def load_trajectory(self):
        """
        Load LAMMPS trajectory file
        Expected format: timestep, then atoms with id type x y z
        """
        print("Loading trajectory file...")
        
        with open(self.trajectory_file, 'r') as f:
            lines = f.readlines()
        
        i = 0
        while i < len(lines):
            line = lines[i].strip()
            
            # Look for timestep information
            if 'ITEM: TIMESTEP' in line:
                i += 1
                timestep = int(lines[i].strip())
                self.timesteps.append(timestep)
                
                # Skip to atoms section
                while i < len(lines) and 'ITEM: ATOMS' not in lines[i]:
                    i += 1
                
                if i >= len(lines):
                    break
                    
                i += 1  # Skip the ATOMS header line
                
                # Read atom data for this timestep
                frame_data = []
                while i < len(lines) and not lines[i].strip().startswith('ITEM:'):
                    if lines[i].strip():  # Skip empty lines
                        parts = lines[i].strip().split()
                        if len(parts) >= 5:  # id type x y z
                            atom_id = int(parts[0])
                            atom_type = int(parts[1])
                            x, y, z = float(parts[2]), float(parts[3]), float(parts[4])
                            frame_data.append([atom_id, atom_type, x, y, z])
                    i += 1
                
                if frame_data:
                    self.trajectory_data.append(np.array(frame_data))
            else:
                i += 1
        
        print(f"Loaded {len(self.trajectory_data)} frames with {len(self.timesteps)} timesteps")
        
    def extract_atom_trajectory(self, target_atom_id):
        """
        Extract position trajectory for a specific atom
        
        Parameters:
        target_atom_id (int): ID of the atom to track
        
        Returns:
        numpy.ndarray: Array of positions [timestep, x, y, z]
        """
        positions = []
        valid_timesteps = []
        
        for i, frame in enumerate(self.trajectory_data):
            # Find the target atom in this frame
            atom_mask = frame[:, 0] == target_atom_id
            if np.any(atom_mask):
                atom_data = frame[atom_mask][0]
                positions.append([self.timesteps[i], atom_data[2], atom_data[3], atom_data[4]])
                valid_timesteps.append(i)
        
        if not positions:
            raise ValueError(f"Atom {target_atom_id} not found in trajectory")
            
        return np.array(positions), valid_timesteps
    
    def detect_diffusion_jumps(self, positions, jump_threshold=1.1):
        """
        Detect diffusion jumps and compute jump distances
        
        Parameters:
        positions (numpy.ndarray): Position array [timestep, x, y, z]
        jump_threshold (float): Minimum displacement to consider as jump (Angstrom)
        
        Returns:
        tuple: (jump_timesteps, jump_distances, all_displacements)
        """
        if len(positions) < 2:
            return [], [], []
        
        # Calculate displacements between consecutive timesteps
        coords = positions[:, 1:4]  # x, y, z coordinates
        displacements = np.sqrt(np.sum(np.diff(coords, axis=0)**2, axis=1))
        
        # Identify jumps
        jump_mask = displacements > jump_threshold
        jump_indices = np.where(jump_mask)[0] + 1  # +1 because diff shifts indices
        jump_timesteps = positions[jump_indices, 0] if len(jump_indices) > 0 else []
        jump_distances = displacements[jump_mask] if len(jump_indices) > 0 else []
        
        return jump_timesteps, jump_distances, displacements
    
    def compute_msd(self, positions):
        """
        Compute Mean Square Displacement
        
        Parameters:
        positions (numpy.ndarray): Position array [timestep, x, y, z]
        
        Returns:
        tuple: (time_points, msd_values)
        """
        coords = positions[:, 1:4]  # x, y, z coordinates
        time_points = positions[:, 0]  # timesteps
        
        # Reference position (first timestep)
        ref_pos = coords[0]
        
        # Calculate MSD
        msd = np.sum(np.sqrt((coords - ref_pos)**2), axis=1)
        
        return time_points, msd
    
    def compute_total_jump_distance(self, positions):
        """
        Compute cumulative jump distance over time
        
        Parameters:
        positions (numpy.ndarray): Position array [timestep, x, y, z]
        
        Returns:
        tuple: (time_points, cumulative_distances)
        """
        if len(positions) < 2:
            return positions[:, 0], np.zeros(len(positions))
        
        coords = positions[:, 1:4]
        time_points = positions[:, 0]
        
        # Calculate step-wise distances
        step_distances = np.sqrt(np.sum(np.diff(coords, axis=0)**2, axis=1))
        
        # Cumulative sum (prepend 0 for first timestep)
        cumulative_distances = np.concatenate([[0], np.cumsum(step_distances)])
        
        return time_points, cumulative_distances
    
    def find_neighboring_atoms(self, target_atom_id, valid_timestep_indices, radius_type5=4.5):
        """
        Find neighboring type 5 atoms within specified radius
        
        Parameters:
        target_atom_id (int): ID of the target atom
        valid_timestep_indices (list): Indices of frames where target atom exists
        radius_type5 (float): Search radius for type 5 atoms in Angstrom
        
        Returns:
        tuple: (type5_neighbors, type5_timesteps)
        """
        type5_neighbors = []
        type5_timesteps = []
        
        for i in valid_timestep_indices:
            frame = self.trajectory_data[i]
            timestep = self.timesteps[i]
            
            # Find target atom position
            target_mask = frame[:, 0] == target_atom_id
            if not np.any(target_mask):
                continue
                
            target_pos = frame[target_mask][0, 2:5]  # x, y, z
            
            # Calculate distances to all other atoms
            other_atoms = frame[frame[:, 0] != target_atom_id]
            if len(other_atoms) == 0:
                type5_neighbors.append(0)
                continue
            
            distances = cdist([target_pos], other_atoms[:, 2:5])[0]
            
            # Count type 5 neighbors within radius
            neighbors_mask_type5 = distances <= radius_type5
            neighbor_atoms_type5 = other_atoms[neighbors_mask_type5]
            
            type5_count = np.sum(neighbor_atoms_type5[:, 1] == 5)
            
            type5_neighbors.append(type5_count)
            
            # Mark timesteps with type 5 neighbors
            if type5_count > 0:
                type5_timesteps.append(timestep)
        
        return type5_neighbors, type5_timesteps
    
    def analyze_atom(self, target_atom_id, jump_threshold=3, neighbor_radius=4.5):
        """
        Complete analysis for a specific atom
        
        Parameters:
        target_atom_id (int): ID of the atom to analyze
        jump_threshold (float): Jump detection threshold in Angstrom
        neighbor_radius (float): Neighbor search radius in Angstrom
        
        Returns:
        dict: Complete analysis results
        """
        print(f"Analyzing atom {target_atom_id}...")
        
        # Extract atom trajectory
        positions, valid_indices = self.extract_atom_trajectory(target_atom_id)
        
        # Detect jumps
        jump_timesteps, jump_distances, all_displacements = self.detect_diffusion_jumps(
            positions, jump_threshold)

        if len(jump_distances) == 0 or np.max(jump_distances) > 50:
            print("Jump distance > 50. Skipping...")
            return None, None, None, None
        
        # Compute MSD
        time_points, msd = self.compute_msd(positions)
        
        # Compute total jump distance
        _, total_jump_dist = self.compute_total_jump_distance(positions)
        
        # Find type 5 neighbors
        type5_neighbors, type5_timesteps = self.find_neighboring_atoms(
            target_atom_id, valid_indices, radius_type5=neighbor_radius)
        
        results = {
            'positions': positions,
            'time_points': time_points,
            'msd': msd,
            'total_jump_distance': total_jump_dist,
            'jump_timesteps': jump_timesteps,
            'jump_distances': jump_distances,
            'all_displacements': all_displacements,
            'type5_neighbors': type5_neighbors,
            'type5_timesteps': type5_timesteps
        }
        
        ave_n5 = round(sum(results['type5_neighbors'])/len(results['type5_neighbors']),3)
        final_msd = results['msd'][-1]
        final_distance = np.sum(results['jump_distances'])
        
        return results, ave_n5, final_msd, final_distance
    
    def plot_analysis(self, results, target_atom_id, temp, i):
        
        if results == None:
            print("Jump distance > 50. Skipping...")
            return None
        
        # Assuming singlecol is defined elsewhere, default to 3.5 if not
        try:
            singlecol
        except NameError:
            singlecol = 3.5
             
        fig, ((ax1, ax2)) = plt.subplots(2, 1, figsize=(singlecol*2, singlecol*2), sharex=True)
        fig.suptitle(f'{temp}K_{i} Atom {target_atom_id}', fontsize=16)
        
        # Plot 1: MSD vs Time
        ax1.set_xlabel('Timestep')
        ax1.set_ylabel('MSD (Ų)')
        ax1.grid(True, alpha=0.3)
        
        # Plot 2: Total Jump Distance vs Time
        correlation_factor = round(results['msd'][-1] / np.sum(results['jump_distances']), 3)
        line1, = ax1.plot(results['jump_timesteps'], np.cumsum(results['jump_distances']), 'r-', linewidth=2, alpha=0.7,
                 label=f'Total Jump Distance\nf = |Displacement| / Distance = {correlation_factor}')
        ax1.set_xlabel('Timestep')
        ax1.set_ylabel('Jump Distance (Å)')

        ax12=ax1.twinx()
        line2, = ax12.plot(results['time_points'], results['msd'], 'b-', linewidth=2, label='Displacement', alpha=0.7)
        
        ax12.set_ylabel('Displacement (Å)')
        
        handles1, labels1 = ax1.get_legend_handles_labels()
        handles2, labels2 = ax12.get_legend_handles_labels()
        ax1.legend(handles1 + handles2, labels1 + labels2, loc='best', frameon=False)
        
        # Plot 2: Neighbor count
        ave_n5 = round(sum(results['type5_neighbors'])/len(results['type5_neighbors']),3)
        line1 = ax2.scatter(results['time_points'], results['type5_neighbors'], color='purple', alpha=0.3, 
                    label=f'Type 5 neighbors\nave={ave_n5}')

        ax22=ax2.twinx()

        # Plot 3: Displacement per step
        if len(results['all_displacements']) > 0:
            time_diffs = np.diff(results['time_points'])
            line3, = ax22.plot(results['time_points'][1:], results['all_displacements'], 'g-', alpha=0.4, label='Step Displacement')
            
            # Mark jump events
            if len(results['jump_timesteps']) > 0:
                jump_indices = []
                for jt in results['jump_timesteps']:
                    idx = np.where(results['time_points'] == jt)[0]
                    if len(idx) > 0:
                        jump_indices.append(idx[0] - 1)  # -1 because displacement is between steps
                
                if jump_indices:
                    line4 = ax22.scatter(results['time_points'][np.array(jump_indices)+1], 
                              results['all_displacements'][jump_indices], 
                              color='red', s=20, alpha=0.4, label='Jumps', zorder=5)
            
            line5 = ax22.axhline(y=3, color='red', linestyle='--', alpha=0.5, label='Jump Threshold')

        ax22.set_ylabel('Displacement (Å)')
        
        handles1, labels1 = ax2.get_legend_handles_labels()
        handles2, labels2 = ax22.get_legend_handles_labels()
        ax2.legend(handles1 + handles2, labels1 + labels2, loc='best', frameon=False)
        
        ax2.set_xlabel('Timestep')
        ax2.set_ylabel('Number of Neighbors')
        
        # Jump distance histogram
        fig, ax3 = plt.subplots(1, 1, figsize=(singlecol*2, singlecol*1), sharex=True)
        fig.suptitle(f'{temp}K_{i} Atom {target_atom_id}', fontsize=16)
        counts, edges, bars = ax3.hist(results['jump_distances'], edgecolor='black')
        ax3.set_xlabel('Jump Distance (Å)')
        ax3.set_ylabel('Count')
        
        plt.tight_layout()

        # Print summary statistics
        print("\n=== Analysis Summary ===")
        print(f"Total timesteps analyzed: {len(results['time_points'])}")
        print(f"Number of jumps detected: {len(results['jump_timesteps'])}")
        if len(results['jump_distances']) > 0:
            print(f"Average jump distance: {np.mean(results['jump_distances']):.2f} Å")
            print(f"Maximum jump distance: {np.max(results['jump_distances']):.2f} Å")
        print(f"Final displacement: {results['msd'][-1]:.2f} Ų")
        print(f"Total distance traveled: {np.sum(results['jump_distances'])} Å")
        print(f"Timesteps with Type 5 neighbors: {len(results['type5_timesteps'])}")
        
        return

In [ ]:
import numpy as np
import pandas as pd

def load_trj(filename):
    """
    Load LAMMPS-style .trj file assuming columns: id type x y z
    and separated by timestep blocks.
    """
    timesteps = []
    with open(filename) as f:
        lines = f.readlines()

    i = 0
    while i < len(lines):
        if "ITEM: TIMESTEP" in lines[i]:
            step = int(lines[i+1])
            natoms = int(lines[i+3])
            start = i+9
            block = lines[start:start+natoms]
            data = np.loadtxt(block, usecols=(0,1,2,3,4))
            df = pd.DataFrame(data, columns=['id','type','x','y','z'])
            df['timestep'] = step
            timesteps.append(df)
            i = start + natoms
        else:
            i += 1

    return pd.concat(timesteps, ignore_index=True)

In [ ]:
def downsample_trj(input_file, output_file, target_interval=4000, original_interval=1000, max_timestep=None):
    """
    Downsample LAMMPS trajectory file to a different timestep interval.
    
    Parameters:
    -----------
    input_file : str
        Path to input .trj file
    output_file : str
        Path to output .trj file
    target_interval : int
        Desired timestep interval in output (default: 4000)
    original_interval : int
        Timestep interval in input file (default: 1000)
    max_timestep : int or None
        Maximum timestep to include (exclusive). Frames >= max_timestep are skipped.
    """
    skip_factor = target_interval // original_interval
    
    with open(input_file) as f:
        lines = f.readlines()
    
    with open(output_file, 'w') as out:
        i = 0
        frame_count = 0
        
        while i < len(lines):
            if "ITEM: TIMESTEP" in lines[i]:
                step = int(lines[i+1])
                natoms = int(lines[i+3])
                
                # Check if we've exceeded max_timestep
                if max_timestep is not None and step >= max_timestep:
                    break
                
                # Only write frames at multiples of target_interval
                if step % target_interval == 0:
                    # Write the entire frame (header + atoms)
                    frame_end = i + 9 + natoms
                    out.writelines(lines[i:frame_end])
                    frame_count += 1
                
                i = i + 9 + natoms
            else:
                i += 1
    
    print(f"Wrote {frame_count} frames to {output_file}")

In [ ]:
temp, i = 400, 9
input_file = f'./{temp}_diffusion_4/dump.{temp}_diffuse_{i}.trj'
output_file = f'./{temp}_diffusion_4/dump.{temp}_diffuse_{i}_4000.trj'

# Usage
downsample_trj(input_file, output_file, target_interval=4000, original_interval=1000, max_timestep=2000000)

## 500K

### 500K_3

In [ ]:
temp, i = 500, 3
df = load_trj(f'./{temp}_diffusion_/dump.{temp}_diffuse_{i}.trj')
Na_atom = df[(df['type'] == 5.0) & (df['timestep'] == 0)]
Na_ids = Na_atom['id'].to_numpy()
ints = np.random.randint(0, 1511, size=500)
Na_atom.iloc[ints]
Na_id_ls = Na_ids[ints].astype(int)

In [ ]:
# Usage example:
temp,i=500,3
ave_n5_ls_500_3=[]
ave_n6_ls_500_3=[]
msd_ls_500_3=[]
distance_ls_500_3=[]

if __name__ == "__main__":
    # Initialize analyzer
    analyzer = MDTrajectoryAnalyzer(f'./{temp}_diffusion_3/dump.{temp}_diffuse_{i}.trj')  # Replace with your file
    analyzer.load_trajectory()

    for i in Na_id_ls:
        target_atom_id = i
        results, ave_n5, ave_n6, msd, distance = analyzer.analyze_atom(target_atom_id)
        #ave_n5, ave_n6, msd, distance = analyzer.plot_analysis(results, target_atom_id)
        ave_n5_ls_500_3.append(ave_n5)
        ave_n6_ls_500_3.append(ave_n6)
        msd_ls_500_3.append(msd)
        distance_ls_500_3.append(distance)

In [ ]:
plt.figure()
distance_ls_500_3 = [x for x in distance_ls_500_3 if x is not None]
msd_ls_500_3 = [x for x in msd_ls_500_3 if x is not None]
corr_matrix = np.corrcoef(distance_ls_500_3, msd_ls_500_3)
corr = round(corr_matrix[0, 1],3)
f = np.array(msd_ls_500_3)/np.array(distance_ls_500_3)
plt.scatter(distance_ls_500_3, msd_ls_500_3, label=f'Pearson correlation:{corr}\nf = {round(f.mean(),3)}')
plt.legend()

In [ ]:
%matplotlib inline
fig, axes = plt.subplots(2, 2, figsize=(singlecol*2, singlecol*2))

# axes is a 2x2 array, so access with [row, col]
distance_ls_500_3 = [x for x in distance_ls_500_3 if x is not None]
ave_n5_ls_500_3 = [x for x in ave_n5_ls_500_3 if x is not None]
ave_n6_ls_500_3= [x for x in ave_n6_ls_500_3 if x is not None]
msd_ls_500_3 = [x for x in msd_ls_500_3 if x is not None]

corr_matrix = np.corrcoef(distance_ls_500_3, ave_n5_ls_500_3)
corr = round(corr_matrix[0, 1],3)
axes[0, 0].scatter(distance_ls_500_3, ave_n5_ls_500_3, label=f'Pearson correlation:{corr}')
axes[0, 0].set_title('distance vs ave_n5')
axes[0, 0].legend()

corr_matrix = np.corrcoef(distance_ls_500_3, ave_n6_ls_500_3)
corr = round(corr_matrix[0, 1],3)
axes[0, 1].scatter(distance_ls_500_3, ave_n6_ls_500_3, label=f'Pearson correlation:{corr}')
axes[0, 1].set_title('distance vs ave_n6')
axes[0, 1].legend()

corr_matrix = np.corrcoef(msd_ls_500_3, ave_n5_ls_500_3)
corr = round(corr_matrix[0, 1],3)
axes[1, 0].scatter(msd_ls_500_3, ave_n5_ls_500_3, label=f'Pearson correlation:{corr}')
axes[1, 0].set_title('displacement vs ave_n5')
axes[1, 0].legend()

corr_matrix = np.corrcoef(msd_ls_500_3, ave_n6_ls_500_3)
corr = round(corr_matrix[0, 1],3)
axes[1, 1].scatter(msd_ls_500_3, ave_n6_ls_500_3, label=f'Pearson correlation:{corr}')
axes[1, 1].set_title('displacement vs ave_n6')
axes[1, 1].legend()

### 500K_4

In [ ]:
temp, i = 500, 4
df = load_trj(f'./{temp}_diffusion_3/dump.{temp}_diffuse_{i}.trj')
Na_atom = df[(df['type'] == 5.0) & (df['timestep'] == 0)]
Na_ids = Na_atom['id'].to_numpy()
ints = np.random.randint(0, 200, size=150)
Na_atom.iloc[ints]
Na_id_ls = Na_ids[ints].astype(int)

In [ ]:
# Usage example:
temp,i=500,4
ave_n5_ls_500_4=[]
ave_n6_ls_500_4=[]
msd_ls_500_4=[]
distance_ls_500_4=[]

if __name__ == "__main__":
    # Initialize analyzer
    analyzer = MDTrajectoryAnalyzer(f'./{temp}_diffusion_3/dump.{temp}_diffuse_{i}.trj')  # Replace with your file
    analyzer.load_trajectory()

    for i in Na_id_ls:
        target_atom_id = i
        results, ave_n5, ave_n6, msd, distance = analyzer.analyze_atom(target_atom_id)
        #ave_n5, ave_n6, msd, distance = analyzer.plot_analysis(results, target_atom_id)
        ave_n5_ls_500_4.append(ave_n5)
        ave_n6_ls_500_4.append(ave_n6)
        msd_ls_500_4.append(msd)
        distance_ls_500_4.append(distance)

In [ ]:
plt.figure()
distance_ls_500_4 = [x for x in distance_ls_500_4 if x is not None]
msd_ls_500_4 = [x for x in msd_ls_500_4 if x is not None]
corr_matrix = np.corrcoef(distance_ls_500_4, msd_ls_500_4)
corr = round(corr_matrix[0, 1],3)
f = np.array(msd_ls_500_4)/np.array(distance_ls_500_4)
plt.scatter(distance_ls_500_4, msd_ls_500_4, label=f'Pearson correlation:{corr}\nf = {round(f.mean(),3)}')
plt.legend()


In [ ]:
%matplotlib inline
fig, axes = plt.subplots(2, 2, figsize=(singlecol*2, singlecol*2))

# axes is a 2x2 array, so access with [row, col]
distance_ls_500_4 = [x for x in distance_ls_500_4 if x is not None]
ave_n5_ls_500_4 = [x for x in ave_n5_ls_500_4 if x is not None]
ave_n6_ls_500_4= [x for x in ave_n6_ls_500_4 if x is not None]
msd_ls_500_4 = [x for x in msd_ls_500_4 if x is not None]

corr_matrix = np.corrcoef(distance_ls_500_4, ave_n5_ls_500_4)
corr = round(corr_matrix[0, 1],3)
axes[0, 0].scatter(distance_ls_500_4, ave_n5_ls_500_4, label=f'Pearson correlation:{corr}')
axes[0, 0].set_title('distance vs ave_n5')
axes[0, 0].legend()

corr_matrix = np.corrcoef(distance_ls_500_4, ave_n6_ls_500_4)
corr = round(corr_matrix[0, 1],3)
axes[0, 1].scatter(distance_ls_500_4, ave_n6_ls_500_4, label=f'Pearson correlation:{corr}')
axes[0, 1].set_title('distance vs ave_n6')
axes[0, 1].legend()

corr_matrix = np.corrcoef(msd_ls_500_4, ave_n5_ls_500_4)
corr = round(corr_matrix[0, 1],3)
axes[1, 0].scatter(msd_ls_500_4, ave_n5_ls_500_4, label=f'Pearson correlation:{corr}')
axes[1, 0].set_title('displacement vs ave_n5')
axes[1, 0].legend()

corr_matrix = np.corrcoef(msd_ls_500_4, ave_n6_ls_500_4)
corr = round(corr_matrix[0, 1],3)
axes[1, 1].scatter(msd_ls_500_4, ave_n6_ls_500_4, label=f'Pearson correlation:{corr}')
axes[1, 1].set_title('displacement vs ave_n6')
axes[1, 1].legend()

### 500K_5

In [ ]:
temp, i = 500, 5
df = load_trj(f'./{temp}_diffusion_3/dump.{temp}_diffuse_{i}.trj')
Na_atom = df[(df['type'] == 5.0) & (df['timestep'] == 0)]
Na_ids = Na_atom['id'].to_numpy()
ints = random.sample(range(0, 860), 500)
Na_atom.iloc[ints]
Na_id_ls = Na_ids[ints].astype(int)

In [ ]:
# Usage example:
temp,i=500,5
ave_n5_ls_500_5=[]
ave_n6_ls_500_5=[]
msd_ls_500_5=[]
distance_ls_500_5=[]

if __name__ == "__main__":
    # Initialize analyzer
    analyzer = MDTrajectoryAnalyzer(f'./{temp}_diffusion_3/dump.{temp}_diffuse_{i}.trj')  # Replace with your file
    analyzer.load_trajectory()

    for i in Na_id_ls:
        target_atom_id = i
        results, ave_n5, ave_n6, msd, distance = analyzer.analyze_atom(target_atom_id)
        #ave_n5, ave_n6, msd, distance = analyzer.plot_analysis(results, target_atom_id)
        ave_n5_ls_500_5.append(ave_n5)
        ave_n6_ls_500_5.append(ave_n6)
        msd_ls_500_5.append(msd)
        distance_ls_500_5.append(distance)

In [ ]:
plt.figure()
distance_ls_500_5 = [x for x in distance_ls_500_5 if x is not None]
msd_ls_500_5 = [x for x in msd_ls_500_5 if x is not None]
corr_matrix = np.corrcoef(distance_ls_500_5, msd_ls_500_5)
corr = round(corr_matrix[0, 1],3)
f = np.array(msd_ls_500_5)/np.array(distance_ls_500_5)
plt.scatter(distance_ls_500_5, msd_ls_500_5, label=f'Pearson correlation:{corr}\nf = {round(f.mean(),3)}')
plt.legend()


In [ ]:
np.array(msd_ls_500_5).mean(), np.array(distance_ls_500_5).mean()

In [ ]:
%matplotlib inline
fig, axes = plt.subplots(2, 2, figsize=(singlecol*2, singlecol*2))

# axes is a 2x2 array, so access with [row, col]
distance_ls_500_5 = [x for x in distance_ls_500_5 if x is not None]
ave_n5_ls_500_5 = [x for x in ave_n5_ls_500_5 if x is not None]
ave_n6_ls_500_5= [x for x in ave_n6_ls_500_5 if x is not None]
msd_ls_500_5 = [x for x in msd_ls_500_5 if x is not None]

corr_matrix = np.corrcoef(distance_ls_500_5, ave_n5_ls_500_5)
corr = round(corr_matrix[0, 1],3)
axes[0, 0].scatter(distance_ls_500_5, ave_n5_ls_500_5, label=f'Pearson correlation:{corr}')
axes[0, 0].set_title('distance vs ave_n5')
axes[0, 0].legend()

corr_matrix = np.corrcoef(distance_ls_500_5, ave_n6_ls_500_5)
corr = round(corr_matrix[0, 1],3)
axes[0, 1].scatter(distance_ls_500_5, ave_n6_ls_500_5, label=f'Pearson correlation:{corr}')
axes[0, 1].set_title('distance vs ave_n6')
axes[0, 1].legend()

corr_matrix = np.corrcoef(msd_ls_500_5, ave_n5_ls_500_5)
corr = round(corr_matrix[0, 1],3)
axes[1, 0].scatter(msd_ls_500_5, ave_n5_ls_500_5, label=f'Pearson correlation:{corr}')
axes[1, 0].set_title('displacement vs ave_n5')
axes[1, 0].legend()

corr_matrix = np.corrcoef(msd_ls_500_5, ave_n6_ls_500_5)
corr = round(corr_matrix[0, 1],3)
axes[1, 1].scatter(msd_ls_500_5, ave_n6_ls_500_5, label=f'Pearson correlation:{corr}')
axes[1, 1].set_title('displacement vs ave_n6')
axes[1, 1].legend()

### 500K_6

In [ ]:
temp, i = 500, 6
df = load_trj(f'./{temp}_diffusion_3/dump.{temp}_diffuse_{i}.trj')

In [ ]:
Na_atom = df[(df['type'] == 5.0) & (df['timestep'] == 0)]
Na_ids = Na_atom['id'].to_numpy()
ints = np.random.randint(0, 1511, size=500)
Na_atom.iloc[ints]
Na_id_ls = Na_ids[ints].astype(int)

In [ ]:
Na_atom.iloc[ints]

In [ ]:
# Usage example:
temp,i=500,6
ave_n5_ls_500_6=[]
ave_n6_ls_500_6=[]
msd_ls_500_6=[]
distance_ls_500_6=[]

if __name__ == "__main__":
    # Initialize analyzer
    analyzer = MDTrajectoryAnalyzer(f'./{temp}_diffusion_3/dump.{temp}_diffuse_{i}.trj')  # Replace with your file
    analyzer.load_trajectory()

    for i in Na_id_ls:
        target_atom_id = i
        results, ave_n5, ave_n6, msd, distance = analyzer.analyze_atom(target_atom_id)
        #ave_n5, ave_n6, msd, distance = analyzer.plot_analysis(results, target_atom_id)
        ave_n5_ls_500_6.append(ave_n5)
        ave_n6_ls_500_6.append(ave_n6)
        msd_ls_500_6.append(msd)
        distance_ls_500_6.append(distance)

In [ ]:
# Usage example:
temp,i=500,6

if __name__ == "__main__":
    # Initialize analyzer
    analyzer = MDTrajectoryAnalyzer(f'./{temp}_diffusion_3/dump.{temp}_diffuse_{i}.trj')  # Replace with your file
    analyzer.load_trajectory()

    for i in Na_id_ls:
        target_atom_id = i
        results, ave_n5, ave_n6, msd, distance = analyzer.analyze_atom(target_atom_id)
        analyzer.plot_analysis(results, target_atom_id)


In [ ]:
plt.figure()
distance_ls_500_6 = [x for x in distance_ls_500_6 if x is not None]
msd_ls_500_6 = [x for x in msd_ls_500_6 if x is not None]
corr_matrix = np.corrcoef(distance_ls_500_6, msd_ls_500_6)
corr = round(corr_matrix[0, 1],3)
f = np.array(msd_ls_500_6)/np.array(distance_ls_500_6)
plt.scatter(distance_ls_500_6, msd_ls_500_6, label=f'Pearson correlation:{corr}\nf = {round(f.mean(),3)}')
plt.legend()


In [ ]:
%matplotlib inline
fig, axes = plt.subplots(2, 2, figsize=(singlecol*2, singlecol*2))

# axes is a 2x2 array, so access with [row, col]
distance_ls_500_6 = [x for x in distance_ls_500_6 if x is not None]
ave_n5_ls_500_6 = [x for x in ave_n5_ls_500_6 if x is not None]
ave_n6_ls_500_6= [x for x in ave_n6_ls_500_6 if x is not None]
msd_ls_500_6 = [x for x in msd_ls_500_6 if x is not None]

corr_matrix = np.corrcoef(distance_ls_500_6, ave_n5_ls_500_6)
corr = round(corr_matrix[0, 1],3)
axes[0, 0].scatter(distance_ls_500_6, ave_n5_ls_500_6, label=f'Pearson correlation:{corr}')
axes[0, 0].set_title('distance vs ave_n5')
axes[0, 0].legend()

corr_matrix = np.corrcoef(distance_ls_500_6, ave_n6_ls_500_6)
corr = round(corr_matrix[0, 1],3)
axes[0, 1].scatter(distance_ls_500_6, ave_n6_ls_500_6, label=f'Pearson correlation:{corr}')
axes[0, 1].set_title('distance vs ave_n6')
axes[0, 1].legend()

corr_matrix = np.corrcoef(msd_ls_500_6, ave_n5_ls_500_6)
corr = round(corr_matrix[0, 1],3)
axes[1, 0].scatter(msd_ls_500_6, ave_n5_ls_500_6, label=f'Pearson correlation:{corr}')
axes[1, 0].set_title('displacement vs ave_n5')
axes[1, 0].legend()

corr_matrix = np.corrcoef(msd_ls_500_6, ave_n6_ls_500_6)
corr = round(corr_matrix[0, 1],3)
axes[1, 1].scatter(msd_ls_500_6, ave_n6_ls_500_6, label=f'Pearson correlation:{corr}')
axes[1, 1].set_title('displacement vs ave_n6')
axes[1, 1].legend()

### 500K_8

In [ ]:
temp, i = 500, 8
df = load_trj(f'./{temp}_diffusion_3/dump.{temp}_diffuse_{i}.trj')
Na_atom = df[(df['type'] == 5.0) & (df['timestep'] == 0)]
Na_ids = Na_atom['id'].to_numpy()
ints = np.random.randint(0, 866, size=500)
Na_atom.iloc[ints]
Na_id_ls = Na_ids[ints].astype(int)

In [ ]:
# Usage example:
temp,i=500,8
ave_n5_ls_500_8=[]
ave_n6_ls_500_8=[]
msd_ls_500_8=[]
distance_ls_500_8=[]

if __name__ == "__main__":
    # Initialize analyzer
    analyzer = MDTrajectoryAnalyzer(f'./{temp}_diffusion_3/dump.{temp}_diffuse_{i}.trj')  # Replace with your file
    analyzer.load_trajectory()

    for i in Na_id_ls:
        target_atom_id = i
        results, ave_n5, ave_n6, msd, distance = analyzer.analyze_atom(target_atom_id)
        #ave_n5, ave_n6, msd, distance = analyzer.plot_analysis(results, target_atom_id)
        ave_n5_ls_500_8.append(ave_n5)
        ave_n6_ls_500_8.append(ave_n6)
        msd_ls_500_8.append(msd)
        distance_ls_500_8.append(distance)

In [ ]:
plt.figure()
distance_ls_500_8 = [x for x in distance_ls_500_8 if x is not None]
msd_ls_500_8 = [x for x in msd_ls_500_8 if x is not None]
corr_matrix = np.corrcoef(distance_ls_500_8, msd_ls_500_8)
corr = round(corr_matrix[0, 1],3)
plt.scatter(distance_ls_500_8, msd_ls_500_8, label=f'Pearson correlation:{corr}')
plt.legend()


In [ ]:
f = np.array(msd_ls_500_8)/np.array(distance_ls_500_8)
f.mean()

In [ ]:
%matplotlib inline
fig, axes = plt.subplots(2, 2, figsize=(singlecol*2, singlecol*2))

# axes is a 2x2 array, so access with [row, col]
distance_ls_500_8 = [x for x in distance_ls_500_8 if x is not None]
ave_n5_ls_500_8 = [x for x in ave_n5_ls_500_8 if x is not None]
ave_n6_ls_500_8= [x for x in ave_n6_ls_500_8 if x is not None]
msd_ls_500_8 = [x for x in msd_ls_500_8 if x is not None]

corr_matrix = np.corrcoef(distance_ls_500_8, ave_n5_ls_500_8)
corr = round(corr_matrix[0, 1],3)
axes[0, 0].scatter(distance_ls_500_8, ave_n5_ls_500_8, label=f'Pearson correlation:{corr}')
axes[0, 0].set_title('distance vs ave_n5')
axes[0, 0].legend()

corr_matrix = np.corrcoef(distance_ls_500_8, ave_n6_ls_500_8)
corr = round(corr_matrix[0, 1],3)
axes[0, 1].scatter(distance_ls_500_8, ave_n6_ls_500_8, label=f'Pearson correlation:{corr}')
axes[0, 1].set_title('distance vs ave_n6')
axes[0, 1].legend()

corr_matrix = np.corrcoef(msd_ls_500_8, ave_n5_ls_500_8)
corr = round(corr_matrix[0, 1],3)
axes[1, 0].scatter(msd_ls_500_8, ave_n5_ls_500_8, label=f'Pearson correlation:{corr}')
axes[1, 0].set_title('displacement vs ave_n5')
axes[1, 0].legend()

corr_matrix = np.corrcoef(msd_ls_500_8, ave_n6_ls_500_8)
corr = round(corr_matrix[0, 1],3)
axes[1, 1].scatter(msd_ls_500_8, ave_n6_ls_500_8, label=f'Pearson correlation:{corr}')
axes[1, 1].set_title('displacement vs ave_n6')
axes[1, 1].legend()

### 500K_9

In [ ]:
temp, i = 500, 9
df = load_trj(f'./{temp}_diffusion_4/dump.{temp}_diffuse_{i}.trj')
Na_atom = df[(df['type'] == 5.0) & (df['timestep'] == 0)]
Na_ids = Na_atom['id'].to_numpy()
ints = np.random.randint(0, 1511, size=500)
Na_atom.iloc[ints]
Na_id_ls = Na_ids[ints].astype(int)

In [ ]:
# Usage example:
temp,i=500,9
ave_n5_ls_500_9=[]
ave_n6_ls_500_9=[]
msd_ls_500_9=[]
distance_ls_500_9=[]

if __name__ == "__main__":
    # Initialize analyzer
    analyzer = MDTrajectoryAnalyzer(f'./{temp}_diffusion_3/dump.{temp}_diffuse_{i}.trj')  # Replace with your file
    analyzer.load_trajectory()

    for i in Na_id_ls:
        target_atom_id = i
        results, ave_n5, ave_n6, msd, distance = analyzer.analyze_atom(target_atom_id)
        #ave_n5, ave_n6, msd, distance = analyzer.plot_analysis(results, target_atom_id)
        ave_n5_ls_500_9.append(ave_n5)
        ave_n6_ls_500_9.append(ave_n6)
        msd_ls_500_9.append(msd)
        distance_ls_500_9.append(distance)

In [ ]:
plt.figure()
distance_ls_500_9 = [x for x in distance_ls_500_9 if x is not None]
msd_ls_500_9 = [x for x in msd_ls_500_9 if x is not None]
corr_matrix = np.corrcoef(distance_ls_500_9, msd_ls_500_9)
corr = round(corr_matrix[0, 1],3)
plt.scatter(distance_ls_500_9, msd_ls_500_9, label=f'Pearson correlation:{corr}')
plt.legend()


In [ ]:
f = np.array(msd_ls_500_9)/np.array(distance_ls_500_9)
f.mean()

In [ ]:
%matplotlib inline
fig, axes = plt.subplots(2, 2, figsize=(singlecol*2, singlecol*2))

# axes is a 2x2 array, so access with [row, col]
distance_ls_500_9 = [x for x in distance_ls_500_9 if x is not None]
ave_n5_ls_500_9 = [x for x in ave_n5_ls_500_9 if x is not None]
ave_n6_ls_500_9= [x for x in ave_n6_ls_500_9 if x is not None]
msd_ls_500_9 = [x for x in msd_ls_500_9 if x is not None]

corr_matrix = np.corrcoef(distance_ls_500_9, ave_n5_ls_500_9)
corr = round(corr_matrix[0, 1],3)
axes[0, 0].scatter(distance_ls_500_9, ave_n5_ls_500_9, label=f'Pearson correlation:{corr}')
axes[0, 0].set_title('distance vs ave_n5')
axes[0, 0].legend()

corr_matrix = np.corrcoef(distance_ls_500_9, ave_n6_ls_500_9)
corr = round(corr_matrix[0, 1],3)
axes[0, 1].scatter(distance_ls_500_9, ave_n6_ls_500_9, label=f'Pearson correlation:{corr}')
axes[0, 1].set_title('distance vs ave_n6')
axes[0, 1].legend()

corr_matrix = np.corrcoef(msd_ls_500_9, ave_n5_ls_500_9)
corr = round(corr_matrix[0, 1],3)
axes[1, 0].scatter(msd_ls_500_9, ave_n5_ls_500_9, label=f'Pearson correlation:{corr}')
axes[1, 0].set_title('displacement vs ave_n5')
axes[1, 0].legend()

corr_matrix = np.corrcoef(msd_ls_500_9, ave_n6_ls_500_9)
corr = round(corr_matrix[0, 1],3)
axes[1, 1].scatter(msd_ls_500_9, ave_n6_ls_500_9, label=f'Pearson correlation:{corr}')
axes[1, 1].set_title('displacement vs ave_n6')
axes[1, 1].legend()

In [ ]:
# Usage example:
temp,i=500,9

if __name__ == "__main__":
    # Initialize analyzer
    #analyzer = MDTrajectoryAnalyzer(f'./{temp}_diffusion_4/dump.{temp}_diffuse_{i}.trj')  # Replace with your file
    #analyzer.load_trajectory()

    for i in Na_id_ls[11:15]:
        target_atom_id = i
        results, ave_n5, ave_n6, msd, distance = analyzer.analyze_atom(target_atom_id)
        analyzer.plot_analysis(results, target_atom_id, 500, i)


## 450K

### 450K_1

In [ ]:
%matplotlib widget

In [ ]:
import random
temp, i = 450, 1
df = load_trj(f'./{temp}_diffusion_3/dump.{temp}_diffuse_{i}.trj')
Na_atom = df[(df['type'] == 5.0) & (df['timestep'] == 0)]
Na_ids = Na_atom['id'].to_numpy()
ints = random.sample(range(1, 210), 150)
Na_atom.iloc[ints]
Na_id_ls = Na_ids[ints].astype(int)

In [ ]:
ave_n5_ls_450_1=[]
ave_n6_ls_450_1=[]
msd_ls_450_1=[]
distance_ls_450_1=[]

if __name__ == "__main__":
    # Initialize analyzer
    analyzer = MDTrajectoryAnalyzer(f'./{temp}_diffusion_3/dump.{temp}_diffuse_{i}.trj')  # Replace with your file
    analyzer.load_trajectory()

    for j in Na_id_ls:
        target_atom_id = j
        results, ave_n5, ave_n6, msd, distance = analyzer.analyze_atom(target_atom_id)
        #ave_n5, ave_n6, msd, distance = analyzer.plot_analysis(results, target_atom_id)
        ave_n5_ls_450_1.append(ave_n5)
        ave_n6_ls_450_1.append(ave_n6)
        msd_ls_450_1.append(msd)
        distance_ls_450_1.append(distance)

In [ ]:
if __name__ == "__main__":
    # Initialize analyzer
    analyzer = MDTrajectoryAnalyzer(f'./{temp}_diffusion_3/dump.{temp}_diffuse_{i}.trj')  # Replace with your file
    analyzer.load_trajectory()

    for j in Na_id_ls[:10]:
        target_atom_id = j
        results, ave_n5, ave_n6, msd, distance = analyzer.analyze_atom(target_atom_id)
        analyzer.plot_analysis(results, target_atom_id, 450, 1)

In [ ]:
np.array(msd_ls_450_1).mean(), np.array(distance_ls_450_1).mean()

In [ ]:
plt.figure(figsize=(singlecol, singlecol))
distance_ls_450_1 = [x for x in distance_ls_450_1 if x is not None]
msd_ls_450_1 = [x for x in msd_ls_450_1 if x is not None]
corr_matrix = np.corrcoef(distance_ls_450_1, msd_ls_450_1)
corr = round(corr_matrix[0, 1],3)
f = np.array(msd_ls_450_1)/np.array(distance_ls_450_1)
plt.scatter(distance_ls_450_1, msd_ls_450_1, label=f'Pearson correlation:{corr}\nf = {round(f.mean(),3)}')
plt.legend()

In [ ]:
%matplotlib inline
fig, axes = plt.subplots(2, 2, figsize=(singlecol*2, singlecol*2))

# axes is a 2x2 array, so access with [row, col]
distance_ls_450_1 = [x for x in distance_ls_450_1 if x is not None]
ave_n5_ls_450_1 = [x for x in ave_n5_ls_450_1 if x is not None]
ave_n6_ls_450_1= [x for x in ave_n6_ls_450_1 if x is not None]
msd_ls_450_1 = [x for x in msd_ls_450_1 if x is not None]

corr_matrix = np.corrcoef(distance_ls_450_1, ave_n5_ls_450_1)
corr = round(corr_matrix[0, 1],3)
axes[0, 0].scatter(distance_ls_450_1, ave_n5_ls_450_1, label=f'Pearson correlation:{corr}')
axes[0, 0].set_title('distance vs ave_n5')
axes[0, 0].legend()

corr_matrix = np.corrcoef(distance_ls_450_1, ave_n6_ls_450_1)
corr = round(corr_matrix[0, 1],3)
axes[0, 1].scatter(distance_ls_450_1, ave_n6_ls_450_1, label=f'Pearson correlation:{corr}')
axes[0, 1].set_title('distance vs ave_n6')
axes[0, 1].legend()

corr_matrix = np.corrcoef(msd_ls_450_1, ave_n5_ls_450_1)
corr = round(corr_matrix[0, 1],3)
axes[1, 0].scatter(msd_ls_450_1, ave_n5_ls_450_1, label=f'Pearson correlation:{corr}')
axes[1, 0].set_title('displacement vs ave_n5')
axes[1, 0].legend()

corr_matrix = np.corrcoef(msd_ls_450_1, ave_n6_ls_450_1)
corr = round(corr_matrix[0, 1],3)
axes[1, 1].scatter(msd_ls_450_1, ave_n6_ls_450_1, label=f'Pearson correlation:{corr}')
axes[1, 1].set_title('displacement vs ave_n6')
axes[1, 1].legend()

### 450K_2

In [ ]:
temp, i = 450,2
df = load_trj(f'./{temp}_diffusion_3/dump.{temp}_diffuse_{i}.trj')
Na_atom = df[(df['type'] == 5.0) & (df['timestep'] == 0)]
Na_ids = Na_atom['id'].to_numpy()
ints = random.sample(range(1, 800), 500)
Na_atom.iloc[ints]
Na_id_ls = Na_ids[ints].astype(int)

In [ ]:
ave_n5_ls_450_2=[]
ave_n6_ls_450_2=[]
msd_ls_450_2=[]
distance_ls_450_2=[]

if __name__ == "__main__":
    # Initialize analyzer
    analyzer = MDTrajectoryAnalyzer(f'./{temp}_diffusion_3/dump.{temp}_diffuse_{i}.trj')  # Replace with your file
    analyzer.load_trajectory()

    for j in Na_id_ls:
        target_atom_id = j
        results, ave_n5, ave_n6, msd, distance = analyzer.analyze_atom(target_atom_id)
        #ave_n5, ave_n6, msd, distance = analyzer.plot_analysis(results, target_atom_id)
        ave_n5_ls_450_2.append(ave_n5)
        ave_n6_ls_450_2.append(ave_n6)
        msd_ls_450_2.append(msd)
        distance_ls_450_2.append(distance)

In [ ]:
results

In [ ]:
results['jump_distances']

In [ ]:
if __name__ == "__main__":
    # Initialize analyzer
    analyzer = MDTrajectoryAnalyzer(f'./{temp}_diffusion_3/dump.{temp}_diffuse_{i}.trj')  # Replace with your file
    analyzer.load_trajectory()

    for i in Na_id_ls[:10]:
        target_atom_id = i
        results, ave_n5, ave_n6, msd, distance = analyzer.analyze_atom(target_atom_id)
        analyzer.plot_analysis(results, target_atom_id, 450, 2)

In [ ]:
plt.figure()
distance_ls_450_2 = [x for x in distance_ls_450_2 if x is not None]
msd_ls_450_2 = [x for x in msd_ls_450_2 if x is not None]
corr_matrix = np.corrcoef(distance_ls_450_2, msd_ls_450_2)
corr = round(corr_matrix[0, 1],3)
f = np.array(msd_ls_450_2)/np.array(distance_ls_450_2)
plt.scatter(distance_ls_450_2, msd_ls_450_2, label=f'Pearson correlation:{corr}\nf = {round(f.mean(),3)}')
plt.legend()

In [ ]:
np.array(msd_ls_450_2).mean(), np.array(distance_ls_450_2).mean()

In [ ]:
%matplotlib inline
fig, axes = plt.subplots(2, 2, figsize=(singlecol*2, singlecol*2))

# axes is a 2x2 array, so access with [row, col]
distance_ls_450_2 = [x for x in distance_ls_450_2 if x is not None]
ave_n5_ls_450_2 = [x for x in ave_n5_ls_450_2 if x is not None]
ave_n6_ls_450_2= [x for x in ave_n6_ls_450_2 if x is not None]
msd_ls_450_2 = [x for x in msd_ls_450_2 if x is not None]

corr_matrix = np.corrcoef(distance_ls_450_2, ave_n5_ls_450_2)
corr = round(corr_matrix[0, 1],3)
axes[0, 0].scatter(distance_ls_450_2, ave_n5_ls_450_2, label=f'Pearson correlation:{corr}')
axes[0, 0].set_title('distance vs ave_n5')
axes[0, 0].legend()

corr_matrix = np.corrcoef(distance_ls_450_2, ave_n6_ls_450_2)
corr = round(corr_matrix[0, 1],3)
axes[0, 1].scatter(distance_ls_450_2, ave_n6_ls_450_2, label=f'Pearson correlation:{corr}')
axes[0, 1].set_title('distance vs ave_n6')
axes[0, 1].legend()

corr_matrix = np.corrcoef(msd_ls_450_2, ave_n5_ls_450_2)
corr = round(corr_matrix[0, 1],3)
axes[1, 0].scatter(msd_ls_450_2, ave_n5_ls_450_2, label=f'Pearson correlation:{corr}')
axes[1, 0].set_title('displacement vs ave_n5')
axes[1, 0].legend()

corr_matrix = np.corrcoef(msd_ls_450_2, ave_n6_ls_450_2)
corr = round(corr_matrix[0, 1],3)
axes[1, 1].scatter(msd_ls_450_2, ave_n6_ls_450_2, label=f'Pearson correlation:{corr}')
axes[1, 1].set_title('displacement vs ave_n6')
axes[1, 1].legend()

### 450K_3

In [ ]:
temp, i = 450,3
df = load_trj(f'./{temp}_diffusion_3/dump.{temp}_diffuse_{i}.trj')
Na_atom = df[(df['type'] == 5.0) & (df['timestep'] == 0)]
Na_ids = Na_atom['id'].to_numpy()
ints = random.sample(range(1, 1500), 500)
Na_atom.iloc[ints]
Na_id_ls = Na_ids[ints].astype(int)

In [ ]:
ave_n5_ls_450_3=[]
ave_n6_ls_450_3=[]
msd_ls_450_3=[]
distance_ls_450_3=[]

if __name__ == "__main__":
    # Initialize analyzer
    analyzer = MDTrajectoryAnalyzer(f'./{temp}_diffusion_3/dump.{temp}_diffuse_{i}.trj')  # Replace with your file
    analyzer.load_trajectory()

    for i in Na_id_ls:
        target_atom_id = i
        results, ave_n5, ave_n6, msd, distance = analyzer.analyze_atom(target_atom_id)
        #ave_n5, ave_n6, msd, distance = analyzer.plot_analysis(results, target_atom_id)
        ave_n5_ls_450_3.append(ave_n5)
        ave_n6_ls_450_3.append(ave_n6)
        msd_ls_450_3.append(msd)
        distance_ls_450_3.append(distance)

In [ ]:
if __name__ == "__main__":
    # Initialize analyzer
    analyzer = MDTrajectoryAnalyzer(f'./{temp}_diffusion_3/dump.{temp}_diffuse_{i}.trj')  # Replace with your file
    analyzer.load_trajectory()

    for i in Na_id_ls[:10]:
        target_atom_id = i
        results, ave_n5, ave_n6, msd, distance = analyzer.analyze_atom(target_atom_id)
        analyzer.plot_analysis(results, target_atom_id, 450, 3)


In [ ]:
plt.figure()
distance_ls_450_3 = [x for x in distance_ls_450_3 if x is not None]
msd_ls_450_3 = [x for x in msd_ls_450_3 if x is not None]
corr_matrix = np.corrcoef(distance_ls_450_3, msd_ls_450_3)
corr = round(corr_matrix[0, 1],3)
f = np.array(msd_ls_450_3)/np.array(distance_ls_450_3)
plt.scatter(distance_ls_450_3, msd_ls_450_3, label=f'Pearson correlation:{corr}\nf = {round(f.mean(),3)}')
plt.legend()

In [ ]:
np.array(msd_ls_450_3).mean(), np.array(distance_ls_450_3).mean()

In [ ]:
%matplotlib inline
fig, axes = plt.subplots(2, 2, figsize=(singlecol*2, singlecol*2))

# axes is a 2x2 array, so access with [row, col]
distance_ls_450_3 = [x for x in distance_ls_450_3 if x is not None]
ave_n5_ls_450_3 = [x for x in ave_n5_ls_450_3 if x is not None]
ave_n6_ls_450_3= [x for x in ave_n6_ls_450_3 if x is not None]
msd_ls_450_3 = [x for x in msd_ls_450_3 if x is not None]

corr_matrix = np.corrcoef(distance_ls_450_3, ave_n5_ls_450_3)
corr = round(corr_matrix[0, 1],3)
axes[0, 0].scatter(distance_ls_450_3, ave_n5_ls_450_3, label=f'Pearson correlation:{corr}')
axes[0, 0].set_title('distance vs ave_n5')
axes[0, 0].legend()

corr_matrix = np.corrcoef(distance_ls_450_3, ave_n6_ls_450_3)
corr = round(corr_matrix[0, 1],3)
axes[0, 1].scatter(distance_ls_450_3, ave_n6_ls_450_3, label=f'Pearson correlation:{corr}')
axes[0, 1].set_title('distance vs ave_n6')
axes[0, 1].legend()

corr_matrix = np.corrcoef(msd_ls_450_3, ave_n5_ls_450_3)
corr = round(corr_matrix[0, 1],3)
axes[1, 0].scatter(msd_ls_450_3, ave_n5_ls_450_3, label=f'Pearson correlation:{corr}')
axes[1, 0].set_title('displacement vs ave_n5')
axes[1, 0].legend()

corr_matrix = np.corrcoef(msd_ls_450_3, ave_n6_ls_450_3)
corr = round(corr_matrix[0, 1],3)
axes[1, 1].scatter(msd_ls_450_3, ave_n6_ls_450_3, label=f'Pearson correlation:{corr}')
axes[1, 1].set_title('displacement vs ave_n6')
axes[1, 1].legend()

### 450K_4

In [ ]:
import random
temp, i = 450, 4
df = load_trj(f'./{temp}_diffusion_3/dump.{temp}_diffuse_{i}.trj')
Na_atom = df[(df['type'] == 5.0) & (df['timestep'] == 0)]
Na_ids = Na_atom['id'].to_numpy()
ints = random.sample(range(0, 200), 150)
Na_atom.iloc[ints]
Na_id_ls = Na_ids[ints].astype(int)

In [ ]:
if __name__ == "__main__":
    # Initialize analyzer
    analyzer = MDTrajectoryAnalyzer(f'./{temp}_diffusion_3/dump.{temp}_diffuse_{i}.trj')  # Replace with your file
    analyzer.load_trajectory()

    for i in Na_id_ls[:10]:
        target_atom_id = i
        results, ave_n5, ave_n6, msd, distance = analyzer.analyze_atom(target_atom_id)
        analyzer.plot_analysis(results, target_atom_id, 450, 4)


In [ ]:
ave_n5_ls_450_4=[]
ave_n6_ls_450_4=[]
msd_ls_450_4=[]
distance_ls_450_4=[]

if __name__ == "__main__":
    # Initialize analyzer
    analyzer = MDTrajectoryAnalyzer(f'./{temp}_diffusion_3/dump.{temp}_diffuse_{i}.trj')  # Replace with your file
    analyzer.load_trajectory()

    for i in Na_id_ls:
        target_atom_id = i
        results, ave_n5, ave_n6, msd, distance = analyzer.analyze_atom(target_atom_id)
        #ave_n5, ave_n6, msd, distance = analyzer.plot_analysis(results, target_atom_id)
        ave_n5_ls_450_4.append(ave_n5)
        ave_n6_ls_450_4.append(ave_n6)
        msd_ls_450_4.append(msd)
        distance_ls_450_4.append(distance)

In [ ]:
np.array(msd_ls_450_4).mean(), np.array(distance_ls_450_4).mean()

In [ ]:
plt.figure()
distance_ls_450_4 = [x for x in distance_ls_450_4 if x is not None]
msd_ls_450_4 = [x for x in msd_ls_450_4 if x is not None]
corr_matrix = np.corrcoef(distance_ls_450_4, msd_ls_450_4)
corr = round(corr_matrix[0, 1],3)
f = np.array(msd_ls_450_4)/np.array(distance_ls_450_4)
plt.scatter(distance_ls_450_4, msd_ls_450_4, label=f'Pearson correlation:{corr}\nf = {round(f.mean(),3)}')
plt.legend()

In [ ]:
%matplotlib inline
fig, axes = plt.subplots(2, 2, figsize=(singlecol*2, singlecol*2))

# axes is a 2x2 array, so access with [row, col]
distance_ls_450_4 = [x for x in distance_ls_450_4 if x is not None]
ave_n5_ls_450_4 = [x for x in ave_n5_ls_450_4 if x is not None]
ave_n6_ls_450_4= [x for x in ave_n6_ls_450_4 if x is not None]
msd_ls_450_4 = [x for x in msd_ls_450_4 if x is not None]

corr_matrix = np.corrcoef(distance_ls_450_4, ave_n5_ls_450_4)
corr = round(corr_matrix[0, 1],3)
axes[0, 0].scatter(distance_ls_450_4, ave_n5_ls_450_4, label=f'Pearson correlation:{corr}')
axes[0, 0].set_title('distance vs ave_n5')
axes[0, 0].legend()

corr_matrix = np.corrcoef(distance_ls_450_4, ave_n6_ls_450_4)
corr = round(corr_matrix[0, 1],3)
axes[0, 1].scatter(distance_ls_450_4, ave_n6_ls_450_4, label=f'Pearson correlation:{corr}')
axes[0, 1].set_title('distance vs ave_n6')
axes[0, 1].legend()

corr_matrix = np.corrcoef(msd_ls_450_4, ave_n5_ls_450_4)
corr = round(corr_matrix[0, 1],3)
axes[1, 0].scatter(msd_ls_450_4, ave_n5_ls_450_4, label=f'Pearson correlation:{corr}')
axes[1, 0].set_title('displacement vs ave_n5')
axes[1, 0].legend()

corr_matrix = np.corrcoef(msd_ls_450_4, ave_n6_ls_450_4)
corr = round(corr_matrix[0, 1],3)
axes[1, 1].scatter(msd_ls_450_4, ave_n6_ls_450_4, label=f'Pearson correlation:{corr}')
axes[1, 1].set_title('displacement vs ave_n6')
axes[1, 1].legend()

### 450K_5

In [ ]:
temp, i = 450, 5
df = load_trj(f'./{temp}_diffusion_3/dump.{temp}_diffuse_{i}.trj')
Na_atom = df[(df['type'] == 5.0) & (df['timestep'] == 0)]
Na_ids = Na_atom['id'].to_numpy()
ints = random.sample(range(1, 800), 500)
Na_atom.iloc[ints]
Na_id_ls = Na_ids[ints].astype(int)

In [ ]:
ave_n5_ls_450_5=[]
ave_n6_ls_450_5=[]
msd_ls_450_5=[]
distance_ls_450_5=[]

if __name__ == "__main__":
    # Initialize analyzer
    analyzer = MDTrajectoryAnalyzer(f'./{temp}_diffusion_3/dump.{temp}_diffuse_{i}.trj')  # Replace with your file
    analyzer.load_trajectory()

    for i in Na_id_ls:
        target_atom_id = i
        results, ave_n5, ave_n6, msd, distance = analyzer.analyze_atom(target_atom_id)
        #ave_n5, ave_n6, msd, distance = analyzer.plot_analysis(results, target_atom_id)
        ave_n5_ls_450_5.append(ave_n5)
        ave_n6_ls_450_5.append(ave_n6)
        msd_ls_450_5.append(msd)
        distance_ls_450_5.append(distance)

In [ ]:
if __name__ == "__main__":
    # Initialize analyzer
    analyzer = MDTrajectoryAnalyzer(f'./{temp}_diffusion_3/dump.{temp}_diffuse_{i}.trj')  # Replace with your file
    analyzer.load_trajectory()

    for i in Na_id_ls[:10]:
        target_atom_id = i
        results, ave_n5, ave_n6, msd, distance = analyzer.analyze_atom(target_atom_id)
        analyzer.plot_analysis(results, target_atom_id,450,5)


In [ ]:
plt.figure()
distance_ls_450_5 = [x for x in distance_ls_450_5 if x is not None]
msd_ls_450_5 = [x for x in msd_ls_450_5 if x is not None]
corr_matrix = np.corrcoef(distance_ls_450_5, msd_ls_450_5)
corr = round(corr_matrix[0, 1],3)
f = np.array(msd_ls_450_5)/np.array(distance_ls_450_5)
plt.scatter(distance_ls_450_5, msd_ls_450_5, label=f'Pearson correlation:{corr}\nf = {round(f.mean(),3)}')
plt.legend()

In [ ]:
np.array(msd_ls_450_5).mean(), np.array(distance_ls_450_5).mean()

In [ ]:
%matplotlib inline
fig, axes = plt.subplots(2, 2, figsize=(singlecol*2, singlecol*2))

# axes is a 2x2 array, so access with [row, col]
distance_ls_450_5 = [x for x in distance_ls_450_5 if x is not None]
ave_n5_ls_450_5 = [x for x in ave_n5_ls_450_5 if x is not None]
ave_n6_ls_450_5= [x for x in ave_n6_ls_450_5 if x is not None]
msd_ls_450_5 = [x for x in msd_ls_450_5 if x is not None]

corr_matrix = np.corrcoef(distance_ls_450_5, ave_n5_ls_450_5)
corr = round(corr_matrix[0, 1],3)
axes[0, 0].scatter(distance_ls_450_5, ave_n5_ls_450_5, label=f'Pearson correlation:{corr}')
axes[0, 0].set_title('distance vs ave_n5')
axes[0, 0].legend()

corr_matrix = np.corrcoef(distance_ls_450_5, ave_n6_ls_450_5)
corr = round(corr_matrix[0, 1],3)
axes[0, 1].scatter(distance_ls_450_5, ave_n6_ls_450_5, label=f'Pearson correlation:{corr}')
axes[0, 1].set_title('distance vs ave_n6')
axes[0, 1].legend()

corr_matrix = np.corrcoef(msd_ls_450_5, ave_n5_ls_450_5)
corr = round(corr_matrix[0, 1],3)
axes[1, 0].scatter(msd_ls_450_5, ave_n5_ls_450_5, label=f'Pearson correlation:{corr}')
axes[1, 0].set_title('displacement vs ave_n5')
axes[1, 0].legend()

corr_matrix = np.corrcoef(msd_ls_450_5, ave_n6_ls_450_5)
corr = round(corr_matrix[0, 1],3)
axes[1, 1].scatter(msd_ls_450_5, ave_n6_ls_450_5, label=f'Pearson correlation:{corr}')
axes[1, 1].set_title('displacement vs ave_n6')
axes[1, 1].legend()

### 450K_6

In [ ]:
temp, i = 450, 6
df = load_trj(f'./{temp}_diffusion_3/dump.{temp}_diffuse_{i}.trj')
Na_atom = df[(df['type'] == 5.0) & (df['timestep'] == 0)]
Na_ids = Na_atom['id'].to_numpy()
ints = random.sample(range(1, 1500), 500)
Na_atom.iloc[ints]
Na_id_ls = Na_ids[ints].astype(int)

In [ ]:
ave_n5_ls_450_6=[]
ave_n6_ls_450_6=[]
msd_ls_450_6=[]
distance_ls_450_6=[]

if __name__ == "__main__":
    # Initialize analyzer
    analyzer = MDTrajectoryAnalyzer(f'./{temp}_diffusion_3/dump.{temp}_diffuse_{i}.trj')  # Replace with your file
    analyzer.load_trajectory()

    for i in Na_id_ls:
        target_atom_id = i
        results, ave_n5, ave_n6, msd, distance = analyzer.analyze_atom(target_atom_id)
        #ave_n5, ave_n6, msd, distance = analyzer.plot_analysis(results, target_atom_id)
        ave_n5_ls_450_6.append(ave_n5)
        ave_n6_ls_450_6.append(ave_n6)
        msd_ls_450_6.append(msd)
        distance_ls_450_6.append(distance)

In [ ]:
if __name__ == "__main__":
    # Initialize analyzer
    analyzer = MDTrajectoryAnalyzer(f'./{temp}_diffusion_3/dump.{temp}_diffuse_{i}.trj')  # Replace with your file
    analyzer.load_trajectory()

    for i in Na_id_ls[:10]:
        target_atom_id = i
        results, ave_n5, ave_n6, msd, distance = analyzer.analyze_atom(target_atom_id)
        analyzer.plot_analysis(results, target_atom_id,450,6)


In [ ]:
np.array(msd_ls_450_6).mean(), np.array(distance_ls_450_6).mean()

In [ ]:
plt.figure()
distance_ls_450_6 = [x for x in distance_ls_450_6 if x is not None]
msd_ls_450_6 = [x for x in msd_ls_450_6 if x is not None]
corr_matrix = np.corrcoef(distance_ls_450_6, msd_ls_450_6)
corr = round(corr_matrix[0, 1],3)
f = np.array(msd_ls_450_6)/np.array(distance_ls_450_6)
plt.scatter(distance_ls_450_6, msd_ls_450_6, label=f'Pearson correlation:{corr}\nf = {round(f.mean(),3)}')
plt.legend()

In [ ]:
%matplotlib inline
fig, axes = plt.subplots(2, 2, figsize=(singlecol*2, singlecol*2))

# axes is a 2x2 array, so access with [row, col]
distance_ls_450_6 = [x for x in distance_ls_450_6 if x is not None]
ave_n5_ls_450_6 = [x for x in ave_n5_ls_450_6 if x is not None]
ave_n6_ls_450_6= [x for x in ave_n6_ls_450_6 if x is not None]
msd_ls_450_6 = [x for x in msd_ls_450_6 if x is not None]

corr_matrix = np.corrcoef(distance_ls_450_6, ave_n5_ls_450_6)
corr = round(corr_matrix[0, 1],3)
axes[0, 0].scatter(distance_ls_450_6, ave_n5_ls_450_6, label=f'Pearson correlation:{corr}')
axes[0, 0].set_title('distance vs ave_n5')
axes[0, 0].legend()

corr_matrix = np.corrcoef(distance_ls_450_6, ave_n6_ls_450_6)
corr = round(corr_matrix[0, 1],3)
axes[0, 1].scatter(distance_ls_450_6, ave_n6_ls_450_6, label=f'Pearson correlation:{corr}')
axes[0, 1].set_title('distance vs ave_n6')
axes[0, 1].legend()

corr_matrix = np.corrcoef(msd_ls_450_6, ave_n5_ls_450_6)
corr = round(corr_matrix[0, 1],3)
axes[1, 0].scatter(msd_ls_450_6, ave_n5_ls_450_6, label=f'Pearson correlation:{corr}')
axes[1, 0].set_title('displacement vs ave_n5')
axes[1, 0].legend()

corr_matrix = np.corrcoef(msd_ls_450_6, ave_n6_ls_450_6)
corr = round(corr_matrix[0, 1],3)
axes[1, 1].scatter(msd_ls_450_6, ave_n6_ls_450_6, label=f'Pearson correlation:{corr}')
axes[1, 1].set_title('displacement vs ave_n6')
axes[1, 1].legend()

### 450K_7

In [ ]:
temp, i = 450, 7
df = load_trj(f'./{temp}_diffusion_3/dump.{temp}_diffuse_{i}.trj')
Na_atom = df[(df['type'] == 5.0) & (df['timestep'] == 0)]
Na_ids = Na_atom['id'].to_numpy()
ints = random.sample(range(1, 200), 150)
Na_atom.iloc[ints]
Na_id_ls = Na_ids[ints].astype(int)

In [ ]:
ave_n5_ls_450_7=[]
ave_n6_ls_450_7=[]
msd_ls_450_7=[]
distance_ls_450_7=[]

if __name__ == "__main__":
    # Initialize analyzer
    analyzer = MDTrajectoryAnalyzer(f'./{temp}_diffusion_3/dump.{temp}_diffuse_{i}.trj')  # Replace with your file
    analyzer.load_trajectory()

    for i in Na_id_ls:
        target_atom_id = i
        results, ave_n5, ave_n6, msd, distance = analyzer.analyze_atom(target_atom_id)
        #ave_n5, ave_n6, msd, distance = analyzer.plot_analysis(results, target_atom_id)
        ave_n5_ls_450_7.append(ave_n5)
        ave_n6_ls_450_7.append(ave_n6)
        msd_ls_450_7.append(msd)
        distance_ls_450_7.append(distance)

In [ ]:
plt.figure()
distance_ls_450_7 = [x for x in distance_ls_450_7 if x is not None]
msd_ls_450_7 = [x for x in msd_ls_450_7 if x is not None]
corr_matrix = np.corrcoef(distance_ls_450_7, msd_ls_450_7)
corr = round(corr_matrix[0, 1],3)
f = np.array(msd_ls_450_7)/np.array(distance_ls_450_7)
plt.scatter(distance_ls_450_7, msd_ls_450_7, label=f'Pearson correlation:{corr}\nf = {round(f.mean(),3)}')
plt.legend()

In [ ]:
np.array(msd_ls_450_7).mean(), np.array(distance_ls_450_7).mean()

In [ ]:
%matplotlib inline
fig, axes = plt.subplots(2, 2, figsize=(singlecol*2, singlecol*2))

# axes is a 2x2 array, so access with [row, col]
distance_ls_450_7 = [x for x in distance_ls_450_7 if x is not None]
ave_n5_ls_450_7 = [x for x in ave_n5_ls_450_7 if x is not None]
ave_n6_ls_450_7= [x for x in ave_n6_ls_450_7 if x is not None]
msd_ls_450_7 = [x for x in msd_ls_450_7 if x is not None]

corr_matrix = np.corrcoef(distance_ls_450_7, ave_n5_ls_450_7)
corr = round(corr_matrix[0, 1],3)
axes[0, 0].scatter(distance_ls_450_7, ave_n5_ls_450_7, label=f'Pearson correlation:{corr}')
axes[0, 0].set_title('distance vs ave_n5')
axes[0, 0].legend()

corr_matrix = np.corrcoef(distance_ls_450_7, ave_n6_ls_450_7)
corr = round(corr_matrix[0, 1],3)
axes[0, 1].scatter(distance_ls_450_7, ave_n6_ls_450_7, label=f'Pearson correlation:{corr}')
axes[0, 1].set_title('distance vs ave_n6')
axes[0, 1].legend()

corr_matrix = np.corrcoef(msd_ls_450_7, ave_n5_ls_450_7)
corr = round(corr_matrix[0, 1],3)
axes[1, 0].scatter(msd_ls_450_7, ave_n5_ls_450_7, label=f'Pearson correlation:{corr}')
axes[1, 0].set_title('displacement vs ave_n5')
axes[1, 0].legend()

corr_matrix = np.corrcoef(msd_ls_450_7, ave_n6_ls_450_7)
corr = round(corr_matrix[0, 1],3)
axes[1, 1].scatter(msd_ls_450_7, ave_n6_ls_450_7, label=f'Pearson correlation:{corr}')
axes[1, 1].set_title('displacement vs ave_n6')
axes[1, 1].legend()

### 450K_8

In [ ]:
temp, i = 450, 8
df = load_trj(f'./{temp}_diffusion_3/dump.{temp}_diffuse_{i}.trj')
Na_atom = df[(df['type'] == 5.0) & (df['timestep'] == 0)]
Na_ids = Na_atom['id'].to_numpy()
ints = random.sample(range(1, 860), 500)
Na_atom.iloc[ints]
Na_id_ls = Na_ids[ints].astype(int)

In [ ]:
ave_n5_ls_450_8=[]
ave_n6_ls_450_8=[]
msd_ls_450_8=[]
distance_ls_450_8=[]

if __name__ == "__main__":
    # Initialize analyzer
    analyzer = MDTrajectoryAnalyzer(f'./{temp}_diffusion_3/dump.{temp}_diffuse_{i}.trj')  # Replace with your file
    analyzer.load_trajectory()

    for i in Na_id_ls:
        target_atom_id = i
        results, ave_n5, ave_n6, msd, distance = analyzer.analyze_atom(target_atom_id)
        #ave_n5, ave_n6, msd, distance = analyzer.plot_analysis(results, target_atom_id)
        ave_n5_ls_450_8.append(ave_n5)
        ave_n6_ls_450_8.append(ave_n6)
        msd_ls_450_8.append(msd)
        distance_ls_450_8.append(distance)

In [ ]:
if __name__ == "__main__":
    # Initialize analyzer
    analyzer = MDTrajectoryAnalyzer(f'./{temp}_diffusion_3/dump.{temp}_diffuse_{i}.trj')  # Replace with your file
    analyzer.load_trajectory()

    for i in Na_id_ls[:10]:
        target_atom_id = i
        results, ave_n5, ave_n6, msd, distance = analyzer.analyze_atom(target_atom_id)
        analyzer.plot_analysis(results, target_atom_id)


In [ ]:
plt.figure()
distance_ls_450_8 = [x for x in distance_ls_450_8 if x is not None]
msd_ls_450_8 = [x for x in msd_ls_450_8 if x is not None]
corr_matrix = np.corrcoef(distance_ls_450_8, msd_ls_450_8)
corr = round(corr_matrix[0, 1],3)
f = np.array(msd_ls_450_8)/np.array(distance_ls_450_8)
plt.scatter(distance_ls_450_8, msd_ls_450_8, label=f'Pearson correlation:{corr}\nf = {round(f.mean(),3)}')
plt.legend()

In [ ]:
np.array(msd_ls_450_8).mean(), np.array(distance_ls_450_8).mean()

In [ ]:
ave_n6_ls_450_8

In [ ]:
%matplotlib inline
fig, axes = plt.subplots(2, 2, figsize=(singlecol*2, singlecol*2))

# axes is a 2x2 array, so access with [row, col]
distance_ls_450_8 = [x for x in distance_ls_450_8 if x is not None]
ave_n5_ls_450_8 = [x for x in ave_n5_ls_450_8 if x is not None]
ave_n6_ls_450_8= [x for x in ave_n6_ls_450_8 if x is not None]
msd_ls_450_8 = [x for x in msd_ls_450_8 if x is not None]

corr_matrix = np.corrcoef(distance_ls_450_8, ave_n5_ls_450_8)
corr = round(corr_matrix[0, 1],3)
axes[0, 0].scatter(distance_ls_450_8, ave_n5_ls_450_8, label=f'Pearson correlation:{corr}')
axes[0, 0].set_title('distance vs ave_n5')
axes[0, 0].legend()

corr_matrix = np.corrcoef(distance_ls_450_8, ave_n6_ls_450_8)
corr = round(corr_matrix[0, 1],3)
axes[0, 1].scatter(distance_ls_450_8, ave_n6_ls_450_8, label=f'Pearson correlation:{corr}')
axes[0, 1].set_title('distance vs ave_n6')
axes[0, 1].legend()

corr_matrix = np.corrcoef(msd_ls_450_8, ave_n5_ls_450_8)
corr = round(corr_matrix[0, 1],3)
axes[1, 0].scatter(msd_ls_450_8, ave_n5_ls_450_8, label=f'Pearson correlation:{corr}')
axes[1, 0].set_title('displacement vs ave_n5')
axes[1, 0].legend()

corr_matrix = np.corrcoef(msd_ls_450_8, ave_n6_ls_450_8)
corr = round(corr_matrix[0, 1],3)
axes[1, 1].scatter(msd_ls_450_8, ave_n6_ls_450_8, label=f'Pearson correlation:{corr}')
axes[1, 1].set_title('displacement vs ave_n6')
axes[1, 1].legend()

### 450K_9

In [ ]:
temp, i = 450,9
df = load_trj(f'./{temp}_diffusion_3/dump.{temp}_diffuse_{i}.trj')
Na_atom = df[(df['type'] == 6.0) & (df['timestep'] == 0)]
Na_ids = Na_atom['id'].to_numpy()
ints = random.sample(range(1, 1500), 10)
Na_atom.iloc[ints]
Na_id_ls = Na_ids[ints].astype(int)

In [ ]:
Na_id_ls

In [ ]:
ave_n5_ls_450_9=[]
ave_n6_ls_450_9=[]
msd_ls_450_9=[]
distance_ls_450_9=[]

if __name__ == "__main__":
    # Initialize analyzer
    analyzer = MDTrajectoryAnalyzer(f'./{temp}_diffusion_3/dump.{temp}_diffuse_{i}.trj')  # Replace with your file
    analyzer.load_trajectory()

    for j in Na_id_ls:
        target_atom_id = j
        results, ave_n5, ave_n6, msd, distance = analyzer.analyze_atom(target_atom_id)
        #ave_n5, ave_n6, msd, distance = analyzer.plot_analysis(results, target_atom_id)
        ave_n5_ls_450_9.append(ave_n5)
        ave_n6_ls_450_9.append(ave_n6)
        msd_ls_450_9.append(msd)
        distance_ls_450_9.append(distance)

In [ ]:
if __name__ == "__main__":
    # Initialize analyzer
    analyzer = MDTrajectoryAnalyzer(f'./{temp}_diffusion_3/dump.{temp}_diffuse_{i}.trj')  # Replace with your file
    analyzer.load_trajectory()

    for i in Na_id_ls[:10]:
        target_atom_id = i
        results, ave_n5, ave_n6, msd, distance = analyzer.analyze_atom(target_atom_id)
        analyzer.plot_analysis(results, target_atom_id,450,9)


In [ ]:
import numpy as np
import re

def collect_means(pattern):
    # Filter only matching items first
    matched = [(k, v) for k, v in globals().items() if re.fullmatch(pattern, k)]
    # Sort them by the numeric suffix captured from the name
    matched_sorted = sorted(matched, key=lambda kv: int(re.search(r"(\d+)$", kv[0]).group(1)))
    # Compute mean while filtering out None values
    return [
        np.mean([x for x in np.ravel(v) if x is not None])
        for _, v in matched_sorted
    ]

# Use the helper function for each group
msd_ls_450K = collect_means(r"msd_ls_450_\d+")
distance_ls_450K = collect_means(r"distance_ls_450_\d+")
n5_ls_450K = collect_means(r"ave_n5_ls_450_\d+")
n6_ls_450K = collect_means(r"ave_n6_ls_450_\d+")

In [ ]:
for i in range(1, 10):
    np.save(f'msd_ls_450_{i}.npy', globals()[f'msd_ls_450_{i}'])
    np.save(f'distance_ls_450_{i}.npy', globals()[f'distance_ls_450_{i}'])
    np.save(f'ave_n5_ls_450_{i}.npy', globals()[f'ave_n5_ls_450_{i}'])
    np.save(f'ave_n6_ls_450_{i}.npy', globals()[f'ave_n6_ls_450_{i}'])

In [ ]:
msd_vars = [v for k, v in globals().items() if re.fullmatch(r"msd_ls_450_\d+", k)] 
msd_ls_450K = [np.array(v).mean() for v in msd_vars] 

distance_vars = [v for k, v in globals().items() if re.fullmatch(r"distance_ls_450_\d+", k)] 
distance_ls_450K = [np.array(v).mean() for v in distance_vars] 

n5_vars = [v for k, v in globals().items() if re.fullmatch(r"ave_n5_ls_450_\d+", k)] 
n5_ls_450K = [np.mean([x for x in np.ravel(v) if x is not None]) for v in n5_vars] 

n6_vars = [v for k, v in globals().items() if re.fullmatch(r"ave_n6_ls_450_\d+", k)] 
n6_ls_450K = [np.mean([x for x in np.ravel(v) if x is not None]) for v in n6_vars]

In [ ]:
n5_vars[7]

In [ ]:
plt.figure()
for i in range(1):
    print(i)
    plt.hist(n5_vars[8])

In [ ]:
distance_vs_n5_ls_450K = []
distance_vs_n6_ls_450K = []
for i in range(len(msd_ls_450K)):
    print(i)
    corr_matrix = np.corrcoef(distance_vars[i], n5_vars[i])
    distance_vs_n5_ls_450K.append(round(corr_matrix[0, 1],3))

    corr_matrix = np.corrcoef(distance_vars[i], n6_vars[i])
    distance_vs_n6_ls_450K.append(round(corr_matrix[0, 1],3))


In [ ]:
len(distance_vars[8])

In [ ]:
distance_vs_n6_ls_450K

In [ ]:
distance_vs_n5_ls_450K

In [ ]:
df = pd.DataFrame({
    "ave_msd": msd_ls_450K,
    "ave_distance": distance_ls_450K,
    "ave_n5": n5_ls_450K,
    "ave_n6": n6_ls_450K,
    "distance_vs_n5": distance_vs_n5_ls_450K,
    "distance_vs_n6": distance_vs_n6_ls_450K,
})

In [ ]:
df

In [ ]:
msd_ls_450K = []
distance_ls_450K = []
distance_vs_n5_ls_450K = []
distance_vs_n6_ls_450K = []
np.array(msd_ls_450_9).mean()
np.array(distance_ls_450_9).mean()
corr_matrix = np.corrcoef(distance_ls_450_9, ave_n5_ls_450_9)
corr = round(corr_matrix[0, 1],3)

In [ ]:
plt.figure()
distance_ls_450_9 = [x for x in distance_ls_450_9 if x is not None]
msd_ls_450_9 = [x for x in msd_ls_450_9 if x is not None]
corr_matrix = np.corrcoef(distance_ls_450_9, msd_ls_450_9)
corr = round(corr_matrix[0, 1],3)
f = np.array(msd_ls_450_9)/np.array(distance_ls_450_9)
plt.scatter(distance_ls_450_9, msd_ls_450_9, label=f'Pearson correlation:{corr}\nf = {round(f.mean(),3)}')
plt.legend()

In [ ]:
%matplotlib inline
fig, axes = plt.subplots(2, 2, figsize=(singlecol*2, singlecol*2))

# axes is a 2x2 array, so access with [row, col]
distance_ls_450_9 = [x for x in distance_ls_450_9 if x is not None]
ave_n5_ls_450_9 = [x for x in ave_n5_ls_450_9 if x is not None]
ave_n6_ls_450_9= [x for x in ave_n6_ls_450_9 if x is not None]
msd_ls_450_9 = [x for x in msd_ls_450_9 if x is not None]

corr_matrix = np.corrcoef(distance_ls_450_9, ave_n5_ls_450_9)
corr = round(corr_matrix[0, 1],3)
axes[0, 0].scatter(distance_ls_450_9, ave_n5_ls_450_9, label=f'Pearson correlation:{corr}')
axes[0, 0].set_title('distance vs ave_n5')
axes[0, 0].legend()

corr_matrix = np.corrcoef(distance_ls_450_9, ave_n6_ls_450_9)
corr = round(corr_matrix[0, 1],3)
axes[0, 1].scatter(distance_ls_450_9, ave_n6_ls_450_9, label=f'Pearson correlation:{corr}')
axes[0, 1].set_title('distance vs ave_n6')
axes[0, 1].legend()

corr_matrix = np.corrcoef(msd_ls_450_9, ave_n5_ls_450_9)
corr = round(corr_matrix[0, 1],3)
axes[1, 0].scatter(msd_ls_450_9, ave_n5_ls_450_9, label=f'Pearson correlation:{corr}')
axes[1, 0].set_title('displacement vs ave_n5')
axes[1, 0].legend()

corr_matrix = np.corrcoef(msd_ls_450_9, ave_n6_ls_450_9)
corr = round(corr_matrix[0, 1],3)
axes[1, 1].scatter(msd_ls_450_9, ave_n6_ls_450_9, label=f'Pearson correlation:{corr}')
axes[1, 1].set_title('displacement vs ave_n6')
axes[1, 1].legend()

In [ ]:
import numpy as np
import pandas as pd

def load_trj(filename):
    """
    Load LAMMPS-style .trj file assuming columns: id type x y z
    and separated by timestep blocks.
    """
    timesteps = []
    with open(filename) as f:
        lines = f.readlines()

    i = 0
    while i < len(lines):
        if "ITEM: TIMESTEP" in lines[i]:
            step = int(lines[i+1])
            natoms = int(lines[i+3])
            start = i+9
            block = lines[start:start+natoms]
            data = np.loadtxt(block, usecols=(0,1,2,3,4))
            df = pd.DataFrame(data, columns=['id','type','x','y','z'])
            df['timestep'] = step
            timesteps.append(df)
            i = start + natoms
        else:
            i += 1

    return pd.concat(timesteps, ignore_index=True)

temp, i = 400, 6
df = load_trj(f'./{temp}_diffusion_3/dump.{temp}_diffuse_{i}.trj')
Na_atom = df[(df['type'] == 5.0) & (df['timestep'] == 0)]
Na_ids = Na_atom['id'].to_numpy()
ints = np.random.randint(0, 1511, size=10)
Na_atom.iloc[ints]
Na_ids[ints].astype(int)

In [ ]:
def extract_atom_traj(df, atom_id):
    return df[df['id'] == atom_id].sort_values('timestep')

In [ ]:
from numpy.linalg import norm

def detect_jumps(atom_df, threshold=1):
    coords = atom_df[['x','y','z']].to_numpy()
    timesteps = atom_df['timestep'].to_numpy()
    displacements = np.linalg.norm(np.diff(coords, axis=0), axis=1)

    jumps = displacements > threshold
    jump_distances = displacements[jumps]
    jump_timesteps = timesteps[1:][jumps]

    return jump_timesteps, jump_distances, displacements, timesteps


In [ ]:
def compute_msd(atom_df):
    coords = atom_df[['x','y','z']].to_numpy()
    r0 = coords[0]
    disp = coords - r0
    squared = np.sum(disp**2, axis=1)
    return squared


In [ ]:
def neighbor_counts(df, atom_id, radius=4.0):
    atom_pos = df[df['id']==atom_id][['timestep','x','y','z']].to_numpy()
    result = []

    for ts, x, y, z in atom_pos:
        # get all atoms at this timestep
        frame = df[df['timestep'] == ts]
        dx = frame['x'] - x
        dy = frame['y'] - y
        dz = frame['z'] - z
        dist = np.sqrt(dx**2 + dy**2 + dz**2)
        mask = (dist <= radius) & (frame['id'] != atom_id)
        types = frame[mask]['type']
        n5 = (types == 5).sum()
        n6 = (types == 6).sum()
        result.append((int(ts), n5, n6))
    return pd.DataFrame(result, columns=['timestep','n_type5','n_type6'])


In [ ]:
import matplotlib.pyplot as plt

def plot_results(msd, displacements, timesteps, nbh_df):
    fig, axs = plt.subplots(2,1, figsize=(8,6), sharex=True)

    axs[0].plot(timesteps, msd, label='MSD')
    axs[0].set_ylabel('MSD ($\mathrm{\AA^2}$)')
    axs[0].legend()

    cumulative_jump_dist = np.cumsum(displacements)
    axs[1].plot(timesteps[1:], cumulative_jump_dist[:], label='Cumulative displacement')
    
    # Mark timesteps with any type-6 neighbors
    has_type6 = nbh_df[nbh_df['n_type6']>0]['timestep']
    axs[1].scatter(has_type6, cumulative_jump_dist[np.isin(timesteps[1:], has_type6)], 
                   color='red', s=15, label='Has type-6 neighbor')

    axs[1].set_ylabel('Cumulative jump distance (Å)')
    axs[1].set_xlabel('Timestep')
    axs[1].legend()
    plt.tight_layout()
    plt.show()


In [ ]:
temp, i = 400, 6
df = load_trj(f'./{temp}_diffusion_3/dump.{temp}_diffuse_{i}.trj')

In [ ]:
Na_atom = df[(df['type'] == 5.0) & (df['timestep'] == 0)]
Na_ids = Na_atom['id'].to_numpy()

In [ ]:
ints = np.random.randint(0, 1511, size=10)
Na_atom.iloc[ints]

## 400K

### 400K_1

In [ ]:
import random
temp, i = 400,1
df = load_trj(f'./{temp}_diffusion_4/dump.{temp}_diffuse_{i}.trj')
Na_atom = df[(df['type'] == 5.0) & (df['timestep'] == 0)]
Na_ids = Na_atom['id'].to_numpy()
ints = random.sample(range(1, 210), 150)
Na_atom.iloc[ints]
Na_id_ls_400_1 = Na_ids[ints].astype(int)
Na_id_ls = Na_id_ls_400_1

In [ ]:
ave_n5_ls_400_1=[]
ave_n6_ls_400_1=[]
msd_ls_400_1=[]
distance_ls_400_1=[]
result_ls_400_1=[]

if __name__ == "__main__":
    # Initialize analyzer
    analyzer = MDTrajectoryAnalyzer(f'./{temp}_diffusion_4/dump.{temp}_diffuse_{i}.trj')  # Replace with your file
    analyzer.load_trajectory()

    for j in Na_id_ls:
        target_atom_id = j
        results, ave_n5, ave_n6, msd, distance = analyzer.analyze_atom(target_atom_id)
        #ave_n5, ave_n6, msd, distance = analyzer.plot_analysis(results, target_atom_id)
        result_ls_400_1.append(results)
        ave_n5_ls_400_1.append(ave_n5)
        ave_n6_ls_400_1.append(ave_n6)
        msd_ls_400_1.append(msd)
        distance_ls_400_1.append(distance)

In [ ]:
import numpy as np
result_ls_400_1 = [x for x in result_ls_400_1 if x is not None]
all_jump_distances = []
for result in result_ls_400_1:
    jump_dists = result['jump_distances']
    if isinstance(jump_dists, np.ndarray):
        if jump_dists.ndim > 0:  # Not a scalar
            all_jump_distances.extend(jump_dists.flatten())
    elif isinstance(jump_dists, (list, tuple)):
        for item in jump_dists:
            if hasattr(item, '__iter__') and not isinstance(item, str):
                all_jump_distances.extend(item)
            else:
                all_jump_distances.append(item)

all_jump_distances_400_1 = np.array(all_jump_distances)

In [ ]:
from scipy.stats import expon
plt.rcParams.update({'font.size': 16})
plt.rcParams.update({'lines.linewidth':1})
plt.rcParams.update({'axes.linewidth':1})
plt.rcParams.update({'lines.markersize':5})
plt.rcParams.update({'savefig.dpi':200})
plt.rcParams.update({'legend.fontsize': 16})

# Fit exponential distribution
loc, scale = expon.fit(all_jump_distances_400_1)
print(f"Location parameter: {loc:.4f}")
print(f"Scale parameter (mean): {scale:.4f}")

# Plot
plt.figure(figsize=(singlecol*2, singlecol*2))

# Histogram with counts (not density)
counts, bins, patches = plt.hist(all_jump_distances_400_1, bins=50, alpha=0.7, 
                                  edgecolor='black', label='Data')

# Scale the fitted curve to match counts
x_fit = np.linspace(all_jump_distances_400_1.min(), all_jump_distances_400_1.max(), 1000)
bin_width = bins[1] - bins[0]
n_total = len(all_jump_distances_400_1)

# Scale PDF to counts: PDF * bin_width * total_count
plt.plot(x_fit, expon.pdf(x_fit, loc, scale) * bin_width * n_total, 
         'r-', linewidth=2, label=f'Exponential fit')

plt.xlabel('Jump distance (Å)')
plt.ylabel('Counts')
plt.legend(frameon=False)
#plt.savefig('jump_distribution_400_6.svg')
plt.show()

In [ ]:
from scipy.optimize import curve_fit

# Define Gaussian function
def gaussian(x, amplitude, mean, stddev):
    return amplitude * np.exp(-((x - mean) ** 2) / (2 * stddev ** 2))

plt.rcParams.update({'font.size': 16})
plt.rcParams.update({'lines.linewidth':1})
plt.rcParams.update({'axes.linewidth':1})
plt.rcParams.update({'lines.markersize':5})
plt.rcParams.update({'savefig.dpi':200})
plt.rcParams.update({'legend.fontsize': 16})

# Plot
plt.figure(figsize=(singlecol*2, singlecol*2))

# Histogram with counts
counts, bins, patches = plt.hist(all_jump_distances_400_1, bins=50, alpha=0.7, 
                                  edgecolor='black', label='Data')

# Get bin centers for fitting
bin_centers = (bins[:-1] + bins[1:]) / 2

# Initial guess with mean = 0.5
initial_guess = [counts.max(), 0.5, np.std(all_jump_distances)]

# Fit Gaussian
popt, pcov = curve_fit(gaussian, bin_centers, counts, p0=initial_guess)
amplitude, mean, stddev = popt

print(f"Amplitude: {amplitude:.4f}")
print(f"Mean (μ): {mean:.4f}")
print(f"Standard deviation (σ): {stddev:.4f}")

# Calculate R-squared
residuals = counts - gaussian(bin_centers, *popt)
ss_res = np.sum(residuals**2)
ss_tot = np.sum((counts - np.mean(counts))**2)
r_squared = 1 - (ss_res / ss_tot)
print(f"R-squared: {r_squared:.4f}")

# Plot fitted curve
x_fit = np.linspace(all_jump_distances_400_1.min(), all_jump_distances_400_1.max(), 1000)
plt.plot(x_fit, gaussian(x_fit, *popt), 'r-', linewidth=2, label=f'Gaussian fit')

plt.xlabel('Jump distance (Å)')
plt.ylabel('Counts')
plt.legend(frameon=False)
plt.savefig('jump_distribution_400_1.svg')
plt.show()

In [ ]:
%matplotlib inline

plt.rcParams.update({'font.size': 8})
plt.rcParams.update({'lines.linewidth':1})
plt.rcParams.update({'axes.linewidth':1})
plt.rcParams.update({'lines.markersize':3})
plt.rcParams.update({'savefig.dpi':300})
plt.rcParams.update({'legend.fontsize': 8})
fig, axes = plt.subplots(2, 2, figsize=(singlecol*2, singlecol*2))

# axes is a 2x2 array, so access with [row, col]
distance_ls_400_1 = [x for x in distance_ls_400_1 if x is not None]
ave_n5_ls_400_1 = [x for x in ave_n5_ls_400_1 if x is not None]
ave_n6_ls_400_1= [x for x in ave_n6_ls_400_1 if x is not None]
msd_ls_400_1 = [x for x in msd_ls_400_1 if x is not None]

corr_matrix = np.corrcoef(distance_ls_400_1, ave_n5_ls_400_1)
corr = round(corr_matrix[0, 1],3)
axes[0, 0].scatter(distance_ls_400_1, ave_n5_ls_400_1, label=f'Pearson correlation:{corr}')
#axes[0, 0].set_title('distance vs ave_n5')
axes[0, 0].set_xlabel('Jump distance (Å)')
axes[0, 0].set_ylabel('Average number of Na neighbor')
axes[0, 0].legend(frameon=False)

corr_matrix = np.corrcoef(distance_ls_400_1, ave_n6_ls_400_1)
corr = round(corr_matrix[0, 1],3)
axes[0, 1].scatter(distance_ls_400_1, ave_n6_ls_400_1, label=f'Pearson correlation:{corr}')
axes[0, 1].set_xlabel('Jump distance (Å)')
axes[0, 1].set_ylabel('Average number of water neighbor')
axes[0, 1].legend(frameon=False)

corr_matrix = np.corrcoef(msd_ls_400_1, ave_n5_ls_400_1)
corr = round(corr_matrix[0, 1],3)
axes[1, 0].scatter(msd_ls_400_1, ave_n5_ls_400_1, label=f'Pearson correlation:{corr}')
axes[1, 0].set_xlabel('Displacement (Å)')
axes[1, 0].set_ylabel('Average number of Na neighbor')
axes[1, 0].legend(frameon=False)

corr_matrix = np.corrcoef(msd_ls_400_1, ave_n6_ls_400_1)
corr = round(corr_matrix[0, 1],3)
axes[1, 1].scatter(msd_ls_400_1, ave_n6_ls_400_1, label=f'Pearson correlation:{corr}')
axes[1, 1].set_xlabel('Displacement (Å)')
axes[1, 1].set_ylabel('Average number of water neighbor')
axes[1, 1].legend(frameon=False)

plt.tight_layout(w_pad=2.0)

#plt.savefig('jump_correlation_400_1.svg')

In [ ]:
temp, i = 400,1
if __name__ == "__main__":
    # Initialize analyzer
    analyzer_400_1 = MDTrajectoryAnalyzer(f'./{temp}_diffusion_4/dump.{temp}_diffuse_{i}.trj')  # Replace with your file
    analyzer_400_1.load_trajectory()

    for j in Na_id_ls[40:50]:
        target_atom_id = j
        results, ave_n5, ave_n6, msd, distance = analyzer_400_1.analyze_atom(target_atom_id)
        analyzer_400_1.plot_analysis(results, target_atom_id, 400, 1)

### 400K_1 Tetragonal

In [ ]:
import random
temp, i = 400,1
df = load_trj(f'./../production_run/free_E/test/diffusion_400/dump.{temp}_diffuse_400.trj')
Na_atom = df[(df['type'] == 5.0) & (df['timestep'] == 0)]
Na_ids = Na_atom['id'].to_numpy()
ints = random.sample(range(1, 210), 150)
Na_atom.iloc[ints]
Na_id_ls = Na_ids[ints].astype(int)

In [ ]:
%matplotlib widget
result_ls_400_T=[]
temp, i = 400,1
if __name__ == "__main__":
    # Initialize analyzer
    #analyzer = MDTrajectoryAnalyzer_dehydrate(f'./../production_run/free_E/test/diffusion_400/dump.{temp}_diffuse_400.trj')  # Replace with your file
    #analyzer.load_trajectory()

    for j in Na_id_ls[40:50]:
        target_atom_id = j
        results, ave_n5, msd, distance = analyzer.analyze_atom(target_atom_id)
        #analyzer.plot_analysis(results, target_atom_id, 400, 1)
        result_ls_400_T.append(results)

In [ ]:
%matplotlib widget
plt.rcParams.update({'font.size': 8})
plt.rcParams.update({'savefig.dpi':300})
plt.rcParams.update({'legend.fontsize': 8})

fig, ((ax1, ax2)) = plt.subplots(2, 1, figsize=(singlecol*2, singlecol*1.2), sharex=True)#,dpi=200)
#fig.suptitle(f'{temp}K_{i} Atom {target_atom_id}', fontsize=16)

results=result_ls_400_T[2]

# Plot 1: MSD vs Time
ax1.set_xlim(0,5e6)
ax1.set_ylabel('MSD (Ų)')

# Plot 2: Total Jump Distance vs Time
correlation_factor = round(results['msd'][-1] / np.sum(results['jump_distances']), 3)
line1, = ax1.plot(results['jump_timesteps'], np.cumsum(results['jump_distances']), 'r-', linewidth=2, alpha=0.7,
         label=f'Accumulative Jump Distance')

ax1.set_ylabel('Jump Distance (Å)', color='red')

ax12=ax1.twinx()
line2, = ax12.plot(results['time_points'], results['msd'], 'b-', linewidth=2, label=f'Displacement', alpha=0.7)
#\nf = |Displacement| / Distance = {correlation_factor}

ax12.set_ylabel('Displacement (Å)', color='blue')

ax1.tick_params(axis='y', colors='red')   # x-axis tick labels
ax12.tick_params(axis='y', colors='blue')  # y-axis tick labels

handles1, labels1 = ax1.get_legend_handles_labels()
handles2, labels2 = ax12.get_legend_handles_labels()
#ax1.legend(handles1 + handles2, labels1 + labels2, loc='best', frameon=False)

ax22=ax2.twinx()

if len(results['all_displacements']) > 0:
    time_diffs = np.diff(results['time_points'])
    line3, = ax22.plot(results['time_points'][1:], results['all_displacements'], 'g-', alpha=0.4, label='Step Displacement')
    
    # Mark jump events
    if len(results['jump_timesteps']) > 0:
        jump_indices = []
        for jt in results['jump_timesteps']:
            idx = np.where(results['time_points'] == jt)[0]
            if len(idx) > 0:
                jump_indices.append(idx[0] - 1)  # -1 because displacement is between steps
        
        if jump_indices:
            line4 = ax22.scatter(results['time_points'][np.array(jump_indices)+1], 
                      results['all_displacements'][jump_indices], 
                      color='red', s=10, alpha=0.4, label='Jumps', zorder=5)
    
    line5 = ax22.axhline(y=1.5, color='red', linestyle='--', alpha=0.5)#, label='Jump Threshold')

ax22.set_ylabel('Displacement (Å)', color='g')
ax22.tick_params(axis='y', colors='g')
# Plot 4: Neighbor count and Type 6 presence
ave_n5 = round(sum(results['type5_neighbors'])/len(results['type5_neighbors']),3)
line1 = ax2.scatter(results['time_points'], results['type5_neighbors'], color='purple', alpha=0.5, 
            label=f'Na neighbors', s=5)


handles1, labels1 = ax2.get_legend_handles_labels()
handles2, labels2 = ax22.get_legend_handles_labels()
ax2.legend(handles1 + handles2, labels1 + labels2, loc='best', frameon=False ,ncol=4)
#ax2.set_ylim(-0.1,2.5)
#ax12.set_ylim(5,12)
#ax1.set_ylim(7,12)
ax2.set_xlabel('Simulation time (fs)')
ax2.set_ylabel('Number of Neighbors')
plt.tight_layout()
#plt.savefig('400K_T_Na_jump_2.svg')

### 400K_1 Rhombohedral

In [ ]:
import random
temp, i = 400,1
df = load_trj(f'./../production_run/free_E/test/diffusion_400/dump.{temp}_diffuse_3600.trj')
Na_atom = df[(df['type'] == 5.0) & (df['timestep'] == 0)]
Na_ids = Na_atom['id'].to_numpy()
ints = random.sample(range(1, 210), 150)
Na_atom.iloc[ints]
Na_id_ls = Na_ids[ints].astype(int)

In [ ]:
temp, i = 400,1
if __name__ == "__main__":
    # Initialize analyzer
    analyzer = MDTrajectoryAnalyzer_dehydrate(f'./../production_run/free_E/test/diffusion_400/dump.{temp}_diffuse_3600.trj')  # Replace with your file
    analyzer.load_trajectory()

    for j in Na_id_ls[:10]:
        target_atom_id = j
        results, ave_n5, msd, distance = analyzer.analyze_atom(target_atom_id)
        analyzer.plot_analysis(results, target_atom_id, 400, 1)

### 400K_6

In [ ]:
import random
temp, i = 400, 6
df = load_trj(f'./{temp}_diffusion_4/dump.{temp}_diffuse_{i}.trj')
Na_atom = df[(df['type'] == 5.0) & (df['timestep'] == 0)]
Na_ids = Na_atom['id'].to_numpy()
ints = random.sample(range(1, 1500), 500)
Na_atom.iloc[ints]
Na_id_ls_400_6 = Na_ids[ints].astype(int)
Na_id_ls = Na_id_ls_400_6

In [ ]:
ave_n5_ls_400_6=[]
ave_n6_ls_400_6=[]
msd_ls_400_6=[]
distance_ls_400_6=[]
result_ls_400_6=[]
if __name__ == "__main__":
    # Initialize analyzer
    #analyzer = MDTrajectoryAnalyzer(f'./{temp}_diffusion_4/dump.{temp}_diffuse_{i}.trj')  # Replace with your file
    #analyzer.load_trajectory()

    for i in Na_id_ls[10:20]:
        target_atom_id = i
        results, ave_n5, ave_n6, msd, distance = analyzer.analyze_atom(target_atom_id)
        #ave_n5, ave_n6, msd, distance = analyzer.plot_analysis(results, target_atom_id)
        result_ls_400_6.append(results)
        ave_n5_ls_400_6.append(ave_n5)
        ave_n6_ls_400_6.append(ave_n6)
        msd_ls_400_6.append(msd)
        distance_ls_400_6.append(distance)

In [ ]:
import numpy as np
result_ls_400_9 = [x for x in result_ls_400_9 if x is not None]
all_jump_distances = []
for result in result_ls_400_9:
    jump_dists = result['jump_distances']
    if isinstance(jump_dists, np.ndarray):
        if jump_dists.ndim > 0:  # Not a scalar
            all_jump_distances.extend(jump_dists.flatten())
    elif isinstance(jump_dists, (list, tuple)):
        for item in jump_dists:
            if hasattr(item, '__iter__') and not isinstance(item, str):
                all_jump_distances.extend(item)
            else:
                all_jump_distances.append(item)

all_jump_distances_400_6 = np.array(all_jump_distances)

In [ ]:
from scipy.optimize import curve_fit

# Define Gaussian function
def gaussian(x, amplitude, mean, stddev):
    return amplitude * np.exp(-((x - mean) ** 2) / (2 * stddev ** 2))

plt.rcParams.update({'font.size': 16})
plt.rcParams.update({'lines.linewidth':1})
plt.rcParams.update({'axes.linewidth':1})
plt.rcParams.update({'lines.markersize':5})
plt.rcParams.update({'savefig.dpi':200})
plt.rcParams.update({'legend.fontsize': 16})

# Plot
plt.figure(figsize=(singlecol*2, singlecol*2))

# Histogram with counts
counts, bins, patches = plt.hist(all_jump_distances_400_6, bins=50, alpha=0.7, 
                                  edgecolor='black', label='Data')

# Get bin centers for fitting
bin_centers = (bins[:-1] + bins[1:]) / 2

# Initial guess with mean = 0.5
initial_guess = [counts.max(), 0.5, np.std(all_jump_distances)]

# Fit Gaussian
popt, pcov = curve_fit(gaussian, bin_centers, counts, p0=initial_guess)
amplitude, mean, stddev = popt

print(f"Amplitude: {amplitude:.4f}")
print(f"Mean (μ): {mean:.4f}")
print(f"Standard deviation (σ): {stddev:.4f}")

# Calculate R-squared
residuals = counts - gaussian(bin_centers, *popt)
ss_res = np.sum(residuals**2)
ss_tot = np.sum((counts - np.mean(counts))**2)
r_squared = 1 - (ss_res / ss_tot)
print(f"R-squared: {r_squared:.4f}")

# Plot fitted curve
x_fit = np.linspace(all_jump_distances_400_6.min(), all_jump_distances_400_6.max(), 1000)
plt.plot(x_fit, gaussian(x_fit, *popt), 'r-', linewidth=2, label=f'Gaussian fit')

plt.xlabel('Jump distance (Å)')
plt.ylabel('Counts')
plt.legend(frameon=False)
plt.savefig('jump_distribution_400_6.svg')
plt.show()

In [ ]:
from scipy.optimize import curve_fit

# Define bi-exponential function
def biexponential(x, a1, b1, a2, b2):
    """
    Bi-exponential: a1 * exp(-b1 * x) + a2 * exp(-b2 * x)
    a1, a2: amplitudes
    b1, b2: decay rates
    """
    return a1 * np.exp(-b1 * x) + a2 * np.exp(-b2 * x)

plt.rcParams.update({'font.size': 16})
plt.rcParams.update({'lines.linewidth':1})
plt.rcParams.update({'axes.linewidth':1})
plt.rcParams.update({'lines.markersize':5})
plt.rcParams.update({'savefig.dpi':200})
plt.rcParams.update({'legend.fontsize': 16})

# Plot
plt.figure(figsize=(singlecol*2, singlecol*2))

# Histogram with counts
counts, bins, patches = plt.hist(all_jump_distances_400_6, bins=50, alpha=0.7, 
                                  edgecolor='black', label='Data')

# Get bin centers for fitting
bin_centers = (bins[:-1] + bins[1:]) / 2
bin_width = bins[1] - bins[0]
n_total = len(all_jump_distances_400_6)

# Initial guess: [a1, b1, a2, b2]
# Fast decay component and slow decay component
initial_guess = [counts.max() * 0.5, 5.0, counts.max() * 0.5, 0.5]

try:
    # Fit bi-exponential
    popt, pcov = curve_fit(biexponential, bin_centers, counts, 
                           p0=initial_guess, maxfev=10000)
    a1, b1, a2, b2 = popt
    perr = np.sqrt(np.diag(pcov))
    
    print(f"Component 1:")
    print(f"  Amplitude (a1): {a1:.4f} ± {perr[0]:.4f}")
    print(f"  Decay rate (b1): {b1:.4f} ± {perr[1]:.4f}")
    print(f"  Mean distance: {1/b1:.4f} Å")
    print(f"\nComponent 2:")
    print(f"  Amplitude (a2): {a2:.4f} ± {perr[2]:.4f}")
    print(f"  Decay rate (b2): {b2:.4f} ± {perr[3]:.4f}")
    print(f"  Mean distance: {1/b2:.4f} Å")
    
    # Calculate R-squared
    residuals = counts - biexponential(bin_centers, *popt)
    ss_res = np.sum(residuals**2)
    ss_tot = np.sum((counts - np.mean(counts))**2)
    r_squared = 1 - (ss_res / ss_tot)
    print(f"\nR-squared: {r_squared:.4f}")
    
    # Plot fitted curve
    x_fit = np.linspace(all_jump_distances_400_6.min(), all_jump_distances_400_6.max(), 1000)
    plt.plot(x_fit, biexponential(x_fit, *popt), 'r-', linewidth=2, 
             label=f'Bi-exponential fit')
    
    # Optionally plot individual components
    plt.plot(x_fit, a1 * np.exp(-b1 * x_fit), 'g--', linewidth=1.5, alpha=0.7,
             label=f'Fast component')
    plt.plot(x_fit, a2 * np.exp(-b2 * x_fit), 'b--', linewidth=1.5, alpha=0.7,
             label=f'Slow component')
    
except Exception as e:
    print(f"Fitting failed: {e}")

plt.xlabel('Jump distance (Å)')
plt.ylabel('Counts')
plt.legend(frameon=False)
#plt.savefig('jump_distribution_400_9.svg')
plt.show()

In [ ]:
from scipy.stats import expon
plt.rcParams.update({'font.size': 16})
plt.rcParams.update({'lines.linewidth':1})
plt.rcParams.update({'axes.linewidth':1})
plt.rcParams.update({'lines.markersize':5})
plt.rcParams.update({'savefig.dpi':200})
plt.rcParams.update({'legend.fontsize': 16})

# Fit exponential distribution
loc, scale = expon.fit(all_jump_distances_400_6)
print(f"Location parameter: {loc:.4f}")
print(f"Scale parameter (mean): {scale:.4f}")

# Plot
plt.figure(figsize=(singlecol*2, singlecol*2))

# Histogram with counts (not density)
counts, bins, patches = plt.hist(all_jump_distances_400_6, bins=50, alpha=0.7, 
                                  edgecolor='black', label='Data')

# Scale the fitted curve to match counts
x_fit = np.linspace(all_jump_distances_400_6.min(), all_jump_distances_400_6.max(), 1000)
bin_width = bins[1] - bins[0]
n_total = len(all_jump_distances_400_6)

# Scale PDF to counts: PDF * bin_width * total_count
plt.plot(x_fit, expon.pdf(x_fit, loc, scale) * bin_width * n_total, 
         'r-', linewidth=2, label=f'Exponential fit')

plt.xlabel('Jump distance (Å)')
plt.ylabel('Counts')
plt.legend(frameon=False)
#plt.savefig('jump_distribution_400_6.svg')
plt.show()

In [ ]:
%matplotlib inline

plt.rcParams.update({'font.size': 8})
plt.rcParams.update({'lines.linewidth':1})
plt.rcParams.update({'axes.linewidth':1})
plt.rcParams.update({'lines.markersize':3})
plt.rcParams.update({'savefig.dpi':300})
plt.rcParams.update({'legend.fontsize': 8})
fig, axes = plt.subplots(2, 2, figsize=(singlecol*2, singlecol*2))

# axes is a 2x2 array, so access with [row, col]
distance_ls_400_6 = [x for x in distance_ls_400_6 if x is not None]
ave_n5_ls_400_6 = [x for x in ave_n5_ls_400_6 if x is not None]
ave_n6_ls_400_6= [x for x in ave_n6_ls_400_6 if x is not None]
msd_ls_400_6 = [x for x in msd_ls_400_6 if x is not None]

corr_matrix = np.corrcoef(distance_ls_400_6, ave_n5_ls_400_6)
corr = round(corr_matrix[0, 1],3)
axes[0, 0].scatter(distance_ls_400_6, ave_n5_ls_400_6, label=f'Pearson correlation:{corr}')
#axes[0, 0].set_title('distance vs ave_n5')
axes[0, 0].set_xlabel('Jump distance (Å)')
axes[0, 0].set_ylabel('Average number of Na neighbor')
axes[0, 0].legend(frameon=False)

corr_matrix = np.corrcoef(distance_ls_400_6, ave_n6_ls_400_6)
corr = round(corr_matrix[0, 1],3)
axes[0, 1].scatter(distance_ls_400_6, ave_n6_ls_400_6, label=f'Pearson correlation:{corr}')
axes[0, 1].set_xlabel('Jump distance (Å)')
axes[0, 1].set_ylabel('Average number of water neighbor')
axes[0, 1].legend(frameon=False)

corr_matrix = np.corrcoef(msd_ls_400_6, ave_n5_ls_400_6)
corr = round(corr_matrix[0, 1],3)
axes[1, 0].scatter(msd_ls_400_6, ave_n5_ls_400_6, label=f'Pearson correlation:{corr}')
axes[1, 0].set_xlabel('Displacement (Å)')
axes[1, 0].set_ylabel('Average number of Na neighbor')
axes[1, 0].legend(frameon=False)

corr_matrix = np.corrcoef(msd_ls_400_6, ave_n6_ls_400_6)
corr = round(corr_matrix[0, 1],3)
axes[1, 1].scatter(msd_ls_400_6, ave_n6_ls_400_6, label=f'Pearson correlation:{corr}')
axes[1, 1].set_xlabel('Displacement (Å)')
axes[1, 1].set_ylabel('Average number of water neighbor')
axes[1, 1].legend(frameon=False)

plt.tight_layout(w_pad=2.0)

plt.savefig('jump_correlation_400_6.svg')

In [ ]:
%matplotlib widget
temp, i = 400,6
if __name__ == "__main__":
    # Initialize analyzer
    #analyzer_400_6 = MDTrajectoryAnalyzer(f'./{temp}_diffusion_4/dump.{temp}_diffuse_{i}.trj')  # Replace with your file
    #analyzer_400_6.load_trajectory()

    for j in Na_id_ls[21:30]:
        target_atom_id = j
        results, ave_n5, ave_n6, msd, distance = analyzer_400_6.analyze_atom(target_atom_id)
        analyzer_400_6.plot_analysis(results, target_atom_id, 400, 6)

In [ ]:
%matplotlib widget
plt.rcParams.update({'font.size': 8})
plt.rcParams.update({'savefig.dpi':300})
plt.rcParams.update({'legend.fontsize': 8})

fig, ((ax1, ax2)) = plt.subplots(2, 1, figsize=(singlecol*2, singlecol*1.2), sharex=True)#,dpi=200)
#fig.suptitle(f'{temp}K_{i} Atom {target_atom_id}', fontsize=16)

results=result_ls_400_8[0]

# Plot 1: MSD vs Time
ax1.set_xlim(300000,340000)#,2e6)
ax1.set_ylabel('MSD (Ų)')

# Plot 2: Total Jump Distance vs Time
correlation_factor = round(results['msd'][-1] / np.sum(results['jump_distances']), 3)
line1, = ax1.plot(results['jump_timesteps'], np.cumsum(results['jump_distances']), 'r-', linewidth=2, alpha=0.7,
         label=f'Accumulative Jump Distance')

ax1.set_ylabel('Jump Distance (Å)', color='red')

ax12=ax1.twinx()
line2, = ax12.plot(results['time_points'], results['msd'], 'b-', linewidth=2, label=f'Displacement', alpha=0.7)
#\nf = |Displacement| / Distance = {correlation_factor}

ax12.set_ylabel('Displacement (Å)', color='blue')

ax1.tick_params(axis='y', colors='red')   # x-axis tick labels
ax12.tick_params(axis='y', colors='blue')  # y-axis tick labels

handles1, labels1 = ax1.get_legend_handles_labels()
handles2, labels2 = ax12.get_legend_handles_labels()
#ax1.legend(handles1 + handles2, labels1 + labels2, loc='best', frameon=False)

ax22=ax2.twinx()

if len(results['all_displacements']) > 0:
    time_diffs = np.diff(results['time_points'])
    line3, = ax22.plot(results['time_points'][1:], results['all_displacements'], 'g-', alpha=0.4, label='Step Displacement')
    
    # Mark jump events
    if len(results['jump_timesteps']) > 0:
        jump_indices = []
        for jt in results['jump_timesteps']:
            idx = np.where(results['time_points'] == jt)[0]
            if len(idx) > 0:
                jump_indices.append(idx[0] - 1)  # -1 because displacement is between steps
        
        if jump_indices:
            line4 = ax22.scatter(results['time_points'][np.array(jump_indices)+1], 
                      results['all_displacements'][jump_indices], 
                      color='red', s=10, alpha=0.4, label='Jumps', zorder=5)
    
    line5 = ax22.axhline(y=1.5, color='red', linestyle='--', alpha=0.5)#, label='Jump Threshold')

ax22.set_ylabel('Displacement (Å)', color='g')
ax22.tick_params(axis='y', colors='g')
# Plot 4: Neighbor count and Type 6 presence
ave_n5 = round(sum(results['type5_neighbors'])/len(results['type5_neighbors']),3)
line1 = ax2.scatter(results['time_points'], results['type5_neighbors'], color='purple', alpha=0.5, 
            label=f'Na neighbors', s=5)

ave_n6 = round(sum(results['type6_neighbors'])/len(results['type6_neighbors']),3)
line2 = ax2.scatter(results['time_points'], results['type6_neighbors'], color='orange', alpha=0.5, 
            label=f'H2O neighbors', s=5)



handles1, labels1 = ax2.get_legend_handles_labels()
handles2, labels2 = ax22.get_legend_handles_labels()
ax1.legend(handles1 + handles2, labels1 + labels2, loc='best', frameon=False ,ncol=4)
ax2.set_ylim(-0.1,2.5)
ax12.set_ylim(5,12)
ax1.set_ylim(7,12)
ax2.set_xlabel('Simulation time (fs)')
ax2.set_ylabel('Number of Neighbors')
plt.tight_layout()
plt.savefig('400K_8_Na_jump_7352.svg')

### 400K_7

In [ ]:
temp, i = 400, 7
df = load_trj(f'./{temp}_diffusion_4/dump.{temp}_diffuse_{i}.trj')
Na_atom = df[(df['type'] == 5.0) & (df['timestep'] == 0)]
Na_ids = Na_atom['id'].to_numpy()
ints = random.sample(range(1, 210), 150)
Na_atom.iloc[ints]
Na_id_ls_400_7 = Na_ids[ints].astype(int)
Na_id_ls = Na_id_ls_400_7

In [ ]:
ave_n5_ls_400_7=[]
ave_n6_ls_400_7=[]
msd_ls_400_7=[]
distance_ls_400_7=[]
result_ls_400_7=[]

temp, i = 400, 7
if __name__ == "__main__":
    # Initialize analyzer
    analyzer = MDTrajectoryAnalyzer(f'./{temp}_diffusion_4/dump.{temp}_diffuse_{i}.trj')  # Replace with your file
    analyzer.load_trajectory()

    for i in Na_id_ls_400_7[:10]:
        target_atom_id = i
        results, ave_n5, ave_n6, msd, distance = analyzer.analyze_atom(target_atom_id)
        #ave_n5, ave_n6, msd, distance = analyzer.plot_analysis(results, target_atom_id)
        result_ls_400_7.append(results)
        ave_n5_ls_400_7.append(ave_n5)
        ave_n6_ls_400_7.append(ave_n6)
        msd_ls_400_7.append(msd)
        distance_ls_400_7.append(distance)

### 400K_8

In [ ]:
temp, i = 400, 8
df = load_trj(f'./{temp}_diffusion_4/dump.{temp}_diffuse_{i}.trj')
Na_atom = df[(df['type'] == 5.0) & (df['timestep'] == 0)]
Na_ids = Na_atom['id'].to_numpy()
ints = random.sample(range(1, 860), 400)
Na_atom.iloc[ints]
Na_id_ls_400_8 = Na_ids[ints].astype(int)
Na_id_ls = Na_id_ls_400_8

In [ ]:
ave_n5_ls_400_8=[]
ave_n6_ls_400_8=[]
msd_ls_400_8=[]
distance_ls_400_8=[]
result_ls_400_8=[]

temp, i = 400, 8
if __name__ == "__main__":
    # Initialize analyzer
    #analyzer = MDTrajectoryAnalyzer(f'./{temp}_diffusion_4/dump.{temp}_diffuse_{i}.trj')  # Replace with your file
    #analyzer.load_trajectory()

    for i in Na_id_ls_400_8[:1]:
        target_atom_id = 7352
        results, ave_n5, ave_n6, msd, distance = analyzer.analyze_atom(target_atom_id)
        #ave_n5, ave_n6, msd, distance = analyzer.plot_analysis(results, target_atom_id)
        result_ls_400_8.append(results)
        ave_n5_ls_400_8.append(ave_n5)
        ave_n6_ls_400_8.append(ave_n6)
        msd_ls_400_8.append(msd)
        distance_ls_400_8.append(distance)

### 400K_9

In [ ]:
temp, i = 400, 9
df = load_trj(f'./{temp}_diffusion_4/dump.{temp}_diffuse_{i}.trj')
Na_atom = df[(df['type'] == 7.0) & (df['timestep'] == 0)]
Na_ids = Na_atom['id'].to_numpy()
ints = random.sample(range(1, 1500), 500)
Na_atom.iloc[ints]
Na_id_ls_400_9 = Na_ids[ints].astype(int)
Na_id_ls = Na_id_ls_400_9

In [ ]:
ave_n5_ls_400_9=[]
ave_n6_ls_400_9=[]
msd_ls_400_9=[]
distance_ls_400_9=[]
result_ls_400_9=[]

temp, i = 400, 9
if __name__ == "__main__":
    # Initialize analyzer
    #analyzer = MDTrajectoryAnalyzer(f'./{temp}_diffusion_4/dump.{temp}_diffuse_{i}.trj')  # Replace with your file
    #analyzer.load_trajectory()

    for i in Na_id_ls_400_9[40:50]:
        target_atom_id = i
        results, ave_n5, ave_n6, msd, distance = analyzer.analyze_atom(target_atom_id)
        #ave_n5, ave_n6, msd, distance = analyzer.plot_analysis(results, target_atom_id)
        result_ls_400_9.append(results)
        ave_n5_ls_400_9.append(ave_n5)
        ave_n6_ls_400_9.append(ave_n6)
        msd_ls_400_9.append(msd)
        distance_ls_400_9.append(distance)

In [ ]:
%matplotlib widget
plt.rcParams.update({'font.size': 8})
plt.rcParams.update({'savefig.dpi':300})
plt.rcParams.update({'legend.fontsize': 8})

fig, ((ax1, ax2)) = plt.subplots(2, 1, figsize=(singlecol*2, singlecol*1.2), sharex=True)#,dpi=200)
#fig.suptitle(f'{temp}K_{i} Atom {target_atom_id}', fontsize=16)

results=result_ls_400_8[0]

# Plot 1: MSD vs Time
ax1.set_xlim(300000,340000)#,2e6)
ax1.set_ylabel('MSD (Ų)')

# Plot 2: Total Jump Distance vs Time
correlation_factor = round(results['msd'][-1] / np.sum(results['jump_distances']), 3)
line1, = ax1.plot(results['jump_timesteps'], np.cumsum(results['jump_distances']), 'r-', linewidth=2, alpha=0.7,
         label=f'Accumulative Jump Distance')

ax1.set_ylabel('Jump Distance (Å)', color='red')

ax12=ax1.twinx()
line2, = ax12.plot(results['time_points'], results['msd'], 'b-', linewidth=2, label=f'Displacement', alpha=0.7)
#\nf = |Displacement| / Distance = {correlation_factor}

ax12.set_ylabel('Displacement (Å)', color='blue')

ax1.tick_params(axis='y', colors='red')   # x-axis tick labels
ax12.tick_params(axis='y', colors='blue')  # y-axis tick labels

handles1, labels1 = ax1.get_legend_handles_labels()
handles2, labels2 = ax12.get_legend_handles_labels()
#ax1.legend(handles1 + handles2, labels1 + labels2, loc='best', frameon=False)

ax22=ax2.twinx()

if len(results['all_displacements']) > 0:
    time_diffs = np.diff(results['time_points'])
    line3, = ax22.plot(results['time_points'][1:], results['all_displacements'], 'g-', alpha=0.4, label='Step Displacement')
    
    # Mark jump events
    if len(results['jump_timesteps']) > 0:
        jump_indices = []
        for jt in results['jump_timesteps']:
            idx = np.where(results['time_points'] == jt)[0]
            if len(idx) > 0:
                jump_indices.append(idx[0] - 1)  # -1 because displacement is between steps
        
        if jump_indices:
            line4 = ax22.scatter(results['time_points'][np.array(jump_indices)+1], 
                      results['all_displacements'][jump_indices], 
                      color='red', s=10, alpha=0.4, label='Jumps', zorder=5)
    
    line5 = ax22.axhline(y=1.5, color='red', linestyle='--', alpha=0.5)#, label='Jump Threshold')

ax22.set_ylabel('Displacement (Å)', color='g')
ax22.tick_params(axis='y', colors='g')
# Plot 4: Neighbor count and Type 6 presence
ave_n5 = round(sum(results['type5_neighbors'])/len(results['type5_neighbors']),3)
line1 = ax2.scatter(results['time_points'], results['type5_neighbors'], color='purple', alpha=0.5, 
            label=f'Na neighbors', s=5)

ave_n6 = round(sum(results['type6_neighbors'])/len(results['type6_neighbors']),3)
line2 = ax2.scatter(results['time_points'], results['type6_neighbors'], color='orange', alpha=0.5, 
            label=f'H2O neighbors', s=5)



handles1, labels1 = ax2.get_legend_handles_labels()
handles2, labels2 = ax22.get_legend_handles_labels()
ax1.legend(handles1 + handles2, labels1 + labels2, loc='best', frameon=False ,ncol=4)
ax2.set_ylim(-0.1,2.5)
ax12.set_ylim(5,12)
ax1.set_ylim(7,12)
ax2.set_xlabel('Simulation time (fs)')
ax2.set_ylabel('Number of Neighbors')
plt.tight_layout()
plt.savefig('400K_8_Na_jump_7352.svg')

In [ ]:
import numpy as np
result_ls_400_9 = [x for x in result_ls_400_9 if x is not None]
all_jump_distances = []
for result in result_ls_400_9:
    jump_dists = result['jump_distances']
    if isinstance(jump_dists, np.ndarray):
        if jump_dists.ndim > 0:  # Not a scalar
            all_jump_distances.extend(jump_dists.flatten())
    elif isinstance(jump_dists, (list, tuple)):
        for item in jump_dists:
            if hasattr(item, '__iter__') and not isinstance(item, str):
                all_jump_distances.extend(item)
            else:
                all_jump_distances.append(item)

all_jump_distances_400_9 = np.array(all_jump_distances)

In [ ]:
all_jump_distances_400_9_Na = all_jump_distances_400_9

In [ ]:
all_jump_distances_400_9_H = all_jump_distances_400_9

In [ ]:
#np.save('jump_distances_400_9_Na.py', all_jump_distances_400_9_Na) ### 144 atoms
#np.save('jump_distances_400_9_H.py', all_jump_distances_400_9_H)  ### 81 atoms

In [ ]:
all_jump_distances_400_9_Na = np.load('jump_distances_400_9_Na.npy')
all_jump_distances_400_9_H = np.load('jump_distances_400_9_H.npy')

In [ ]:
len(result_ls_400_9)

In [ ]:
2000/(all_jump_distances_400_9.shape[0]/144)

In [ ]:
count = sum(x > 2.5 for x in all_jump_distances_400_9_Na)
t = 2000/(count/144)

In [ ]:
count = sum(x > 1.5 for x in all_jump_distances_400_9_Na)

In [ ]:
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(singlecol * 2, singlecol * 1),  dpi=300)
# l range
l_values = np.linspace(0, 2.5, 100)

t_values = []

for l in l_values:
    count = np.sum(all_jump_distances_400_9_Na > l)
    t = 2000 / (count / 144)
    t_values.append(t)

t_values = np.array(t_values)

axes[0].plot(l_values, t_values, color='tab:blue')
axes[0].set_xlabel('Threshold (Å)')
axes[0].set_ylabel('Residence time (ps)')

l_values = np.linspace(0, 2.5, 100)
t_values = []
for l in l_values:
    count = np.sum(all_jump_distances_400_9_H > l)
    t = 2000 / (count / 81)
    t_values.append(t)

t_values = np.array(t_values)

axes[1].plot(l_values, t_values, color='tab:orange')
axes[1].set_xlabel('Threshold (Å)')
axes[1].set_ylabel('Residence time (ps)')

plt.tight_layout()
#plt.savefig('residence_time.svg')

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# l range
l_values = np.linspace(1, 1.5, 100)

t_values = []

for l in l_values:
    count = np.sum(all_jump_distances_400_9_H > l)
    t = 2000 / (count / 81)
    t_values.append(t)

t_values = np.array(t_values)

# Plot
plt.figure(figsize=(6, 4))
plt.plot(l_values, t_values)
plt.xlabel('Threshold (Å)')
plt.ylabel('Residence time (ps)')
plt.tight_layout()
plt.show()


In [ ]:
plt.rcParams.update({'font.size': 8})
plt.rcParams.update({'lines.linewidth': 2})
plt.rcParams.update({'axes.linewidth': 1})
plt.rcParams.update({'lines.markersize': 5})
plt.rcParams.update({'savefig.dpi': 300})
plt.rcParams.update({'legend.fontsize': 8})

fig, axes = plt.subplots(1, 2, figsize=(singlecol * 2, singlecol * 1), sharex=True, dpi=300)

# --- Top panel: H ---
axes[1].hist(all_jump_distances_400_9_H, bins=50, edgecolor='black', color='tab:orange', alpha=0.7)
axes[1].axvline(all_jump_distances_400_9_H.mean(),linestyle='--', color='r')
axes[1].text(all_jump_distances_400_9_H.mean() - 0.5, axes[1].get_ylim()[1] * 1.02,
             f'Mean: {all_jump_distances_400_9_H.mean():.3f} Å', color='r')
axes[1].set_ylabel('Counts')
#axes[0].set_title('H')

# --- Bottom panel: Na ---
axes[0].hist(all_jump_distances_400_9_Na, bins=50, edgecolor='black', color='tab:blue', alpha=0.7)
axes[0].axvline(all_jump_distances_400_9_Na.mean(), linestyle='--', color='r')
axes[0].text(all_jump_distances_400_9_Na.mean() - 0.5, axes[0].get_ylim()[1] * 1.02,
             f'Mean: {all_jump_distances_400_9_Na.mean():.3f} Å', color='r')
axes[0].set_xlabel('Jump distance (Å)')
axes[1].set_xlabel('Jump distance (Å)')
axes[].set_ylabel('Counts')
axes[0].set_xlim(0,4)

plt.tight_layout()

plt.savefig('Na_H_jump_hist.svg')

In [ ]:
plt.rcParams.update({'font.size': 16})
plt.rcParams.update({'lines.linewidth':3})
plt.rcParams.update({'axes.linewidth':1})
plt.rcParams.update({'lines.markersize':5})
plt.rcParams.update({'savefig.dpi':200})
plt.rcParams.update({'legend.fontsize': 16})
plt.figure(figsize=(singlecol*2, singlecol*2))
plt.hist(all_jump_distances_400_9, bins=50, edgecolor='black', color='tab:orange', alpha=0.7)
plt.vlines(all_jump_distances_400_9.mean(), 0, 8000, linestyles='--', color = 'r')
plt.text(all_jump_distances_400_9.mean()-0.5, 8050, f'Mean: {round(all_jump_distances_400_9.mean(),3)} Å', color='r')

plt.hist(all_jump_distances_400_9, bins=50, edgecolor='black', color='tab:blue', alpha=0.7)
plt.vlines(all_jump_distances_400_9.mean(), 0, 8000, linestyles='--', color = 'r')
plt.text(all_jump_distances_400_9.mean()-0.5, 8050, f'Mean: {round(all_jump_distances_400_9.mean(),3)} Å', color='r')
plt.xlabel('Jump distance (Å)')
plt.ylabel('Counts')
plt.tight_layout()

In [ ]:
from scipy.stats import expon
plt.rcParams.update({'font.size': 16})
plt.rcParams.update({'lines.linewidth':1})
plt.rcParams.update({'axes.linewidth':1})
plt.rcParams.update({'lines.markersize':5})
plt.rcParams.update({'savefig.dpi':200})
plt.rcParams.update({'legend.fontsize': 16})

# Fit exponential distribution
loc, scale = expon.fit(all_jump_distances_400_9)
print(f"Location parameter: {loc:.4f}")
print(f"Scale parameter (mean): {scale:.4f}")

# Plot
plt.figure(figsize=(singlecol*2, singlecol*2))

# Histogram with counts (not density)
counts, bins, patches = plt.hist(all_jump_distances_400_9, bins=50, alpha=0.7, 
                                  edgecolor='black', label='Data')

# Scale the fitted curve to match counts
x_fit = np.linspace(all_jump_distances_400_9.min(), all_jump_distances_400_9.max(), 1000)
bin_width = bins[1] - bins[0]
n_total = len(all_jump_distances_400_9)

# Scale PDF to counts: PDF * bin_width * total_count
plt.plot(x_fit, expon.pdf(x_fit, loc, scale) * bin_width * n_total, 
         'r-', linewidth=2, label=f'Exponential fit')

plt.xlabel('Jump distance (Å)')
plt.ylabel('Counts')
plt.legend(frameon=False)
plt.savefig('jump_distribution_400_9_E.svg')
plt.show()
plt.tight_layout()

In [ ]:
from scipy.optimize import curve_fit

# Define Gaussian function
def gaussian(x, amplitude, mean, stddev):
    return amplitude * np.exp(-((x - mean) ** 2) / (2 * stddev ** 2))

plt.rcParams.update({'font.size': 16})
plt.rcParams.update({'lines.linewidth':1})
plt.rcParams.update({'axes.linewidth':1})
plt.rcParams.update({'lines.markersize':5})
plt.rcParams.update({'savefig.dpi':200})
plt.rcParams.update({'legend.fontsize': 16})

# Plot
plt.figure(figsize=(singlecol*2, singlecol*2))

# Histogram with counts
counts, bins, patches = plt.hist(all_jump_distances_400_9, bins=50, alpha=0.7, 
                                  edgecolor='black', label='Data')

# Get bin centers for fitting
bin_centers = (bins[:-1] + bins[1:]) / 2

# Initial guess with mean = 0.5
initial_guess = [counts.max(), 0.5, np.std(all_jump_distances_400_9)]

# Fit Gaussian
popt, pcov = curve_fit(gaussian, bin_centers, counts, p0=initial_guess)
amplitude, mean, stddev = popt

print(f"Amplitude: {amplitude:.4f}")
print(f"Mean (μ): {mean:.4f}")
print(f"Standard deviation (σ): {stddev:.4f}")

# Calculate R-squared
residuals = counts - gaussian(bin_centers, *popt)
ss_res = np.sum(residuals**2)
ss_tot = np.sum((counts - np.mean(counts))**2)
r_squared = 1 - (ss_res / ss_tot)
print(f"R-squared: {r_squared:.4f}")

# Plot fitted curve
x_fit = np.linspace(all_jump_distances_400_9.min(), all_jump_distances_400_9.max(), 1000)
plt.plot(x_fit, gaussian(x_fit, *popt), 'r-', linewidth=2, label=f'Gaussian fit')

plt.xlabel('Jump distance (Å)')
plt.ylabel('Counts')
plt.legend(frameon=False)
plt.savefig('jump_distribution_400_9_G.svg')
plt.show()
plt.tight_layout()

In [ ]:
distance_ls_400_9 = [x for x in distance_ls_400_9 if x is not None]
ave_n5_ls_400_9 = [x for x in ave_n5_ls_400_9 if x is not None]
ave_n6_ls_400_9= [x for x in ave_n6_ls_400_9 if x is not None]
msd_ls_400_9 = [x for x in msd_ls_400_9 if x is not None]

In [ ]:
len(distance_ls_400_9)

In [ ]:
%matplotlib inline

plt.rcParams.update({'font.size': 8})
plt.rcParams.update({'lines.linewidth':1})
plt.rcParams.update({'axes.linewidth':1})
plt.rcParams.update({'lines.markersize':3})
plt.rcParams.update({'savefig.dpi':300})
plt.rcParams.update({'legend.fontsize': 8})
fig, axes = plt.subplots(2, 2, figsize=(singlecol*2, singlecol*2))

# axes is a 2x2 array, so access with [row, col]
distance_ls_400_9 = [x for x in distance_ls_400_9 if x is not None]
ave_n5_ls_400_9 = [x for x in ave_n5_ls_400_9 if x is not None]
ave_n6_ls_400_9= [x for x in ave_n6_ls_400_9 if x is not None]
msd_ls_400_9 = [x for x in msd_ls_400_9 if x is not None]

corr_matrix = np.corrcoef(distance_ls_400_9, ave_n5_ls_400_9)
corr = round(corr_matrix[0, 1],3)
axes[0, 0].scatter(distance_ls_400_9, ave_n5_ls_400_9, label=f'Pearson correlation:{corr}')
#axes[0, 0].set_title('distance vs ave_n5')
axes[0, 0].set_xlabel('Jump distance (Å)')
axes[0, 0].set_ylabel('Average number of Na neighbor')
axes[0, 0].legend(frameon=False)

corr_matrix = np.corrcoef(distance_ls_400_9, ave_n6_ls_400_9)
corr = round(corr_matrix[0, 1],3)
axes[0, 1].scatter(distance_ls_400_9, ave_n6_ls_400_9, label=f'Pearson correlation:{corr}')
axes[0, 1].set_xlabel('Jump distance (Å)')
axes[0, 1].set_ylabel('Average number of water neighbor')
axes[0, 1].legend(frameon=False)

corr_matrix = np.corrcoef(msd_ls_400_9, ave_n5_ls_400_9)
corr = round(corr_matrix[0, 1],3)
axes[1, 0].scatter(msd_ls_400_9, ave_n5_ls_400_9, label=f'Pearson correlation:{corr}')
axes[1, 0].set_xlabel('Displacement (Å)')
axes[1, 0].set_ylabel('Average number of Na neighbor')
axes[1, 0].legend(frameon=False)

corr_matrix = np.corrcoef(msd_ls_400_9, ave_n6_ls_400_9)
corr = round(corr_matrix[0, 1],3)
axes[1, 1].scatter(msd_ls_400_9, ave_n6_ls_400_9, label=f'Pearson correlation:{corr}')
axes[1, 1].set_xlabel('Displacement (Å)')
axes[1, 1].set_ylabel('Average number of water neighbor')
axes[1, 1].legend(frameon=False)

plt.tight_layout(w_pad=2.0)

plt.savefig('jump_correlation_400_9.svg')

In [ ]:
        results = {
            'positions': positions,
            'time_points': time_points,
            'msd': msd,
            'total_jump_distance': total_jump_dist,
            'jump_timesteps': jump_timesteps,
            'jump_distances': jump_distances,
            'all_displacements': all_displacements,
            'type5_neighbors': type5_neighbors,
            'type6_neighbors': type6_neighbors,
            'type5_timesteps': type5_timesteps,
            'type6_timesteps': type6_timesteps
        }

In [ ]:
%matplotlib widget
temp, i = 400,9
if __name__ == "__main__":
    # Initialize analyzer
    analyzer_400_9 = MDTrajectoryAnalyzer(f'./{temp}_diffusion_4/dump.{temp}_diffuse_{i}.trj')  # Replace with your file
    analyzer_400_9.load_trajectory()

    for j in Na_id_ls[90:105]:
        target_atom_id = j
        results, ave_n5, ave_n6, msd, distance = analyzer_400_9.analyze_atom(target_atom_id)
        analyzer_400_9.plot_analysis(results, target_atom_id, 400, 9)

### 400K_9 H

In [ ]:
temp, i = 400, 9
df = load_trj(f'./{temp}_diffusion_4/dump.{temp}_diffuse_{i}.trj')
Na_atom = df[(df['type'] == .0) & (df['timestep'] == 0)]
Na_ids = Na_atom['id'].to_numpy()
ints = random.sample(range(1, 1500), 500)
Na_atom.iloc[ints]
H_id_ls_400_9 = Na_ids[ints].astype(int)
H_id_ls = H_id_ls_400_9

In [ ]:
ave_n5_ls_400_9=[]
ave_n6_ls_400_9=[]
msd_ls_400_9=[]
distance_ls_400_9=[]
result_ls_400_9=[]

temp, i = 400, 9
if __name__ == "__main__":
    # Initialize analyzer
    #analyzer_400_9 = MDTrajectoryAnalyzer(f'./{temp}_diffusion_4/dump.{temp}_diffuse_{i}.trj')  # Replace with your file
    #analyzer_400_9.load_trajectory()

    for i in H_id_ls_400_9[:100]:
    #Na_id_ls_400_9[:300]:#H_id_ls_400_9[:100]:
        target_atom_id = i
        results, ave_n5, ave_n6, msd, distance = analyzer_400_9.analyze_atom(target_atom_id)
        #ave_n5, ave_n6, msd, distance = analyzer.plot_analysis(results, target_atom_id)
        result_ls_400_9.append(results)
        ave_n5_ls_400_9.append(ave_n5)
        ave_n6_ls_400_9.append(ave_n6)
        msd_ls_400_9.append(msd)
        distance_ls_400_9.append(distance)

In [ ]:
%matplotlib widget
temp, i = 400,9
if __name__ == "__main__":
    # Initialize analyzer
    analyzer_400_9 = MDTrajectoryAnalyzer(f'./{temp}_diffusion_4/dump.{temp}_diffuse_{i}.trj')  # Replace with your file
    analyzer_400_9.load_trajectory()

    for j in H_id_ls_400_9[200:210]:
        target_atom_id = j
        results, ave_n5, ave_n6, msd, distance = analyzer_400_9.analyze_atom(target_atom_id)
        analyzer_400_9.plot_analysis(results, target_atom_id, 400, 9)

### 400K_vac_1

In [ ]:
temp, i = 400, 9
df = load_trj(f'./{temp}_diffusion_20_MTP_2/dump.{temp}_diffuse_{i}.trj')
Na_atom = df[(df['type'] == 5.0) & (df['timestep'] == 0)]
Na_ids = Na_atom['id'].to_numpy()
ints = random.sample(range(1, 1500), 10)
Na_atom.iloc[ints]
H_id_ls_400_9 = Na_ids[ints].astype(int)
H_id_ls = H_id_ls_400_9

In [ ]:
Na_ids

In [ ]:
ave_n5_ls_400_9=[]
ave_n6_ls_400_9=[]
msd_ls_400_9=[]
distance_ls_400_9=[]
result_ls_400_9=[]

temp, i = 400, 9
if __name__ == "__main__":
    # Initialize analyzer
    analyzer = MDTrajectoryAnalyzer(f'./{temp}_diffusion_4/dump.{temp}_diffuse_{i}.trj')  # Replace with your file
    analyzer.load_trajectory()

    for i in Na_id_ls_400_9[40:50]:
        target_atom_id = i
        results, ave_n5, ave_n6, msd, distance = analyzer.analyze_atom(target_atom_id)
        #ave_n5, ave_n6, msd, distance = analyzer.plot_analysis(results, target_atom_id)
        result_ls_400_9.append(results)
        ave_n5_ls_400_9.append(ave_n5)
        ave_n6_ls_400_9.append(ave_n6)
        msd_ls_400_9.append(msd)
        distance_ls_400_9.append(distance)

In [ ]:
H_id_ls_400_9[2:]

In [ ]:
ave_n5_ls_400_9=[]
ave_n6_ls_400_9=[]
msd_ls_400_9=[]
distance_ls_400_9=[]
result_ls_400_9=[]

temp, i = 400, 9
if __name__ == "__main__":
    # Initialize analyzer
    analyzer_400_9 = MDTrajectoryAnalyzer(f'./{temp}_diffusion_20_MTP/dump.{temp}_diffuse_{i}.trj')  # Replace with your file
    analyzer_400_9.load_trajectory()

    for i in H_id_ls_400_9[:]:
    #Na_id_ls_400_9[:300]:#H_id_ls_400_9[:100]:
        target_atom_id = i
        results, ave_n5, ave_n6, msd, distance = analyzer_400_9.analyze_atom(target_atom_id)
        ave_n5, ave_n6, msd, distance = analyzer_400_9.plot_analysis(results, target_atom_id, 400, 9)
        #result_ls_400_9.append(results)
        #ave_n5_ls_400_9.append(ave_n5)
        #ave_n6_ls_400_9.append(ave_n6)
        #msd_ls_400_9.append(msd)
        #distance_ls_400_9.append(distance)

### 400_diffusion_vac_lv20_r02_65_2

In [ ]:
temp, i = 400, 11
df = load_trj(f'./{temp}_diffusion_vac_lv20_r02_65_2/dump.{temp}_diffuse_{i}.trj')
Na_atom = df[(df['type'] == 6.0) & (df['timestep'] == 0)]
Na_ids = Na_atom['id'].to_numpy()
ints = random.sample(range(1, 400), 10)
Na_atom.iloc[ints]
O_id_ls_400_vac_11 = Na_ids[ints].astype(int)

In [ ]:
trj = load_trj('400_diffusion_vac_lv20_r02_65_2/dump.400_diffuse_12.trj')
first_step = trj['timestep'].min()
last_step = trj['timestep'].max()

first = (
    trj.loc[(trj['timestep'] == first_step) & (trj['type'] == 6)]
    .set_index('id')[['x', 'y', 'z']]
)
last = (
    trj.loc[(trj['timestep'] == last_step) & (trj['type'] == 6)]
    .set_index('id')[['x', 'y', 'z']]
)

# Align the coordinates by atom ID
positions = first.join(last, lsuffix='_first', rsuffix='_last', how='inner')

delta = (
    positions[['x_last', 'y_last', 'z_last']].to_numpy()
    - positions[['x_first', 'y_first', 'z_first']].to_numpy()
)

positions['displacement_A'] = np.linalg.norm(delta, axis=1)

selected = positions.loc[positions['displacement_A'] > 1].copy()
selected.index = selected.index.astype(int)
selected_ids = selected.index.tolist()

print(f'{len(selected_ids)} type-6 atoms have displacement > 4 Å')
print(selected[['displacement_A']])
print('Selected atom IDs:', selected_ids)

In [ ]:
plt.figure()
plt.plot([6,5,4,3,2],[58,81,100,108,118])
plt.plot([5,4,3,2],[20,30,36,45])

In [ ]:
%matplotlib widget

In [ ]:
ave_n5_ls_400_9=[]
ave_n6_ls_400_9=[]
msd_ls_400_9=[]
distance_ls_400_9=[]
result_ls_400_9=[]

temp, i = 400, 12
if __name__ == "__main__":
    # Initialize analyzer
    #analyzer = MDTrajectoryAnalyzer(f'./{temp}_diffusion_vac_lv20_r02_65_2/dump.{temp}_diffuse_{i}.trj')  # Replace with your file
    #analyzer.load_trajectory()
    for i in [7, 3760, 4465, 861, 2209, 1022, 2292, 458, 470, 1099, 1089, 2946, 1172, 3688, 2221, 4948, 3055, 3710, 4333, 2323, 4275, 752, 2110, 2845, 3085, 2028, 3740]:
        target_atom_id = i
        results, ave_n5, ave_n6, msd, distance = analyzer.analyze_atom(target_atom_id)
        if results is None:          # skipped atom (no jumps or jump > 50 Å)
            continue
        analyzer.plot_analysis(results, target_atom_id, 400, 11)
        result_ls_400_9.append(results)
        ave_n5_ls_400_9.append(ave_n5)
        ave_n6_ls_400_9.append(ave_n6)
        msd_ls_400_9.append(msd)
        distance_ls_400_9.append(distance)

In [ ]:
result_ls_400_9 = [x for x in result_ls_400_9 if x is not None]
all_jump_distances = []
for result in result_ls_400_9:
    jump_dists = result['jump_distances']
    if isinstance(jump_dists, np.ndarray):
        if jump_dists.ndim > 0:  # Not a scalar
            all_jump_distances.extend(jump_dists.flatten())
    elif isinstance(jump_dists, (list, tuple)):
        for item in jump_dists:
            if hasattr(item, '__iter__') and not isinstance(item, str):
                all_jump_distances.extend(item)
            else:
                all_jump_distances.append(item)

all_jump_distances_400_9 = np.array(all_jump_distances)

In [ ]:
mask = np.logical_and(all_jump_distances_400_9 <= 10, all_jump_distances_400_9 > 3)
all_jump_distances_400_9 = all_jump_distances_400_9[mask]

In [ ]:
from scipy.stats import expon
plt.rcParams.update({'font.size': 16})
plt.rcParams.update({'lines.linewidth':1})
plt.rcParams.update({'axes.linewidth':1})
plt.rcParams.update({'lines.markersize':5})
plt.rcParams.update({'savefig.dpi':200})
plt.rcParams.update({'legend.fontsize': 16})

# Plot
plt.figure(figsize=(singlecol*2, singlecol*2))

# Histogram with counts (not density)
counts, bins, patches = plt.hist(all_jump_distances_400_9, bins=50, alpha=0.7, 
                                  edgecolor='black', label='Data')

plt.xlabel('Jump distance (Å)')
plt.ylabel('Counts')
plt.legend(frameon=False)
#plt.savefig('jump_distribution_400_6.svg')
plt.show()

In [ ]:
plt.figure()
plt.hist(all_jump_distances_400_9)

# Free energy

In [ ]:
def parse_print_file(file_path):
    """
    Parse a LAMMPS print file with Step and Atoms columns.
    
    Args:
        file_path (str): Path to the LAMMPS output file
        
    Returns:
        dict: Dictionary with 'Step' and 'atoms' arrays
    """
    data = {'Step': [], 'atoms': []}
    try:
        with open(file_path, 'r') as file:
            # Skip lines until we find the header
            for line in file:
                if re.match(r"^\s*Step\s+Atoms", line):
                    # Try to skip the nex|t line if it's a separator
                    next_line = next(file, '').strip()
                    if not re.match(r"^\s*\d+", next_line) and len(next_line.split()) != 2:
                        continue
                    else:
                        # If it's a data line, process it
                        values = next_line.split()
                        data['Step'].append(int(values[0]))
                        data['atoms'].append(float(values[1]))
                    
                    # Process remaining data lines
                    for data_line in file:
                        data_line = data_line.strip()
                        if re.match(r"^\s*\d+", data_line) and len(data_line.split()) == 2:
                            values = data_line.split()
                            data['Step'].append(int(values[0]))
                            data['atoms'].append(float(values[1]))
                        else:
                            break
    except Exception as e:
        print(f"Error parsing file {file_path}: {e}")
    
    # Convert lists to numpy arrays for easier manipulation
    data['Step'] = np.array(data['Step'])
    data['atoms'] = np.array(data['atoms'])
    
    return data

In [ ]:
import os

def plot_loop(temp, Nt0, mu_b, fi, step, serie, curr_N):
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12,5))
    plt.subplots_adjust(wspace=0.35)

    Nf_ls = []
    mu_ls = []

    for i in range(1, curr_N):
        print(i)
        step = step
        Nt = Nt0 - (i-1)*step
        atoms_file = f'./Nt_{temp}/{serie}/atoms_log_{i}.txt'
        
        if not os.path.exists(atoms_file) or os.path.getsize(atoms_file) == 0:
            print(f"Skipped empty file: {atoms_file}")
            continue
        
        log_data = parse_print_file(atoms_file)
        step = np.array(log_data['Step'])
        n_Na = np.array(log_data['atoms']) - 17280
        ax1.plot(step, n_Na, label=f'Nf_{i}={n_Na[-1]}, Nt={Nt}')
        ax1.plot([step[0], step[-1]], [Nt, Nt], color='black')
        
        converge_file = f'./Nt_{temp}/{serie}/{temp}_converge_{i}.txt'
        if not os.path.exists(converge_file) or os.path.getsize(converge_file) == 0:
            print(f"Skipped empty file: {converge_file}")
            continue
        
        data = np.loadtxt(converge_file, skiprows=2)
        if data.size == 0:
            print(f"Skipped empty file: {converge_file}")
            continue
        
        if data.ndim == 1:
            ts = data[0] if len(data) > 0 else 0
            Nf = data[-1] - 17280 if len(data) > 0 else 0
        else:
            ts = data[-1, 0]
            Nf = data[-1, -1] - 17280

        print(Nf,Nt)
        mu = mu_b - 2 * fi * (Nf - Nt)
        Nf_ls.append(Nf)
        mu_ls.append(mu)
        print(ts, Nf, mu)
        ax1.scatter(ts, Nf)
        ax1.text(ts, Nf-5, s=f'{Nf}', fontsize=8)
        ax2.scatter(Nf, mu)
        
    print(Nf_ls, mu_ls)
    ax2.plot(Nf_ls, mu_ls)
    plt.title(f'{temp} K mu_bias = -3')
    ax1.grid()
    ax1.set_xlabel('Timestep (0.4 fs/step)')
    ax1.set_ylabel('Number of Na+')
    ax2.set_xlabel('Number of Na+')
    ax2.set_ylabel('Chemical potential (eV)')
    plt.tight_layout()
    plt.savefig(f'./Nt_{temp}/{serie}/loop_{temp}.png')

    return Nf_ls, mu_ls


In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt

def plot_loop(temp, Nt0, mu_b, fi, step_size, serie, curr_N, vac=False):
    """
    Plot analysis of Na+ atom count vs timestep and chemical potential vs final atom count.
    
    Parameters:
    -----------
    temp : float
        Temperature in Kelvin
    Nt0 : int
        Initial target number of atoms
    mu_b : float
        Base chemical potential in eV
    fi : float
        Coupling parameter for chemical potential calculation
    step_size : int
        Step size for target atom count reduction
    serie : str
        Series identifier for file paths
    curr_N : int
        Current simulation number (upper limit for loop)
    
    Returns:
    --------
    tuple: (Nf_ls, mu_ls) - lists of final atom counts and chemical potentials
    """
    # Create figure with two subplots
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))
    plt.subplots_adjust(wspace=0.35)
    
    # Initialize lists to store results
    Nf_ls = []
    mu_ls = []
    
    # Loop through simulation runs
    for i in range(1, curr_N):
        #print(f"Processing run {i}/{curr_N-1}")
        
        # Calculate target number of atoms for this run
        Nt = Nt0 + (i-1) * step_size

        if vac == False:
            # Define file paths
            atoms_file = f'./Nt_{temp}/{serie}/atoms_log_{i}.txt'
            converge_file = f'./Nt_{temp}/{serie}/{temp}_converge_{i}.txt'
        else:
            atoms_file = f'./Nt_{temp}_vac/{serie}/atoms_log_{i}.txt'
            converge_file = f'./Nt_{temp}_vac/{serie}/{temp}_converge_{i}.txt'
            
        # Check and process atoms log file
        if not os.path.exists(atoms_file) or os.path.getsize(atoms_file) == 0:
            print(f"Warning: Skipped empty or missing file: {atoms_file}")
            continue
        
        try:
            # Parse atom count data over time
            log_data = parse_print_file(atoms_file)
            timesteps = np.array(log_data['Step'])
            n_Na = np.array(log_data['atoms']) - 17280  # Subtract baseline
            
            # Plot time evolution of atom count
            ax1.plot(timesteps, n_Na, label=f'Run {i}: Nf={n_Na[-1]}, Nt={Nt}')
            ax1.axhline(y=Nt, color='black', linestyle='-', alpha=0.5, linewidth=0.8)
            
        except Exception as e:
            print(f"Error processing {atoms_file}: {e}")
            continue
        
        # Check and process convergence file
        if not os.path.exists(converge_file) or os.path.getsize(converge_file) == 0:
            print(f"Warning: Skipped empty or missing file: {converge_file}")
            continue
        
        try:
            # Load convergence data
            data = np.loadtxt(converge_file, skiprows=2)
            if data.size == 0:
                print(f"Warning: No data in file: {converge_file}")
                continue
            
            # Extract final timestep and atom count
            if data.ndim == 1:
                ts_final = data[0] if len(data) > 0 else 0
                Nf = data[-1] - 17280 if len(data) > 0 else 0
                Nf = n_Na[:].mean()
            else:
                ts_final = data[-1, 0]
                Nf = data[-1, -1] - 17280
                Nf = n_Na[:].mean()
            
            # Calculate chemical potential
            mu = mu_b - 2 * fi * (Nf - Nt)
            
            # Store results
            Nf_ls.append(Nf)
            mu_ls.append(mu)
            
            print(f"{i}  Final: ts={ts_final}, Nf={Nf}, Nt={Nt}, mu={mu:.4f}")
            
            # Add markers and annotations to plots
            ax1.scatter(ts_final, Nf, color='red', s=30, zorder=5)
            ax1.annotate(f'{Nf}', (ts_final, Nf), xytext=(5, 5), 
                        textcoords='offset points', fontsize=8, 
                        bbox=dict(boxstyle='round,pad=0.3', facecolor='white', alpha=0.7))
            
            ax2.scatter(Nf, mu, s=50, alpha=0.7)
            
        except Exception as e:
            print(f"Error processing {converge_file}: {e}")
            continue
    
    # Plot chemical potential trend
    if len(Nf_ls) > 1:
        ax2.plot(Nf_ls, mu_ls, 'b-', linewidth=2, alpha=0.7)
    
    # Format plots
    ax1.grid(True, alpha=0.3)
    ax1.set_xlabel('Timestep (0.4 fs/step)')
    ax1.set_ylabel('Number of Na+ (relative to baseline)')
    ax1.set_title(f'Atom Count Evolution - {temp} K')
    #ax1.legend(bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=8)
    
    ax2.grid(True, alpha=0.3)
    ax2.set_xlabel('Final Number of Na+ (relative to baseline)')
    ax2.set_ylabel('Chemical Potential (eV)')
    ax2.set_title(f'Chemical Potential vs Final Count - {temp} K')
    
    # Add overall title
    fig.suptitle(f'Simulation Analysis: {temp} K, μ_bias = {mu_b} eV', fontsize=14)
    
    plt.tight_layout()
    
    # Save figure
    output_dir = f'./Nt_{temp}/{serie}/'
    os.makedirs(output_dir, exist_ok=True)
    plt.savefig(f'{output_dir}loop_{temp}.png', dpi=300, bbox_inches='tight')
    plt.show()
    
    #print(f"\nSummary:")
    #print(f"  Final atom counts: {Nf_ls}")
    #print(f"  Chemical potentials: {[f'{mu:.4f}' for mu in mu_ls]}")
    
    return Nf_ls, mu_ls

# Additional helper function for better error handling
def safe_file_check(filepath):
    """Safely check if file exists and has content."""
    return os.path.exists(filepath) and os.path.getsize(filepath) > 0

### plot_loop_vac

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt

def plot_loop_vac(temp, Nt0, mu_b, fi, step_size, serie, curr_N, num, vac=False):
    """
    Plot analysis of Na+ atom count vs timestep and chemical potential vs final atom count.
    
    Parameters:
    -----------
    temp : float
        Temperature in Kelvin
    Nt0 : int
        Initial target number of atoms
    mu_b : float
        Base chemical potential in eV
    fi : float
        Coupling parameter for chemical potential calculation
    step_size : int
        Step size for target atom count reduction
    serie : str
        Series identifier for file paths
    curr_N : int
        Current simulation number (upper limit for loop)
    
    Returns:
    --------
    tuple: (Nf_ls, mu_ls) - lists of final atom counts and chemical potentials
    """
    # Create figure with two subplots
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))
    plt.subplots_adjust(wspace=0.35)
    
    # Initialize lists to store results
    Nf_ls = []
    mu_ls = []
    
    # Loop through simulation runs
    for i in range(1, curr_N):
        #print(f"Processing run {i}/{curr_N-1}")
        
        # Calculate target number of atoms for this run
        Nt = Nt0 + (i-1) * step_size

        if vac == False:
            # Define file paths
            atoms_file = f'./Nt_{temp}/{serie}/atoms_log_{i}.txt'
            converge_file = f'./Nt_{temp}/{serie}/{temp}_converge_{i}.txt'
        else:
            atoms_file = f'./Nt_{temp}_vac/{serie}/atoms_log_{i}.txt'
            converge_file = f'./Nt_{temp}_vac/{serie}/{temp}_converge_{i}.txt'
            
        # Check and process atoms log file
        if not os.path.exists(atoms_file) or os.path.getsize(atoms_file) == 0:
            print(f"Warning: Skipped empty or missing file: {atoms_file}")
            continue
        
        try:
            # Parse atom count data over time
            log_data = parse_print_file(atoms_file)
            timesteps = np.array(log_data['Step'])
            
            if vac == False:
                n_Na = np.array(log_data['atoms']) - 17280  # Subtract baseline
            else:
                n_Na = np.array(log_data['atoms']) - 9872
            
            # Plot time evolution of atom count
            ax1.plot(timesteps, n_Na, label=f'Run {i}: Nf={n_Na[-1]}, Nt={Nt}')
            ax1.axhline(y=Nt, color='black', linestyle='-', alpha=0.5, linewidth=0.8)
            
        except Exception as e:
            print(f"Error processing {atoms_file}: {e}")
            continue
        
        # Check and process convergence file
        if not os.path.exists(converge_file) or os.path.getsize(converge_file) == 0:
            print(f"Warning: Skipped empty or missing file: {converge_file}")
            continue
        
        try:
            # Load convergence data
            data = np.loadtxt(converge_file, skiprows=2)
            if data.size == 0:
                print(f"Warning: No data in file: {converge_file}")
                continue
            
            # Extract final timestep and atom count
            if data.ndim == 1:
                ts_final = data[0] if len(data) > 0 else 0
                Nf = data[-1] - 9872 if len(data) > 0 else 0
                Nf = n_Na[-num:].mean()
            else:
                ts_final = data[-1, 0]
                Nf = data[-1, -1] - 9872
                Nf = n_Na[-num:].mean()
            
            # Calculate chemical potential
            mu = mu_b - 2 * fi * (Nf - Nt)
            
            # Store results
            Nf_ls.append(Nf)
            mu_ls.append(mu)
            
            print(f"{i}  Final: ts={ts_final}, Nf={Nf}, Nt={Nt}, mu={mu:.4f}")
            
            # Add markers and annotations to plots
            ax1.scatter(ts_final, Nf, color='red', s=30, zorder=5)
            ax1.annotate(f'{Nf}', (ts_final, Nf), xytext=(5, 5), 
                        textcoords='offset points', fontsize=8, 
                        bbox=dict(boxstyle='round,pad=0.3', facecolor='white', alpha=0.7))
            
            ax2.scatter(Nf, mu, s=50, alpha=0.7)
            
        except Exception as e:
            print(f"Error processing {converge_file}: {e}")
            continue
    
    # Plot chemical potential trend
    if len(Nf_ls) > 1:
        ax2.plot(Nf_ls, mu_ls, 'b-', linewidth=2, alpha=0.7)
    
    # Format plots
    ax1.grid(True, alpha=0.3)
    ax1.set_xlabel('Timestep (0.4 fs/step)')
    ax1.set_ylabel('Number of Na+ (relative to baseline)')
    ax1.set_title(f'Atom Count Evolution - {temp} K')
    #ax1.legend(bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=8)
    
    ax2.grid(True, alpha=0.3)
    ax2.set_xlabel('Final Number of Na+ (relative to baseline)')
    ax2.set_ylabel('Chemical Potential (eV)')
    ax2.set_title(f'Chemical Potential vs Final Count - {temp} K')
    
    # Add overall title
    fig.suptitle(f'Simulation Analysis: {temp} K, μ_bias = {mu_b} eV', fontsize=14)
    
    plt.tight_layout()
    
    # Save figure
    output_dir = f'./Nt_{temp}/{serie}/'
    os.makedirs(output_dir, exist_ok=True)
    plt.savefig(f'{output_dir}loop_{temp}.png', dpi=300, bbox_inches='tight')
    plt.show()
    
    #print(f"\nSummary:")
    #print(f"  Final atom counts: {Nf_ls}")
    #print(f"  Chemical potentials: {[f'{mu:.4f}' for mu in mu_ls]}")
    
    return Nf_ls, mu_ls

# Additional helper function for better error handling
def safe_file_check(filepath):
    """Safely check if file exists and has content."""
    return os.path.exists(filepath) and os.path.getsize(filepath) > 0

In [ ]:
fig, (ax1) = plt.subplots(1, 1, figsize=(singlecol, singlecol),dpi=200)
plt.subplots_adjust(wspace=0.35)
#300, 1700, -0.5, 0.005, -25, 1, 71
#300, 1700, -2.5, 0.005, -25, 4, 71
temp, Nt0, mu_b, fi, step_size, serie, curr_N = 300, 1700, -2.5, 0.005, -25, 5, 70
cmap = plt.get_cmap("viridis")
# Initialize lists to store results
Nf_ls = []
mu_ls = []

# Loop through simulation runs
for i in range(1, curr_N):
    color = cmap(i / curr_N)
    
    #print(f"Processing run {i}/{curr_N-1}")
    
    # Calculate target number of atoms for this run
    Nt = Nt0 + (i-1) * step_size
    
    # Define file paths
    atoms_file = f'./Nt_{temp}/{serie}/atoms_log_{i}.txt'
    converge_file = f'./Nt_{temp}/{serie}/{temp}_converge_{i}.txt'
    
    # Check and process atoms log file
    if not os.path.exists(atoms_file) or os.path.getsize(atoms_file) == 0:
        print(f"Warning: Skipped empty or missing file: {atoms_file}")
        continue
    
    try:
        # Parse atom count data over time
        log_data = parse_print_file(atoms_file)
        timesteps = np.array(log_data['Step'])
        n_Na = np.array(log_data['atoms']) - 17280  # Subtract baseline
        
        # Plot time evolution of atom count
        ax1.plot(timesteps, n_Na/864, label=f'Run {i}: Nf={n_Na[-1]}, Nt={Nt}', color=color)
        #ax1.axhline(y=Nt, color='black', linestyle='-', alpha=0.5, linewidth=0.8)
        
    except Exception as e:
        print(f"Error processing {atoms_file}: {e}")
        continue
    
    # Check and process convergence file
    if not os.path.exists(converge_file) or os.path.getsize(converge_file) == 0:
        print(f"Warning: Skipped empty or missing file: {converge_file}")
        continue
    
    try:
        # Load convergence data
        data = np.loadtxt(converge_file, skiprows=2)
        if data.size == 0:
            print(f"Warning: No data in file: {converge_file}")
            continue
        
        # Extract final timestep and atom count
        if data.ndim == 1:
            ts_final = data[0] if len(data) > 0 else 0
            Nf = data[-1] - 17280 if len(data) > 0 else 0
        else:
            ts_final = data[-1, 0]
            Nf = data[-1, -1] - 17280
        
        # Calculate chemical potential
        mu = mu_b - 2 * fi * (Nf - Nt)
        
        # Store results
        Nf_ls.append(Nf)
        mu_ls.append(mu)
        
        print(f"{i}  Final: ts={ts_final}, Nf={Nf}, Nt={Nt}, mu={mu:.4f}")
        
        # Add markers and annotations to plots
        ax1.scatter(ts_final, Nf/864, color='red', s=10, zorder=5)
        #ax1.annotate(f'{Nf}', (ts_final, Nf), xytext=(5, 5), 
        #            textcoords='offset points', fontsize=8, 
        #            bbox=dict(boxstyle='round,pad=0.3', facecolor='white', alpha=0.7))
        
        
    except Exception as e:
        print(f"Error processing {converge_file}: {e}")
        continue

# Format plots
#ax1.grid(True, alpha=0.3)
ax1.set_xlabel('Timestep (0.4 fs/step)')
#ax1.set_xlim(0,3e5)
ax1.set_ylabel('x')
ax1.ticklabel_format(style='sci', axis='x', scilimits=(0,0))
ax1.set_ylabel(r"$x$ in Na$_x$Mn[Fe(CN)$_6$]$\cdot$YH$_2$O")
#ax1.set_title(f'Atom Count Evolution - {temp} K')
#ax1.legend(bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=8)
plt.tight_layout()

# Save figure
plt.savefig(f'./free_E_loop_{temp}_{serie}.svg', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
timesteps

In [ ]:
fig, (ax1) = plt.subplots(1, 1, figsize=(singlecol, singlecol), dpi=200)
plt.subplots_adjust(wspace=0.35)

temp, Nt0, mu_b, fi, step_size, curr_N = 300, 1700, -2.5, 0.005, -25, 71
series_list = [2, 4]  # concatenate series 3 then 4

cmap = plt.get_cmap("viridis")

# Initialize lists to store results
Nf_ls = []
mu_ls = []

# Build a flat list of (global_index, serie, local_i, Nt) across both series
runs = []
for serie in series_list:
    for i in range(1, curr_N):
        Nt = Nt0 + (i - 1) * step_size
        runs.append((serie, i, Nt))

total_runs = len(runs)

for global_idx, (serie, i, Nt) in enumerate(runs):
    color = cmap(global_idx / total_runs)

    atoms_file    = f'./Nt_{temp}/{serie}/atoms_log_{i}.txt'
    converge_file = f'./Nt_{temp}/{serie}/{temp}_converge_{i}.txt'

    # --- atoms log ---
    if not os.path.exists(atoms_file) or os.path.getsize(atoms_file) == 0:
        print(f"Warning: Skipped empty or missing file: {atoms_file}")
        continue

    try:
        log_data  = parse_print_file(atoms_file)
        timesteps = np.array(log_data['Step'])
        n_Na      = np.array(log_data['atoms']) - 17280

        ax1.plot(timesteps, n_Na / 864,
                 label=f'S{serie} Run {i}: Nf={n_Na[-1]}, Nt={Nt}',
                 color=color)
    except Exception as e:
        print(f"Error processing {atoms_file}: {e}")
        continue

    # --- convergence file ---
    if not os.path.exists(converge_file) or os.path.getsize(converge_file) == 0:
        print(f"Warning: Skipped empty or missing file: {converge_file}")
        continue

    try:
        data = np.loadtxt(converge_file, skiprows=2)
        if data.size == 0:
            print(f"Warning: No data in file: {converge_file}")
            continue

        if data.ndim == 1:
            ts_final = data[0]  if len(data) > 0 else 0
            Nf       = data[-1] - 17280 if len(data) > 0 else 0
        else:
            ts_final = data[-1, 0]
            Nf       = data[-1, -1] - 17280

        mu = mu_b - 2 * fi * (Nf - Nt)

        Nf_ls.append(Nf)
        mu_ls.append(mu)

        print(f"S{serie}-{i}  Final: ts={ts_final}, Nf={Nf}, Nt={Nt}, mu={mu:.4f}")

        ax1.scatter(ts_final, Nf / 864, color='red', s=10, zorder=5)

    except Exception as e:
        print(f"Error processing {converge_file}: {e}")
        continue

# Format
ax1.set_xlabel('Timestep (0.4 fs/step)')
ax1.set_ylabel(r"$x$ in Na$_x$Mn[Fe(CN)$_6$]$\cdot$YH$_2$O")
ax1.ticklabel_format(style='sci', axis='x', scilimits=(0, 0))

plt.tight_layout()
plt.savefig(f'./free_E_loop_{temp}_s3s4.svg', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
fig, (ax1) = plt.subplots(1, 1, figsize=(singlecol, singlecol), dpi=200)
plt.subplots_adjust(wspace=0.35)

temp, Nt0, mu_b, fi, step_size, curr_N = 300, 1700, -2.5, 0.005, -25, 71
series_list = [2, 4]  # concatenate series 3 then 4

cmap = plt.get_cmap("viridis")

# Initialize lists to store results
Nf_ls = []
mu_ls = []

# Build a flat list of (global_index, serie, local_i, Nt) across both series
runs = []
for serie in series_list:
    for i in range(1, curr_N):
        Nt = Nt0 + (i - 1) * step_size
        runs.append((serie, i, Nt))

total_runs = len(runs)

for global_idx, (serie, i, Nt) in enumerate(runs):
    color = cmap(global_idx / total_runs)

    atoms_file    = f'./Nt_{temp}/{serie}/atoms_log_{i}.txt'
    converge_file = f'./Nt_{temp}/{serie}/{temp}_converge_{i}.txt'

    # --- atoms log ---
    if not os.path.exists(atoms_file) or os.path.getsize(atoms_file) == 0:
        print(f"Warning: Skipped empty or missing file: {atoms_file}")
        continue

    try:
        log_data  = parse_print_file(atoms_file)
        timesteps = np.array(log_data['Step'])
        n_Na      = np.array(log_data['atoms']) - 17280

        ax1.plot(timesteps, n_Na / 864,
                 label=f'S{serie} Run {i}: Nf={n_Na[-1]}, Nt={Nt}',
                 color=color)
    except Exception as e:
        print(f"Error processing {atoms_file}: {e}")
        continue

    # --- convergence file ---
    if not os.path.exists(converge_file) or os.path.getsize(converge_file) == 0:
        print(f"Warning: Skipped empty or missing file: {converge_file}")
        continue

    try:
        data = np.loadtxt(converge_file, skiprows=2)
        if data.size == 0:
            print(f"Warning: No data in file: {converge_file}")
            continue

        if data.ndim == 1:
            ts_final = data[0]  if len(data) > 0 else 0
            Nf       = data[-1] - 17280 if len(data) > 0 else 0
        else:
            ts_final = data[-1, 0]
            Nf       = data[-1, -1] - 17280

        mu = mu_b - 2 * fi * (Nf - Nt)

        Nf_ls.append(Nf)
        mu_ls.append(mu)

        print(f"S{serie}-{i}  Final: ts={ts_final}, Nf={Nf}, Nt={Nt}, mu={mu:.4f}")

        ax1.scatter(ts_final, Nf / 864, color='red', s=10, zorder=5)

    except Exception as e:
        print(f"Error processing {converge_file}: {e}")
        continue

# Format
ax1.set_xlabel('Timestep (0.4 fs/step)')
ax1.set_ylabel(r"$x$ in Na$_x$Mn[Fe(CN)$_6$]$\cdot$YH$_2$O")
ax1.ticklabel_format(style='sci', axis='x', scilimits=(0, 0))

plt.tight_layout()
plt.savefig(f'./free_E_loop_{temp}_s3s4.svg', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
%matplotlib inline

In [ ]:
file = f'./RDF_lv16_E0_sf0.1_f0.1_2/Na_32_O_32_temp.log'
plot_log_free(file)

In [ ]:
data = parse_lammps_free(f'Nt_300/5/loop_bias_5.log_{i}')

In [ ]:
data

In [ ]:
%matplotlib inline
for i in [1,4]:
    plot_log_free_E(f'Nt_300/1/loop_bias_1.log_{i}')

### 300K_1

In [ ]:
%matplotlib widget
Nf_ls_300_1, mu_ls_300_1 = plot_loop_vac(300, 1000, -4, 0.005, -20, 1, 51)

In [ ]:
%matplotlib widget
Nf_ls_300_1, mu_ls_300_1 = plot_loop_vac(300, 1000, -4, 0.005, -20, 1, 50, True)

### 300K_2

In [ ]:
%matplotlib widget
Nf_ls_300_2, mu_ls_300_2 = plot_loop(300, 1700, -2.5, 0.005, -25, 2, 71)

In [ ]:
%matplotlib widget
Nf_ls_300_3, mu_ls_300_3 = plot_loop(300, 1700, -2.5, 0.005, -25, 3, 71)

In [ ]:
%matplotlib widget
Nf_ls_300_4, mu_ls_300_4 = plot_loop(300, 1700, -2.5, 0.005, -25, 4, 70)

In [ ]:
%matplotlib widget
Nf_ls_300_5, mu_ls_300_5 = plot_loop_vac(300, 1700, -2.5, 0.005, -25, 5, 71, -500)

### 300_6

In [ ]:
%matplotlib widget
Nf_ls_300_6, mu_ls_300_6 = plot_loop_vac(300, 1700, -4, 0.005, -25, 6, 70, 500)

In [ ]:
fig, (ax1) = plt.subplots(1, 1, figsize=(singlecol, singlecol),dpi=200)
colors=['tab:blue','tab:orange','tab:green']
Nf_ls_300_3= np.array(Nf_ls_300_3)
ax1.plot(Nf_ls_300_3/864, mu_ls_300_3, label='3rd sampling', color=colors[2], lw=0.8)
Nf_ls_300_4= np.array(Nf_ls_300_4)
ax1.plot(Nf_ls_300_4/864, mu_ls_300_4, label='4th sampling', color=colors[0], lw=0.8)

#ax1.plot(Nf_ls_300_4/864, (np.array(mu_ls_300_3)+np.array(mu_ls_300_4))/2, label='3+4th sampling', color=colors[0], lw=0.8)
ax1.legend(frameon=False)

ax1.set_ylim(-3,-2.3)

In [ ]:
fig, (ax1) = plt.subplots(1, 1, figsize=(singlecol, singlecol),dpi=200)
colors=['tab:blue','tab:orange','tab:green']
Nf_ls_300_5= np.array(Nf_ls_300_5)
ax1.plot(Nf_ls_300_5/864, mu_ls_300_5, label='5th sampling', color=colors[2], lw=0.8)
Nf_ls_300_4= np.array(Nf_ls_300_4)
ax1.plot(Nf_ls_300_4/864, mu_ls_300_4, label='4th sampling', color=colors[0], lw=0.8)
ax1.legend(frameon=False)


In [ ]:
d

In [ ]:
fig, (ax1) = plt.subplots(1, 1, figsize=(singlecol, singlecol),dpi=200)
plt.subplots_adjust(wspace=0.35)

colors=['tab:blue','tab:orange','tab:green', 'tab:purple']

Nf_ls_300_1= np.array(Nf_ls_300_1)
ax1.scatter(Nf_ls_300_1/864, mu_ls_300_1, alpha=0.3, color=colors[0], s=5)
ax1.plot(Nf_ls_300_1/864, mu_ls_300_1, label='1st sampling', color=colors[0], lw=0.8)

Nf_ls_300_2= np.array(Nf_ls_300_2)
ax1.scatter(Nf_ls_300_2/864, mu_ls_300_2, alpha=0.3, color=colors[1], s=5)
ax1.plot(Nf_ls_300_2/864, mu_ls_300_2, label='2nd sampling' , color=colors[1], lw=0.8)

#Nf_ls_300_3= np.array(Nf_ls_300_3)
#ax1.scatter(Nf_ls_300_3/864, mu_ls_300_3, alpha=0.3, color=colors[2], s=5)
#ax1.plot(Nf_ls_300_3/864, mu_ls_300_3, label='3rd sampling', color=colors[2], lw=0.8)

Nf_ls_300_5= np.array(Nf_ls_300_5)
ax1.scatter(Nf_ls_300_5/864, mu_ls_300_5, alpha=0.3, color=colors[2], s=5)
ax1.plot(Nf_ls_300_5/864, mu_ls_300_5, label='5th sampling', color=colors[2], lw=0.8)

ax1.set_ylim(-3,-2.2)
ax1.set_xlabel(r"$x$ in Na$_x$Mn[Fe(CN)$_6$]$\cdot$YH$_2$O")
ax1.set_ylabel('Sodium chemical potential (eV)')
ax1.legend(frameon=False)

plt.tight_layout()
plt.savefig('sodium_mu.svg')
#for i in ls:
    #plt.scatter(Nf_ls_300_2[i-1], mu_ls_300_2[i-1], color='r')

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.interpolate import CubicSpline
from scipy.integrate import cumulative_trapezoid

x = np.array(Nf_ls_300_5/864)
y = np.array(mu_ls_300_5)

# Your x is decreasing, CubicSpline requires increasing x
idx = np.argsort(x)
x = x[idx]
y = np.array(y)[idx]

# Fit spline
spline = CubicSpline(x, y)

# Fine grid
x_fit = np.linspace(x.min(), x.max(), 1000)
y_fit = spline(x_fit)

# Cumulative integral
integral = cumulative_trapezoid(y_fit, x_fit, initial=)

# Plot
fig, ax = plt.subplots(1, 2, figsize=(10,4))

# Fit
ax[0].scatter(x, y, color='k', s=20, label='Data')
ax[0].plot(x_fit, y_fit, 'r', lw=2, label='Spline fit')
ax[0].set_xlabel('x')
ax[0].set_ylabel('y')
ax[0].legend()

x0, x1 = x_fit[0], x_fit[-1]
y0, y1 = integral[0], integral[-1]
# Linear function
slope = (y1 - y0) / (x1 - x0)
linear = y0 + slope * (x_fit - x0)

# Difference
diff = integral - linear

# Integral
ax[1].plot(x_fit, integral, 'b', lw=2, label='Integral')
ax[1].plot([x_fit[0], x_fit[-1]],
           [integral[0], integral[-1]],
           '-r', lw=2, label='Linear')

ax12 = ax[1].twinx()
ax12.plot(x_fit, diff, 'k', lw=2, label='Difference')

ax[1].set_xlabel('x')
ax[1].set_ylabel(r'$\int y(x)\,dx$')

plt.tight_layout()
plt.show()

In [ ]:
[x_fit[0], x_fit[-1]]
[integral[0], integral[-1]]

In [ ]:
Nf_ls_400_3, mu_ls_400_3 = plot_loop(400, 1700, -0.5, 0.005, -25, 3, 71)

In [ ]:
mu_ls_300

In [ ]:
Nf_ls_300_6, mu_ls_300_6 = plot_loop(300, 1700, -0.5, 0.01, -25, 6, 71)

In [ ]:
Nf_ls_300_7, mu_ls_300_7 = plot_loop(300, 1700, -0.5, 0.01, -25, 7, 71)

In [ ]:
Nf_ls_300_8, mu_ls_300_8 = plot_loop(300, 1700, -0.5, 0.005, -25, 8, 71)

In [ ]:
Nf_ls_400_1, mu_ls_400_1 = plot_loop(400, 1700, -0.5, 0.005, -25, 1, 71)

In [ ]:
Nf_ls_400_1, mu_ls_400_1 = plot_loop(400, 1700, -2.5, 0.005, -25, 2, 71)

In [ ]:
plt.figure()
plt.plot(Nf_ls_300_1, mu_ls_300_1, label='1')
plt.plot(Nf_ls_300_2, mu_ls_300_2, label='2')

In [ ]:
plt.figure()
plt.plot(Nf_ls_400_1, mu_ls_400_1, label='1')
plt.plot(Nf_ls_400_2, mu_ls_400_2, label='1')
plt.plot(Nf_ls_400_3, mu_ls_400_3, label='2')

In [ ]:
plt.figure()
plt.plot(Nf_ls_300_1, mu_ls_300_1, label='1')
plt.plot(Nf_ls_300_2, mu_ls_300_2, label='2')
#plt.plot(Nf_ls_300_6, mu_ls_300_6, label='6')
plt.plot(Nf_ls_300_7, mu_ls_300_7, label='7')
plt.plot(Nf_ls_300_8, mu_ls_300_8, label='8')
plt.plot(Nf_ls_400_1, mu_ls_400_1, label='1')
plt.legend()

## Modify data file

### Remove H2O

In [ ]:
import numpy as np
import random
from collections import defaultdict

class LAMMPSWaterRemover:
    def __init__(self, filename):
        self.filename = filename
        self.atoms = []
        self.header = []
        self.footer = []
        self.atom_types = {1: 'C', 2: 'Mn', 3: 'Fe', 4: 'N', 5: 'Na', 6: 'O', 7: 'H'}
        
    def read_lammps_data(self):
        """Read LAMMPS data file and parse atoms"""
        with open(self.filename, 'r') as f:
            lines = f.readlines()
        
        # Find Atoms section
        atoms_start = None
        atoms_end = None
        
        for i, line in enumerate(lines):
            if 'Atoms' in line:
                atoms_start = i + 2  # Skip header and blank line
                break
        
        if atoms_start is None:
            raise ValueError("Could not find 'Atoms' section in the file")
        
        # Find end of atoms section (blank line or end of file)
        atoms_end = len(lines)
        for i in range(atoms_start, len(lines)):
            if lines[i].strip() == '':
                atoms_end = i
                break
        
        # Store header (everything before Atoms section)
        self.header = lines[:atoms_start-2]
        
        # Parse atoms
        for i in range(atoms_start, atoms_end):
            line = lines[i].strip()
            if line:
                parts = line.split()
                atom_id = int(parts[0])
                atom_type = int(parts[1])
                x, y, z = float(parts[2]), float(parts[3]), float(parts[4])
                
                self.atoms.append({
                    'id': atom_id,
                    'type': atom_type,
                    'x': x, 'y': y, 'z': z
                })
        
        # Store footer (everything after atoms section)
        self.footer = lines[atoms_end:]
        
        print(f"Read {len(self.atoms)} atoms")
    
    def calculate_distance(self, atom1, atom2):
        """Calculate Euclidean distance between two atoms"""
        dx = atom1['x'] - atom2['x']
        dy = atom1['y'] - atom2['y']
        dz = atom1['z'] - atom2['z']
        return np.sqrt(dx*dx + dy*dy + dz*dz)
    
    def find_water_molecules(self, oh_distance_cutoff=1.2, hoh_angle_tolerance=30.0):
        """
        Identify water molecules using distance-based criteria
        
        Parameters:
        - oh_distance_cutoff: Maximum O-H bond distance (default 1.2 Å)
        - hoh_angle_tolerance: Tolerance for H-O-H angle deviation from 104.5° (default ±30°)
        """
        # Separate atoms by type
        oxygen_atoms = [atom for atom in self.atoms if atom['type'] == 6]
        hydrogen_atoms = [atom for atom in self.atoms if atom['type'] == 7]
        
        print(f"Found {len(oxygen_atoms)} oxygen atoms and {len(hydrogen_atoms)} hydrogen atoms")
        
        water_molecules = []
        used_hydrogens = set()
        
        for o_atom in oxygen_atoms:
            # Find hydrogen atoms within bonding distance
            nearby_hydrogens = []
            
            for h_atom in hydrogen_atoms:
                if h_atom['id'] not in used_hydrogens:
                    distance = self.calculate_distance(o_atom, h_atom)
                    if distance <= oh_distance_cutoff:
                        nearby_hydrogens.append((h_atom, distance))
            
            # Sort by distance and take the closest ones
            nearby_hydrogens.sort(key=lambda x: x[1])
            
            # Check if we have exactly 2 hydrogens for a water molecule
            if len(nearby_hydrogens) >= 2:
                h1, h1_dist = nearby_hydrogens[0]
                h2, h2_dist = nearby_hydrogens[1]
                
                # Optional: Check H-O-H angle (approximately 104.5°)
                if self._is_valid_water_geometry(o_atom, h1, h2, hoh_angle_tolerance):
                    water_mol = {
                        'oxygen': o_atom,
                        'hydrogens': [h1, h2],
                        'oh_distances': [h1_dist, h2_dist]
                    }
                    water_molecules.append(water_mol)
                    
                    # Mark hydrogens as used
                    used_hydrogens.add(h1['id'])
                    used_hydrogens.add(h2['id'])
        
        print(f"Identified {len(water_molecules)} water molecules")
        return water_molecules
    
    def _is_valid_water_geometry(self, o_atom, h1_atom, h2_atom, angle_tolerance):
        """
        Check if the H-O-H angle is reasonable for a water molecule
        Expected angle is ~104.5°
        """
        # Vectors from O to each H
        vec1 = np.array([h1_atom['x'] - o_atom['x'], 
                        h1_atom['y'] - o_atom['y'], 
                        h1_atom['z'] - o_atom['z']])
        vec2 = np.array([h2_atom['x'] - o_atom['x'], 
                        h2_atom['y'] - o_atom['y'], 
                        h2_atom['z'] - o_atom['z']])
        
        # Calculate angle between vectors
        dot_product = np.dot(vec1, vec2)
        norms = np.linalg.norm(vec1) * np.linalg.norm(vec2)
        
        if norms == 0:
            return False
        
        cos_angle = dot_product / norms
        # Clamp to valid range for arccos
        cos_angle = np.clip(cos_angle, -1.0, 1.0)
        angle_rad = np.arccos(cos_angle)
        angle_deg = np.degrees(angle_rad)
        
        # Water H-O-H angle is approximately 104.5°
        expected_angle = 104.5
        return abs(angle_deg - expected_angle) <= angle_tolerance
    
    def remove_random_water(self, num_to_remove, oh_distance_cutoff=1.2, hoh_angle_tolerance=30.0):
        """Remove a specified number of random water molecules"""
        water_molecules = self.find_water_molecules(oh_distance_cutoff, hoh_angle_tolerance)
        
        if num_to_remove > len(water_molecules):
            print(f"Warning: Requested to remove {num_to_remove} water molecules, but only {len(water_molecules)} found")
            num_to_remove = len(water_molecules)
        
        if num_to_remove == 0:
            print("No water molecules to remove")
            return 0
        
        # Randomly select water molecules to remove
        molecules_to_remove = random.sample(water_molecules, num_to_remove)
        
        # Collect atom IDs to remove
        atoms_to_remove = set()
        
        for mol in molecules_to_remove:
            atoms_to_remove.add(mol['oxygen']['id'])
            for h_atom in mol['hydrogens']:
                atoms_to_remove.add(h_atom['id'])
        
        print(f"Removing {num_to_remove} water molecules ({len(atoms_to_remove)} atoms)")
        
        # Remove atoms
        original_atom_count = len(self.atoms)
        self.atoms = [atom for atom in self.atoms if atom['id'] not in atoms_to_remove]
        
        print(f"Removed {original_atom_count - len(self.atoms)} atoms")
        
        # Renumber atoms
        self._renumber_atoms()
        
        return len(atoms_to_remove)
    
    def _renumber_atoms(self):
        """Renumber atoms after removal"""
        for i, atom in enumerate(self.atoms):
            atom['id'] = i + 1
    
    def write_lammps_data(self, output_filename):
        """Write modified LAMMPS data file"""
        with open(output_filename, 'w') as f:
            # Write header with updated atom count
            for line in self.header:
                if 'atoms' in line.lower():
                    f.write(f"{len(self.atoms)} atoms\n")
                else:
                    f.write(line)
            
            # Write Atoms section
            f.write("Atoms # atomic\n\n")
            for atom in self.atoms:
                f.write(f"{atom['id']} {atom['type']} {atom['x']:.6f} {atom['y']:.6f} {atom['z']:.6f}\n")
            
            # Write footer
            for line in self.footer:
                f.write(line)
        
        print(f"Written modified structure to {output_filename}")
    
    def analyze_structure(self):
        """Analyze the structure and provide statistics"""
        type_counts = defaultdict(int)
        for atom in self.atoms:
            type_counts[atom['type']] += 1
        
        print("\nStructure Analysis:")
        print("-" * 30)
        for atom_type, count in sorted(type_counts.items()):
            element = self.atom_types.get(atom_type, f"Type{atom_type}")
            print(f"{element} (type {atom_type}): {count} atoms")
        
        # Estimate potential water molecules
        oxygen_count = type_counts.get(6, 0)
        hydrogen_count = type_counts.get(7, 0)
        max_possible_water = min(oxygen_count, hydrogen_count // 2)
        print(f"\nMaximum possible water molecules: {max_possible_water}")

# Example usage
def main():
    # Initialize the water remover
    remover = LAMMPSWaterRemover('Na_28_O_32.data')
    
    # Read the LAMMPS data file
    remover.read_lammps_data()
    
    # Analyze the structure
    remover.analyze_structure()
    
    # Remove 10 random water molecules
    # You can adjust the distance cutoff and angle tolerance as needed
    num_removed_atoms = remover.remove_random_water(
        num_to_remove=864,
        oh_distance_cutoff=1.5,    # Å
        hoh_angle_tolerance=50.0   # degrees
    )
    
    # Write the modified structure
    remover.write_lammps_data('test_Na_28_O_16.data')
    
    print(f"\nSummary:")
    print(f"Atoms removed: {num_removed_atoms}")
    print(f"Final atom count: {len(remover.atoms)}")

if __name__ == "__main__":
    main()

In [ ]:
def main():
    # Initialize the water remover
    remover = LAMMPSWaterRemover('./RDF_lv12_E0_4/Na_16_O_4_0.data')
    remover.read_lammps_data()
    remover.analyze_structure()
if __name__ == "__main__":
    main()

In [ ]:
def main():
    # Initialize the water remover
    remover = LAMMPSWaterRemover('./Na_16_O_32.data')
    remover.read_lammps_data()
    remover.analyze_structure()
if __name__ == "__main__":
    main()

In [ ]:
for i in range(14):
    def main():
        # Initialize the water remover
        remover = LAMMPSWaterRemover(f'./RDF_lv12_E0_sf0.1_f0.1/Na_28_O_4_{i+1}.data')
        remover.read_lammps_data()
        remover.analyze_structure()
    if __name__ == "__main__":
        main()

In [ ]:
for O in [4,16,32]:
    for Na in [4,16,28]:
        filename = f'./Na_{Na}_O_{O}.data'
        def main():
            # Initialize the water remover
            remover = LAMMPSWaterRemover(filename)
            remover.read_lammps_data()
            remover.analyze_structure()
        if __name__ == "__main__":
            main()

In [ ]:
# Define mapping from old type to new type
type_map = {
    1: 1,  # C stays 1
    2: 3,  # Fe becomes 3
    3: 7,  # H becomes 7
    4: 2,  # Mn becomes 2
    5: 4,  # N becomes 4
    6: 5,  # Na becomes 5
    7: 6,  # O becomes 6
}

input_file = "Na_0_O_32.data"
output_file = "new_Na_0_O_32.data"

with open(input_file, "r") as fin, open(output_file, "w") as fout:
    in_atoms_section = False

    for line in fin:
        if "Atoms" in line:
            fout.write(line)
            in_atoms_section = True
            continue
        if in_atoms_section and line.strip() == "":
            in_atoms_section = False
            fout.write(line)
            continue

        if in_atoms_section:
            parts = line.strip().split()
            if len(parts) < 3:
                fout.write(line)
                continue
            atom_id = int(parts[0])
            old_type = int(parts[1])
            new_type = type_map[old_type]
            new_line = f"{atom_id} {new_type} " + " ".join(parts[2:]) + "\n"
            fout.write(new_line)
        else:
            fout.write(line)


In [ ]:
for i in range(8):
    file = f'gcmc_{i+1}.log'
    plot_log_no_V(file)

In [ ]:
element_map = {
    1: "C",
    2: "Mn", 
    3: "Fe",
    4: "N",
    5: "Na",
    6: "O",
    7: "H"
}
for i in [32]:
    input_file = f'./RDF/dump.Na_32_O_{i}_temp.trj'
    output_file = f'./RDF/new_dump.Na_32_O_{i}_temp.trj'
# Convert the file
    convert_dump_file(input_file, output_file, element_map)
    print(f"Conversion complete. Modified file saved as {output_file}")

In [ ]:
plot_log('gcmc_6.log')

In [ ]:
# tri P
for i in range(21):
    file = f'./test_lammps/gcmc_{i+1}.log'
    plot_log_no_V(file)

In [ ]:
# tri P
for i in [6,8]:
    file = f'./test_lammps/gcmc_{i}.log'
    plot_log(file)

In [ ]:
file = f'./test_lammps/gcmc_8.log'
plot_log(file)

In [ ]:
for i in range(1,7):
    file = f'./pot_save/gcmc_{i}_520.log'
    plot_log(file)

In [ ]:
for i in [1,2,5]:
    file = f'./test_almtp_lv20/gcmc_{i}.log'
    plot_log(file)

In [ ]:
for i in [3,4,6]:
    file = f'./test_almtp_lv20/gcmc_{i}.log'
    plot_log(file)

In [ ]:
for i in range(2):
    file = f'./test_almtp_nbh/gcmc_{i+1}.log'
    plot_log_no_V(file)

In [ ]:
for i in range(7):
    file = f'./test_training_set/gcmc_{i+1}.log'
    plot_log(file)

In [ ]:
for i in range(7,11):
    file = f'./test_training_set/gcmc_{i+1}.log'
    plot_log(file)

In [ ]:
for i in range(11,14):
    file = f'./test_training_set/gcmc_{i+1}.log'
    plot_log(file)

In [ ]:
for i in range(14,21):
    file = f'./test_training_set/gcmc_{i+1}.log'
    plot_log(file)

In [ ]:
for i in range(21,27):
    file = f'./test_training_set/gcmc_{i+1}.log'
    plot_log(file)

In [ ]:
for i in range(27,33):
    file = f'./test_training_set/gcmc_{i+1}.log'
    plot_log(file)

In [ ]:
for i in range(10):
    file = f'./test_training_set/make_cfg_{i+1}.log'
    plot_log(file)

### 400K

In [ ]:
for i in [2,4,6,8]:
    file = f'./test_training_set/diffusion/400_diffuse_{i+1}.log'
    plot_log_diff(file)

### 500K

In [ ]:
for i in [2,4,6,8]:
    file = f'./test_training_set/diffusion/500K_in_diffuse_{i}.log'
    plot_log_diff(file)

### 600K

In [ ]:
for i in [2,4,6,8]:
    file = f'./test_training_set/diffusion/600_diffuse_{i}.log'
    plot_log_diff(file)

### 700K

In [ ]:
for i in [2,4,6,8]:
    file = f'./test_training_set/diffusion/700_diffuse_{i}.log'
    plot_log_diff(file)

### 800K

In [ ]:
for i in [2,4,6,8]:
    file = f'./test_training_set/diffusion/800_diffuse_{i}.log'
    plot_log_diff(file)

In [ ]:
file = f'raise_temp.log'
path = './test_training_set/phase_diagram/'
log_data = parse_lammps_log_diff(path+file)
log_data_2 = parse_lammps_log(path+file)

In [ ]:
file = f'raise_temp.log'
path = './test_training_set/phase_diagram/'
log_data = parse_lammps_log_diff(path+file)
log_data_2 = parse_lammps_log(path+file)
num_Na = int(log_data['v_nNa'][0])

fig, ax1 = plt.subplots(figsize=(10, 6))

# Plot RMS(E/atom) on the first y-axis
color = 'tab:blue'
ax1.set_xlabel('Timestep (1 fs/step)')
ax1.set_ylabel('Volume (A^3)', color=color)
ax1.plot(log_data['Step'], log_data['Volume'], color=color, marker='o', markersize=3)
ax1.plot(log_data_2['Step'], log_data_2['Volume'], color=color, marker='o', markersize=3)
ax1.tick_params(axis='y', labelcolor=color)

# Create the second y-axis and plot RMS(f)
ax2 = ax1.twinx()
color = 'tab:orange'
ax2.set_ylabel('Temperature (K) (A^3)', color=color)
ax2.plot(log_data['Step'][10:], log_data['Temp'][10:], color=color, marker='s', markersize=3)
ax2.plot(log_data_2['Step'][10:], log_data_2['Temp'][10:], color=color, marker='s', markersize=3)
ax2.tick_params(axis='y', labelcolor=color)

# MSD(PB) (Third y-axis, second right)
ax3 = ax1.twinx()
color3 = 'tab:green'
ax3.spines['right'].set_position(('outward', 80))
ax3.set_ylabel('PB MSD A^2', color=color3)
ax3.plot(log_data['Step'], log_data['MSD(PB)'], color=color3, marker='*', markersize=3)
ax3.hlines(3,log_data['Step'][0],log_data['Step'][-1], color='r')
ax3.tick_params(axis='y')
ax3.grid(True, linestyle='--', alpha=0.7)

ax4 = ax1.twinx()
color4 = 'tab:purple'
ax4.spines['left'].set_position(('outward', 80))
ax4.set_ylabel('Na MSD A^2', color=color3)
ax4.plot(log_data['Step'], log_data['MSD(Na)'], color=color4, marker='*', markersize=3)
#ax4.tick_params(axis='y')

plt.title(file + f'    Number of Na = {num_Na}')
plt.tight_layout()
#plt.savefig(log_file_path+'.jpg', dpi=300, bbox_inches='tight')

In [ ]:
plt.plot(log_data['Step'], log_data['Temp'], label='Temperature')
plt2=plt.twinx()
plt2.plot(log_data['Step'], log_data['Volume'],color='orange', label'Volume')

In [ ]:
plt.plot(log_data['Step'], log_data['Temp'])
plt2=plt.twinx()
plt2.plot(log_data['Step'], log_data['MSD(Na)'],color='orange')

In [ ]:
filename = f'dump_raise_temp_Na.dat'
path = './test_training_set/diffusion/'
data_Na = pd.read_csv(path + filename, delim_whitespace=True, comment='#', header=None)
data_Na.columns = ['TimeStep', 'c_msd_Na[1]', 'c_msd_Na[2]', 'c_msd_Na[3]', 'c_msd_Na[4]']

filename = f'dump_raise_temp_PB.dat'
path = './test_training_set/diffusion/'
data_PB = pd.read_csv(path + filename, delim_whitespace=True, comment='#', header=None)
data_PB.columns = ['TimeStep', 'c_msd_PB[1]', 'c_msd_PB[2]', 'c_msd_PB[3]', 'c_msd_PB[4]']

In [ ]:
time = range(0, data_PB['TimeStep'].iloc[-1]+40000,40000)
temp = range(800,1600,50)
len(time),len(temp)

In [ ]:
plt.figure(figsize=(8,5))
#plt.plot(data_Na['TimeStep'], data_Na['c_msd_Na[1]'], label='MSD(x)')
#plt.plot(data_Na['TimeStep'], data_Na['c_msd_Na[2]'], label='MSD(y)')
#plt.plot(data_Na['TimeStep'], data_Na['c_msd_Na[3]'], label='MSD(z)')
plt.plot(data_Na['TimeStep'], data_Na['c_msd_Na[4]'], label='TMSD_Na')
#plt.plot(data_PB['TimeStep'], data_PB['c_msd_PB[1]'], label='MSD(x)')
#plt.plot(data_PB['TimeStep'], data_PB['c_msd_PB[2]'], label='MSD(y)')
#plt.plot(data_PB['TimeStep'], data_PB['c_msd_PB[3]'], label='MSD(z)')
plt.plot(data_PB['TimeStep'], data_PB['c_msd_PB[4]'], label='TMSD_PB')
plt.xlabel('Timestep (1 fs/step)')
plt.grid()
plt.ylabel('MSD (A^2)')

plt2 = plt.twinx()
plt2.scatter(time, temp)
plt.legend()


In [ ]:
#data = {'Step': [], 'Temp': [], 'TotEng': [], 'Press': [], 'Volume': [], 'v_nNa': [], 'MSD(PB)' : [], 'MSD(Na)' : []}

In [ ]:
i=2
file = f'./test_training_set/diffusion/800_diffuse_{i}.log'
data = parse_lammps_log(file)


In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(18,8))
plt.subplots_adjust(wspace=1)

for i in [2, 4, 6, 8]:
    file = f'./test_training_set/diffusion/500K_in_diffuse_{i}.log'
    log_data = parse_lammps_log_diff(file)
    log_data_2 = parse_lammps_log(file)
    num_Na = int(log_data_2['v_nNa'][0])

    # Plot RMS(E/atom) on the first y-axis
    ax1.set_xlabel('Timestep (1 fs/step)')
    ax1.set_ylabel('MSD A^2')
    ax1.plot(log_data['Step'], log_data['MSD(Na)'],  marker='*', markersize=3, label=f'{temp}K SoC = {round(num_Na/4096,3)}')


ax1.grid(True, linestyle='--', alpha=0.7)
plt.title('MSD 500K')
ax1.legend()
plt.tight_layout()

delta_t_ls = np.array([1, 2, 5, 10, 20, 50, 100, 200, 500, 1000,2000,5000]) #1, 2.5, 10, 25, 100, 250 ps
temp = 500
for index in [2, 4, 6, 8]:
    file = f'./test_training_set/diffusion/{temp}K_in_diffuse_{index}.log'
    log_data = parse_lammps_log_diff(file)
    MSD_Na = np.array(log_data['MSD(Na)'])
    timestep = np.array(log_data['Step'])
    num_Na = int(log_data['v_nNa'][0])
    t_intival = []
    MSD_intival = []
    for i in range(delta_t_ls.shape[0]):
        #print(i, delta_t_ls[i]*0.5)
        t_intival.append(np.arange(timestep[0], timestep[-1], int(delta_t_ls[i])*500, dtype='int'))
        MSD_intival.append(np.linspace(MSD_Na[0], MSD_Na[-1], t_intival[i].shape[0]))

    delta_t_all = []
    delta_MSD_all = []
    for i in range(delta_t_ls.shape[0]):
        delta_t = []
        delta_MSD = []
        for j in range(t_intival[i].shape[0]-1):
            delta_t.append(t_intival[i][j+1] - t_intival[i][j])
            delta_MSD.append(MSD_intival[i][j+1] - MSD_intival[i][j])
        delta_t_all.append(delta_t)
        delta_MSD_all.append(delta_MSD)

    final_MSD = []
    final_t = []
    for i in range(delta_t_ls.shape[0]):
        final_t.append(sum(delta_t_all[i])/len(delta_t_all[i]))
        final_MSD.append(sum(delta_MSD_all[i])/len(delta_MSD_all[i]))
        
    ax2.plot(final_t, final_MSD, marker='*', markersize=5, label=f'{temp}K SoC = {round(num_Na/4096,3)}')
    
ax2.grid(True, linestyle='--', alpha=0.7)
plt.tight_layout()
ax2.legend()
ax2.set_xscale('log')
ax2.set_yscale('log')
ax2.set_xlabel('delta t (fs)')
ax2.set_ylabel('MSD (A^2)')
plt.savefig(f'Diffusion {temp}K'+'.png', bbox_inches='tight')

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(18,8))
plt.subplots_adjust(wspace=3)
temp= 800
for i in [2, 4, 6, 8]:
    file = f'./test_training_set/diffusion/{temp}_diffuse_{i}.log'
    log_data = parse_lammps_log_diff(file)
    log_data_2 = parse_lammps_log(file)
    num_Na = int(log_data_2['v_nNa'][0])

    # Plot RMS(E/atom) on the first y-axis
    ax1.set_xlabel('Timestep (1 fs/step)')
    ax1.set_ylabel('MSD A^2')
    ax1.plot(log_data['Step'], log_data['MSD(Na)'],  marker='*', markersize=3, label=f'{temp}K SoC = {round(num_Na/4096,3)}')


ax1.grid(True, linestyle='--', alpha=0.7)
plt.title(f'MSD {temp} K')
ax1.legend()
plt.tight_layout()

delta_t_ls = np.array([1, 2, 5, 10, 20, 50, 100, 200, 500, 1000,2000,5000]) #1, 2.5, 10, 25, 100, 250 ps
for index in [2, 4, 6, 8]:
    file = f'./test_training_set/diffusion/{temp}_diffuse_{index}.log'
    log_data = parse_lammps_log_diff(file)
    MSD_Na = np.array(log_data['MSD(Na)'])
    timestep = np.array(log_data['Step'])
    num_Na = int(log_data_2['v_nNa'][0])
    t_intival = []
    MSD_intival = []
    for i in range(delta_t_ls.shape[0]):
        #print(i, delta_t_ls[i]*0.5)
        t_intival.append(np.arange(timestep[0], timestep[-1], int(delta_t_ls[i])*500, dtype='int'))
        MSD_intival.append(np.linspace(MSD_Na[0], MSD_Na[-1], t_intival[i].shape[0]))

    delta_t_all = []
    delta_MSD_all = []
    for i in range(delta_t_ls.shape[0]):
        delta_t = []
        delta_MSD = []
        for j in range(t_intival[i].shape[0]-1):
            delta_t.append(t_intival[i][j+1] - t_intival[i][j])
            delta_MSD.append(MSD_intival[i][j+1] - MSD_intival[i][j])
        delta_t_all.append(delta_t)
        delta_MSD_all.append(delta_MSD)

    final_MSD = []
    final_t = []
    for i in range(delta_t_ls.shape[0]):
        final_t.append(sum(delta_t_all[i])/len(delta_t_all[i]))
        final_MSD.append(sum(delta_MSD_all[i])/len(delta_MSD_all[i]))
        
    ax2.plot(final_t, final_MSD, marker='*', markersize=5, label=f'{temp}K SoC = {round(num_Na/4096,3)}')
    
ax2.grid(True, linestyle='--', alpha=0.7)
plt.tight_layout()
ax2.legend()
ax2.set_xscale('log')
ax2.set_yscale('log')
ax2.set_xlabel('delta t (fs)')
ax2.set_ylabel('MSD (A^2)')
plt.savefig(f'Diffusion {temp}K'+'.png', bbox_inches='tight')

In [ ]:
fig, ax1 = plt.subplots(figsize=(10, 6))

for i in [2, 4, 6, 8]:
    file = f'./test_training_set/diffusion/600_diffuse_{i}.log'
    log_data = parse_lammps_log_diff(file)
    log_data_2 = parse_lammps_log(file)
    num_Na = int(log_data_2['v_nNa'][0])

    # Plot RMS(E/atom) on the first y-axis
    ax1.set_xlabel('Timestep (1 fs/step)')
    ax1.set_ylabel('MSD A^2')
    ax1.plot(log_data['Step'], log_data['MSD(Na)'],  marker='*', markersize=3, label=f'SoC {round(num_Na/4096,3)}')


plt.grid(True, linestyle='--', alpha=0.7)
plt.tight_layout()
ax1.legend()

In [ ]:
fig, ax1 = plt.subplots(figsize=(10, 6))

for i in [2, 4, 6, 8]:
    file = f'./test_training_set/diffusion/700_diffuse_{i}.log'
    log_data = parse_lammps_log_diff(file)
    log_data_2 = parse_lammps_log(file)
    num_Na = int(log_data_2['v_nNa'][0])

    # Plot RMS(E/atom) on the first y-axis
    ax1.set_xlabel('Timestep (1 fs/step)')
    ax1.set_ylabel('MSD A^2')
    ax1.plot(log_data['Step'], log_data['MSD(Na)'],  marker='*', markersize=3, label=f'SoC {round(num_Na/4096,3)}')


plt.grid(True, linestyle='--', alpha=0.7)
plt.tight_layout()
ax1.legend()

In [ ]:
fig, ax1 = plt.subplots(figsize=(10, 6))

for i in [2, 4, 6, 8]:
    file = f'./test_training_set/diffusion/800_diffuse_{i}.log'
    log_data = parse_lammps_log_diff(file)
    log_data_2 = parse_lammps_log(file)
    num_Na = int(log_data_2['v_nNa'][0])

    # Plot RMS(E/atom) on the first y-axis
    ax1.set_xlabel('Timestep (1 fs/step)')
    ax1.set_ylabel('MSD A^2')
    ax1.plot(log_data['Step'], log_data['MSD(Na)'],  marker='*', markersize=3, label=f'SoC {round(num_Na/4096,3)}')


plt.grid(True, linestyle='--', alpha=0.7)
plt.tight_layout()
ax1.legend()

In [ ]:
t_intival[8]

In [ ]:
delta_t_ls = np.array([1, 2, 5, 10, 20, 50, 100, 200, 500, 1000,2000,5000]) #1, 2.5, 10, 25, 100, 250 ps

fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(18,5))
plt.subplots_adjust(wspace=1)

temp = 500

diffusivity_list = []
num_Na_list = []
for index in [2, 4, 6, 8]:
    file = f'./test_training_set/diffusion/{temp}K_in_diffuse_{index}.log'
    log_data = parse_lammps_log_diff(file)
    MSD_Na = np.array(log_data['MSD(Na)'])
    timestep = np.array(log_data['Step'])
    num_Na = int(log_data['v_nNa'][0])
    num_Na_list.append(num_Na)
    t_intival = []
    MSD_intival = []
    for i in range(delta_t_ls.shape[0]):
        #print(i, delta_t_ls[i]*0.5)
        t_intival.append(np.arange(timestep[0], timestep[-1], int(delta_t_ls[i])*500, dtype='int'))
        MSD_intival.append(np.linspace(MSD_Na[0], MSD_Na[-1], t_intival[i].shape[0]))

    delta_t_all = []
    delta_MSD_all = []
    for i in range(delta_t_ls.shape[0]):
        delta_t = []
        delta_MSD = []
        for j in range(t_intival[i].shape[0]-1):
            delta_t.append(t_intival[i][j+1] - t_intival[i][j])
            delta_MSD.append(MSD_intival[i][j+1] - MSD_intival[i][j])
        delta_t_all.append(delta_t)
        delta_MSD_all.append(delta_MSD)

    final_MSD = []
    final_t = []
    diffusivity = []
    for i in range(delta_t_ls.shape[0]):
        final_t.append(sum(delta_t_all[i])/len(delta_t_all[i]))
        final_MSD.append(sum(delta_MSD_all[i])/len(delta_MSD_all[i]))
        
    final_MSD = np.array(final_MSD)
    final_t = np.array(final_t)
    diffusivity_list.append(final_MSD/final_t/6*1e-5)
    
    ax2.plot(final_t, final_MSD, marker='*', markersize=3, label=f'{temp}K SoC = {round(num_Na/4096,3)}')
    ax1.set_xlabel('Timestep (1 fs/step)')
    ax1.set_ylabel('MSD A^2')
    ax1.plot(log_data['Step'], log_data['MSD(Na)'],  marker='*', markersize=3, label=f'SoC {round(num_Na/4096,3)}')
    
ax1.grid(True, linestyle='--', alpha=0.7)
ax2.grid(True, linestyle='--', alpha=0.7)
ax2.legend()
ax1.legend()
ax2.set_xscale('log')
ax2.set_yscale('log')
ax2.set_xlabel('delta t (fs)')
ax2.set_ylabel('MSD (A^2)')
plt.title(f'{temp} K')

for i in range(len(diffusivity_list)):
    ax3.plot(final_t, diffusivity_list[i], marker='*', markersize=3, label=f'{temp}K SoC = {round(num_Na_list[i]/4096,3)}')

ax3.grid(True, linestyle='--', alpha=0.7)
ax3.legend()
#ax3.set_xscale('log')
#ax3.set_yscale('log')
ax3.set_xlabel('delta t (fs)')
ax3.set_ylabel('Diffusivity (m^2/s)')

plt.tight_layout()

In [ ]:
diffusivity_list_500K = diffusivity_list

In [ ]:
delta_t_ls = np.array([1, 2, 5, 10, 20, 50, 100, 200, 500, 1000,2000,5000]) #1, 2.5, 10, 25, 100, 250 ps

fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(18,5))
plt.subplots_adjust(wspace=1)

temp = 600

diffusivity_list = []
num_Na_list = []
for index in [2, 4, 6, 8]:
    file = f'./test_training_set/diffusion/{temp}_diffuse_{index}.log'
    log_data = parse_lammps_log_diff(file)
    MSD_Na = np.array(log_data['MSD(Na)'])
    timestep = np.array(log_data['Step'])
    num_Na = int(log_data['v_nNa'][0])
    num_Na_list.append(num_Na)
    t_intival = []
    MSD_intival = []
    for i in range(delta_t_ls.shape[0]):
        #print(i, delta_t_ls[i]*0.5)
        t_intival.append(np.arange(timestep[0], timestep[-1], int(delta_t_ls[i])*500, dtype='int'))
        MSD_intival.append(np.linspace(MSD_Na[0], MSD_Na[-1], t_intival[i].shape[0]))

    delta_t_all = []
    delta_MSD_all = []
    for i in range(delta_t_ls.shape[0]):
        delta_t = []
        delta_MSD = []
        for j in range(t_intival[i].shape[0]-1):
            delta_t.append(t_intival[i][j+1] - t_intival[i][j])
            delta_MSD.append(MSD_intival[i][j+1] - MSD_intival[i][j])
        delta_t_all.append(delta_t)
        delta_MSD_all.append(delta_MSD)

    final_MSD = []
    final_t = []
    diffusivity = []
    for i in range(delta_t_ls.shape[0]):
        final_t.append(sum(delta_t_all[i])/len(delta_t_all[i]))
        final_MSD.append(sum(delta_MSD_all[i])/len(delta_MSD_all[i]))
        
    final_MSD = np.array(final_MSD)
    final_t = np.array(final_t)
    diffusivity_list.append(final_MSD/final_t/6*1e-5)
    
    ax2.plot(final_t, final_MSD, marker='*', markersize=3, label=f'{temp}K SoC = {round(num_Na/4096,3)}')
    ax1.set_xlabel('Timestep (1 fs/step)')
    ax1.set_ylabel('MSD A^2')
    ax1.plot(log_data['Step'], log_data['MSD(Na)'],  marker='*', markersize=3, label=f'SoC {round(num_Na/4096,3)}')
    
ax1.grid(True, linestyle='--', alpha=0.7)
ax2.grid(True, linestyle='--', alpha=0.7)
ax2.legend()
ax1.legend()
ax2.set_xscale('log')
ax2.set_yscale('log')
ax2.set_xlabel('delta t (fs)')
ax2.set_ylabel('MSD (A^2)')

for i in range(len(diffusivity_list)):
    ax3.plot(final_t, diffusivity_list[i], marker='*', markersize=3, label=f'{temp}K SoC = {round(num_Na_list[i]/4096,3)}')

ax3.grid(True, linestyle='--', alpha=0.7)
ax3.legend()
#ax3.set_xscale('log')
#ax3.set_yscale('log')
ax3.set_xlabel('delta t (fs)')
ax3.set_ylabel('Diffusivity (m^2/s)')
plt.title(f'{temp} K')
plt.tight_layout()

diffusivity_list_600K = diffusivity_list

In [ ]:
delta_t_ls = np.array([1, 2, 5, 10, 20, 50, 100, 200, 500, 1000,2000,5000]) #1, 2.5, 10, 25, 100, 250 ps

fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(18,5))
plt.subplots_adjust(wspace=1)

temp = 700

diffusivity_list = []
num_Na_list = []
for index in [2, 4, 6]:
    file = f'./test_training_set/diffusion/{temp}_diffuse_{index}.log'
    log_data = parse_lammps_log_diff(file)
    MSD_Na = np.array(log_data['MSD(Na)'])
    timestep = np.array(log_data['Step'])
    num_Na = int(log_data['v_nNa'][0])
    num_Na_list.append(num_Na)
    t_intival = []
    MSD_intival = []
    for i in range(delta_t_ls.shape[0]):
        #print(i, delta_t_ls[i]*0.5)
        t_intival.append(np.arange(timestep[0], timestep[-1], int(delta_t_ls[i])*500, dtype='int'))
        MSD_intival.append(np.linspace(MSD_Na[0], MSD_Na[-1], t_intival[i].shape[0]))

    delta_t_all = []
    delta_MSD_all = []
    for i in range(delta_t_ls.shape[0]):
        delta_t = []
        delta_MSD = []
        for j in range(t_intival[i].shape[0]-1):
            delta_t.append(t_intival[i][j+1] - t_intival[i][j])
            delta_MSD.append(MSD_intival[i][j+1] - MSD_intival[i][j])
        delta_t_all.append(delta_t)
        delta_MSD_all.append(delta_MSD)

    final_MSD = []
    final_t = []
    diffusivity = []
    for i in range(delta_t_ls.shape[0]):
        final_t.append(sum(delta_t_all[i])/len(delta_t_all[i]))
        final_MSD.append(sum(delta_MSD_all[i])/len(delta_MSD_all[i]))
        
    final_MSD = np.array(final_MSD)
    final_t = np.array(final_t)
    diffusivity_list.append(final_MSD/final_t/6*1e-5)
    
    ax2.plot(final_t, final_MSD, marker='*', markersize=3, label=f'{temp}K SoC = {round(num_Na/4096,3)}')
    ax1.set_xlabel('Timestep (1 fs/step)')
    ax1.set_ylabel('MSD A^2')
    ax1.plot(log_data['Step'], log_data['MSD(Na)'],  marker='*', markersize=3, label=f'SoC {round(num_Na/4096,3)}')
    
ax1.grid(True, linestyle='--', alpha=0.7)
ax2.grid(True, linestyle='--', alpha=0.7)
ax2.legend()
ax1.legend()
ax2.set_xscale('log')
ax2.set_yscale('log')
ax2.set_xlabel('delta t (fs)')
ax2.set_ylabel('MSD (A^2)')

for i in range(len(diffusivity_list)):
    ax3.plot(final_t, diffusivity_list[i], marker='*', markersize=3, label=f'{temp}K SoC = {round(num_Na_list[i]/4096,3)}')

ax3.grid(True, linestyle='--', alpha=0.7)
ax3.legend()
#ax3.set_xscale('log')
#ax3.set_yscale('log')
ax3.set_xlabel('delta t (fs)')
ax3.set_ylabel('Diffusivity (m^2/s)')

plt.tight_layout()

diffusivity_list_700K = diffusivity_list

In [ ]:
delta_t_ls = np.array([1, 2, 5, 10, 20, 50, 100, 200, 500, 1000,2000,5000]) #1, 2.5, 10, 25, 100, 250 ps

fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(18,5))
plt.subplots_adjust(wspace=1)

temp = 800

diffusivity_list = []
num_Na_list = []
for index in [2, 4, 6]:
    file = f'./test_training_set/diffusion/{temp}_diffuse_{index}.log'
    log_data = parse_lammps_log_diff(file)
    MSD_Na = np.array(log_data['MSD(Na)'])
    timestep = np.array(log_data['Step'])
    num_Na = int(log_data['v_nNa'][0])
    num_Na_list.append(num_Na)
    t_intival = []
    MSD_intival = []
    for i in range(delta_t_ls.shape[0]):
        #print(i, delta_t_ls[i]*0.5)
        t_intival.append(np.arange(timestep[0], timestep[-1], int(delta_t_ls[i])*500, dtype='int'))
        MSD_intival.append(np.linspace(MSD_Na[0], MSD_Na[-1], t_intival[i].shape[0]))

    delta_t_all = []
    delta_MSD_all = []
    for i in range(delta_t_ls.shape[0]):
        delta_t = []
        delta_MSD = []
        for j in range(t_intival[i].shape[0]-1):
            delta_t.append(t_intival[i][j+1] - t_intival[i][j])
            delta_MSD.append(MSD_intival[i][j+1] - MSD_intival[i][j])
        delta_t_all.append(delta_t)
        delta_MSD_all.append(delta_MSD)

    final_MSD = []
    final_t = []
    diffusivity = []
    for i in range(delta_t_ls.shape[0]):
        final_t.append(sum(delta_t_all[i])/len(delta_t_all[i]))
        final_MSD.append(sum(delta_MSD_all[i])/len(delta_MSD_all[i]))
        
    final_MSD = np.array(final_MSD)
    final_t = np.array(final_t)
    diffusivity_list.append(final_MSD/final_t/6*1e-5)
    
    ax2.plot(final_t, final_MSD, marker='*', markersize=3, label=f'{temp}K SoC = {round(num_Na/4096,3)}')
    ax1.set_xlabel('Timestep (1 fs/step)')
    ax1.set_ylabel('MSD A^2')
    ax1.plot(log_data['Step'], log_data['MSD(Na)'],  marker='*', markersize=3, label=f'SoC {round(num_Na/4096,3)}')
    
ax1.grid(True, linestyle='--', alpha=0.7)
ax2.grid(True, linestyle='--', alpha=0.7)
ax2.legend()
ax1.legend()
ax2.set_xscale('log')
ax2.set_yscale('log')
ax2.set_xlabel('delta t (fs)')
ax2.set_ylabel('MSD (A^2)')

for i in range(len(diffusivity_list)):
    ax3.plot(final_t, diffusivity_list[i], marker='*', markersize=3, label=f'{temp}K SoC = {round(num_Na_list[i]/4096,3)}')

ax3.grid(True, linestyle='--', alpha=0.7)
ax3.legend()
#ax3.set_xscale('log')
#ax3.set_yscale('log')
ax3.set_xlabel('delta t (fs)')
ax3.set_ylabel('Diffusivity (m^2/s)')
diffusivity_list_800K = diffusivity_list
plt.tight_layout()

In [ ]:
big_diff = [diffusivity_list_500K, diffusivity_list_600K, diffusivity_list_700K, diffusivity_list_800K]
diffusivity_1 = []
diffusivity_2 = []
diffusivity_3 = []
temp = np.array([500, 600, 700, 800])

for i in range(temp.shape[0]):
    diffusivity_1.append(big_diff[i][0][9])
    diffusivity_2.append(big_diff[i][1][9])
    diffusivity_3.append(big_diff[i][2][9])


In [ ]:
fig, ax1 = plt.subplots(figsize=(10, 6))
ax1.plot(1/temp*1000, diffusivity_1, marker='*', markersize=3)
ax1.plot(1/temp*1000, diffusivity_2, marker='*', markersize=3)
ax1.plot(1/temp*1000, diffusivity_3, marker='*', markersize=3)
ax1.set_xscale('log')
ax1.set_yscale('log')
ax1.set_xlabel('1000/T (1/K)')
ax1.set_ylabel('Diffusivity (m^2/s)')

In [ ]:
x_data, y_data

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit

# Define the exponential function
def exp_func(x, a, b):
    return a * np.exp(b * x)

# Prepare your data
x_data = 1/temp  # Reciprocal of temperature
y_data = diffusivity_1  # Diffusivity values

# Perform curve fitting
popt, pcov = curve_fit(exp_func, x_data, y_data, p0=(1, 1))
a_fit, b_fit = popt

# Calculate standard errors
perr = np.sqrt(np.diag(pcov))
a_err, b_err = perr

# Print fitted parameters with their uncertainties
print(f"Fitted Parameters:")
print(f"a = {a_fit:.5f} ± {a_err:.5f}")
print(f"b = {b_fit:.5f} ± {b_err:.5f}")

# Calculate R-squared to assess goodness of fit
residuals = y_data - exp_func(x_data, a_fit, b_fit)
ss_res = np.sum(residuals**2)
ss_tot = np.sum((y_data - np.mean(y_data))**2)
r_squared = 1 - (ss_res / ss_tot)
print(f"R-squared: {r_squared:.5f}")

# Plotting
plt.figure(figsize=(10, 6))
plt.scatter(x_data, y_data, label='Experimental Data')
x_fit = np.linspace(x_data.min(), x_data.max(), 100)
y_fit = exp_func(x_fit, a_fit, b_fit)
plt.plot(x_fit, y_fit, 'r-', label='Fitted Curve')
plt.xlabel('1/T')
plt.ylabel('Diffusivity')
plt.title('Diffusivity vs 1/T - Exponential Fit')
plt.legend()
plt.xscale('log')
plt.yscale('log')
plt.grid(True)
plt.show()

In [ ]:
-b_fit*8.617e-5


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit

# Define the exponential function
def exp_func(x, a, b):
    return a * np.exp(b * x)

# Prepare your data
x_data = 1/temp[1:]  # Reciprocal of temperature
y_data = diffusivity_3[:-1]  # Diffusivity values

# Perform curve fitting
popt, pcov = curve_fit(exp_func, x_data, y_data, p0=(1, 1))
a_fit, b_fit = popt

# Calculate standard errors
perr = np.sqrt(np.diag(pcov))
a_err, b_err = perr

# Print fitted parameters with their uncertainties
print(f"Fitted Parameters:")
print(f"a = {a_fit:.5f} ± {a_err:.5f}")
print(f"b = {b_fit:.5f} ± {b_err:.5f}")

# Calculate R-squared to assess goodness of fit
residuals = y_data - exp_func(x_data, a_fit, b_fit)
ss_res = np.sum(residuals**2)
ss_tot = np.sum((y_data - np.mean(y_data))**2)
r_squared = 1 - (ss_res / ss_tot)
print(f"R-squared: {r_squared:.5f}")

# Plotting
plt.figure(figsize=(10, 6))
plt.scatter(x_data, y_data, label='Experimental Data')
x_fit = np.linspace(x_data.min(), x_data.max(), 100)
y_fit = exp_func(x_fit, a_fit, b_fit)
plt.plot(x_fit, y_fit, 'r-', label='Fitted Curve')
plt.xlabel('1/T')
plt.ylabel('Diffusivity')
plt.title('Diffusivity vs 1/T - Exponential Fit')
plt.legend()
plt.xscale('log')
plt.yscale('log')
plt.grid(True)
plt.show()

In [ ]:
Ea = -b_fit*8.617e-5
Ea

In [ ]:
y_fit = exp_func(x_data, *popt)

plt.scatter(x_data, y_data, label="Data", color="blue")
plt.plot(x_data, y_fit, label="Exponential Fit", color="red")
plt.xlabel("x")
plt.ylabel("y")
plt.legend()
plt.title("Exponential Fit")
plt.show()

In [ ]:
delta_t_ls = np.array([1, 2, 5, 10, 20, 50, 100, 200, 500, 1000,2000,5000]) #1, 2.5, 10, 25, 100, 250 ps

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(18,8))
plt.subplots_adjust(wspace=3)

temp = 700

diffusivity_list = []
num_Na_list = []
for index in [2, 4, 6]:
    file = f'./test_training_set/diffusion/{temp}K_in_diffuse_{index}.log'
    log_data = parse_lammps_log_diff(file)
    MSD_Na = np.array(log_data['MSD(Na)'])
    timestep = np.array(log_data['Step'])
    num_Na = int(log_data['v_nNa'][0])
    num_Na_list.append(num_Na)
    t_intival = []
    MSD_intival = []
    for i in range(delta_t_ls.shape[0]):
        #print(i, delta_t_ls[i]*0.5)
        t_intival.append(np.arange(timestep[0], timestep[-1], int(delta_t_ls[i])*500, dtype='int'))
        MSD_intival.append(np.linspace(MSD_Na[0], MSD_Na[-1], t_intival[i].shape[0]))

    delta_t_all = []
    delta_MSD_all = []
    for i in range(delta_t_ls.shape[0]):
        delta_t = []
        delta_MSD = []
        for j in range(t_intival[i].shape[0]-1):
            delta_t.append(t_intival[i][j+1] - t_intival[i][j])
            delta_MSD.append(MSD_intival[i][j+1] - MSD_intival[i][j])
        delta_t_all.append(delta_t)
        delta_MSD_all.append(delta_MSD)

    final_MSD = []
    final_t = []
    diffusivity = []
    for i in range(delta_t_ls.shape[0]):
        final_t.append(sum(delta_t_all[i])/len(delta_t_all[i]))
        final_MSD.append(sum(delta_MSD_all[i])/len(delta_MSD_all[i]))
        
    final_MSD = np.array(final_MSD)
    final_t = np.array(final_t)
    diffusivity_list.append(final_MSD/final_t/6*1e-5)
    
    ax1.plot(final_t, final_MSD, marker='*', markersize=3, label=f'{temp}K SoC = {round(num_Na/4096,3)}')
    
ax1.grid(True, linestyle='--', alpha=0.7)
ax1.legend()
ax1.set_xscale('log')
ax1.set_yscale('log')
ax1.set_xlabel('delta t (fs)')
ax1.set_ylabel('MSD (A^2)')

for i in range(len(diffusivity_list)):
    plt.plot(final_t, diffusivity_list[i], marker='*', markersize=3, label=f'{temp}K SoC = {round(num_Na_list[i]/4096,3)}')

ax2.grid(True, linestyle='--', alpha=0.7)
ax2.legend()
ax2.set_xscale('log')
ax2.set_yscale('log')
ax2.set_xlabel('delta t (fs)')
ax2.set_ylabel('Diffusivity (m^2/s)')

plt.tight_layout()

In [ ]:
final_t

In [ ]:
for i in range(len(diffusivity_list)):
    plt.plot(diffusivity_list[i], )

In [ ]:
delta_t_ls = np.array([1, 2, 5, 10, 20, 50, 100, 200, 500, 1000,2000,5000]) #1, 2.5, 10, 25, 100, 250 ps
fig, ax1 = plt.subplots(figsize=(8, 6))
temp = 600
for index in [2, 4, 6, 8]:
    file = f'./test_training_set/diffusion/{temp}_diffuse_{index}.log'
    log_data = parse_lammps_log_diff(file)
    MSD_Na = np.array(log_data['MSD(Na)'])
    timestep = np.array(log_data['Step'])
    num_Na = int(log_data['v_nNa'][0])
    t_intival = []
    MSD_intival = []
    for i in range(delta_t_ls.shape[0]):
        #print(i, delta_t_ls[i]*0.5)
        t_intival.append(np.arange(timestep[0], timestep[-1], int(delta_t_ls[i])*500, dtype='int'))
        MSD_intival.append(np.linspace(MSD_Na[0], MSD_Na[-1], t_intival[i].shape[0]))

    delta_t_all = []
    delta_MSD_all = []
    for i in range(delta_t_ls.shape[0]):
        delta_t = []
        delta_MSD = []
        for j in range(t_intival[i].shape[0]-1):
            delta_t.append(t_intival[i][j+1] - t_intival[i][j])
            delta_MSD.append(MSD_intival[i][j+1] - MSD_intival[i][j])
        delta_t_all.append(delta_t)
        delta_MSD_all.append(delta_MSD)

    final_MSD = []
    final_t = []
    for i in range(delta_t_ls.shape[0]):
        final_t.append(sum(delta_t_all[i])/len(delta_t_all[i]))
        final_MSD.append(sum(delta_MSD_all[i])/len(delta_MSD_all[i]))
        
    ax1.plot(final_t, final_MSD, marker='*', markersize=3, label=f'{temp}K SoC = {round(num_Na/4096,3)}')
    
plt.grid(True, linestyle='--', alpha=0.7)
plt.tight_layout()
ax1.legend()
ax1.set_xscale('log')
ax1.set_yscale('log')
ax1.set_xlabel('delta t (fs)')
ax1.set_ylabel('MSD (A^2)')

In [ ]:
delta_t_ls = np.array([1, 2, 5, 10, 20, 50, 100, 200, 500, 1000,2000,5000]) #1, 2.5, 10, 25, 100, 250 ps
fig, ax1 = plt.subplots(figsize=(8, 6))
temp = 700
for index in [2, 4, 6]:
    file = f'./test_training_set/diffusion/{temp}_diffuse_{index}.log'
    log_data = parse_lammps_log_diff(file)
    MSD_Na = np.array(log_data['MSD(Na)'])
    timestep = np.array(log_data['Step'])
    num_Na = int(log_data['v_nNa'][0])
    t_intival = []
    MSD_intival = []
    for i in range(delta_t_ls.shape[0]):
        #print(i, delta_t_ls[i]*0.5)
        t_intival.append(np.arange(timestep[0], timestep[-1], int(delta_t_ls[i])*500, dtype='int'))
        MSD_intival.append(np.linspace(MSD_Na[0], MSD_Na[-1], t_intival[i].shape[0]))

    delta_t_all = []
    delta_MSD_all = []
    for i in range(delta_t_ls.shape[0]):
        delta_t = []
        delta_MSD = []
        for j in range(t_intival[i].shape[0]-1):
            delta_t.append(t_intival[i][j+1] - t_intival[i][j])
            delta_MSD.append(MSD_intival[i][j+1] - MSD_intival[i][j])
        delta_t_all.append(delta_t)
        delta_MSD_all.append(delta_MSD)

    final_MSD = []
    final_t = []
    for i in range(delta_t_ls.shape[0]):
        final_t.append(sum(delta_t_all[i])/len(delta_t_all[i]))
        final_MSD.append(sum(delta_MSD_all[i])/len(delta_MSD_all[i]))
        
    ax1.plot(final_t, final_MSD, marker='*', markersize=3, label=f'{temp}K SoC = {round(num_Na/4096,3)}')
    
plt.grid(True, linestyle='--', alpha=0.7)
plt.tight_layout()
ax1.legend()
ax1.set_xscale('log')
ax1.set_yscale('log')
ax1.set_xlabel('delta t (fs)')
ax1.set_ylabel('MSD (A^2)')

In [ ]:
delta_t_ls = np.array([1, 2, 5, 10, 20, 50, 100, 200, 500, 1000,2000]) #1, 2.5, 10, 25, 100, 250 ps
fig, ax1 = plt.subplots(figsize=(8, 6))
temp = 800
for index in [2, 4, 6]:
    file = f'./test_training_set/diffusion/{temp}_diffuse_{index}.log'
    log_data = parse_lammps_log_diff(file)
    MSD_Na = np.array(log_data['MSD(Na)'])
    timestep = np.array(log_data['Step'])
    num_Na = int(log_data['v_nNa'][0])
    t_intival = []
    MSD_intival = []
    for i in range(delta_t_ls.shape[0]):
        #print(i, delta_t_ls[i]*0.5)
        t_intival.append(np.arange(timestep[0], timestep[-1], int(delta_t_ls[i])*500, dtype='int'))
        MSD_intival.append(np.linspace(MSD_Na[0], MSD_Na[-1], t_intival[i].shape[0]))

    delta_t_all = []
    delta_MSD_all = []
    for i in range(delta_t_ls.shape[0]):
        delta_t = []
        delta_MSD = []
        for j in range(t_intival[i].shape[0]-1):
            delta_t.append(t_intival[i][j+1] - t_intival[i][j])
            delta_MSD.append(MSD_intival[i][j+1] - MSD_intival[i][j])
        delta_t_all.append(delta_t)
        delta_MSD_all.append(delta_MSD)

    final_MSD = []
    final_t = []
    for i in range(delta_t_ls.shape[0]):
        final_t.append(sum(delta_t_all[i])/len(delta_t_all[i]))
        final_MSD.append(sum(delta_MSD_all[i])/len(delta_MSD_all[i]))
        
    ax1.plot(final_t, final_MSD, marker='*', markersize=3, label=f'{temp}K SoC = {round(num_Na/4096,3)}')
    
plt.grid(True, linestyle='--', alpha=0.7)
plt.tight_layout()
ax1.legend()
ax1.set_xscale('log')
ax1.set_yscale('log')
ax1.set_xlabel('delta t (fs)')
ax1.set_ylabel('MSD (A^2)')

In [ ]:
final_t

In [ ]:
delta_t_ls = np.array([1, 2, 5, 10, 20, 50, 100, 200, 500, 1000]) #1, 2.5, 10, 25, 100, 250 ps
i=6
file = f'./test_training_set/diffusion/600_diffuse_{i}.log'
log_data = parse_lammps_log_diff(file)
MSD_Na = np.array(log_data['MSD(Na)'])
timestep = np.array(log_data['Step'])

In [ ]:
t_intival = []
MSD_intival = []
for i in range(delta_t_ls.shape[0]):
    print(i, delta_t_ls[i]*0.5)
    t_intival.append(np.arange(timestep[0], timestep[-1], int(delta_t_ls[i])*500, dtype='int'))
    MSD_intival.append(np.linspace(MSD_Na[0], MSD_Na[-1], t_intival[i].shape[0]))


In [ ]:
i=0
delta_t_ls.shape[0], t_intival[i].shape[0]-1

In [ ]:
delta_t_all = []
delta_MSD_all = []
for i in range(delta_t_ls.shape[0]):
    delta_t = []
    delta_MSD = []
    for j in range(t_intival[i].shape[0]-1):
        delta_t.append(t_intival[i][j+1] - t_intival[i][j])
        delta_MSD.append(MSD_intival[i][j+1] - MSD_intival[i][j])
    delta_t_all.append(delta_t)
    delta_MSD_all.append(delta_MSD)

In [ ]:
final_MSD = []
final_t = []
for i in range(delta_t_ls.shape[0]):
    final_t.append(sum(delta_t_all[i])/len(delta_t_all[i]))
    final_MSD.append(sum(delta_MSD_all[i])/len(delta_MSD_all[i]))

In [ ]:
fig, ax1 = plt.subplots(figsize=(8, 6))

ax1.plot(final_t, final_MSD,  marker='*', markersize=5)
ax1.set_xlabel('delta t (ps)')
ax1.set_ylabel('MSD (A^2)')
ax1.set_xscale('log')
ax1.set_yscale('log')

## 300K

In [ ]:
for i in range(11):
    file = f'./test_training_set/phase_diagram/300_phase_{i+1}.log'
    plot_log_no_V(file)

## 400K

In [ ]:
for i in range(11):
    file = f'./test_training_set/phase_diagram/400_phase_{i+1}.log'
    plot_log(file)

## 500K

In [ ]:
for i in range(11):
    file = f'./test_training_set/phase_diagram/500_phase_{i+1}.log'
    plot_log(file)

## 600K

In [ ]:
for i in range(11):
    file = f'./test_training_set/phase_diagram/600_phase_{i+1}.log'
    plot_log(file)

## 700K

In [ ]:
for i in range(19):
    file = f'./test_training_set/phase_diagram/700_phase_{i+1}.log'
    plot_log(file)

## 800K

In [ ]:
for i in range(19):
    file = f'./test_training_set/phase_diagram/800_phase_{i+1}.log'
    plot_log(file)

## 900K

In [ ]:
for i in range(19):
    file = f'./test_training_set/phase_diagram/900_phase_{i+1}.log'
    plot_log(file)

## 1000K

In [ ]:
for i in range(19):
    file = f'./test_training_set/phase_diagram/1000_phase_{i+1}.log'
    plot_log(file)

## 1100K

In [ ]:
for i in range(8,19):
    file = f'./test_training_set/phase_diagram/1100_phase_{i+1}.log'
    plot_log(file)

## 1200K

In [ ]:
for i in range(8,19):
    file = f'./test_training_set/phase_diagram/1200_phase_{i+1}.log'
    plot_log(file)

## 1300K

In [ ]:
for i in range(8,19):
    file = f'./test_training_set/phase_diagram/1300_phase_{i+1}.log'
    plot_log(file)

## Na profiles

In [ ]:
def plot_Na_profile(temp, n_1, n_2, mu_1, mu_2):
    fig, ax1 = plt.subplots(figsize=(10, 6))
    temp=temp
    mu_1, mu_2, step = mu_1, mu_2, 0.05
    for i in range(n_1, n_2):
        file = f'./test_training_set/phase_diagram/{temp}_phase_{i+1}.log'
        log_data = parse_lammps_log(file)
     
        E_Na = -.14023100E+01
        my_list = log_data['TotEng'][20:40]
        E_i = sum(my_list) / len(my_list)
        my_list = log_data['TotEng'][-20:]
        E_f = sum(my_list) / len(my_list)
        my_list = log_data['v_nNa'][20:40]
        n_Na_i = sum(my_list) / len(my_list)
        my_list = log_data['v_nNa'][-20:]
        n_Na_f = sum(my_list) / len(my_list)   
        n_Na = n_Na_i-n_Na_f
        V_ave = -(E_i-E_f-n_Na*E_Na)/n_Na
        
        ax1.plot(log_data['Step'], log_data['v_nNa'], color=cm.jet((i-n_1)/n_2), label=f'mu_{i+1} = {round(np.arange(mu_1, mu_2+step, step)[i],2)}  V = {round(V_ave,2)}')#, marker='*')
    
    
    #ax1.hlines(512,0,3e6, linestyle='--', color='black')
    #ax1.hlines(0,0,3e6, linestyle='--', color='black')
    
    ax1.set_xlabel('Timestep (0.4 fs/step)')
    ax1.set_ylabel('Number of Na+')
    #ax1.set_xlim(0,3e6)
    plt.title(f'{temp} K  mu = {mu_1} ~ {mu_2}')
    #ax1.tick_params(axis='y', labelcolor=color)
    #ax1.set_ylim(-5,515)
    ax1.legend()
    plt.grid()

In [ ]:
plot_Na_profile(300)

In [ ]:
plot_Na_profile(400)

In [ ]:
plot_Na_profile(500)

In [ ]:
plot_Na_profile(600)

In [ ]:
plot_Na_profile(700, 0, 19, -2.2, -1.3)

In [ ]:
plot_Na_profile(800, 0, 19, -2.2, -1.3)

In [ ]:
plot_Na_profile(900, 0, 19, -2.2, -1.3)

In [ ]:
plot_Na_profile(1000, 0, 19, -2.2, -1.3)

In [ ]:
plot_Na_profile(1100, 0, 19, -2.2, -1.3)

In [ ]:
plot_Na_profile(1200, 8, 19, -2.2, -1.3)

In [ ]:
plot_Na_profile(1300, 8, 19, -2.2, -1.3)

In [ ]:
fig, ax1 = plt.subplots(figsize=(10, 6))
temp=300
mu_1, mu_2, step = -2.2, -1.7, 0.05
for i in range(11):
    file = f'./test_training_set/phase_diagram/{temp}_phase_{i+1}.log'
    log_data = parse_lammps_log(file)
 
    E_Na = -.14023100E+01
    my_list = log_data['TotEng'][20:40]
    E_i = sum(my_list) / len(my_list)
    my_list = log_data['TotEng'][-20:]
    E_f = sum(my_list) / len(my_list)
    my_list = log_data['v_nNa'][20:40]
    n_Na_i = sum(my_list) / len(my_list)
    my_list = log_data['v_nNa'][-20:]
    n_Na_f = sum(my_list) / len(my_list)   
    n_Na = n_Na_i-n_Na_f
    V_ave = -(E_i-E_f-n_Na*E_Na)/n_Na
    
    ax1.plot(log_data['Step'], log_data['v_nNa'], color=cm.jet(i/11), label=f'mu_{i+1} = {round(np.arange(mu_1, mu_2+step, step)[i],2)}  V = {round(V_ave,2)}')#, marker='*')


#ax1.hlines(512,0,3e6, linestyle='--', color='black')
#ax1.hlines(0,0,3e6, linestyle='--', color='black')

ax1.set_xlabel('Timestep (0.4 fs/step)')
ax1.set_ylabel('Number of Na+')
#ax1.set_xlim(0,3e6)
plt.title(f'{temp} K  mu = {mu_1} ~ {mu_2}')
#ax1.tick_params(axis='y', labelcolor=color)
ax1.set_ylim(-5,515)
ax1.legend()
plt.grid()

In [ ]:
fig, ax1 = plt.subplots(figsize=(10, 6))
temp=400
mu_1, mu_2, step = -2.2, -1.7, 0.05
for i in range(11):
    file = f'./test_training_set/phase_diagram/{temp}_phase_{i+1}.log'
    log_data = parse_lammps_log(file)
 
    E_Na = -.14023100E+01
    my_list = log_data['TotEng'][20:40]
    E_i = sum(my_list) / len(my_list)
    my_list = log_data['TotEng'][-20:]
    E_f = sum(my_list) / len(my_list)
    my_list = log_data['v_nNa'][20:40]
    n_Na_i = sum(my_list) / len(my_list)
    my_list = log_data['v_nNa'][-20:]
    n_Na_f = sum(my_list) / len(my_list)   
    n_Na = n_Na_i-n_Na_f
    V_ave = -(E_i-E_f-n_Na*E_Na)/n_Na
    
    ax1.plot(log_data['Step'], log_data['v_nNa'], color=cm.jet(i/11), label=f'mu_{i+1} = {round(np.arange(mu_1, mu_2+step, step)[i],2)}  V = {round(V_ave,2)}')#, marker='*')


#ax1.hlines(512,0,3e6, linestyle='--', color='black')
#ax1.hlines(0,0,3e6, linestyle='--', color='black')

ax1.set_xlabel('Timestep (0.4 fs/step)')
ax1.set_ylabel('Number of Na+')
#ax1.set_xlim(0,3e6)
plt.title(f'{temp} K  mu = {mu_1} ~ {mu_2}')
#ax1.tick_params(axis='y', labelcolor=color)
ax1.set_ylim(-5,515)
ax1.legend()
plt.grid()

In [ ]:
fig, ax1 = plt.subplots(figsize=(10, 6))
temp=500
mu_1, mu_2, step = -2.2, -1.7, 0.05
for i in range(11):
    file = f'./test_training_set/phase_diagram/{temp}_phase_{i+1}.log'
    log_data = parse_lammps_log(file)
 
    E_Na = -.14023100E+01
    my_list = log_data['TotEng'][20:40]
    E_i = sum(my_list) / len(my_list)
    my_list = log_data['TotEng'][-20:]
    E_f = sum(my_list) / len(my_list)
    my_list = log_data['v_nNa'][20:40]
    n_Na_i = sum(my_list) / len(my_list)
    my_list = log_data['v_nNa'][-20:]
    n_Na_f = sum(my_list) / len(my_list)   
    n_Na = n_Na_i-n_Na_f
    V_ave = -(E_i-E_f-n_Na*E_Na)/n_Na
    
    ax1.plot(log_data['Step'], log_data['v_nNa'], color=cm.jet(i/11), label=f'mu_{i+1} = {round(np.arange(mu_1, mu_2+step, step)[i],2)}  V = {round(V_ave,2)}')#, marker='*')


ax1.hlines(512,0,3e6, linestyle='--', color='black')
ax1.hlines(0,0,3e6, linestyle='--', color='black')

ax1.set_xlabel('Timestep (0.4 fs/step)')
ax1.set_ylabel('Number of Na+')
ax1.set_xlim(0,3e6)
plt.title(f'{temp} K  mu = {mu_1} ~ {mu_2}')
#ax1.tick_params(axis='y', labelcolor=color)
#ax1.set_xlim(0,1e6)
ax1.legend()
plt.grid()

In [ ]:
fig, ax1 = plt.subplots(figsize=(10, 6))
temp=600
mu_1, mu_2, step = -2.2, -1.7, 0.05
for i in range(11):
    file = f'./test_training_set/phase_diagram/{temp}_phase_{i+1}.log'
    log_data = parse_lammps_log(file)
 
    E_Na = -.14023100E+01
    my_list = log_data['TotEng'][20:40]
    E_i = sum(my_list) / len(my_list)
    my_list = log_data['TotEng'][-20:]
    E_f = sum(my_list) / len(my_list)
    my_list = log_data['v_nNa'][20:40]
    n_Na_i = sum(my_list) / len(my_list)
    my_list = log_data['v_nNa'][-20:]
    n_Na_f = sum(my_list) / len(my_list)   
    n_Na = n_Na_i-n_Na_f
    V_ave = -(E_i-E_f-n_Na*E_Na)/n_Na
    
    ax1.plot(log_data['Step'], log_data['v_nNa'], color=cm.jet(i/11), label=f'mu_{i+1} = {round(np.arange(mu_1, mu_2+step, step)[i],2)}  V = {round(V_ave,2)}')#, marker='*')


#ax1.hlines(512,0,3e6, linestyle='--', color='black')
#ax1.hlines(0,0,3e6, linestyle='--', color='black')

ax1.set_xlabel('Timestep (0.4 fs/step)')
ax1.set_ylabel('Number of Na+')
#ax1.set_xlim(0,3e6)
plt.title(f'{temp} K  mu = {mu_1} ~ {mu_2}')
#ax1.tick_params(axis='y', labelcolor=color)
#ax1.set_xlim(0,1e6)
ax1.legend()
plt.grid()

In [ ]:
fig, ax1 = plt.subplots(figsize=(10, 6))
temp=400
mu_1, mu_2, step = -2.2, -1.7, 0.05
for i in range(11):
    file = f'./test_training_set/phase_diagram/{temp}_phase_{i+1}.log'
    log_data_nbh = parse_lammps_log(file)
    ax1.plot(log_data_nbh['Step'], log_data_nbh['v_nNa'], color=cm.jet(i/11), label=f'mu = {round(np.arange(mu_1, mu_2+step, step)[i],2)}')#, marker='*')

#ax1.hlines(512,0,3e6, linestyle='--', color='black')
#ax1.hlines(0,0,3e6, linestyle='--', color='black')

ax1.set_xlabel('Timestep (0.4 fs/step)')
ax1.set_ylabel('Number of Na+')
#ax1.set_xlim(0,3e6)
plt.title(f'{temp} K  mu = {mu_1} ~ {mu_2}')
#ax1.tick_params(axis='y', labelcolor=color)
#ax1.set_xlim(0,1e6)
ax1.legend()
plt.grid()

In [ ]:
fig, ax1 = plt.subplots(figsize=(10, 6))
temp=500
mu_1, mu_2, step = -2.2, -1.7, 0.05
for i in range(11):
    file = f'./test_training_set/phase_diagram/{temp}_phase_{i+1}.log'
    log_data_nbh = parse_lammps_log(file)
    ax1.plot(log_data_nbh['Step'], log_data_nbh['v_nNa'], color=cm.jet(i/11), label=f'mu = {round(np.arange(mu_1, mu_2+step, step)[i],2)}')#, marker='*')

#ax1.hlines(512,0,3e6, linestyle='--', color='black')
#ax1.hlines(0,0,3e6, linestyle='--', color='black')

ax1.set_xlabel('Timestep (0.4 fs/step)')
ax1.set_ylabel('Number of Na+')
#ax1.set_xlim(0,3e6)
plt.title(f'{temp} K  mu = {mu_1} ~ {mu_2}')
#ax1.tick_params(axis='y', labelcolor=color)
#ax1.set_xlim(0,1e6)
ax1.legend()
plt.grid()

In [ ]:
fig, ax1 = plt.subplots(figsize=(10, 6))
temp=600
mu_1, mu_2, step = -2.2, -1.7, 0.05
for i in range(7):
    file = f'./test_training_set/phase_diagram/{temp}_phase_{i+1}.log'
    log_data_nbh = parse_lammps_log(file)
    ax1.plot(log_data_nbh['Step'], log_data_nbh['v_nNa'], color=cm.jet(i/11), label=f'mu = {round(np.arange(mu_1, mu_2+step, step)[i],2)}')#, marker='*')

#ax1.hlines(512,0,3e6, linestyle='--', color='black')
#ax1.hlines(0,0,3e6, linestyle='--', color='black')

ax1.set_xlabel('Timestep (0.4 fs/step)')
ax1.set_ylabel('Number of Na+')
#ax1.set_xlim(0,3e6)
plt.title(f'{temp} K  mu = {mu_1} ~ {mu_2}')
#ax1.tick_params(axis='y', labelcolor=color)
#ax1.set_xlim(0,1e6)
ax1.legend()
plt.grid()

In [ ]:
fig, ax1 = plt.subplots(figsize=(10, 6))
temp=700
mu_1, mu_2, step = -2.2, -1.7, 0.05
for i in range(11):
    file = f'./test_training_set/phase_diagram/{temp}_phase_{i+1}.log'
    log_data_nbh = parse_lammps_log(file)
    ax1.plot(log_data_nbh['Step'], log_data_nbh['v_nNa'], color=cm.jet(i/11), label=f'mu = {round(np.arange(mu_1, mu_2+step, step)[i],2)}')#, marker='*')

#ax1.hlines(512,0,3e6, linestyle='--', color='black')
#ax1.hlines(0,0,3e6, linestyle='--', color='black')

ax1.set_xlabel('Timestep (0.4 fs/step)')
ax1.set_ylabel('Number of Na+')
#ax1.set_xlim(0,3e6)
plt.title(f'{temp} K  mu = {mu_1} ~ {mu_2}')
#ax1.tick_params(axis='y', labelcolor=color)
#ax1.set_xlim(0,1e6)
ax1.legend()
plt.grid()

In [ ]:
np.arange(mu_2-2*step, mu_2, step)

In [ ]:
fig, ax1 = plt.subplots(figsize=(10, 6))
temp=300
mu_1, mu_2, step = -2, -1, 0.1
for i in range(11,22):
    file = f'./test_training_set/phase_diagram/{temp}_phase_{i+1}.log'
    log_data_nbh = parse_lammps_log(file)
    ax1.plot(log_data_nbh['Step'], log_data_nbh['v_nNa'], color=cm.jet((i-11)/11), label=f'mu = {round(np.arange(mu_1, mu_2+step, step)[i-11],2)}')#, marker='*')

ax1.set_xlabel('Timestep (0.4 fs/step)')
ax1.set_ylabel('Number of Na+')
plt.title(f'{temp} K  mu = {mu_1} ~ {mu_2}')
#ax1.tick_params(axis='y', labelcolor=color)
#ax1.set_xlim(0,1e6)
ax1.legend()
plt.grid()

In [ ]:
fig, ax1 = plt.subplots(figsize=(10, 6))
temp=350
mu_1, mu_2, step = -3, -2, 0.1
for i in range(11):
    file = f'./test_training_set/phase_diagram/{temp}_phase_{i+1}.log'
    log_data_nbh = parse_lammps_log(file)
    ax1.plot(log_data_nbh['Step'], log_data_nbh['v_nNa'], color=cm.jet(i/11), label=f'mu = {round(np.arange(mu_1, mu_2+step, step)[i],2)}')#, marker='*')

ax1.set_xlabel('Timestep (0.4 fs/step)')
ax1.set_ylabel('Number of Na+')
plt.title(f'{temp} K  mu = {mu_1} ~ {mu_2}')
#ax1.tick_params(axis='y', labelcolor=color)
#ax1.set_xlim(0,1e6)
ax1.legend()
plt.grid()

In [ ]:
fig, ax1 = plt.subplots(figsize=(10, 6))
temp=400
mu_1, mu_2, step = -3, -2, 0.1
for i in range(11):
    file = f'./test_training_set/phase_diagram/{temp}_phase_{i+1}.log'
    log_data_nbh = parse_lammps_log(file)
    ax1.plot(log_data_nbh['Step'], log_data_nbh['v_nNa'], color=cm.jet(i/11), label=f'mu = {round(np.arange(mu_1, mu_2+step, step)[i],2)}')#, marker='*')

ax1.set_xlabel('Timestep (0.4 fs/step)')
ax1.set_ylabel('Number of Na+')
plt.title(f'{temp} K  mu = {mu_1} ~ {mu_2}')
#ax1.tick_params(axis='y', labelcolor=color)
#ax1.set_xlim(0,1e6)
ax1.legend()
plt.grid()

In [ ]:
fig, ax1 = plt.subplots(figsize=(10, 6))
temp=450
mu_1, mu_2, step = -3, -2, 0.1
for i in range(11):
    file = f'./test_training_set/phase_diagram/{temp}_phase_{i+1}.log'
    log_data_nbh = parse_lammps_log(file)
    ax1.plot(log_data_nbh['Step'], log_data_nbh['v_nNa'], color=cm.jet(i/11), label=f'mu = {round(np.arange(mu_1, mu_2+step, step)[i],2)}')#, marker='*')

ax1.set_xlabel('Timestep (0.4 fs/step)')
ax1.set_ylabel('Number of Na+')
plt.title(f'{temp} K  mu = {mu_1} ~ {mu_2}')
#ax1.tick_params(axis='y', labelcolor=color)
#ax1.set_xlim(0,1e6)
ax1.legend()
plt.grid()

In [ ]:
fig, ax1 = plt.subplots(figsize=(10, 6))
temp=500
mu_1, mu_2, step = -3, -2, 0.1
for i in range(11):
    file = f'./test_training_set/phase_diagram/{temp}_phase_{i+1}.log'
    log_data_nbh = parse_lammps_log(file)
    ax1.plot(log_data_nbh['Step'], log_data_nbh['v_nNa'], color=cm.jet(i/11), label=f'mu = {round(np.arange(mu_1, mu_2+step, step)[i],2)}')#, marker='*')

ax1.set_xlabel('Timestep (0.4 fs/step)')
ax1.set_ylabel('Number of Na+')
plt.title(f'{temp} K  mu = {mu_1} ~ {mu_2}')
#ax1.tick_params(axis='y', labelcolor=color)
#ax1.set_xlim(0,1e6)
ax1.legend()
plt.grid()

In [ ]:
for i in range(len(data)-1):
    delta_r_square_1.append(abs(data['c_msd_Na[4]'].iloc[i+1]-data['c_msd_Na[4]'].iloc[i]))
for i in range(0, len(data)-1, 2):
    delta_r_square_2.append(abs(data['c_msd_Na[4]'].iloc[i+1]-data['c_msd_Na[4]'].iloc[i]))
for i in range(0, len(data)-1, 100):
    delta_r_square_5.append(abs(data['c_msd_Na[4]'].iloc[i+1]-data['c_msd_Na[4]'].iloc[i]))

In [ ]:
for i in range(8,16):
    file = f'./test_training_set/in_diffuse_{i+1}.log'
    plot_log_diff(file)

In [ ]:
num_Na_ls/512

In [ ]:
num_Na_ls = np.array([101.0, 143.0, 194.0, 243.0, 284.0, 410.0, 449.0, 494.0])*8
p1, p2 = 4,8
for i in range(p1, p2):
    filename = f'600K_msd_Na_{i+1}.dat'
    path = './test_training_set/'
    data = pd.read_csv(path + filename, delim_whitespace=True, comment='#', header=None)
    data.columns = ['TimeStep', 'c_msd_Na[1]', 'c_msd_Na[2]', 'c_msd_Na[3]', 'c_msd_Na[4]']
    plt.figure(figsize=(8,5))
    plt.plot(data['TimeStep'], data['c_msd_Na[1]']*num_Na_ls[i-p1], label='MSD(x)')
    plt.plot(data['TimeStep'], data['c_msd_Na[2]']*num_Na_ls[i-p1], label='MSD(y)')
    plt.plot(data['TimeStep'], data['c_msd_Na[3]']*num_Na_ls[i-p1], label='MSD(z)')
    plt.plot(data['TimeStep'], data['c_msd_Na[4]']*num_Na_ls[i-p1], label='TMSD')
    plt.legend()
    plt.xlabel('Timestep (1 fs/step)')
    plt.ylabel('MSD (A^2)')
    plt.title(filename+ f'  {int(num_Na_ls[i-p1])} Na+')


In [ ]:
# Path to your LAMMPS log file
i=4
log_file_path = f'./test_almtp_lv20/gcmc_{i}.log'
#log_file_path = 'gcmc_3.log'
log_data = parse_lammps_log(log_file_path)

#E_Na = -.13023100E+01
#V_ave = -(log_data['TotEng'][20]-log_data['TotEng'][-1]-(log_data['v_nNa'][20]-log_data['v_nNa'][-1])*E_Na)/(log_data['v_nNa'][20]-log_data['v_nNa'][-1])

fig, ax1 = plt.subplots(figsize=(10, 6))

# Plot RMS(E/atom) on the first y-axis
color = 'tab:blue'
ax1.set_xlabel('Timestep (0.4 fs/step)')
ax1.set_ylabel('Total E (eV)', color=color)
ax1.plot(log_data['Step'], log_data['TotEng'], color=color, marker='o', markersize=3)
ax1.tick_params(axis='y', labelcolor=color)

# Create the second y-axis and plot RMS(f)
ax2 = ax1.twinx()
color = 'tab:orange'
ax2.set_ylabel('Volume (A^3)', color=color)
ax2.plot(log_data['Step'][10:], log_data['Volume'][10:], color=color, marker='s', markersize=3)
ax2.tick_params(axis='y', labelcolor=color)

ax3 = ax1.twinx()
color = 'tab:green'
ax3.spines['right'].set_position(('outward', 80)) 
ax3.set_ylabel('Number of Na+', color=color)
ax3.plot(log_data['Step'], log_data['v_nNa'], color=color, marker='*')
ax3.tick_params(axis='y', labelcolor=color)
plt.grid()
#plt.title(log_file_path+f'    Average V = {round(V_ave,3)}')
#plt.xlim(10000,30000)
#ax1.set_ylim(-31500, -31000)
#ax2.set_ylim(54000,59000)


In [ ]:
i=5
file = f'./test_almtp_lv20/gcmc_{i}.log'
plot_log(file)

In [ ]:
for i in range(4):
    file = f'./test_almtp_stress/gcmc_{i+1}.log'
    plot_log(file)

In [ ]:
plot_log('gcmc_3.log')

In [ ]:
plot_log('gcmc_2.log')

In [ ]:
plot_log('gcmc_2.log')

In [ ]:
plot_log('gcmc_2.log')

In [ ]:
# Path to your LAMMPS log file
log_file_path = './pot_save/gcmc_6_497.log'
log_data = parse_lammps_log(log_file_path)

fig, ax1 = plt.subplots(figsize=(10, 6))

# Plot RMS(E/atom) on the first y-axis
color = 'tab:blue'
ax1.set_xlabel('Timestep (0.1 fs/step)')
ax1.set_ylabel('Total E (eV)', color=color)
ax1.plot(log_data['Step'], log_data['TotEng'], color=color, marker='o', markersize=3)
ax1.tick_params(axis='y', labelcolor=color)

# Create the second y-axis and plot RMS(f)
ax2 = ax1.twinx()
color = 'tab:orange'
ax2.set_ylabel('Volume (A^3)', color=color)
ax2.plot(log_data['Step'][10:], log_data['Volume'][10:], color=color, marker='s', markersize=3)
ax2.tick_params(axis='y', labelcolor=color)

ax3 = ax1.twinx()
color = 'tab:green'
ax3.spines['right'].set_position(('outward', 60)) 
ax3.set_ylabel('Number of Na+', color=color)
ax3.plot(log_data['Step'], log_data['v_nNa'], color=color, marker='*')
ax3.tick_params(axis='y', labelcolor=color)
plt.grid()
plt.title(log_file_path)
#plt.xlim(10000,30000)
#ax1.set_ylim(-31500, -31000)
#ax2.set_ylim(54000,59000)
E_Na = -.13023100E+01
V_ave = -(log_data['TotEng'][20]-log_data['TotEng'][-1]-(log_data['v_nNa'][20]-log_data['v_nNa'][-1])*E_Na)/(log_data['v_nNa'][20]-log_data['v_nNa'][-1])
V_ave

In [ ]:
log_data['TotEng'][-1],log_data['v_nNa'][-1],log_data['Volume'][-1]

In [ ]:
# Path to your LAMMPS log file
log_file_path = './pot_save/gcmc_3_488.log'
log_data = parse_lammps_log(log_file_path)

fig, ax1 = plt.subplots(figsize=(10, 6))

# Plot RMS(E/atom) on the first y-axis
color = 'tab:blue'
ax1.set_xlabel('Timestep (0.1 fs/step)')
ax1.set_ylabel('Total E (eV)', color=color)
ax1.plot(log_data['Step'], log_data['TotEng'], color=color, marker='o', markersize=3)
ax1.tick_params(axis='y', labelcolor=color)

# Create the second y-axis and plot RMS(f)
ax2 = ax1.twinx()
color = 'tab:orange'
ax2.set_ylabel('Volume (A^3)', color=color)
ax2.plot(log_data['Step'][10:], log_data['Volume'][10:], color=color, marker='s', markersize=3)
ax2.tick_params(axis='y', labelcolor=color)

ax3 = ax1.twinx()
color = 'tab:green'
ax3.spines['right'].set_position(('outward', 60)) 
ax3.set_ylabel('Number of Na+', color=color)
ax3.plot(log_data['Step'], log_data['v_nNa'], color=color, marker='*')
ax3.tick_params(axis='y', labelcolor=color)
plt.grid()
plt.title(log_file_path)
#plt.xlim(10000,30000)
#ax1.set_ylim(-31500, -31000)
#ax2.set_ylim(54000,59000)
E_Na = -.13023100E+01
V_ave = -(log_data['TotEng'][20]-log_data['TotEng'][-1]-(log_data['v_nNa'][20]-log_data['v_nNa'][-1])*E_Na)/(log_data['v_nNa'][20]-log_data['v_nNa'][-1])
V_ave

In [ ]:
# Path to your LAMMPS log file
log_file_path = './pot_save/gcmc_2_466.log'
log_data = parse_lammps_log(log_file_path)

fig, ax1 = plt.subplots(figsize=(10, 6))

# Plot RMS(E/atom) on the first y-axis
color = 'tab:blue'
ax1.set_xlabel('Timestep (0.1 fs/step)')
ax1.set_ylabel('Total E (eV)', color=color)
ax1.plot(log_data['Step'], log_data['TotEng'], color=color, marker='o', markersize=3)
ax1.tick_params(axis='y', labelcolor=color)

# Create the second y-axis and plot RMS(f)
ax2 = ax1.twinx()
color = 'tab:orange'
ax2.set_ylabel('Volume (A^3)', color=color)
ax2.plot(log_data['Step'][10:], log_data['Volume'][10:], color=color, marker='s', markersize=3)
ax2.tick_params(axis='y', labelcolor=color)

ax3 = ax1.twinx()
color = 'tab:green'
ax3.spines['right'].set_position(('outward', 60)) 
ax3.set_ylabel('Number of Na+', color=color)
ax3.plot(log_data['Step'], log_data['v_nNa'], color=color, marker='*')
ax3.tick_params(axis='y', labelcolor=color)
plt.grid()
plt.title(log_file_path)
#plt.xlim(10000,30000)
#ax1.set_ylim(-31500, -31000)
#ax2.set_ylim(54000,59000)
E_Na = -.13023100E+01
V_ave = -(log_data['TotEng'][20]-log_data['TotEng'][-1]-(log_data['v_nNa'][20]-log_data['v_nNa'][-1])*E_Na)/(log_data['v_nNa'][20]-log_data['v_nNa'][-1])
V_ave

In [ ]:
# Path to your LAMMPS log file
log_file_path = './pot_save/gcmc_4_468.log'
log_data = parse_lammps_log(log_file_path)

fig, ax1 = plt.subplots(figsize=(10, 6))

# Plot RMS(E/atom) on the first y-axis
color = 'tab:blue'
ax1.set_xlabel('Timestep (0.1 fs/step)')
ax1.set_ylabel('Total E (eV)', color=color)
ax1.plot(log_data['Step'], log_data['TotEng'], color=color, marker='o', markersize=3)
ax1.tick_params(axis='y', labelcolor=color)

# Create the second y-axis and plot RMS(f)
ax2 = ax1.twinx()
color = 'tab:orange'
ax2.set_ylabel('Volume (A^3)', color=color)
ax2.plot(log_data['Step'][10:], log_data['Volume'][10:], color=color, marker='s', markersize=3)
ax2.tick_params(axis='y', labelcolor=color)

ax3 = ax1.twinx()
color = 'tab:green'
ax3.spines['right'].set_position(('outward', 60)) 
ax3.set_ylabel('Number of Na+', color=color)
ax3.plot(log_data['Step'], log_data['v_nNa'], color=color, marker='*')
ax3.tick_params(axis='y', labelcolor=color)
plt.grid()
plt.title(log_file_path)
#plt.xlim(10000,30000)
#ax1.set_ylim(-31500, -31000)
#ax2.set_ylim(54000,59000)


In [ ]:
# Path to your LAMMPS log file
log_file_path = './pot_save/gcmc_235.log'
log_data = parse_lammps_log(log_file_path)

fig, ax1 = plt.subplots(figsize=(10, 6))

# Plot RMS(E/atom) on the first y-axis
color = 'tab:blue'
ax1.set_xlabel('Timestep (0.1 fs/step)')
ax1.set_ylabel('Total E (eV)', color=color)
ax1.plot(log_data['Step'], log_data['TotEng'], color=color, marker='o', markersize=3)
ax1.tick_params(axis='y', labelcolor=color)

# Create the second y-axis and plot RMS(f)
ax2 = ax1.twinx()
color = 'tab:orange'
ax2.set_ylabel('Volume (A^3)', color=color)
ax2.plot(log_data['Step'][10:], log_data['Volume'][10:], color=color, marker='s', markersize=3)
ax2.tick_params(axis='y', labelcolor=color)

ax3 = ax1.twinx()
color = 'tab:green'
ax3.spines['right'].set_position(('outward', 60)) 
ax3.set_ylabel('Number of Na+', color=color)
ax3.plot(log_data['Step'], log_data['v_nNa'], color=color, marker='*')
ax3.tick_params(axis='y', labelcolor=color)
plt.grid()
plt.title(log_file_path)
#plt.xlim(10000,30000)
#ax1.set_ylim(-31500, -31000)
#ax2.set_ylim(54000,59000)


In [ ]:
# Path to your LAMMPS log file
log_file_path = './pot_save/gcmc_257.log'
log_data = parse_lammps_log(log_file_path)

fig, ax1 = plt.subplots(figsize=(10, 6))

# Plot RMS(E/atom) on the first y-axis
color = 'tab:blue'
ax1.set_xlabel('Timestep (0.1 fs/step)')
ax1.set_ylabel('Total E (eV)', color=color)
ax1.plot(log_data['Step'], log_data['TotEng'], color=color, marker='o', markersize=3)
ax1.tick_params(axis='y', labelcolor=color)

# Create the second y-axis and plot RMS(f)
ax2 = ax1.twinx()
color = 'tab:orange'
ax2.set_ylabel('Volume (A^3)', color=color)
ax2.plot(log_data['Step'][10:], log_data['Volume'][10:], color=color, marker='s', markersize=3)
ax2.tick_params(axis='y', labelcolor=color)

ax3 = ax1.twinx()
color = 'tab:green'
ax3.spines['right'].set_position(('outward', 60)) 
ax3.set_ylabel('Number of Na+', color=color)
ax3.plot(log_data['Step'], log_data['v_nNa'], color=color, marker='*')
ax3.tick_params(axis='y', labelcolor=color)
plt.grid()
plt.title(log_file_path)
#plt.xlim(10000,30000)
#ax1.set_ylim(-31500, -31000)
#ax2.set_ylim(54000,59000)


In [ ]:
# Path to your LAMMPS log file
log_file_path = './pot_save/gcmc_274.log'
log_data = parse_lammps_log(log_file_path)

fig, ax1 = plt.subplots(figsize=(10, 6))

# Plot RMS(E/atom) on the first y-axis
color = 'tab:blue'
ax1.set_xlabel('Timestep (0.1 fs/step)')
ax1.set_ylabel('Total E (eV)', color=color)
ax1.plot(log_data['Step'], log_data['TotEng'], color=color, marker='o', markersize=3)
ax1.tick_params(axis='y', labelcolor=color)

# Create the second y-axis and plot RMS(f)
ax2 = ax1.twinx()
color = 'tab:orange'
ax2.set_ylabel('Volume (A^3)', color=color)
ax2.plot(log_data['Step'][10:], log_data['Volume'][10:], color=color, marker='s', markersize=3)
ax2.tick_params(axis='y', labelcolor=color)

ax3 = ax1.twinx()
color = 'tab:green'
ax3.spines['right'].set_position(('outward', 60)) 
ax3.set_ylabel('Number of Na+', color=color)
ax3.plot(log_data['Step'], log_data['v_nNa'], color=color, marker='*')
ax3.tick_params(axis='y', labelcolor=color)
plt.grid()
plt.title(log_file_path)
#plt.xlim(10000,30000)
#ax1.set_ylim(-31500, -31000)
#ax2.set_ylim(54000,59000)


In [ ]:
# Path to your LAMMPS log file
log_file_path = './pot_save/gcmc_279.log'
log_data = parse_lammps_log(log_file_path)

fig, ax1 = plt.subplots(figsize=(10, 6))

# Plot RMS(E/atom) on the first y-axis
color = 'tab:blue'
ax1.set_xlabel('Timestep (0.1 fs/step)')
ax1.set_ylabel('Total E (eV)', color=color)
ax1.plot(log_data['Step'], log_data['TotEng'], color=color, marker='o', markersize=3)
ax1.tick_params(axis='y', labelcolor=color)

# Create the second y-axis and plot RMS(f)
ax2 = ax1.twinx()
color = 'tab:orange'
ax2.set_ylabel('Volume (A^3)', color=color)
ax2.plot(log_data['Step'][10:], log_data['Volume'][10:], color=color, marker='s', markersize=3)
ax2.tick_params(axis='y', labelcolor=color)

ax3 = ax1.twinx()
color = 'tab:green'
ax3.spines['right'].set_position(('outward', 60)) 
ax3.set_ylabel('Number of Na+', color=color)
ax3.plot(log_data['Step'], log_data['v_nNa'], color=color, marker='*')
ax3.tick_params(axis='y', labelcolor=color)
plt.grid()
plt.title(log_file_path)
#plt.xlim(10000,30000)
#ax1.set_ylim(-31500, -31000)
#ax2.set_ylim(54000,59000)


In [ ]:
# Path to your LAMMPS log file
log_file_path = './pot_save/gcmc_288.log'
log_data = parse_lammps_log(log_file_path)

fig, ax1 = plt.subplots(figsize=(10, 6))

# Plot RMS(E/atom) on the first y-axis
color = 'tab:blue'
ax1.set_xlabel('Timestep (0.1 fs/step)')
ax1.set_ylabel('Total E (eV)', color=color)
ax1.plot(log_data['Step'], log_data['TotEng'], color=color, marker='o', markersize=3)
ax1.tick_params(axis='y', labelcolor=color)

# Create the second y-axis and plot RMS(f)
ax2 = ax1.twinx()
color = 'tab:orange'
ax2.set_ylabel('Volume (A^3)', color=color)
ax2.plot(log_data['Step'][10:], log_data['Volume'][10:], color=color, marker='s', markersize=3)
ax2.tick_params(axis='y', labelcolor=color)

ax3 = ax1.twinx()
color = 'tab:green'
ax3.spines['right'].set_position(('outward', 60)) 
ax3.set_ylabel('Number of Na+', color=color)
ax3.plot(log_data['Step'], log_data['v_nNa'], color=color, marker='*')
ax3.tick_params(axis='y', labelcolor=color)
plt.grid()
plt.title(log_file_path)
#plt.xlim(10000,30000)
#ax1.set_ylim(-31500, -31000)
#ax2.set_ylim(54000,59000)


In [ ]:
# Path to your LAMMPS log file
log_file_path = './pot_save/gcmc_321.log'
log_data = parse_lammps_log(log_file_path)

fig, ax1 = plt.subplots(figsize=(10, 6))

# Plot RMS(E/atom) on the first y-axis
color = 'tab:blue'
ax1.set_xlabel('Timestep (0.1 fs/step)')
ax1.set_ylabel('Total E (eV)', color=color)
ax1.plot(log_data['Step'], log_data['TotEng'], color=color, marker='o', markersize=3)
ax1.tick_params(axis='y', labelcolor=color)

# Create the second y-axis and plot RMS(f)
ax2 = ax1.twinx()
color = 'tab:orange'
ax2.set_ylabel('Volume (A^3)', color=color)
ax2.plot(log_data['Step'][10:], log_data['Volume'][10:], color=color, marker='s', markersize=3)
ax2.tick_params(axis='y', labelcolor=color)

ax3 = ax1.twinx()
color = 'tab:green'
ax3.spines['right'].set_position(('outward', 60)) 
ax3.set_ylabel('Number of Na+', color=color)
ax3.plot(log_data['Step'], log_data['v_nNa'], color=color, marker='*')
ax3.tick_params(axis='y', labelcolor=color)
plt.grid()
plt.title(log_file_path)
#plt.xlim(10000,30000)
#ax1.set_ylim(-31500, -31000)
#ax2.set_ylim(54000,59000)


In [ ]:
# Path to your LAMMPS log file
log_file_path = './pot_save/gcmc_331.log'
log_data = parse_lammps_log(log_file_path)

fig, ax1 = plt.subplots(figsize=(10, 6))

# Plot RMS(E/atom) on the first y-axis
color = 'tab:blue'
ax1.set_xlabel('Timestep (0.1 fs/step)')
ax1.set_ylabel('Total E (eV)', color=color)
ax1.plot(log_data['Step'], log_data['TotEng'], color=color, marker='o', markersize=3)
ax1.tick_params(axis='y', labelcolor=color)

# Create the second y-axis and plot RMS(f)
ax2 = ax1.twinx()
color = 'tab:orange'
ax2.set_ylabel('Volume (A^3)', color=color)
ax2.plot(log_data['Step'][10:], log_data['Volume'][10:], color=color, marker='s', markersize=3)
ax2.tick_params(axis='y', labelcolor=color)

ax3 = ax1.twinx()
color = 'tab:green'
ax3.spines['right'].set_position(('outward', 60)) 
ax3.set_ylabel('Number of Na+', color=color)
ax3.plot(log_data['Step'], log_data['v_nNa'], color=color, marker='*')
ax3.tick_params(axis='y', labelcolor=color)
plt.grid()
plt.title(log_file_path)
#plt.xlim(10000,30000)
#ax1.set_ylim(-31500, -31000)
#ax2.set_ylim(54000,59000)


In [ ]:
# Path to your LAMMPS log file
log_file_path = './pot_save/gcmc_332.log'
log_data = parse_lammps_log(log_file_path)

fig, ax1 = plt.subplots(figsize=(10, 6))

# Plot RMS(E/atom) on the first y-axis
color = 'tab:blue'
ax1.set_xlabel('Timestep (0.1 fs/step)')
ax1.set_ylabel('Total E (eV)', color=color)
ax1.plot(log_data['Step'], log_data['TotEng'], color=color, marker='o', markersize=3)
ax1.tick_params(axis='y', labelcolor=color)

# Create the second y-axis and plot RMS(f)
ax2 = ax1.twinx()
color = 'tab:orange'
ax2.set_ylabel('Volume (A^3)', color=color)
ax2.plot(log_data['Step'][10:], log_data['Volume'][10:], color=color, marker='s', markersize=3)
ax2.tick_params(axis='y', labelcolor=color)

ax3 = ax1.twinx()
color = 'tab:green'
ax3.spines['right'].set_position(('outward', 60)) 
ax3.set_ylabel('Number of Na+', color=color)
ax3.plot(log_data['Step'], log_data['v_nNa'], color=color, marker='*')
ax3.tick_params(axis='y', labelcolor=color)
plt.grid()
plt.title(log_file_path)
#plt.xlim(10000,30000)
#ax1.set_ylim(-31500, -31000)
#ax2.set_ylim(54000,59000)


In [ ]:
# Path to your LAMMPS log file
log_file_path = './pot_save/gcmc_340.log'
log_data = parse_lammps_log(log_file_path)

fig, ax1 = plt.subplots(figsize=(10, 6))

# Plot RMS(E/atom) on the first y-axis
color = 'tab:blue'
ax1.set_xlabel('Timestep (0.1 fs/step)')
ax1.set_ylabel('Total E (eV)', color=color)
ax1.plot(log_data['Step'], log_data['TotEng'], color=color, marker='o', markersize=3)
ax1.tick_params(axis='y', labelcolor=color)

# Create the second y-axis and plot RMS(f)
ax2 = ax1.twinx()
color = 'tab:orange'
ax2.set_ylabel('Volume (A^3)', color=color)
ax2.plot(log_data['Step'][10:], log_data['Volume'][10:], color=color, marker='s', markersize=3)
ax2.tick_params(axis='y', labelcolor=color)

ax3 = ax1.twinx()
color = 'tab:green'
ax3.spines['right'].set_position(('outward', 60)) 
ax3.set_ylabel('Number of Na+', color=color)
ax3.plot(log_data['Step'], log_data['v_nNa'], color=color, marker='*')
ax3.tick_params(axis='y', labelcolor=color)
plt.grid()
plt.title(log_file_path)


E_Na = -.13023100E+01
V_ave = -(log_data['TotEng'][20]-log_data['TotEng'][-1]-(log_data['v_nNa'][20]-log_data['v_nNa'][-1])*E_Na)/(log_data['v_nNa'][20]-log_data['v_nNa'][-1])
V_ave

In [ ]:
# Path to your LAMMPS log file
log_file_path = './pot_save/gcmc_356.log'
log_data = parse_lammps_log(log_file_path)

fig, ax1 = plt.subplots(figsize=(10, 6))

# Plot RMS(E/atom) on the first y-axis
color = 'tab:blue'
ax1.set_xlabel('Timestep (0.1 fs/step)')
ax1.set_ylabel('Total E (eV)', color=color)
ax1.plot(log_data['Step'], log_data['TotEng'], color=color, marker='o', markersize=3)
ax1.tick_params(axis='y', labelcolor=color)

# Create the second y-axis and plot RMS(f)
ax2 = ax1.twinx()
color = 'tab:orange'
ax2.set_ylabel('Volume (A^3)', color=color)
ax2.plot(log_data['Step'][10:], log_data['Volume'][10:], color=color, marker='s', markersize=3)
ax2.tick_params(axis='y', labelcolor=color)

ax3 = ax1.twinx()
color = 'tab:green'
ax3.spines['right'].set_position(('outward', 60)) 
ax3.set_ylabel('Number of Na+', color=color)
ax3.plot(log_data['Step'], log_data['v_nNa'], color=color, marker='*')
ax3.tick_params(axis='y', labelcolor=color)
plt.grid()
plt.title(log_file_path)
#plt.xlim(10000,30000)
#ax1.set_ylim(-31500, -31000)
#ax2.set_ylim(54000,59000)

E_Na = -.13023100E+01
V_ave = -(log_data['TotEng'][20]-log_data['TotEng'][-1]-(log_data['v_nNa'][20]-log_data['v_nNa'][-1])*E_Na)/(log_data['v_nNa'][20]-log_data['v_nNa'][-1])
V_ave

In [ ]:
# Path to your LAMMPS log file
log_file_path = './pot_save/gcmc_419.log'
log_data = parse_lammps_log(log_file_path)

fig, ax1 = plt.subplots(figsize=(10, 6))

# Plot RMS(E/atom) on the first y-axis
color = 'tab:blue'
ax1.set_xlabel('Timestep (0.1 fs/step)')
ax1.set_ylabel('Total E (eV)', color=color)
ax1.plot(log_data['Step'], log_data['TotEng'], color=color, marker='o', markersize=3)
ax1.tick_params(axis='y', labelcolor=color)

# Create the second y-axis and plot RMS(f)
ax2 = ax1.twinx()
color = 'tab:orange'
ax2.set_ylabel('Volume (A^3)', color=color)
ax2.plot(log_data['Step'][10:], log_data['Volume'][10:], color=color, marker='s', markersize=3)
ax2.tick_params(axis='y', labelcolor=color)

ax3 = ax1.twinx()
color = 'tab:green'
ax3.spines['right'].set_position(('outward', 60)) 
ax3.set_ylabel('Number of Na+', color=color)
ax3.plot(log_data['Step'], log_data['v_nNa'], color=color, marker='*')
ax3.tick_params(axis='y', labelcolor=color)
plt.grid()
plt.title(log_file_path)
#plt.xlim(10000,30000)
#ax1.set_ylim(-31500, -31000)
#ax2.set_ylim(54000,59000)

E_Na = -.13023100E+01
V_ave = -(log_data['TotEng'][20]-log_data['TotEng'][-1]-(log_data['v_nNa'][20]-log_data['v_nNa'][-1])*E_Na)/(log_data['v_nNa'][20]-log_data['v_nNa'][-1])
V_ave



In [ ]:
E_Na = -0.14023
V_ave = -(log_data['TotEng'][20]-log_data['TotEng'][-1]-(log_data['v_nNa'][20]-log_data['v_nNa'][-1])*E_Na)/(log_data['v_nNa'][20]-log_data['v_nNa'][-1])
V_ave

In [ ]:
E_Na = -0.14023
V_ave = -(log_data['TotEng'][20]-log_data['TotEng'][-1]-(log_data['v_nNa'][20]-log_data['v_nNa'][-1])*E_Na)/(log_data['v_nNa'][20]-log_data['v_nNa'][-1])
V_ave

In [ ]:
E_Na = -0.14023
V_ave = -(log_data['TotEng'][20]-log_data['TotEng'][-1]-(log_data['v_nNa'][20]-log_data['v_nNa'][-1])*E_Na)/(log_data['v_nNa'][20]-log_data['v_nNa'][-1])
V_ave

In [ ]:
log_data['v_nNa'][-1]

In [ ]:
x = int((log_data['Step'][-1]-5000)/4000)
a = range(0,x*20,20)

In [ ]:
len(a)

In [ ]:
i=1
log_data['Step'][a[i]-1], log_data['Step'][a[i+1]-1]

In [ ]:
i
log_data['v_nNa'][a[i]] - log_data['v_nNa'][a[i+1]]

In [ ]:
x = int((log_data['Step'][-1]-5000)/4000)
a = range(0,x*20,20)
V = []
E_Na = -.13023100E+01

for i in range(len(a)-1):
    delta_Na = log_data['v_nNa'][a[i]] - log_data['v_nNa'][a[i+1]]
    delta_E =  log_data['TotEng'][a[i]-1] - log_data['TotEng'][a[i+1]-1]
    V.append(-(delta_E-delta_Na*E_Na)/delta_Na)

In [ ]:
V

In [ ]:
fig = plt.figure(figsize=(10, 6))

plt.plot(range(len(V)-1), V[1:])
#plt.xlim(40,300)
plt.ylabel('Voltage')
plt.xlabel('Number of Na (512 Na originally)')
plt.title('')
plt.grid()
#plt.text(50,5, f'Average V = {round(np.mean(V[5:50]),3)}')
#plt.xlim(512,480)
#plt.savefig('desodiation_V.png', dpi=300, bbox_inches='tight')

In [ ]:
np.mean(V[1:])

In [ ]:
log_data['TotEng'][20],log_data['TotEng'][-1],log_data['v_nNa'][20],log_data['v_nNa'][-1]